# Football Managerial Decision Intelligence
### Interactive Player Performance, Starting XI Optimisation and Substitution Analysis Using Match Data
*Big Data Visualization — mini-project notebook*

| | |
|---|---|
| **Student name** | `[enter your name]` |
| **Registration number** | `[enter]` |
| **Course / section** | `[enter]` |
| **Faculty / guide** | `[enter]` |
| **Institution** | `[enter]` |
| **Project review (D4)** | 2-Nov-2026 · **Demo & viva (D2)** | 12-Nov-2026 |

**Research question.** *How can historical football match events, player performance indicators, lineup records and
match-state context be transformed into interactive visual analytics that support evidence-based starting-lineup and
substitution decisions?*

**Objective.** Build a reproducible pipeline — provenance → validation → features → analysis → decision support →
evaluation — over real open event data, and expose it through interactive Plotly visuals.

**Modules.** ① data collection & provenance · ② quality audit · ③ canonical tables · ④ feature engineering ·
⑤ exploratory analysis · ⑥ player intelligence · ⑦ team & tactical intelligence · ⑧ starting-XI decision support ·
⑨ substitution intelligence · ⑩ managerial decision patterns · ⑪ gated modelling · ⑫ findings and viva preparation.

> **Scope and disclaimer.** This is football *performance-analysis decision support*. It is **not** a betting,
> fantasy or player-rating product. Every recommendation is a transparent heuristic computed only from information
> available before the decision, shown with its data coverage. Associations are never presented as causal effects,
> and the coaching staff — who hold information this dataset does not — make the final decision.

## 1 · Environment setup
Everything this notebook needs is defined inside it: each pipeline stage's code appears right before it is used
(cells headed **Pipeline code**). It downloads StatsBomb open data from GitHub on first run and caches it under
`data/raw/` next to the notebook, so later runs (and offline demos) read from the cache. Works in Jupyter, VS Code and
Google Colab (*Runtime → Run all*) with no extra files.

In [1]:
import os, sys, subprocess, importlib, platform, time, warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
HEADLESS = os.environ.get("FMA_HEADLESS") == "1"   # set when executed non-interactively (no live widgets)
ROOT = Path.cwd()

if IN_COLAB:  # Colab ships pandas / numpy / plotly / scikit-learn / scipy; enable interactive widgets
    from google.colab import output
    output.enable_custom_widget_manager()

import numpy as np, pandas as pd
import plotly, plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
pio.renderers.default = "colab" if IN_COLAB else "notebook_connected+plotly_mimetype"

try:
    import ipywidgets as W
    WIDGETS_OK = True
except Exception:
    WIDGETS_OK = False

SEED = 42
np.random.seed(SEED)
print(f"Working folder : {ROOT}")
print(f"Runtime        : {'Google Colab' if IN_COLAB else 'local Jupyter'} | Python {platform.python_version()} | "
      f"pandas {pd.__version__} | plotly {plotly.__version__} | widgets {'on' if WIDGETS_OK else 'off'}")

Working folder : C:\Users\ASUS\Desktop\big data project
Runtime        : local Jupyter | Python 3.10.11 | pandas 2.1.4 | plotly 6.6.0 | widgets on


## 2 · Configuration
Every analytical choice lives here — nothing is hidden in later cells. Change a value and re-run from this cell.

* **Data mode** — `REAL_OPEN_DATA` (StatsBomb open data, default) or `USER_DATA` (your own StatsBomb-format
  JSON in `data/external/user_statsbomb/`).
* **Competition / season** must be a pair that exists in the source catalogue shown in Section 3.
* `MATCH_LIMIT = None` analyses every published match; set e.g. `20` for a quick run (the truncation is reported).
* Thresholds, formation templates, weights, timing bands and substitution rules are editable dictionaries.

In [2]:
from __future__ import annotations

# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · config — all tunable analytical choices
# Central configuration for the football managerial decision-analytics pipeline.
#
# Every tunable number used by the analysis lives here (thresholds, weights,
# timing bands, formation templates, substitution rules) so that the notebook
# never relies on scattered magic numbers. All values are *analytical choices*,
# not football truths; the notebook surfaces them so they can be challenged.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
import copy
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any


# ---------------------------------------------------------------------------
# Data modes
# ---------------------------------------------------------------------------
REAL_OPEN_DATA = "REAL_OPEN_DATA"
USER_DATA = "USER_DATA"
DEMO_SYNTHETIC = "DEMO_SYNTHETIC"
DATA_MODES = (REAL_OPEN_DATA, USER_DATA, DEMO_SYNTHETIC)

SYNTHETIC_WARNING = "Illustrative synthetic data — not real match or player observations."

# StatsBomb event coordinates: 120 x 80 units, origin top-left, every team's
# events recorded attacking left -> right (towards x = 120).
PITCH_LENGTH = 120.0
PITCH_WIDTH = 80.0
COORDINATE_SYSTEM = "statsbomb_120x80_attack_left_to_right"


# ---------------------------------------------------------------------------
# Position -> role mapping (StatsBomb position ids 1-25)
# ---------------------------------------------------------------------------
POSITION_ROLE: dict[str, str] = {
    "Goalkeeper": "GK",
    "Right Back": "FB", "Left Back": "FB",
    "Right Center Back": "CB", "Center Back": "CB", "Left Center Back": "CB",
    "Right Wing Back": "WB", "Left Wing Back": "WB",
    "Right Defensive Midfield": "DM", "Center Defensive Midfield": "DM", "Left Defensive Midfield": "DM",
    "Right Center Midfield": "CM", "Center Midfield": "CM", "Left Center Midfield": "CM",
    "Right Midfield": "WM", "Left Midfield": "WM",
    "Right Attacking Midfield": "AM", "Center Attacking Midfield": "AM", "Left Attacking Midfield": "AM",
    "Right Wing": "W", "Left Wing": "W",
    "Right Center Forward": "CF", "Center Forward": "CF", "Left Center Forward": "CF",
    "Secondary Striker": "CF",
}

ROLE_FAMILY: dict[str, str] = {
    "GK": "Goalkeeper",
    "CB": "Defender", "FB": "Defender", "WB": "Defender",
    "DM": "Midfielder", "CM": "Midfielder", "AM": "Midfielder", "WM": "Midfielder",
    "W": "Forward", "CF": "Forward",
}

ROLE_LABEL: dict[str, str] = {
    "GK": "Goalkeeper", "CB": "Centre-back", "FB": "Full-back", "WB": "Wing-back",
    "DM": "Defensive midfielder", "CM": "Central midfielder", "AM": "Attacking midfielder",
    "WM": "Wide midfielder", "W": "Winger", "CF": "Centre-forward",
}

# Position compatibility (0 = not allowed, 1 = natural role). Rows: player's
# primary role, columns: slot role. Analytical judgement, editable.
ROLE_COMPATIBILITY: dict[str, dict[str, float]] = {
    "GK": {"GK": 1.0},
    "CB": {"CB": 1.0, "FB": 0.6, "WB": 0.45, "DM": 0.55},
    "FB": {"FB": 1.0, "WB": 0.9, "CB": 0.55, "WM": 0.6, "DM": 0.4},
    "WB": {"WB": 1.0, "FB": 0.85, "WM": 0.8, "W": 0.6, "CB": 0.35},
    "DM": {"DM": 1.0, "CM": 0.85, "CB": 0.55, "AM": 0.45, "FB": 0.35},
    "CM": {"CM": 1.0, "DM": 0.8, "AM": 0.8, "WM": 0.6, "WB": 0.35},
    "AM": {"AM": 1.0, "CM": 0.75, "W": 0.8, "WM": 0.65, "CF": 0.6},
    "WM": {"WM": 1.0, "W": 0.85, "WB": 0.7, "AM": 0.65, "CM": 0.55, "FB": 0.5},
    "W": {"W": 1.0, "WM": 0.85, "AM": 0.75, "CF": 0.65, "WB": 0.5},
    "CF": {"CF": 1.0, "W": 0.6, "AM": 0.6},
}


# ---------------------------------------------------------------------------
# Formation templates: slot id, role, pitch x/y (120x80, attacking right)
# ---------------------------------------------------------------------------
FORMATION_TEMPLATES: dict[str, list[dict[str, Any]]] = {
    "4-3-3": [
        {"slot": "GK", "role": "GK", "x": 6, "y": 40},
        {"slot": "RB", "role": "FB", "x": 30, "y": 70}, {"slot": "RCB", "role": "CB", "x": 24, "y": 52},
        {"slot": "LCB", "role": "CB", "x": 24, "y": 28}, {"slot": "LB", "role": "FB", "x": 30, "y": 10},
        {"slot": "CDM", "role": "DM", "x": 44, "y": 40}, {"slot": "RCM", "role": "CM", "x": 58, "y": 56},
        {"slot": "LCM", "role": "CM", "x": 58, "y": 24},
        {"slot": "RW", "role": "W", "x": 84, "y": 70}, {"slot": "ST", "role": "CF", "x": 92, "y": 40},
        {"slot": "LW", "role": "W", "x": 84, "y": 10},
    ],
    "4-2-3-1": [
        {"slot": "GK", "role": "GK", "x": 6, "y": 40},
        {"slot": "RB", "role": "FB", "x": 30, "y": 70}, {"slot": "RCB", "role": "CB", "x": 24, "y": 52},
        {"slot": "LCB", "role": "CB", "x": 24, "y": 28}, {"slot": "LB", "role": "FB", "x": 30, "y": 10},
        {"slot": "RDM", "role": "DM", "x": 46, "y": 52}, {"slot": "LDM", "role": "DM", "x": 46, "y": 28},
        {"slot": "RAM", "role": "W", "x": 76, "y": 68}, {"slot": "CAM", "role": "AM", "x": 74, "y": 40},
        {"slot": "LAM", "role": "W", "x": 76, "y": 12}, {"slot": "ST", "role": "CF", "x": 94, "y": 40},
    ],
    "4-4-2": [
        {"slot": "GK", "role": "GK", "x": 6, "y": 40},
        {"slot": "RB", "role": "FB", "x": 30, "y": 70}, {"slot": "RCB", "role": "CB", "x": 24, "y": 52},
        {"slot": "LCB", "role": "CB", "x": 24, "y": 28}, {"slot": "LB", "role": "FB", "x": 30, "y": 10},
        {"slot": "RM", "role": "WM", "x": 62, "y": 70}, {"slot": "RCM", "role": "CM", "x": 54, "y": 50},
        {"slot": "LCM", "role": "CM", "x": 54, "y": 30}, {"slot": "LM", "role": "WM", "x": 62, "y": 10},
        {"slot": "RST", "role": "CF", "x": 90, "y": 50}, {"slot": "LST", "role": "CF", "x": 90, "y": 30},
    ],
    "3-5-2": [
        {"slot": "GK", "role": "GK", "x": 6, "y": 40},
        {"slot": "RCB", "role": "CB", "x": 24, "y": 58}, {"slot": "CB", "role": "CB", "x": 22, "y": 40},
        {"slot": "LCB", "role": "CB", "x": 24, "y": 22},
        {"slot": "RWB", "role": "WB", "x": 56, "y": 72}, {"slot": "CDM", "role": "DM", "x": 46, "y": 40},
        {"slot": "RCM", "role": "CM", "x": 60, "y": 54}, {"slot": "LCM", "role": "CM", "x": 60, "y": 26},
        {"slot": "LWB", "role": "WB", "x": 56, "y": 8},
        {"slot": "RST", "role": "CF", "x": 90, "y": 50}, {"slot": "LST", "role": "CF", "x": 90, "y": 30},
    ],
    "3-4-3": [
        {"slot": "GK", "role": "GK", "x": 6, "y": 40},
        {"slot": "RCB", "role": "CB", "x": 24, "y": 58}, {"slot": "CB", "role": "CB", "x": 22, "y": 40},
        {"slot": "LCB", "role": "CB", "x": 24, "y": 22},
        {"slot": "RWB", "role": "WB", "x": 54, "y": 72}, {"slot": "RCM", "role": "CM", "x": 50, "y": 50},
        {"slot": "LCM", "role": "CM", "x": 50, "y": 30}, {"slot": "LWB", "role": "WB", "x": 54, "y": 8},
        {"slot": "RW", "role": "W", "x": 84, "y": 66}, {"slot": "ST", "role": "CF", "x": 94, "y": 40},
        {"slot": "LW", "role": "W", "x": 84, "y": 14},
    ],
    "3-4-2-1": [
        {"slot": "GK", "role": "GK", "x": 6, "y": 40},
        {"slot": "RCB", "role": "CB", "x": 24, "y": 58}, {"slot": "CB", "role": "CB", "x": 22, "y": 40},
        {"slot": "LCB", "role": "CB", "x": 24, "y": 22},
        {"slot": "RWB", "role": "WB", "x": 56, "y": 72}, {"slot": "RDM", "role": "DM", "x": 48, "y": 50},
        {"slot": "LDM", "role": "DM", "x": 48, "y": 30}, {"slot": "LWB", "role": "WB", "x": 56, "y": 8},
        {"slot": "RAM", "role": "AM", "x": 78, "y": 54}, {"slot": "LAM", "role": "AM", "x": 78, "y": 26},
        {"slot": "ST", "role": "CF", "x": 96, "y": 40},
    ],
    "4-1-4-1": [
        {"slot": "GK", "role": "GK", "x": 6, "y": 40},
        {"slot": "RB", "role": "FB", "x": 30, "y": 70}, {"slot": "RCB", "role": "CB", "x": 24, "y": 52},
        {"slot": "LCB", "role": "CB", "x": 24, "y": 28}, {"slot": "LB", "role": "FB", "x": 30, "y": 10},
        {"slot": "CDM", "role": "DM", "x": 42, "y": 40},
        {"slot": "RM", "role": "WM", "x": 66, "y": 70}, {"slot": "RCM", "role": "CM", "x": 60, "y": 52},
        {"slot": "LCM", "role": "CM", "x": 60, "y": 28}, {"slot": "LM", "role": "WM", "x": 66, "y": 10},
        {"slot": "ST", "role": "CF", "x": 94, "y": 40},
    ],
}

# ---------------------------------------------------------------------------
# Role-specific metric weights used by the transparent role-fit score.
# Keys are per-90 / rate columns produced by features.player_season_table.
# ---------------------------------------------------------------------------
ROLE_METRIC_WEIGHTS: dict[str, dict[str, float]] = {
    "GK": {"gk_saves_p90": 0.35, "gk_claims_p90": 0.15, "pass_completion_pct": 0.25, "long_pass_completion_pct": 0.25},
    "CB": {"interceptions_p90": 0.15, "clearances_p90": 0.15, "aerials_won_p90": 0.15, "tackles_won_p90": 0.1,
           "ball_recoveries_p90": 0.1, "pass_completion_pct": 0.15, "progressive_passes_p90": 0.2},
    "FB": {"progressive_passes_p90": 0.15, "progressive_carries_p90": 0.15, "crosses_completed_p90": 0.1,
           "tackles_won_p90": 0.15, "interceptions_p90": 0.1, "pressures_p90": 0.1, "key_passes_p90": 0.1,
           "pass_completion_pct": 0.15},
    "WB": {"progressive_carries_p90": 0.2, "crosses_completed_p90": 0.15, "key_passes_p90": 0.15,
           "final_third_passes_p90": 0.1, "tackles_won_p90": 0.1, "pressures_p90": 0.1, "xg_p90": 0.05,
           "dribbles_completed_p90": 0.15},
    "DM": {"pass_completion_pct": 0.2, "progressive_passes_p90": 0.2, "interceptions_p90": 0.15,
           "ball_recoveries_p90": 0.15, "tackles_won_p90": 0.15, "pressures_p90": 0.15},
    "CM": {"progressive_passes_p90": 0.2, "pass_completion_pct": 0.15, "key_passes_p90": 0.15,
           "progressive_carries_p90": 0.15, "pressures_p90": 0.1, "ball_recoveries_p90": 0.1,
           "final_third_passes_p90": 0.15},
    "AM": {"key_passes_p90": 0.2, "xg_p90": 0.15, "xa_p90": 0.15, "progressive_carries_p90": 0.15,
           "dribbles_completed_p90": 0.15, "box_touches_p90": 0.1, "shots_p90": 0.1},
    "WM": {"crosses_completed_p90": 0.15, "key_passes_p90": 0.15, "progressive_carries_p90": 0.2,
           "dribbles_completed_p90": 0.15, "pressures_p90": 0.15, "final_third_passes_p90": 0.2},
    "W": {"xg_p90": 0.2, "key_passes_p90": 0.15, "dribbles_completed_p90": 0.2, "progressive_carries_p90": 0.15,
          "box_touches_p90": 0.15, "shots_p90": 0.15},
    "CF": {"xg_p90": 0.3, "shots_p90": 0.15, "goals_p90": 0.15, "box_touches_p90": 0.15, "key_passes_p90": 0.1,
           "aerials_won_p90": 0.05, "pressures_p90": 0.1},
}

# Tie-breaker added (before compatibility scaling) when a player's usual side
# (Right/Left/Centre from the position name) matches the slot side.
SIDE_MATCH_BONUS = 2.0

# Starting-XI composite score weights (each component already scaled 0-100).
XI_SCORE_WEIGHTS: dict[str, float] = {
    "role_fit": 0.40,          # season-to-date role-specific percentile composite
    "recent_form": 0.20,       # last-N appearance role composite (pre-decision)
    "sample_reliability": 0.15,  # evidence volume (minutes) behind the numbers
    "workload_suitability": 0.15,  # recent-minutes / rest-day proxy (not fatigue)
    "lineup_continuity": 0.10,  # started the team's previous match
}

# Substitution candidate weights and tactical-intent metric sets.
SUB_CANDIDATE_WEIGHTS: dict[str, float] = {
    "role_compatibility": 0.35, "intent_fit": 0.35, "recent_form": 0.15, "sample_reliability": 0.15,
}
SUB_INTENT_METRICS: dict[str, dict[str, float]] = {
    "attack": {"xg_p90": 0.3, "shots_p90": 0.15, "key_passes_p90": 0.2, "box_touches_p90": 0.15,
               "progressive_carries_p90": 0.1, "dribbles_completed_p90": 0.1},
    "protect": {"interceptions_p90": 0.2, "tackles_won_p90": 0.2, "clearances_p90": 0.15,
                "aerials_won_p90": 0.15, "pass_completion_pct": 0.15, "ball_recoveries_p90": 0.15},
    "balance": {"progressive_passes_p90": 0.25, "pass_completion_pct": 0.2, "key_passes_p90": 0.2,
                "pressures_p90": 0.15, "ball_recoveries_p90": 0.2},
}

# Analytical substitution-timing bands (period, inclusive minute range on the
# StatsBomb continuous clock, where stoppage time keeps counting: 45+2 = 47).
SUB_TIMING_BANDS: list[dict[str, Any]] = [
    {"label": "0-30", "periods": [1], "min": 0, "max": 30},
    {"label": "31-45+", "periods": [1], "min": 31, "max": 200},
    {"label": "46-60", "periods": [2], "min": 0, "max": 60},
    {"label": "61-70", "periods": [2], "min": 61, "max": 70},
    {"label": "71-80", "periods": [2], "min": 71, "max": 80},
    {"label": "81-90+", "periods": [2], "min": 81, "max": 200},
    {"label": "Extra time", "periods": [3, 4], "min": 0, "max": 200},
]

MATCH_PHASES: list[dict[str, Any]] = [
    {"label": "0-15", "periods": [1], "min": 0, "max": 15},
    {"label": "16-30", "periods": [1], "min": 16, "max": 30},
    {"label": "31-45+", "periods": [1], "min": 31, "max": 200},
    {"label": "46-60", "periods": [2], "min": 0, "max": 60},
    {"label": "61-75", "periods": [2], "min": 61, "max": 75},
    {"label": "76-90+", "periods": [2], "min": 76, "max": 200},
    {"label": "Extra time", "periods": [3, 4], "min": 0, "max": 200},
]

# Substitution rules are competition-specific. These defaults reflect the
# IFAB Law 3 option of up to five substitutes in three in-play opportunities
# (half-time changes do not use an opportunity), adopted by the Bundesliga
# from 2020/21. They are USER-CONFIGURED, not read from a regulation feed.
DEFAULT_SUB_RULES: dict[str, Any] = {
    "max_substitutions": 5,
    "max_windows": 3,
    "extra_time_additional_substitutions": 1,
    "extra_time_additional_windows": 1,
    "interval_changes_use_window": False,
    "window_grouping_seconds": 90,
    "source": "User-configured default (IFAB Law 3 five-substitute option). Verify against the competition's regulations.",
}


@dataclass
class Config:
    """Run configuration. Edit in the notebook's configuration cell."""

    project_root: Path = field(default_factory=lambda: Path.cwd())
    data_mode: str = REAL_OPEN_DATA
    competition_id: int = 9          # 1. Bundesliga (StatsBomb open data)
    season_id: int = 281             # 2023/2024
    match_limit: int | None = None   # None = every available match in the competition-season
    focus_team: str | None = "Bayer Leverkusen"
    include_360: bool = False        # 360 freeze frames are large; not needed for core analysis
    statsbomb_base_url: str = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"
    user_data_dir: str = "data/external/user_statsbomb"
    managers_csv: str = "data/external/managers.csv"
    request_timeout: int = 60
    max_retries: int = 4
    random_seed: int = 42
    min_minutes: float = 180.0                 # per-90 / percentile eligibility
    min_group_size: int = 8                    # smallest role group for role-level percentiles
    form_window: int = 5                       # appearances in recent-form window
    min_manager_matches: int = 10              # manager comparisons need this many matches
    impact_window_minutes: float = 10.0        # pre/post substitution-decision window (10 or 15 typical)
    reliability_full_minutes: float = 900.0    # minutes at which sample reliability = 100
    formation_templates: dict = field(default_factory=lambda: copy.deepcopy(FORMATION_TEMPLATES))
    role_metric_weights: dict = field(default_factory=lambda: copy.deepcopy(ROLE_METRIC_WEIGHTS))
    xi_score_weights: dict = field(default_factory=lambda: dict(XI_SCORE_WEIGHTS))
    sub_candidate_weights: dict = field(default_factory=lambda: dict(SUB_CANDIDATE_WEIGHTS))
    sub_intent_metrics: dict = field(default_factory=lambda: copy.deepcopy(SUB_INTENT_METRICS))
    sub_timing_bands: list = field(default_factory=lambda: copy.deepcopy(SUB_TIMING_BANDS))
    match_phases: list = field(default_factory=lambda: copy.deepcopy(MATCH_PHASES))
    sub_rules: dict = field(default_factory=lambda: dict(DEFAULT_SUB_RULES))
    role_compatibility: dict = field(default_factory=lambda: copy.deepcopy(ROLE_COMPATIBILITY))

    def __post_init__(self) -> None:
        self.project_root = Path(self.project_root)
        if self.data_mode not in DATA_MODES:
            raise ValueError(f"data_mode must be one of {DATA_MODES}, got {self.data_mode!r}")

    # Paths -----------------------------------------------------------------
    @property
    def raw_dir(self) -> Path:
        sub = {REAL_OPEN_DATA: "statsbomb", USER_DATA: "user", DEMO_SYNTHETIC: "synthetic"}[self.data_mode]
        return self.project_root / "data" / "raw" / sub

    @property
    def _mode_suffix(self) -> str:
        # Synthetic outputs never share folders with real-data outputs.
        return "synthetic" if self.data_mode == DEMO_SYNTHETIC else ""

    @property
    def processed_dir(self) -> Path:
        return self.project_root / "data" / "processed" / self._mode_suffix

    @property
    def figures_dir(self) -> Path:
        return self.project_root / "outputs" / "figures" / self._mode_suffix

    @property
    def exports_dir(self) -> Path:
        return self.project_root / "outputs" / "exports" / self._mode_suffix

    @property
    def reports_dir(self) -> Path:
        """Where auto-generated evidence documents go (synthetic runs never touch reports/)."""
        return self.exports_dir if self.is_synthetic else self.project_root / "reports"

    @property
    def is_synthetic(self) -> bool:
        return self.data_mode == DEMO_SYNTHETIC

    def ensure_dirs(self) -> None:
        self.raw_dir.mkdir(parents=True, exist_ok=True)  # only the raw-data cache is written to disk

    def to_dict(self) -> dict[str, Any]:
        d = asdict(self)
        d["project_root"] = str(self.project_root)
        return d


def format_formation(code: Any) -> str | None:
    """Convert a StatsBomb formation code (e.g. 3421) into '3-4-2-1'."""
    if code is None or (isinstance(code, float) and code != code):
        return None
    s = str(int(code)) if not isinstance(code, str) else code.replace("-", "")
    return "-".join(s) if s.isdigit() else None


def role_for_position(position_name: Any) -> str | None:
    """Map a StatsBomb position name onto a project role code (None if unknown)."""
    if not isinstance(position_name, str):
        return None
    return POSITION_ROLE.get(position_name)


def position_side(name: Any) -> str:
    """'R', 'L' or 'C' from a StatsBomb position name or a slot code (e.g. 'RCB', 'LWB', 'ST')."""
    if not isinstance(name, str) or not name:
        return "C"
    if name.startswith("Right") or (name[0] == "R" and name.isupper() and len(name) > 1):
        return "R"
    if name.startswith("Left") or (name[0] == "L" and name.isupper() and len(name) > 1):
        return "L"
    return "C"


def compatibility(player_role: str | None, slot_role: str, table: dict | None = None) -> float:
    """Position compatibility of a player's primary role for a slot role (0-1)."""
    table = table or ROLE_COMPATIBILITY
    if player_role is None:
        return 0.0
    return float(table.get(player_role, {}).get(slot_role, 0.0))

In [3]:
DATA_MODE      = REAL_OPEN_DATA     # REAL_OPEN_DATA | USER_DATA
COMPETITION_ID = 9                       # 1. Bundesliga
SEASON_ID      = 281                     # 2023/2024
MATCH_LIMIT    = None                    # None = all published matches for the selection
FOCUS_TEAM     = "Bayer Leverkusen"      # team analysed in XI / substitution / manager sections
MIN_MINUTES    = 180                     # per-90 & percentile eligibility
OFFLINE        = False                   # True = use cached raw files only (no network)

cfg = Config(project_root=ROOT, data_mode=DATA_MODE, competition_id=COMPETITION_ID, season_id=SEASON_ID,
                  match_limit=MATCH_LIMIT, focus_team=FOCUS_TEAM, min_minutes=MIN_MINUTES,
                  min_group_size=8)
BANNER = SYNTHETIC_WARNING if cfg.is_synthetic else None

def status_badge():
    color = "#c8553d" if cfg.is_synthetic else "#2a9d8f"
    text = (f"DATA MODE: {cfg.data_mode} — {SYNTHETIC_WARNING}" if cfg.is_synthetic else
            f"DATA MODE: {cfg.data_mode} — real StatsBomb open data (competition {cfg.competition_id}, "
            f"season {cfg.season_id})")
    display(HTML(f'<div style="background:{color};color:white;padding:8px 14px;border-radius:6px;'
                 f'font-weight:600;font-family:sans-serif">{text}</div>'))
status_badge()
display(pd.DataFrame({"setting": ["min_minutes", "min_group_size", "form_window", "reliability_full_minutes",
                                  "min_manager_matches", "impact_window_minutes"],
                      "value": [cfg.min_minutes, cfg.min_group_size, cfg.form_window, cfg.reliability_full_minutes,
                                cfg.min_manager_matches, cfg.impact_window_minutes]}))

display(Markdown("**Starting-XI score weights** (components scaled 0-100, then × position compatibility)"))
display(pd.DataFrame([cfg.xi_score_weights]))
display(Markdown("**Substitution rules** — " + cfg.sub_rules["source"]))
display(pd.DataFrame([{k: v for k, v in cfg.sub_rules.items() if k != "source"}]))
display(Markdown("**Substitution timing bands** (clock minutes; stoppage time keeps counting, e.g. 45+2 = 47)"))
display(pd.DataFrame(cfg.sub_timing_bands))
display(Markdown("**Formation templates available:** " + ", ".join(cfg.formation_templates)))

,setting,value
0,min_minutes,180.0
1,min_group_size,8.0
2,form_window,5.0
3,reliability_full_minutes,900.0
4,min_manager_matches,10.0
5,impact_window_minutes,10.0


**Starting-XI score weights** (components scaled 0-100, then × position compatibility)

,role_fit,recent_form,sample_reliability,workload_suitability,lineup_continuity
0,0.4,0.2,0.15,0.15,0.1


**Substitution rules** — User-configured default (IFAB Law 3 five-substitute option). Verify against the competition's regulations.

,max_substitutions,max_windows,extra_time_additional_substitutions,extra_time_additional_windows,interval_changes_use_window,window_grouping_seconds
0,5,3,1,1,False,90


**Substitution timing bands** (clock minutes; stoppage time keeps counting, e.g. 45+2 = 47)

,label,periods,min,max
0,0-30,[1],0,30
1,31-45+,[1],31,200
2,46-60,[2],0,60
3,61-70,[2],61,70
4,71-80,[2],71,80
5,81-90+,[2],81,200
6,Extra time,"[3, 4]",0,200


**Formation templates available:** 4-3-3, 4-2-3-1, 4-4-2, 3-5-2, 3-4-3, 3-4-2-1, 4-1-4-1

## 2b · Visual design system
One editorial theme for every chart (neutral background, navy / teal / amber / red), a reusable 120 × 80 StatsBomb pitch, and the chart library used throughout. Every chart function checks its inputs and shows a readable message instead of failing when a field is missing.

In [4]:
from __future__ import annotations

# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · player_analysis — metric labels and player summaries
# Player intelligence: profiles, role benchmarks, strengths/weaknesses, threshold sensitivity.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
from typing import Any

import numpy as np
import pandas as pd


METRIC_LABELS: dict[str, str] = {
    "xg_p90": "xG /90", "npxg_p90": "npxG /90", "xa_p90": "xA /90", "goals_p90": "Goals /90",
    "shots_p90": "Shots /90", "key_passes_p90": "Key passes /90", "progressive_passes_p90": "Prog. passes /90",
    "progressive_carries_p90": "Prog. carries /90", "final_third_passes_p90": "Final-third passes /90",
    "box_passes_p90": "Passes into box /90", "crosses_completed_p90": "Completed crosses /90",
    "dribbles_completed_p90": "Completed dribbles /90", "box_touches_p90": "Box touches /90",
    "tackles_won_p90": "Tackles won /90", "interceptions_p90": "Interceptions /90", "clearances_p90": "Clearances /90",
    "ball_recoveries_p90": "Recoveries /90", "pressures_p90": "Pressures /90", "aerials_won_p90": "Aerials won /90",
    "pass_completion_pct": "Pass completion %", "long_pass_completion_pct": "Long-pass completion %",
    "gk_saves_p90": "Saves /90", "gk_claims_p90": "Claims & sweeps /90", "defensive_actions_p90": "Def. actions /90",
    "touches_p90": "Touches /90", "turnovers_p90": "Turnovers /90", "blocks_p90": "Blocks /90",
}


def label(metric: str) -> str:
    return METRIC_LABELS.get(metric, metric.replace("_", " "))


def radar_metrics(role: str | None, config: Config) -> list[str]:
    """Metrics shown on a role's radar: the role-fit components."""
    return list(config.role_metric_weights.get(role or "CM", config.role_metric_weights["CM"]).keys())


def player_profile(season_pct: pd.DataFrame, player_id: Any, team_id: Any = None) -> dict[str, Any]:
    s = season_pct[season_pct["player_id"] == player_id]
    if team_id is not None:
        s = s[s["team_id"] == team_id]
    if s.empty:
        return {}
    r = s.sort_values("minutes_played", ascending=False).iloc[0]
    return {
        "player": r["player_name"], "team": r["team_name"], "primary_position": r.get("primary_position"),
        "role": ROLE_LABEL.get(r.get("primary_role"), "unknown"), "appearances": int(r["appearances"]),
        "starts": int(r["starts"]), "minutes": round(float(r["minutes_played"]), 0),
        "goals": int(r["goals"]), "xG": round(float(r["xg"]), 2), "assists": int(r["assists"]),
        "xA": round(float(r["xa"]), 2), "coverage": r["coverage_label"],
        "benchmark": f"{r.get('benchmark_group')} ({r.get('benchmark_level')}, n={int(r['benchmark_n'])})"
        if pd.notna(r.get("benchmark_n")) else "n/a",
    }


def strengths_weaknesses(season_pct: pd.DataFrame, player_id: Any, config: Config, team_id: Any = None,
                         top: int = 3) -> pd.DataFrame:
    """Highest and lowest role-relevant percentiles (observed metrics only)."""
    s = season_pct[season_pct["player_id"] == player_id]
    if team_id is not None:
        s = s[s["team_id"] == team_id]
    if s.empty:
        return pd.DataFrame(columns=["type", "metric", "value", "percentile"])
    r = s.iloc[0]
    rows = []
    for m in radar_metrics(r.get("primary_role"), config):
        pc = r.get(f"{m}_pct")
        if pd.notna(pc):
            rows.append({"metric": label(m), "value": round(float(r[m]), 2), "percentile": round(float(pc), 0)})
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=["type", "metric", "value", "percentile"])
    df = df.sort_values("percentile", ascending=False)
    st = df.head(top).assign(type="relative strength")
    wk = df.tail(top).sort_values("percentile").assign(type="relative weakness")
    wk = wk[~wk["metric"].isin(st["metric"])]
    return pd.concat([st, wk])[["type", "metric", "value", "percentile"]].reset_index(drop=True)


def threshold_sensitivity(season: pd.DataFrame, metric: str, thresholds: list[float] | None = None,
                          top: int = 5) -> pd.DataFrame:
    """How the leaderboard for ``metric`` changes with the minimum-minutes threshold."""
    thresholds = thresholds or [90, 180, 270, 450, 900]
    rows = []
    for t in thresholds:
        e = season[season["minutes_played"] >= t].dropna(subset=[metric])
        lead = e.nlargest(top, metric)
        rows.append({"min_minutes": t, "eligible_players": len(e),
                     f"top_{top}": ", ".join(f"{n} ({v:.2f})" for n, v in zip(lead["player_name"], lead[metric]))})
    return pd.DataFrame(rows)


def leaderboard(season_pct: pd.DataFrame, metric: str, role_family: str | None = None, min_minutes: float = 0,
                n: int = 15, team: str | None = None) -> pd.DataFrame:
    s = season_pct[season_pct["minutes_played"] >= min_minutes]
    if role_family and role_family != "All":
        s = s[s["role_family"] == role_family]
    if team and team != "All":
        s = s[s["team_name"] == team]
    s = s.dropna(subset=[metric])
    cols = ["player_name", "team_name", "primary_position", "minutes_played", metric, f"{metric}_pct",
            "coverage_label"]
    return s.nlargest(n, metric)[[c for c in cols if c in s.columns]].reset_index(drop=True)


def rolling_form_series(pms: pd.DataFrame, player_id: Any, metrics: list[str], window: int = 5) -> pd.DataFrame:
    """Rolling per-90 over the last ``window`` appearances (chronological)."""
    p = pms[pms["player_id"] == player_id].sort_values("match_date").copy()
    if p.empty:
        return p
    base = [m.replace("_p90", "") for m in metrics if m.endswith("_p90")]
    roll_min = p["minutes_played"].rolling(window, min_periods=1).sum()
    for b in base:
        if b in p.columns:
            with np.errstate(divide="ignore", invalid="ignore"):
                p[f"roll_{b}_p90"] = np.where(roll_min > 0, p[b].rolling(window, min_periods=1).sum() / roll_min * 90,
                                              np.nan)
    p["roll_minutes"] = roll_min
    p["appearance_no"] = range(1, len(p) + 1)
    return p


# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · visualizations — theme, pitch and core chart library
# Plotly visualisations with a consistent editorial sports-analytics theme.
#
# Every public function:
# * checks its required columns and returns a readable "no data" figure instead of raising;
# * never fills missing values with fabricated observations;
# * uses StatsBomb 120 x 80 coordinates (y = 0 at the top touchline, attack left -> right).
# ════════════════════════════════════════════════════════════════════════════════════════════════════
from typing import Any, Iterable

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots


# ---------------------------------------------------------------------------
# Theme
# ---------------------------------------------------------------------------
INK = "#1f2933"
MUTED = "#6b7280"
PAPER = "#fbfaf7"
GRID = "#e7e4dc"
NAVY = "#1d3557"
TEAL = "#2a9d8f"
AMBER = "#e9a23b"
RED = "#c8553d"
SLATE = "#8d99ae"
PURPLE = "#7b5ea7"
PITCH_FILL = "#eef2ec"
PITCH_LINE = "#8f9d93"
COLORWAY = [NAVY, TEAL, AMBER, RED, PURPLE, SLATE, "#457b9d", "#b5838d"]
RESULT_COLORS = {"W": TEAL, "D": SLATE, "L": RED}
STATE_COLORS = {"leading": TEAL, "drawing": SLATE, "trailing": RED}
FAMILY_COLORS = {"Goalkeeper": PURPLE, "Defender": NAVY, "Midfielder": TEAL, "Forward": AMBER}
SHOT_COLORS = {"Goal": AMBER, "Saved": NAVY, "Saved To Post": NAVY, "Off T": SLATE, "Blocked": "#b8c0cc",
               "Post": PURPLE, "Wayward": "#c9ced6", "Saved Off Target": NAVY}
SEQ_SCALE = [[0, "#f1f5f4"], [0.5, "#7fb7ad"], [1, NAVY]]
DIV_SCALE = [[0, RED], [0.5, "#f7f5f0"], [1, NAVY]]


def register_theme(name: str = "fma") -> None:
    tmpl = go.layout.Template()
    tmpl.layout = go.Layout(
        font=dict(family="Inter, Segoe UI, Helvetica, Arial, sans-serif", size=13, color=INK),
        title=dict(font=dict(size=17, color=INK), x=0.01, xanchor="left"),
        paper_bgcolor=PAPER, plot_bgcolor=PAPER, colorway=COLORWAY,
        xaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor="#cfcabd", ticks="outside", tickcolor="#cfcabd"),
        yaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor="#cfcabd", ticks="outside", tickcolor="#cfcabd"),
        legend=dict(bgcolor="rgba(0,0,0,0)", borderwidth=0),
        hoverlabel=dict(bgcolor="white", bordercolor="#cfcabd", font=dict(color=INK)),
        margin=dict(l=60, r=30, t=70, b=50),
    )
    pio.templates[name] = tmpl
    pio.templates.default = name


register_theme()


def _empty(message: str, title: str = "", height: int = 260) -> go.Figure:
    fig = go.Figure()
    fig.add_annotation(text=message, x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False,
                       font=dict(size=14, color=MUTED))
    fig.update_layout(title=title, height=height, xaxis=dict(visible=False), yaxis=dict(visible=False))
    return fig


def _missing(df: pd.DataFrame | None, cols: Iterable[str]) -> str | None:
    """Return a message if the frame is empty or lacks required columns."""
    if df is None or len(df) == 0:
        return "No data available for this selection."
    miss = [c for c in cols if c not in df.columns]
    if miss:
        return f"Required field(s) not available in this dataset: {', '.join(miss)}"
    return None


def _rows(df: pd.DataFrame | None, col: str, value: Any) -> pd.DataFrame | None:
    """Filter ``df[col] == value`` tolerating missing frames/columns (returns None -> empty-state)."""
    if df is None or col not in df.columns:
        return None
    return df[df[col] == value]


def add_banner(fig: go.Figure, text: str | None) -> go.Figure:
    """Persistent banner (e.g. synthetic-data warning) above a chart."""
    if text:
        fig.add_annotation(text=f"<b>{text}</b>", x=0.5, y=1.13, xref="paper", yref="paper", showarrow=False,
                           font=dict(color="white", size=12), bgcolor=RED, borderpad=4)
    return fig


def source_note(fig: go.Figure, text: str) -> go.Figure:
    fig.add_annotation(text=text, x=0, y=-0.12, xref="paper", yref="paper", showarrow=False, xanchor="left",
                       font=dict(size=10, color=MUTED))
    return fig


# ---------------------------------------------------------------------------
# Pitch
# ---------------------------------------------------------------------------
def draw_pitch(fig: go.Figure | None = None, length: float = PITCH_LENGTH, width: float = PITCH_WIDTH,
               row: int | None = None, col: int | None = None, half: bool = False,
               lines_on_top: bool = False) -> go.Figure:
    """Draw a StatsBomb-dimension pitch (120 x 80, y downwards) with standard markings.

    ``lines_on_top`` draws the markings above the data layer (for heatmaps).
    """
    fig = fig or go.Figure()
    layer = "above" if lines_on_top else "below"
    lc = dict(color=PITCH_LINE, width=1.4)
    kw = dict(row=row, col=col) if row else {}
    sx, sy = length / 120, width / 80

    def rect(x0, y0, x1, y1, fill=None, lay=None):
        fig.add_shape(type="rect", x0=x0 * sx, y0=y0 * sy, x1=x1 * sx, y1=y1 * sy, line=lc,
                      fillcolor=fill or "rgba(0,0,0,0)", layer=lay or layer, **kw)

    rect(0, 0, 120, 80, PITCH_FILL, "below")
    if lines_on_top:
        rect(0, 0, 120, 80)
    fig.add_shape(type="line", x0=60 * sx, y0=0, x1=60 * sx, y1=80 * sy, line=lc, layer=layer, **kw)
    fig.add_shape(type="circle", x0=50 * sx, y0=30 * sy, x1=70 * sx, y1=50 * sy, line=lc, layer=layer, **kw)
    for x0, x1 in ((0, 18), (102, 120)):
        rect(x0, 18, x1, 62)
    for x0, x1 in ((0, 6), (114, 120)):
        rect(x0, 30, x1, 50)
    for x0, x1 in ((-2, 0), (120, 122)):
        rect(x0, 36, x1, 44)
    spots = [(60, 40), (12, 40), (108, 40)]
    fig.add_trace(go.Scatter(x=[s[0] * sx for s in spots], y=[s[1] * sy for s in spots], mode="markers",
                             marker=dict(color=PITCH_LINE, size=4), hoverinfo="skip", showlegend=False), **kw)
    # Penalty arcs (portion of r=10 circle outside the box)
    for cx, sign in ((12, 1), (108, -1)):
        th = np.linspace(-0.927, 0.927, 30)
        xs = cx + sign * 10 * np.cos(th)
        ys = 40 + 10 * np.sin(th)
        fig.add_trace(go.Scatter(x=xs * sx, y=ys * sy, mode="lines", line=lc, hoverinfo="skip", showlegend=False),
                      **kw)
    xr = [58 * sx, 124 * sx] if half else [-4, 124 * sx]
    ax = dict(range=xr, visible=False, constrain="domain")
    ay = dict(range=[84 * sy, -4], visible=False, scaleanchor="x" if not row else None, scaleratio=1)
    if row:
        fig.update_xaxes(**ax, row=row, col=col)
        fig.update_yaxes(**ay, row=row, col=col)
    else:
        fig.update_layout(xaxis=ax, yaxis=ay, plot_bgcolor=PAPER)
    return fig


def attack_arrow(fig: go.Figure, text: str = "Attacking direction") -> None:
    fig.add_annotation(x=75, y=85.5, ax=45, ay=85.5, xref="x", yref="y", axref="x", ayref="y", showarrow=True,
                       arrowhead=2, arrowcolor=MUTED, text="")
    fig.add_annotation(x=60, y=88, text=text, showarrow=False, font=dict(size=10, color=MUTED))


# ---------------------------------------------------------------------------
# 1. Coverage cards
# ---------------------------------------------------------------------------
def coverage_cards(scale: pd.DataFrame, banner: str | None = None) -> go.Figure:
    msg = _missing(scale, ["measure", "value"])
    if msg:
        return _empty(msg, "Dataset coverage")
    wanted = ["Matches parsed", "Teams", "Players (unique IDs in squads)", "Events parsed", "Substitutions",
              "Goals (periods 1-4)", "Raw data size (MB)", "Raw JSON files"]
    d = scale.set_index("measure")["value"]
    items = [(w, d[w]) for w in wanted if w in d.index]
    n = len(items)
    cols = 4
    rows = int(np.ceil(n / cols))
    fig = make_subplots(rows=rows, cols=cols, specs=[[{"type": "indicator"}] * cols] * rows,
                        vertical_spacing=0.25)
    for i, (k, v) in enumerate(items):
        fig.add_trace(go.Indicator(mode="number", value=float(v), title=dict(text=k, font=dict(size=13, color=MUTED)),
                                   number=dict(font=dict(size=34, color=NAVY), valueformat=",.0f" if float(v) >= 100
                                               else ",.1f")), row=i // cols + 1, col=i % cols + 1)
    dr = d.get("Date range", "")
    fig.update_layout(title=f"Dataset coverage — {dr}", height=170 * rows + 80, margin=dict(t=90, b=20))
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# 2. Match timeline and cumulative xG
# ---------------------------------------------------------------------------
def match_timeline(tl: pd.DataFrame, title: str = "Match timeline", banner: str | None = None) -> go.Figure:
    msg = _missing(tl, ["elapsed_min", "team_name", "event", "detail"])
    if msg:
        return _empty(msg, title)
    symbols = {"Goal": "star", "Yellow Card": "square", "Second Yellow": "square-x", "Red Card": "square-x",
               "Substitution": "triangle-up", "Tactical shift": "diamond-open"}
    colors = {"Goal": AMBER, "Yellow Card": "#e9c46a", "Second Yellow": RED, "Red Card": RED,
              "Substitution": TEAL, "Tactical shift": MUTED}
    fig = go.Figure()
    for ev, g in tl.groupby("event"):
        fig.add_trace(go.Scatter(x=g["elapsed_min"], y=g["team_name"], mode="markers", name=ev,
                                 marker=dict(symbol=symbols.get(ev, "circle"), size=15 if ev == "Goal" else 12,
                                             color=colors.get(ev, NAVY), line=dict(width=1, color="white")),
                                 customdata=np.stack([g["minute"], g["detail"]], axis=-1),
                                 hovertemplate="%{y}<br>clock %{customdata[0]}'<br>%{customdata[1]}<extra>" + ev
                                               + "</extra>"))
    fig.update_layout(title=title, height=330, xaxis_title="Elapsed playing time (min, incl. stoppage)",
                      yaxis_title="", legend=dict(orientation="h", y=-0.3))
    return add_banner(fig, banner)


def cumulative_xg_chart(cx: pd.DataFrame, title: str = "Cumulative xG", banner: str | None = None) -> go.Figure:
    msg = _missing(cx, ["team_name", "elapsed_min", "cum_xg"])
    if msg:
        return _empty(msg, title)
    fig = go.Figure()
    for i, (team, g) in enumerate(cx.groupby("team_name")):
        x = np.r_[0, g["elapsed_min"].values]
        y = np.r_[0, g["cum_xg"].values]
        fig.add_trace(go.Scatter(x=x, y=y, mode="lines", line_shape="hv", name=team, line=dict(width=2.5,
                                 color=COLORWAY[i % len(COLORWAY)])))
        goals = g[g["outcome"] == "Goal"]
        fig.add_trace(go.Scatter(x=goals["elapsed_min"], y=goals["cum_xg"], mode="markers", showlegend=False,
                                 marker=dict(symbol="star", size=14, color=AMBER, line=dict(color=INK, width=1)),
                                 text=goals["player_name"], hovertemplate="Goal: %{text}<extra></extra>"))
    fig.update_layout(title=title, xaxis_title="Elapsed playing time (min)", yaxis_title="Cumulative StatsBomb xG",
                      height=360)
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# 3. Results overview / 10. Team trend
# ---------------------------------------------------------------------------
def results_overview(team_stats: pd.DataFrame, team: str, banner: str | None = None) -> go.Figure:
    t = _rows(team_stats, "team_name", team)
    msg = _missing(t, ["match_date", "goal_diff", "result", "opponent"])
    if msg:
        return _empty(msg, f"Results — {team}")
    t = t.sort_values("match_date")
    rec = t["result"].value_counts().reindex(["W", "D", "L"], fill_value=0)
    fig = go.Figure(go.Bar(x=t["match_date"], y=t["goal_diff"], marker_color=t["result"].map(RESULT_COLORS),
                           customdata=np.stack([t["opponent"], t["goals_for"], t["goals_against"], t["result"],
                                                t["home_away"]], axis=-1),
                           hovertemplate="%{x|%d %b %Y} vs %{customdata[0]} (%{customdata[4]})<br>"
                                         "%{customdata[1]}-%{customdata[2]} (%{customdata[3]})<extra></extra>"))
    fig.update_layout(title=f"{team}: goal difference by match — W{rec['W']} D{rec['D']} L{rec['L']}",
                      yaxis_title="Goal difference", xaxis_title="", height=340, bargap=0.25)
    return add_banner(fig, banner)


def team_trend(team_stats: pd.DataFrame, team: str, rolling: int = 5, banner: str | None = None) -> go.Figure:
    t = _rows(team_stats, "team_name", team)
    msg = _missing(t, ["match_date", "xg", "xg_against", "goals_for", "goals_against"])
    if msg:
        return _empty(msg, f"Team trend — {team}")
    t = t.sort_values("match_date")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t["match_date"], y=t["xg"].rolling(rolling, min_periods=1).mean(), name="xG for "
                             f"({rolling}-match rolling)", line=dict(color=NAVY, width=3)))
    fig.add_trace(go.Scatter(x=t["match_date"], y=t["xg_against"].rolling(rolling, min_periods=1).mean(),
                             name=f"xG against ({rolling}-match rolling)", line=dict(color=RED, width=3)))
    fig.add_trace(go.Scatter(x=t["match_date"], y=t["goals_for"], mode="markers", name="Goals for (match)",
                             marker=dict(color=NAVY, size=7, opacity=0.5), text=t["opponent"],
                             hovertemplate="%{x|%d %b} vs %{text}: %{y} goals<extra></extra>"))
    fig.add_trace(go.Scatter(x=t["match_date"], y=t["goals_against"], mode="markers", name="Goals against (match)",
                             marker=dict(color=RED, size=7, opacity=0.5, symbol="x"), text=t["opponent"],
                             hovertemplate="%{x|%d %b} vs %{text}: %{y} conceded<extra></extra>"))
    fig.update_layout(title=f"{team}: chance creation and concession over the season", yaxis_title="per match",
                      height=380, legend=dict(orientation="h", y=-0.2))
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# 4. Leaderboard (Plotly dropdown -> works in static HTML)
# ---------------------------------------------------------------------------
def leaderboard_chart(season_pct: pd.DataFrame, metrics: list[str], min_minutes: float, n: int = 15,
                      role_family: str | None = None, banner: str | None = None) -> go.Figure:
    msg = _missing(season_pct, ["player_name", "minutes_played"] + metrics[:1])
    if msg:
        return _empty(msg, "Player contribution leaderboard")
    s = season_pct[season_pct["minutes_played"] >= min_minutes]
    if role_family and role_family != "All":
        s = s[s["role_family"] == role_family]
    metrics = [m for m in metrics if m in s.columns]
    if s.empty or not metrics:
        return _empty("No players meet the filter (try a lower minutes threshold).", "Player contribution leaderboard")
    fig = go.Figure()
    for i, m in enumerate(metrics):
        top = s.dropna(subset=[m]).nlargest(n, m).iloc[::-1]
        fig.add_trace(go.Bar(x=top[m], y=top["player_name"] + " · " + top["team_name"].str.slice(0, 14),
                             orientation="h", visible=(i == 0), name=label(m),
                             marker_color=top["role_family"].map(FAMILY_COLORS).fillna(SLATE),
                             customdata=np.stack([top["minutes_played"].round(0), top["primary_position"].fillna("?"),
                                                  top.get(f"{m}_pct", pd.Series(np.nan, index=top.index)).round(0)],
                                                 axis=-1),
                             hovertemplate="%{y}<br>" + label(m) + ": %{x:.2f}<br>%{customdata[1]}<br>"
                                           "minutes %{customdata[0]}<br>percentile in benchmark %{customdata[2]}"
                                           "<extra></extra>"))
    buttons = [dict(label=label(m), method="update",
                    args=[{"visible": [j == i for j in range(len(metrics))]},
                          {"title": f"Top {n}: {label(m)} (min {min_minutes:.0f} minutes)"}])
               for i, m in enumerate(metrics)]
    fig.update_layout(title=f"Top {n}: {label(metrics[0])} (min {min_minutes:.0f} minutes)", height=560,
                      updatemenus=[dict(buttons=buttons, x=1.0, xanchor="right", y=1.12, direction="down")],
                      showlegend=False, xaxis_title="per 90 minutes / rate", margin=dict(l=230))
    source_note(fig, "Colour = role family (navy defender, teal midfielder, amber forward, purple GK). "
                     "Per-90 uses actual playing time incl. stoppage.")
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# 5. Radar
# ---------------------------------------------------------------------------
def player_radar(season_pct: pd.DataFrame, player_id: Any, metrics: list[str], compare_id: Any = None,
                 banner: str | None = None) -> go.Figure:
    s = _rows(season_pct, "player_id", player_id)
    msg = _missing(s, ["player_name"])
    if msg:
        return _empty(msg, "Player radar")
    pcols = [f"{m}_pct" for m in metrics if f"{m}_pct" in s.columns]
    if not pcols:
        return _empty("No percentile metrics available for this role.", "Player radar")
    r = s.sort_values("minutes_played", ascending=False).iloc[0]
    theta = [label(c[:-4]) for c in pcols]
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=[50] * len(pcols) + [50], theta=theta + theta[:1], name="Benchmark median (50th)",
                                  line=dict(color=MUTED, dash="dot"), fill=None, hoverinfo="skip"))
    vals = [r[c] if pd.notna(r[c]) else 0 for c in pcols]
    raw = [r[c[:-4]] for c in pcols]
    fig.add_trace(go.Scatterpolar(r=vals + vals[:1], theta=theta + theta[:1], fill="toself", name=r["player_name"],
                                  line=dict(color=NAVY, width=2.5), fillcolor="rgba(29,53,87,0.25)",
                                  customdata=np.array(raw + raw[:1], dtype=float),
                                  hovertemplate="%{theta}: %{r:.0f}th pct (value %{customdata:.2f})<extra></extra>"))
    if compare_id is not None:
        c = season_pct[season_pct["player_id"] == compare_id]
        if not c.empty:
            c = c.iloc[0]
            v2 = [c[x] if pd.notna(c[x]) else 0 for x in pcols]
            fig.add_trace(go.Scatterpolar(r=v2 + v2[:1], theta=theta + theta[:1], fill="toself", name=c["player_name"],
                                          line=dict(color=AMBER, width=2), fillcolor="rgba(233,162,59,0.2)"))
    bench = f"{r.get('benchmark_group')} ({r.get('benchmark_level')}, n={int(r['benchmark_n'])} eligible)" \
        if pd.notna(r.get("benchmark_n")) else "n/a"
    fig.update_layout(title=f"{r['player_name']} — percentile profile vs {bench}<br><sup>{r['minutes_played']:.0f} "
                            f"minutes · {r['coverage_label']}</sup>",
                      polar=dict(radialaxis=dict(range=[0, 100], tickvals=[25, 50, 75, 100], gridcolor=GRID),
                                 angularaxis=dict(gridcolor=GRID), bgcolor=PAPER),
                      height=520, legend=dict(orientation="h", y=-0.1))
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# 6. Comparison scatter with x/y dropdowns
# ---------------------------------------------------------------------------
def comparison_scatter(season_pct: pd.DataFrame, metrics: list[str], min_minutes: float, x: str | None = None,
                       y: str | None = None, role_family: str | None = None, highlight_team: str | None = None,
                       banner: str | None = None) -> go.Figure:
    msg = _missing(season_pct, ["player_name", "minutes_played"])
    if msg:
        return _empty(msg, "Player comparison")
    s = season_pct[season_pct["minutes_played"] >= min_minutes]
    if role_family and role_family != "All":
        s = s[s["role_family"] == role_family]
    metrics = [m for m in metrics if m in s.columns]
    if s.empty or len(metrics) < 2:
        return _empty("Not enough players/metrics for this filter.", "Player comparison")
    x, y = x or metrics[0], y or metrics[1]
    colors = s["role_family"].map(FAMILY_COLORS).fillna(SLATE)
    sizes = 6 + 14 * (s["minutes_played"] / s["minutes_played"].max())
    lines = np.where(s["team_name"] == highlight_team, 2, 0.5) if highlight_team else 0.5
    fig = go.Figure(go.Scatter(
        x=s[x], y=s[y], mode="markers", marker=dict(color=colors, size=sizes, opacity=0.8,
                                                     line=dict(color=INK, width=lines)),
        text=s["player_name"], customdata=np.stack([s["team_name"], s["primary_position"].fillna("?"),
                                                    s["minutes_played"].round(0)], axis=-1),
        hovertemplate="<b>%{text}</b> (%{customdata[0]})<br>%{customdata[1]} · %{customdata[2]} min<br>"
                      "x: %{x:.2f}<br>y: %{y:.2f}<extra></extra>"))
    fig.add_vline(x=s[x].median(), line=dict(color=MUTED, dash="dot", width=1))
    fig.add_hline(y=s[y].median(), line=dict(color=MUTED, dash="dot", width=1))
    bx = [dict(label=label(m), method="update", args=[{"x": [s[m]]}, {"xaxis.title.text": label(m)}]) for m in metrics]
    by = [dict(label=label(m), method="update", args=[{"y": [s[m]]}, {"yaxis.title.text": label(m)}]) for m in metrics]
    fig.update_layout(title=f"Player comparison (min {min_minutes:.0f} min; size = minutes; dotted = median)",
                      xaxis_title=label(x), yaxis_title=label(y), height=560,
                      updatemenus=[dict(buttons=bx, x=0.0, xanchor="left", y=1.13, direction="down"),
                                   dict(buttons=by, x=0.32, xanchor="left", y=1.13, direction="down")],
                      annotations=[dict(text="x-axis", x=0.0, y=1.19, xref="paper", yref="paper", showarrow=False,
                                        font=dict(size=10, color=MUTED)),
                                   dict(text="y-axis", x=0.32, y=1.19, xref="paper", yref="paper", showarrow=False,
                                        font=dict(size=10, color=MUTED))],
                      margin=dict(t=110))
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# 7. Form trend / 8. Workload / 9. Role distribution
# ---------------------------------------------------------------------------
def form_trend(series: pd.DataFrame, metrics: list[str], player_name: str, window: int,
               banner: str | None = None) -> go.Figure:
    cols = [f"roll_{m[:-4]}_p90" for m in metrics if m.endswith("_p90")]
    cols = [c for c in cols if series is not None and c in series.columns]
    msg = _missing(series, ["match_date"] + cols[:1])
    if msg or not cols:
        return _empty(msg or "No rolling metrics available.", f"Form — {player_name}")
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(x=series["match_date"], y=series["minutes_played"], name="Minutes (match)",
                         marker_color="#d9d4c7", opacity=0.6), secondary_y=True)
    for i, c in enumerate(cols):
        fig.add_trace(go.Scatter(x=series["match_date"], y=series[c], name=f"{label(c[5:])} ({window}-app rolling)",
                                 mode="lines+markers", line=dict(width=2.5, color=COLORWAY[i % len(COLORWAY)])),
                      secondary_y=False)
    fig.update_yaxes(title_text="rolling per 90", secondary_y=False)
    fig.update_yaxes(title_text="minutes", secondary_y=True, showgrid=False)
    fig.update_layout(title=f"{player_name}: rolling form over appearances (window {window}) — small windows are noisy",
                      height=400, legend=dict(orientation="h", y=-0.2))
    return add_banner(fig, banner)


def workload_bars(season: pd.DataFrame, team: str, banner: str | None = None) -> go.Figure:
    s = _rows(season, "team_name", team)
    msg = _missing(s, ["player_name", "minutes_played", "starts", "sub_appearances"])
    if msg:
        return _empty(msg, f"Minutes — {team}")
    s = s.sort_values("minutes_played", ascending=True)
    fig = go.Figure()
    fig.add_trace(go.Bar(y=s["player_name"], x=s["minutes_played"], orientation="h", name="Minutes",
                         marker_color=s["role_family"].map(FAMILY_COLORS).fillna(SLATE),
                         customdata=np.stack([s["starts"], s["sub_appearances"], s["unused_sub_count"]], axis=-1),
                         text=[f"{a}S / {b}sub" for a, b in zip(s["starts"], s["sub_appearances"])],
                         textposition="outside",
                         hovertemplate="%{y}: %{x:.0f} min<br>starts %{customdata[0]}, sub apps %{customdata[1]}, "
                                       "unused sub %{customdata[2]}<extra></extra>"))
    fig.update_layout(title=f"{team}: minutes, starts and substitute appearances", height=max(380, 22 * len(s)),
                      xaxis_title="Minutes played (actual playing time)", showlegend=False, margin=dict(l=200))
    return add_banner(fig, banner)


def role_distribution(season_pct: pd.DataFrame, metrics: list[str], min_minutes: float,
                      banner: str | None = None) -> go.Figure:
    msg = _missing(season_pct, ["primary_role", "minutes_played"])
    if msg:
        return _empty(msg, "Distribution by role")
    s = season_pct[(season_pct["minutes_played"] >= min_minutes) & season_pct["primary_role"].notna()]
    metrics = [m for m in metrics if m in s.columns]
    if s.empty or not metrics:
        return _empty("No eligible players.", "Distribution by role")
    order = [r for r in ["GK", "CB", "FB", "WB", "DM", "CM", "AM", "WM", "W", "CF"] if r in set(s["primary_role"])]
    fig = go.Figure()
    for i, m in enumerate(metrics):
        fig.add_trace(go.Box(x=s["primary_role"], y=s[m], visible=(i == 0), name=label(m), boxpoints="all",
                             jitter=0.4, pointpos=0, marker=dict(size=5, color=TEAL, opacity=0.6),
                             line=dict(color=NAVY), fillcolor="rgba(42,157,143,0.15)", text=s["player_name"],
                             hovertemplate="%{text}<br>%{y:.2f}<extra></extra>"))
    buttons = [dict(label=label(m), method="update", args=[{"visible": [j == i for j in range(len(metrics))]},
                                                           {"yaxis.title.text": label(m)}])
               for i, m in enumerate(metrics)]
    n_by = s["primary_role"].value_counts()
    fig.update_layout(title=f"Metric distribution by role (min {min_minutes:.0f} min) — n per role: "
                            + ", ".join(f"{r}={n_by.get(r, 0)}" for r in order),
                      xaxis=dict(categoryorder="array", categoryarray=order), yaxis_title=label(metrics[0]),
                      updatemenus=[dict(buttons=buttons, x=1, xanchor="right", y=1.13)], height=440, showlegend=False)
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# 11-12. Formations
# ---------------------------------------------------------------------------
def formation_frequency(fs: pd.DataFrame, team: str, banner: str | None = None) -> go.Figure:
    msg = _missing(fs, ["starting_formation", "wins", "draws", "losses"])
    if msg:
        return _empty(msg, f"Formations — {team}")
    fig = go.Figure()
    for k, c in (("wins", TEAL), ("draws", SLATE), ("losses", RED)):
        fig.add_trace(go.Bar(x=fs["starting_formation"], y=fs[k], name=k.title(), marker_color=c))
    for _, r in fs.iterrows():
        fig.add_annotation(x=r["starting_formation"], y=r["matches"], text=f"{r['points_per_match']:.2f} ppm"
                           + (" *" if r["sample_warning"] else ""), showarrow=False, yshift=12,
                           font=dict(size=11, color=INK))
    fig.update_layout(barmode="stack", title=f"{team}: starting formations and results (* small sample)",
                      yaxis_title="Matches", height=380)
    return add_banner(fig, banner)


def formation_comparison(fs: pd.DataFrame, team: str, banner: str | None = None) -> go.Figure:
    msg = _missing(fs, ["starting_formation", "xg_for", "xg_against"])
    if msg:
        return _empty(msg, f"Formation comparison — {team}")
    fig = go.Figure()
    fig.add_trace(go.Bar(x=fs["starting_formation"], y=fs["xg_for"], name="xG for / match", marker_color=NAVY))
    fig.add_trace(go.Bar(x=fs["starting_formation"], y=fs["xg_against"], name="xG against / match", marker_color=RED))
    fig.add_trace(go.Scatter(x=fs["starting_formation"], y=fs["field_tilt_pct"] / 100 * max(fs["xg_for"].max(), 1),
                             mode="markers", name="Field tilt (scaled)", marker=dict(size=12, color=AMBER,
                                                                                    symbol="diamond"),
                             text=fs["field_tilt_pct"].round(1), hovertemplate="field tilt %{text}%<extra></extra>"))
    for _, r in fs.iterrows():
        fig.add_annotation(x=r["starting_formation"], y=0, text=f"n={r['matches']}", showarrow=False, yshift=-28,
                           font=dict(size=11, color=MUTED))
    fig.update_layout(barmode="group", title=f"{team}: event metrics by starting formation (descriptive; "
                                             f"opponent and squad confound these)", height=400)
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# 13-14. Starting XI
# ---------------------------------------------------------------------------
def xi_pitch(lineup: pd.DataFrame, title: str, banner: str | None = None) -> go.Figure:
    msg = _missing(lineup, ["x", "y", "player_name", "slot", "score"])
    if msg:
        return _empty(msg, title)
    fig = draw_pitch()
    hover = [f"<b>{r.player_name}</b><br>slot {r.slot} ({ROLE_LABEL.get(r.slot_role, r.slot_role)})<br>"
             f"score {r.score:.1f} · compatibility {r.compatibility:.2f}<br>role fit {r.role_fit:.0f} · form "
             f"{r.recent_form:.0f}{' (season fallback)' if r.form_fallback else ''}<br>reliability "
             f"{r.sample_reliability:.0f} · workload {r.workload_suitability:.0f} · continuity "
             f"{r.lineup_continuity:.0f}<br>{r.coverage_label}" for r in lineup.itertuples()]
    fig.add_trace(go.Scatter(x=lineup["x"], y=lineup["y"], mode="markers+text", text=lineup["slot"],
                             textfont=dict(color="white", size=10), marker=dict(size=34, color=lineup["score"],
                             colorscale=SEQ_SCALE, cmin=40, cmax=95, line=dict(color="white", width=2),
                             colorbar=dict(title="slot<br>score", thickness=12, len=0.6)),
                             hovertext=hover, hoverinfo="text", showlegend=False))
    fig.add_trace(go.Scatter(x=lineup["x"], y=lineup["y"] + 6.5, mode="text",
                             text=[str(n).split()[-1] for n in lineup["player_name"]],
                             textfont=dict(size=11, color=INK), hoverinfo="skip", showlegend=False))
    attack_arrow(fig)
    fig.update_layout(title=title, height=560, margin=dict(l=10, r=10, t=80, b=30))
    return add_banner(fig, banner)


def score_decomposition(lineup: pd.DataFrame, weights: dict[str, float], title: str = "Score decomposition",
                        banner: str | None = None) -> go.Figure:
    comps = list(weights)
    msg = _missing(lineup, ["player_name", "compatibility"] + comps)
    if msg:
        return _empty(msg, title)
    fig = go.Figure()
    names = lineup["slot"] + " · " + lineup["player_name"]
    for i, c in enumerate(comps):
        vals = weights[c] * lineup[c].fillna(0) * lineup["compatibility"]
        fig.add_trace(go.Bar(y=names, x=vals, orientation="h", name=f"{c.replace('_', ' ')} (w={weights[c]:.2f})",
                             marker_color=COLORWAY[i % len(COLORWAY)],
                             hovertemplate="%{y}<br>" + c + ": %{x:.1f} weighted points<extra></extra>"))
    fig.update_layout(barmode="stack", title=title, height=480, xaxis_title="Weighted points (x compatibility)",
                      legend=dict(orientation="h", y=-0.15), margin=dict(l=220), yaxis=dict(autorange="reversed"))
    return add_banner(fig, banner)


def alternative_xi_comparison(a: pd.DataFrame, b: pd.DataFrame, label_a: str, label_b: str,
                              banner: str | None = None) -> go.Figure:
    msg = _missing(a, ["slot", "player_name", "score"]) or _missing(b, ["slot", "player_name", "score"])
    if msg:
        return _empty(msg, "Alternative XI comparison")
    m = a[["slot", "player_name", "score"]].merge(b[["slot", "player_name", "score"]], on="slot", how="outer",
                                                  suffixes=("_a", "_b"))
    fig = go.Figure()
    fig.add_trace(go.Bar(x=m["slot"], y=m["score_a"], name=label_a, marker_color=NAVY, text=m["player_name_a"],
                         hovertemplate="%{x}: %{text} (%{y:.1f})<extra></extra>"))
    fig.add_trace(go.Bar(x=m["slot"], y=m["score_b"], name=label_b, marker_color=AMBER, text=m["player_name_b"],
                         hovertemplate="%{x}: %{text} (%{y:.1f})<extra></extra>"))
    changed = m[m["player_name_a"] != m["player_name_b"]]
    fig.update_layout(barmode="group", title=f"{label_a} vs {label_b}: slot scores ({len(changed)} slot(s) differ)",
                      yaxis_title="Slot score", height=400)
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# 15-18. Spatial: shots, passes, network, progressive actions
# ---------------------------------------------------------------------------
def shot_map(shots: pd.DataFrame, title: str, banner: str | None = None) -> go.Figure:
    msg = _missing(shots, ["x", "y", "outcome", "shot_xg"])
    if msg:
        return _empty(msg, title)
    s = shots.dropna(subset=["x", "y"])
    fig = draw_pitch(half=True)
    for oc, g in s.groupby("outcome"):
        fig.add_trace(go.Scatter(x=g["x"], y=g["y"], mode="markers", name=f"{oc} ({len(g)})",
                                 marker=dict(size=6 + 40 * g["shot_xg"].fillna(0.02), color=SHOT_COLORS.get(oc, SLATE),
                                             opacity=0.85, line=dict(color=INK if oc == "Goal" else "white", width=1),
                                             symbol="star" if oc == "Goal" else "circle"),
                                 customdata=np.stack([g["player_name"], g["shot_xg"].round(3), g["minute"],
                                                      g.get("shot_body_part", pd.Series("", index=g.index))], axis=-1),
                                 hovertemplate="%{customdata[0]}<br>xG %{customdata[1]} · %{customdata[3]}<br>"
                                               "minute %{customdata[2]}<extra>" + str(oc) + "</extra>"))
    fig.update_layout(title=f"{title}<br><sup>{len(s)} shots · {s['shot_xg'].sum():.2f} xG · "
                            f"{(s['outcome'] == 'Goal').sum()} goals · marker size = StatsBomb xG</sup>",
                      height=520, legend=dict(orientation="h", y=-0.05))
    return add_banner(fig, banner)


def pass_map(passes: pd.DataFrame, title: str, max_passes: int = 600, banner: str | None = None) -> go.Figure:
    msg = _missing(passes, ["x", "y", "end_x", "end_y", "outcome"])
    if msg:
        return _empty(msg, title)
    p = passes.dropna(subset=["x", "y", "end_x", "end_y"])
    sampled = len(p) > max_passes
    if sampled:
        p = p.sample(max_passes, random_state=0)
    fig = draw_pitch()
    for name, sub, color in (("Completed", p[p["outcome"].isna()], NAVY), ("Incomplete", p[p["outcome"].notna()], RED)):
        xs, ys = [], []
        for r in sub.itertuples():
            xs += [r.x, r.end_x, None]
            ys += [r.y, r.end_y, None]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=color, width=1), opacity=0.45,
                                 name=f"{name} ({len(sub)})", hoverinfo="skip"))
        fig.add_trace(go.Scatter(x=sub["end_x"], y=sub["end_y"], mode="markers", showlegend=False,
                                 marker=dict(size=4, color=color), text=sub["pass_recipient_name"],
                                 hovertemplate="to %{text}<extra></extra>"))
    attack_arrow(fig)
    note = f" (random sample of {max_passes})" if sampled else ""
    fig.update_layout(title=f"{title}{note}", height=520, legend=dict(orientation="h", y=-0.05))
    return add_banner(fig, banner)


def pass_network_chart(nodes: pd.DataFrame, edges: pd.DataFrame, title: str, cutoff: float | None = None,
                       banner: str | None = None) -> go.Figure:
    msg = _missing(nodes, ["x", "y", "passes", "display_name"])
    if msg:
        return _empty(msg, title)
    fig = draw_pitch()
    if edges is not None and not edges.empty:
        mx = edges["count"].max()
        for r in edges.itertuples():
            fig.add_trace(go.Scatter(x=[r.x0, r.x1], y=[r.y0, r.y1], mode="lines", hoverinfo="skip",
                                     line=dict(color=NAVY, width=0.5 + 6 * r.count / mx),
                                     opacity=0.25 + 0.6 * r.count / mx, showlegend=False))
    fig.add_trace(go.Scatter(x=nodes["x"], y=nodes["y"], mode="markers+text", text=nodes["display_name"].str.split()
                             .str[-1], textposition="top center", textfont=dict(size=10, color=INK),
                             marker=dict(size=10 + 25 * nodes["passes"] / nodes["passes"].max(), color=TEAL,
                                         line=dict(color="white", width=2)),
                             customdata=nodes["passes"], hovertext=nodes["display_name"],
                             hovertemplate="%{hovertext}: %{customdata} passes<extra></extra>", showlegend=False))
    attack_arrow(fig)
    sub = f"until first substitution ({cutoff:.0f}' elapsed)" if cutoff else ""
    fig.update_layout(title=f"{title}<br><sup>node = average pass origin, size = passes; edge width = completed "
                            f"passes between pair (min shown) {sub}</sup>", height=540)
    return add_banner(fig, banner)


def progressive_actions_map(flags: pd.DataFrame, title: str, banner: str | None = None) -> go.Figure:
    msg = _missing(flags, ["x", "y", "end_x", "end_y", "progressive_passes", "progressive_carries"])
    if msg:
        return _empty(msg, title)
    fig = draw_pitch()
    for name, mask, color, dash in (("Progressive passes", flags["progressive_passes"] == 1, NAVY, "solid"),
                                    ("Progressive carries", flags["progressive_carries"] == 1, AMBER, "dot")):
        sub = flags[mask]
        xs, ys = [], []
        for r in sub.itertuples():
            xs += [r.x, r.end_x, None]
            ys += [r.y, r.end_y, None]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=color, width=1.6, dash=dash), opacity=0.6,
                                 name=f"{name} ({len(sub)})", hoverinfo="skip"))
    attack_arrow(fig)
    fig.update_layout(title=f"{title}<br><sup>Project definitions: passes start outside the defending 40%; both end "
                            f">=25% closer to goal centre in the attacking 60%</sup>", height=520, legend=dict(orientation="h", y=-0.05))
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# 19-23. Substitutions
# ---------------------------------------------------------------------------
def substitution_timeline(subs: pd.DataFrame, title: str, banner: str | None = None) -> go.Figure:
    msg = _missing(subs, ["elapsed_min", "team_name", "player_on_name", "player_off_name"])
    if msg:
        return _empty(msg, title)
    fig = go.Figure()
    for i, (team, g) in enumerate(subs.groupby("team_name")):
        fig.add_trace(go.Scatter(
            x=g["elapsed_min"], y=[team] * len(g), mode="markers+text", name=team,
            text=g["player_on_name"].str.split().str[-1], textposition="top center",
            marker=dict(symbol="triangle-up", size=14, color=COLORWAY[i % len(COLORWAY)]),
            customdata=np.stack([g["minute"], g["player_off_name"], g["player_on_name"], g["game_state"],
                                 g["sub_reason"]], axis=-1),
            hovertemplate="%{customdata[0]}': OFF %{customdata[1]} → ON %{customdata[2]}<br>state: %{customdata[3]}"
                          " · %{customdata[4]}<extra></extra>"))
    fig.update_layout(title=title, xaxis_title="Elapsed playing time (min)", height=300, showlegend=False,
                      xaxis=dict(range=[0, max(100, subs["elapsed_min"].max() + 5)]))
    return add_banner(fig, banner)


def sub_timing_histogram(subs: pd.DataFrame, bands: list[dict], group: str | None = None,
                         banner: str | None = None) -> go.Figure:
    msg = _missing(subs, ["timing_band", "minute"])
    if msg:
        return _empty(msg, "Substitution timing")
    order = [b["label"] for b in bands]
    fig = go.Figure()
    groups = [(None, subs)] if not group else list(subs.groupby(group))
    for i, (g, d) in enumerate(groups):
        counts = d["timing_band"].value_counts().reindex(order, fill_value=0)
        share = counts / max(counts.sum(), 1) * 100
        fig.add_trace(go.Bar(x=order, y=share, name=str(g) if g is not None else "All",
                             marker_color=COLORWAY[i % len(COLORWAY)], customdata=counts.values,
                             hovertemplate="%{x}: %{y:.1f}% (%{customdata} subs)<extra></extra>"))
    fig.update_layout(title=f"Substitution timing by analytical band (share of substitutions, n={len(subs)})",
                      yaxis_title="% of substitutions", xaxis_title="Band (clock minutes, stoppage counted)",
                      barmode="group", height=380)
    return add_banner(fig, banner)


def subs_by_state(subs: pd.DataFrame, by: str | None = None, banner: str | None = None) -> go.Figure:
    msg = _missing(subs, ["game_state", "minute"])
    if msg:
        return _empty(msg, "Substitutions by score state")
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Share of substitutions", "Minute distribution"))
    counts = subs["game_state"].value_counts().reindex(["leading", "drawing", "trailing"], fill_value=0)
    fig.add_trace(go.Bar(x=counts.index, y=counts / counts.sum() * 100, marker_color=[STATE_COLORS[k] for k in
                                                                                     counts.index],
                         customdata=counts.values, hovertemplate="%{x}: %{y:.1f}% (%{customdata})<extra></extra>",
                         showlegend=False), row=1, col=1)
    for st in ["leading", "drawing", "trailing"]:
        d = subs[subs["game_state"] == st]
        if len(d):
            fig.add_trace(go.Box(y=d["minute"], name=st, marker_color=STATE_COLORS[st], boxpoints="all", jitter=0.4,
                                 pointpos=0, marker=dict(size=4), showlegend=False), row=1, col=2)
    fig.update_yaxes(title_text="%", row=1, col=1)
    fig.update_yaxes(title_text="clock minute", row=1, col=2)
    fig.update_layout(title=f"Substitutions by game state at the moment of the change (n={len(subs)})", height=400)
    return add_banner(fig, banner)


def role_flow_sankey(flow: pd.DataFrame, title: str = "Substitution role flow", banner: str | None = None
                     ) -> go.Figure:
    msg = _missing(flow, ["off_role", "on_role", "count"])
    if msg:
        return _empty(msg, title)
    left = sorted(flow["off_role"].unique())
    right = sorted(flow["on_role"].unique())
    labels = [f"OFF {r}" for r in left] + [f"ON {r}" for r in right]
    src = [left.index(r) for r in flow["off_role"]]
    tgt = [len(left) + right.index(r) for r in flow["on_role"]]
    same = flow["off_role"] == flow["on_role"]
    fig = go.Figure(go.Sankey(
        node=dict(label=labels, pad=14, thickness=16, color=[NAVY] * len(left) + [TEAL] * len(right)),
        link=dict(source=src, target=tgt, value=flow["count"],
                  color=np.where(same, "rgba(141,153,174,0.45)", "rgba(233,162,59,0.55)"))))
    fig.update_layout(title=f"{title} (grey = like-for-like role, amber = role change)", height=460)
    return add_banner(fig, banner)


def impact_chart(impact: pd.DataFrame, metric: str = "xg", banner: str | None = None) -> go.Figure:
    col = f"delta_team_{metric}"
    msg = _missing(impact, [col, "game_state", "contaminated"])
    if msg:
        return _empty(msg, "Pre/post substitution windows")
    fig = go.Figure()
    for st in ["leading", "drawing", "trailing"]:
        for clean, op in ((True, 0.9), (False, 0.45)):
            d = impact[(impact["game_state"] == st) & (impact["contaminated"] != clean)]
            if len(d):
                fig.add_trace(go.Box(y=d[col], x=[st] * len(d), name=("clean window" if clean else "contaminated"),
                                     marker_color=STATE_COLORS[st], opacity=op, boxpoints="all", jitter=0.5,
                                     pointpos=0, legendgroup=str(clean), showlegend=(st == "leading"),
                                     marker=dict(size=4)))
    fig.add_hline(y=0, line=dict(color=INK, width=1))
    fig.update_layout(boxmode="group", title=f"Change in team {metric} per 15 min after vs before substitution decisions — "
                                             f"OBSERVED ASSOCIATION, not causal effect", yaxis_title="post − pre",
                      height=420)
    source_note(fig, "Confounders: score changes, opponent substitutions, fatigue, time left, red cards, regression "
                     "to the mean. 'Contaminated' = goal, red card or another own decision in the window.")
    return add_banner(fig, banner)


def candidate_comparison(cands: pd.DataFrame, weights: dict[str, float], title: str,
                         banner: str | None = None) -> go.Figure:
    comps = list(weights)
    msg = _missing(cands, ["player_name"] + comps)
    if msg:
        return _empty(msg, title)
    c = cands.head(6)
    fig = go.Figure()
    for i, k in enumerate(comps):
        fig.add_trace(go.Bar(y=c["player_name"], x=weights[k] * c[k].fillna(0), orientation="h",
                             name=f"{k.replace('_', ' ')} (w={weights[k]:.2f})", marker_color=COLORWAY[i]))
    fig.update_layout(barmode="stack", title=title, xaxis_title="Weighted points", height=360,
                      yaxis=dict(autorange="reversed"), legend=dict(orientation="h", y=-0.25), margin=dict(l=200))
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# 24-26. Manager
# ---------------------------------------------------------------------------
def manager_formation_distribution(team_stats: pd.DataFrame, manager: str, banner: str | None = None) -> go.Figure:
    t = _rows(team_stats, "manager_name", manager)
    msg = _missing(t, ["match_date", "starting_formation"])
    if msg:
        return _empty(msg, f"Formations — {manager}")
    t = t.sort_values("match_date")
    fig = make_subplots(rows=1, cols=2, column_widths=[0.35, 0.65], specs=[[{"type": "domain"}, {"type": "xy"}]],
                        subplot_titles=("Share of starts", "Formation by match (stability)"))
    vc = t["starting_formation"].value_counts()
    fig.add_trace(go.Pie(labels=vc.index, values=vc.values, hole=0.55, marker=dict(colors=COLORWAY),
                         textinfo="label+percent", showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=t["match_date"], y=t["starting_formation"], mode="markers+lines",
                             line=dict(color=GRID, width=1), marker=dict(size=11, color=t["result"].map(RESULT_COLORS)),
                             text=t["opponent"], hovertemplate="%{x|%d %b} vs %{text}: %{y}<extra></extra>",
                             showlegend=False), row=1, col=2)
    fig.update_layout(title=f"{manager}: observed formation choices (n={len(t)} matches; colour = result)",
                      height=380)
    return add_banner(fig, banner)


def rotation_trend(cont: pd.DataFrame, team: str, banner: str | None = None) -> go.Figure:
    msg = _missing(cont, ["match_date", "lineup_changes", "days_since_prev_dataset_match"])
    if msg:
        return _empty(msg, f"Rotation — {team}")
    c = cont.sort_values("match_date")
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(x=c["match_date"], y=c["lineup_changes"], name="Changes to starting XI",
                         marker_color=TEAL), secondary_y=False)
    fig.add_trace(go.Scatter(x=c["match_date"], y=c["days_since_prev_dataset_match"], name="Days since previous "
                             "(dataset) match", mode="lines+markers", line=dict(color=AMBER)), secondary_y=True)
    fig.update_yaxes(title_text="starters changed", secondary_y=False)
    fig.update_yaxes(title_text="days", secondary_y=True, showgrid=False)
    fig.update_layout(title=f"{team}: lineup rotation between consecutive analysed matches "
                            f"(mean {c['lineup_changes'].mean():.1f} changes)", height=380,
                      legend=dict(orientation="h", y=-0.2))
    source_note(fig, "Only matches in the analysed dataset are visible — cup/European fixtures in between are not.")
    return add_banner(fig, banner)


def manager_sub_profile(subs: pd.DataFrame, manager: str, bands: list[dict], banner: str | None = None
                        ) -> go.Figure:
    s = _rows(subs, "manager_name", manager)
    msg = _missing(s, ["timing_band", "game_state"])
    if msg:
        return _empty(msg, f"Substitution profile — {manager}")
    order = [b["label"] for b in bands]
    states = ["trailing", "drawing", "leading"]
    ct = pd.crosstab(s["game_state"], s["timing_band"]).reindex(index=states, columns=order, fill_value=0)
    fig = go.Figure(go.Heatmap(z=ct.values, x=order, y=states, colorscale=SEQ_SCALE, text=ct.values,
                               texttemplate="%{text}", hovertemplate="%{y}, %{x}: %{z} subs<extra></extra>",
                               colorbar=dict(title="subs")))
    fig.update_layout(title=f"{manager}: substitution timing × game state (n={len(s)} substitutions)",
                      height=340, xaxis_title="Timing band", yaxis_title="Game state")
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# 27-30. EDA, coverage, model
# ---------------------------------------------------------------------------
def correlation_heatmap(df: pd.DataFrame, cols: list[str], title: str = "Correlation (Spearman)",
                        banner: str | None = None) -> go.Figure:
    cols = [c for c in cols if c in (df.columns if df is not None else [])]
    if df is None or len(df) < 5 or len(cols) < 2:
        return _empty("Not enough rows/columns for a correlation matrix.", title)
    c = df[cols].corr(method="spearman")
    lab = [label(x) for x in cols]
    fig = go.Figure(go.Heatmap(z=c.values, x=lab, y=lab, zmin=-1, zmax=1, colorscale=DIV_SCALE,
                               text=c.round(2).values, texttemplate="%{text}", textfont=dict(size=9),
                               colorbar=dict(title="ρ")))
    fig.update_layout(title=f"{title} — n={len(df)}; association only, shared minutes/role drive many pairs",
                      height=620, xaxis=dict(tickangle=-45), margin=dict(l=170, b=150))
    return add_banner(fig, banner)


def feature_distributions(df: pd.DataFrame, metrics: list[str], title: str = "Feature distributions",
                          banner: str | None = None) -> go.Figure:
    metrics = [m for m in metrics if df is not None and m in df.columns]
    if not metrics or df is None or df.empty:
        return _empty("No data for distributions.", title)
    fig = go.Figure()
    for i, m in enumerate(metrics):
        fig.add_trace(go.Histogram(x=df[m].dropna(), nbinsx=30, visible=(i == 0), name=label(m), marker_color=TEAL,
                                   marker_line=dict(color="white", width=0.5)))
    buttons = [dict(label=label(m), method="update", args=[{"visible": [j == i for j in range(len(metrics))]},
                                                           {"xaxis.title.text": label(m)}])
               for i, m in enumerate(metrics)]
    fig.update_layout(title=title, xaxis_title=label(metrics[0]), yaxis_title="players", height=380,
                      updatemenus=[dict(buttons=buttons, x=1, xanchor="right", y=1.15)], showlegend=False)
    return add_banner(fig, banner)


def missingness_heatmap(cov: pd.DataFrame, title: str = "Field completeness by event type (%)",
                        banner: str | None = None) -> go.Figure:
    if cov is None or cov.empty:
        return _empty("No coverage table.", title)
    fig = go.Figure(go.Heatmap(z=cov.values, x=list(cov.columns), y=list(cov.index), zmin=0, zmax=100,
                               colorscale=SEQ_SCALE, text=cov.values.round(0), texttemplate="%{text}",
                               colorbar=dict(title="% present")))
    fig.update_layout(title=title + " — blanks are structural (e.g. pressures have no outcome), not errors",
                      height=520, margin=dict(l=140))
    return add_banner(fig, banner)


def confusion_matrix_chart(cm: np.ndarray, title: str = "Confusion matrix (test period)",
                           banner: str | None = None) -> go.Figure:
    if cm is None:
        return _empty("No fitted model.", title)
    lab = ["did not start", "started"]
    fig = go.Figure(go.Heatmap(z=cm, x=[f"pred: {x}" for x in lab], y=[f"actual: {x}" for x in lab],
                               colorscale=SEQ_SCALE, text=cm, texttemplate="%{text}", showscale=False))
    fig.update_layout(title=title, height=340, yaxis=dict(autorange="reversed"))
    return add_banner(fig, banner)


def roc_chart(roc: pd.DataFrame, auc: float, banner: str | None = None) -> go.Figure:
    msg = _missing(roc, ["fpr", "tpr"])
    if msg:
        return _empty(msg, "ROC curve")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=roc["fpr"], y=roc["tpr"], mode="lines", name=f"Logistic regression (AUC {auc:.3f})",
                             line=dict(color=NAVY, width=3)))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Chance", line=dict(color=MUTED, dash="dot")))
    fig.update_layout(title="ROC — chronological hold-out", xaxis_title="False positive rate",
                      yaxis_title="True positive rate", height=380)
    return add_banner(fig, banner)


def calibration_chart(cal: pd.DataFrame, banner: str | None = None) -> go.Figure:
    msg = _missing(cal, ["mean_pred", "observed_rate", "n"])
    if msg:
        return _empty(msg, "Calibration")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=cal["mean_pred"], y=cal["observed_rate"], mode="lines+markers+text", text=cal["n"],
                             textposition="top center", name="model", marker=dict(size=10, color=TEAL)))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect calibration",
                             line=dict(color=MUTED, dash="dot")))
    fig.update_layout(title="Calibration (labels = rows per bin)", xaxis_title="mean predicted probability",
                      yaxis_title="observed start rate", height=360)
    return add_banner(fig, banner)


def threshold_sensitivity_chart(season: pd.DataFrame, thresholds: list[float] | None = None,
                                banner: str | None = None) -> go.Figure:
    thresholds = thresholds or [0, 90, 180, 270, 450, 900, 1350]
    msg = _missing(season, ["minutes_played"])
    if msg:
        return _empty(msg, "Minutes threshold")
    n = [int((season["minutes_played"] >= t).sum()) for t in thresholds]
    fig = go.Figure(go.Scatter(x=thresholds, y=n, mode="lines+markers+text", text=n, textposition="top center",
                               line=dict(color=NAVY, width=3)))
    fig.update_layout(title="Eligible players vs minimum-minutes threshold", xaxis_title="minimum minutes",
                      yaxis_title="players eligible", height=320)
    return add_banner(fig, banner)


def export_figure(fig: go.Figure, path_html: str, path_png: str | None = None) -> dict[str, str]:
    """Write HTML (always, Plotly JS from CDN) and PNG if kaleido is installed."""
    out = {"html": path_html}
    fig.write_html(path_html, include_plotlyjs="cdn", full_html=True)
    if path_png:
        try:
            fig.write_image(path_png, scale=2)
            out["png"] = path_png
        except Exception as exc:  # kaleido missing or no Chrome
            out["png"] = f"skipped ({type(exc).__name__})"
    return out


# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · extended visual analytics
# Event-stream anatomy, finishing, chance contribution, animated goal-involvement race, unsupervised player archetypes (k-means on standardised per-90 profiles, PCA biplot), parallel coordinates, key-pass map, zone pass-flow matrix and shot anatomy.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
# ---------------------------------------------------------------------------
# Extended visual analytics: data scale, finishing, chance contribution,
# player archetypes (unsupervised), chance-creation maps, pass flow, animation
# ---------------------------------------------------------------------------
EVENT_FAMILY_COLORS = {"Passing": NAVY, "Carrying & dribbling": TEAL, "Defending & duels": RED,
                       "Shooting & goalkeeping": AMBER, "Match administration": SLATE, "Other": PURPLE}


def event_family(t: str) -> str:
    """Group StatsBomb event types into readable families."""
    if t in ("Pass", "Ball Receipt*"):
        return "Passing"
    if t in ("Carry", "Dribble", "Miscontrol", "Dispossessed", "Shield"):
        return "Carrying & dribbling"
    if t in ("Pressure", "Duel", "Interception", "Block", "Clearance", "Ball Recovery", "Dribbled Past", "50/50",
             "Foul Committed", "Foul Won", "Error", "Offside"):
        return "Defending & duels"
    if t in ("Shot", "Goal Keeper", "Own Goal For", "Own Goal Against"):
        return "Shooting & goalkeeping"
    if t in ("Starting XI", "Half Start", "Half End", "Substitution", "Tactical Shift", "Injury Stoppage",
             "Player On", "Player Off", "Referee Ball-Drop", "Bad Behaviour"):
        return "Match administration"
    return "Other"


def event_type_treemap(events: pd.DataFrame, title: str = "Anatomy of the event stream") -> go.Figure:
    """Treemap of every parsed event by family and type: what the 'big data' actually is."""
    msg = _missing(events, ["type"])
    if msg:
        return _empty(msg, title)
    vc = events["type"].value_counts()
    fam = pd.Series([event_family(t) for t in vc.index], index=vc.index)
    ids, labels, parents, values, colors = [], [], [], [], []
    for f, idx in fam.groupby(fam).groups.items():
        ids.append(f); labels.append(f); parents.append(""); values.append(int(vc[idx].sum()))
        colors.append(EVENT_FAMILY_COLORS[f])
        for t in idx:
            ids.append(f"{f}/{t}"); labels.append(t); parents.append(f); values.append(int(vc[t]))
            colors.append(EVENT_FAMILY_COLORS[f])
    fig = go.Figure(go.Treemap(ids=ids, labels=labels, parents=parents, values=values, branchvalues="total",
                               marker=dict(colors=colors, line=dict(color="white", width=1)),
                               textinfo="label+value+percent root",
                               hovertemplate="<b>%{label}</b><br>%{value:,} events<br>%{percentRoot:.1%} of all"
                                             "<extra></extra>"))
    fig.update_layout(title=f"{title} — {len(events):,} events, {vc.size} event types (click to zoom)", height=560,
                      margin=dict(t=70, l=10, r=10, b=10))
    return fig


def events_per_match_chart(events: pd.DataFrame, matches: pd.DataFrame) -> go.Figure:
    """Stacked event volume per match by family."""
    msg = _missing(events, ["type", "match_id"])
    if msg:
        return _empty(msg, "Events per match")
    e = events.assign(family=[event_family(t) for t in events["type"]])
    ct = pd.crosstab(e["match_id"], e["family"])
    m = matches.set_index("match_id").loc[ct.index]
    lab = [f"{d:%d %b} {h[:10]}–{a[:10]}" for d, h, a in zip(m["match_date"], m["home_team"], m["away_team"])]
    fig = go.Figure()
    for f in EVENT_FAMILY_COLORS:
        if f in ct:
            fig.add_trace(go.Bar(x=lab, y=ct[f], name=f, marker_color=EVENT_FAMILY_COLORS[f],
                                 hovertemplate="%{x}<br>" + f + ": %{y:,}<extra></extra>"))
    fig.update_layout(barmode="stack", title=f"Event volume per match (mean {ct.sum(axis=1).mean():,.0f} events/match)",
                      xaxis=dict(tickangle=-60, tickfont=dict(size=8)), yaxis_title="events", height=440,
                      legend=dict(orientation="h", y=1.08), margin=dict(b=130))
    return fig


def finishing_scatter(season_pct: pd.DataFrame, min_minutes: float, highlight_team: str | None = None) -> go.Figure:
    """Goals vs xG per player: who scored more (or less) than their chance quality suggests."""
    msg = _missing(season_pct, ["goals", "xg", "shots", "minutes_played", "player_name"])
    if msg:
        return _empty(msg, "Finishing")
    d = season_pct[(season_pct["minutes_played"] >= min_minutes) & (season_pct["shots"] > 0)].copy()
    if d.empty:
        return _empty("No shooters above the threshold.", "Finishing")
    mx = float(max(d["xg"].max(), d["goals"].max())) + 1
    is_hi = (d["team_name"] == highlight_team) if highlight_team else pd.Series(False, index=d.index)
    fig = go.Figure()
    fig.add_shape(type="path", path=f"M 0 0 L 0 {mx} L {mx} {mx} Z", fillcolor="rgba(42,157,143,0.07)",
                  line=dict(width=0), layer="below")
    fig.add_trace(go.Scatter(x=[0, mx], y=[0, mx], mode="lines", line=dict(color=MUTED, dash="dot"),
                             name="goals = xG", hoverinfo="skip"))
    for flag, name, color in ((False, "Other teams", SLATE), (True, highlight_team or "", TEAL)):
        g = d[is_hi == flag]
        if g.empty:
            continue
        fig.add_trace(go.Scatter(x=g["xg"], y=g["goals"], mode="markers", name=name,
                                 marker=dict(size=7 + 3 * np.sqrt(g["shots"]), color=color, opacity=0.8,
                                             line=dict(color="white", width=1)),
                                 text=g["player_name"], customdata=np.stack([g["shots"], g["minutes_played"]], -1),
                                 hovertemplate="<b>%{text}</b><br>%{y} goals from %{x:.2f} xG<br>%{customdata[0]} "
                                               "shots · %{customdata[1]:.0f} min<extra></extra>"))
    d["diff"] = d["goals"] - d["xg"]
    for _, r in pd.concat([d.nlargest(3, "diff"), d.nsmallest(2, "diff")]).iterrows():
        fig.add_annotation(x=r["xg"], y=r["goals"], text=r["player_name"].split()[-1], showarrow=True, arrowhead=0,
                           ax=25, ay=-20, font=dict(size=11, color=INK))
    fig.update_layout(title=f"Finishing: goals vs expected goals (n={len(d)}; shaded = scored above xG; size = shots)",
                      xaxis=dict(title="xG (StatsBomb, incl. penalties)", range=[0, mx]),
                      yaxis=dict(title="goals", range=[0, mx]), height=520)
    return fig


def goal_involvement_bars(season_pct: pd.DataFrame, team: str, n: int = 15) -> go.Figure:
    """Goals and assists per player with expected involvement (xG + xA) overlaid."""
    s = _rows(season_pct, "team_name", team)
    msg = _missing(s, ["goals", "assists", "xg", "xa", "player_name"])
    if msg:
        return _empty(msg, f"Goal involvement — {team}")
    s = s.assign(gi=s["goals"] + s["assists"], xgi=s["xg"] + s["xa"])
    s = s[s["xgi"] > 0].sort_values(["gi", "xgi"], ascending=False).head(n).iloc[::-1]
    fig = go.Figure()
    fig.add_trace(go.Bar(y=s["player_name"], x=s["goals"], orientation="h", name="Goals", marker_color=NAVY))
    fig.add_trace(go.Bar(y=s["player_name"], x=s["assists"], orientation="h", name="Assists", marker_color=TEAL))
    fig.add_trace(go.Scatter(y=s["player_name"], x=s["xgi"], mode="markers", name="xG + xA (expected)",
                             marker=dict(symbol="diamond", size=12, color=AMBER, line=dict(color=INK, width=1)),
                             hovertemplate="%{y}: %{x:.2f} xG+xA<extra></extra>"))
    fig.update_layout(barmode="stack", title=f"{team}: goal involvement vs expected involvement (top {len(s)})",
                      xaxis_title="count", height=520, margin=dict(l=200), legend=dict(orientation="h", y=-0.12))
    return fig


def chance_treemap(season_pct: pd.DataFrame, team: str) -> go.Figure:
    """Share of the team's chance creation (xG + xA) by role family and player."""
    s = _rows(season_pct, "team_name", team)
    msg = _missing(s, ["xg", "xa", "role_family", "player_name"])
    if msg:
        return _empty(msg, f"Chance contribution — {team}")
    s = s.assign(v=s["xg"] + s["xa"])
    s = s[(s["v"] > 0) & s["role_family"].notna()]
    ids, labels, parents, values, colors, cd = [], [], [], [], [], []
    for fam, g in s.groupby("role_family"):
        ids.append(fam); labels.append(fam); parents.append(""); values.append(g["v"].sum())
        colors.append(FAMILY_COLORS.get(fam, SLATE)); cd.append([g["xg"].sum(), g["xa"].sum()])
        for _, r in g.iterrows():
            ids.append(f"{fam}/{r['player_id']}"); labels.append(r["player_name"]); parents.append(fam)
            values.append(r["v"]); colors.append(FAMILY_COLORS.get(fam, SLATE)); cd.append([r["xg"], r["xa"]])
    fig = go.Figure(go.Treemap(ids=ids, labels=labels, parents=parents, values=values, branchvalues="total",
                               customdata=cd, marker=dict(colors=colors, line=dict(color="white", width=1)),
                               textinfo="label+percent root",
                               hovertemplate="<b>%{label}</b><br>xG %{customdata[0]:.2f} + xA %{customdata[1]:.2f}"
                                             "<br>%{percentRoot:.1%} of team chance creation<extra></extra>"))
    fig.update_layout(title=f"{team}: who creates the chances? (area = xG + xA)", height=520,
                      margin=dict(t=70, l=10, r=10, b=10))
    return fig


def involvement_race(pms: pd.DataFrame, team: str, top: int = 10) -> go.Figure:
    """Animated race of cumulative goals + assists over the season (press ▶)."""
    p = _rows(pms, "team_name", team)
    msg = _missing(p, ["match_date", "player_name", "goals", "assists"])
    if msg:
        return _empty(msg, f"Goal-involvement race — {team}")
    p = p.assign(gi=p["goals"] + p["assists"])
    piv = p.pivot_table(index="match_date", columns="player_name", values="gi", aggfunc="sum").fillna(0).sort_index()
    cum = piv.cumsum()
    leaders = cum.iloc[-1].sort_values(ascending=False).head(top).index
    if cum.empty or cum.iloc[-1].max() == 0:
        return _empty("No goal involvements recorded.", f"Goal-involvement race — {team}")
    color = {pl: COLORWAY[i % len(COLORWAY)] for i, pl in enumerate(leaders)}
    xmax = float(cum[leaders].values.max()) * 1.15

    def bar(i):
        v = cum.iloc[i][leaders]
        return go.Bar(x=v.values, y=list(leaders), orientation="h", text=v.astype(int).values, textposition="outside",
                      marker_color=[color[pl] for pl in leaders], hovertemplate="%{y}: %{x:.0f}<extra></extra>")

    frames = [go.Frame(data=[bar(i)], name=str(i),
                       layout=go.Layout(title_text=f"{team}: cumulative goals + assists — after match {i + 1} "
                                                   f"({cum.index[i]:%d %b %Y})")) for i in range(len(cum))]
    fig = go.Figure(data=[bar(len(cum) - 1)], frames=frames)
    steps = [dict(method="animate", label=str(i + 1), args=[[str(i)], dict(mode="immediate", frame=dict(
        duration=0, redraw=True), transition=dict(duration=0))]) for i in range(len(cum))]
    fig.update_layout(title=f"{team}: cumulative goals + assists — full season (press ▶ to replay)",
                      xaxis=dict(range=[0, xmax], title="goals + assists"),
                      yaxis=dict(categoryorder="total ascending"), height=520, margin=dict(l=200, t=90),
                      updatemenus=[dict(type="buttons", x=0, y=-0.12, xanchor="left", direction="left", buttons=[
                          dict(label="▶ Play", method="animate", args=[None, dict(frame=dict(duration=350, redraw=True),
                                                                                  fromcurrent=True)]),
                          dict(label="❚❚ Pause", method="animate", args=[[None], dict(mode="immediate", frame=dict(
                              duration=0, redraw=False))])])],
                      sliders=[dict(active=len(cum) - 1, x=0.2, len=0.8, y=-0.08, steps=steps,
                                    currentvalue=dict(prefix="match "))])
    return fig


def player_archetypes(season_pct: pd.DataFrame, metrics: list[str], min_minutes: float, seed: int = 0,
                      k_range: range = range(3, 8)) -> dict | None:
    """Unsupervised player archetypes: standardise per-90 profiles, choose k by silhouette, project with PCA.

    Descriptive grouping of playing styles in this sample — not a quality ranking.
    """
    from sklearn.cluster import KMeans
    from sklearn.decomposition import PCA
    from sklearn.metrics import silhouette_score
    from sklearn.preprocessing import StandardScaler

    d = season_pct[(season_pct["minutes_played"] >= min_minutes) & (season_pct["primary_role"] != "GK")]
    d = d.dropna(subset=metrics).copy()
    if len(d) < 12:
        return None
    X = StandardScaler().fit_transform(d[metrics])
    scores = {}
    for k in k_range:
        if k < len(d):
            lab = KMeans(n_clusters=k, n_init=20, random_state=seed).fit_predict(X)
            scores[k] = silhouette_score(X, lab)
    k = max(scores, key=scores.get)
    km = KMeans(n_clusters=k, n_init=20, random_state=seed).fit(X)
    pca = PCA(n_components=2, random_state=seed).fit(X)
    pcs = pca.transform(X)
    centers = pd.DataFrame(km.cluster_centers_, columns=metrics)
    names = {}
    for c in range(k):
        top = centers.loc[c].sort_values(ascending=False).index[:2]
        names[c] = f"A{c + 1}: high {label(top[0])} & {label(top[1])}"
    d["pc1"], d["pc2"], d["cluster"] = pcs[:, 0], pcs[:, 1], km.labels_
    d["archetype"] = d["cluster"].map(names)
    centers.index = [names[c] for c in range(k)]
    return {"frame": d, "centers_z": centers, "k": k, "silhouette": scores, "explained": pca.explained_variance_ratio_,
            "loadings": pd.DataFrame(pca.components_.T, index=metrics, columns=["pc1", "pc2"])}


def archetype_map(res: dict | None, highlight_team: str | None = None) -> go.Figure:
    """PCA map of player profiles coloured by archetype, with metric loading arrows (biplot)."""
    if not res:
        return _empty("Not enough eligible players for archetypes.", "Player archetypes")
    d, ld = res["frame"], res["loadings"]
    fig = go.Figure()
    for i, (name, g) in enumerate(d.groupby("archetype")):
        hi = (g["team_name"] == highlight_team) if highlight_team else pd.Series(False, index=g.index)
        fig.add_trace(go.Scatter(x=g["pc1"], y=g["pc2"], mode="markers+text", name=name,
                                 text=[n.split()[-1] if h else "" for n, h in zip(g["player_name"], hi)],
                                 textposition="top center", textfont=dict(size=9),
                                 marker=dict(size=np.where(hi, 13, 9), color=COLORWAY[i % len(COLORWAY)], opacity=0.85,
                                             symbol=np.where(hi, "circle", "circle-open"), line=dict(width=1.5)),
                                 customdata=np.stack([g["player_name"], g["team_name"], g["primary_position"]], -1),
                                 hovertemplate="<b>%{customdata[0]}</b> (%{customdata[1]})<br>%{customdata[2]}"
                                               "<extra>" + name + "</extra>"))
    span = max(d["pc1"].abs().max(), d["pc2"].abs().max())
    top = ld.assign(n=np.hypot(ld["pc1"], ld["pc2"])).nlargest(6, "n")
    for m, r in top.iterrows():
        fig.add_annotation(x=r["pc1"] * span * 0.9, y=r["pc2"] * span * 0.9, ax=0, ay=0, xref="x", yref="y",
                           axref="x", ayref="y", showarrow=True, arrowhead=2, arrowcolor=MUTED, arrowwidth=1.2,
                           text=label(m), font=dict(size=10, color=MUTED))
    ev = res["explained"]
    fig.update_layout(title=f"Player archetypes — k={res['k']} chosen by silhouette ({res['silhouette'][res['k']]:.2f});"
                            f" n={len(d)} outfield players<br><sup>filled = {highlight_team}; arrows = metric "
                            f"directions (PCA loadings)</sup>",
                      xaxis_title=f"PC1 ({ev[0]:.0%} of variance)", yaxis_title=f"PC2 ({ev[1]:.0%} of variance)",
                      height=620, legend=dict(orientation="h", y=-0.15))
    return fig


def archetype_profile_heatmap(res: dict | None) -> go.Figure:
    if not res:
        return _empty("Not enough eligible players for archetypes.", "Archetype profiles")
    c = res["centers_z"]
    n = res["frame"]["archetype"].value_counts()
    fig = go.Figure(go.Heatmap(z=c.values, x=[label(m) for m in c.columns], y=[f"{i} (n={n.get(i, 0)})" for i in c.index],
                               zmid=0, colorscale=DIV_SCALE, text=c.round(1).values, texttemplate="%{text}",
                               xgap=2, ygap=2, colorbar=dict(title="z-score", thickness=12),
                               hovertemplate="%{y}<br>%{x}: %{z:.2f} SD from the sample mean<extra></extra>"))
    fig.update_layout(title="What defines each archetype (cluster centre, standard deviations from the mean)",
                      height=140 + 60 * len(c), xaxis=dict(tickangle=-35), margin=dict(l=330), plot_bgcolor="white")
    return fig


def role_parallel(season_pct: pd.DataFrame, family: str, metrics: list[str], min_minutes: float,
                  highlight_team: str | None = None) -> go.Figure:
    """Parallel coordinates of percentile profiles within one role family."""
    d = season_pct[(season_pct["role_family"] == family) & (season_pct["minutes_played"] >= min_minutes)]
    cols = [f"{m}_pct" for m in metrics if f"{m}_pct" in season_pct.columns]
    d = d.dropna(subset=cols)
    if len(d) < 3:
        return _empty("Not enough players.", f"Parallel coordinates — {family}")
    flag = (d["team_name"] == highlight_team).astype(int) if highlight_team else np.zeros(len(d))
    fig = go.Figure(go.Parcoords(line=dict(color=flag, colorscale=[[0, "#c9ced6"], [1, TEAL]], cmin=0, cmax=1),
                                 dimensions=[dict(label=label(c[:-4]), values=d[c], range=[0, 100]) for c in cols]))
    fig.update_layout(title=f"{family}s: percentile profiles across metrics (teal = {highlight_team}; drag on an axis "
                            f"to filter; n={len(d)})", height=460, margin=dict(l=60, r=60, t=90))
    return fig


def key_pass_map(flags: pd.DataFrame, team_id: Any, title: str) -> go.Figure:
    """Every pass that led directly to a shot; amber = assist for a goal."""
    msg = _missing(flags, ["key_passes", "assists", "x", "y", "end_x", "end_y"])
    if msg:
        return _empty(msg, title)
    kp = flags[(flags["team_id"] == team_id) & (flags["key_passes"] == 1)].dropna(subset=["x", "y", "end_x", "end_y"])
    fig = draw_pitch()
    for name, sub, color, w in (("Shot assists", kp[kp["assists"] == 0], NAVY, 1.2),
                                ("Goal assists", kp[kp["assists"] == 1], AMBER, 2.6)):
        xs, ys = [], []
        for r in sub.itertuples():
            xs += [r.x, r.end_x, None]
            ys += [r.y, r.end_y, None]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=color, width=w), opacity=0.6,
                                 name=f"{name} ({len(sub)})", hoverinfo="skip"))
        fig.add_trace(go.Scatter(x=sub["end_x"], y=sub["end_y"], mode="markers", showlegend=False,
                                 marker=dict(size=6, color=color), text=sub["player_name"],
                                 hovertemplate="from %{text}<extra></extra>"))
    attack_arrow(fig)
    fig.update_layout(title=f"{title}<br><sup>{len(kp)} key passes · origin → shot location</sup>", height=520,
                      legend=dict(orientation="h", y=-0.05))
    return fig


def pass_flow_matrix(flags: pd.DataFrame, team_id: Any, title: str) -> go.Figure:
    """Completed passes between 9 pitch zones (thirds × lanes): how the team moves the ball."""
    msg = _missing(flags, ["x", "y", "end_x", "end_y", "passes_completed"])
    if msg:
        return _empty(msg, title)
    p = flags[(flags["team_id"] == team_id) & (flags["passes_completed"] == 1)].dropna(subset=["x", "y", "end_x", "end_y"])
    if p.empty:
        return _empty("No completed passes.", title)
    z0 = np.digitize(p["x"], [40, 80]) * 3 + np.digitize(p["y"], [80 / 3, 160 / 3])
    z1 = np.digitize(p["end_x"], [40, 80]) * 3 + np.digitize(p["end_y"], [80 / 3, 160 / 3])
    labs = [f"{t}-{l}" for t in ("Def", "Mid", "Att") for l in ("Left", "Centre", "Right")]
    mat = pd.crosstab(pd.Categorical(z0, categories=range(9)), pd.Categorical(z1, categories=range(9)),
                      dropna=False).to_numpy()
    share = mat / mat.sum() * 100
    fwd = (np.digitize(p["end_x"], [40, 80]) > np.digitize(p["x"], [40, 80])).mean()
    fig = go.Figure(go.Heatmap(z=share, x=labs, y=labs, colorscale=SEQ_SCALE, text=np.round(share, 1),
                               texttemplate="%{text}", xgap=2, ygap=2, colorbar=dict(title="% of passes", thickness=12),
                               hovertemplate="from %{y} to %{x}: %{z:.1f}%<extra></extra>"))
    fig.update_layout(title=f"{title}<br><sup>{len(p):,} completed passes · {fwd:.0%} move the ball into a more "
                            f"advanced third · lanes as seen attacking left → right</sup>",
                      xaxis_title="to zone", yaxis=dict(title="from zone", autorange="reversed"), height=560,
                      plot_bgcolor="white", margin=dict(l=120))
    return fig


def shot_sunburst(flags: pd.DataFrame, team: str) -> go.Figure:
    """Shots by play pattern → body part → outcome."""
    s = flags[(flags["type"] == "Shot") & (flags["team_name"] == team)] if flags is not None and "type" in flags else None
    msg = _missing(s, ["play_pattern", "shot_body_part", "outcome"])
    if msg:
        return _empty(msg, f"Shot anatomy — {team}")
    s = s.fillna({"play_pattern": "Unknown", "shot_body_part": "Unknown", "outcome": "Unknown"})
    ids, labels, parents, values, colors = [], [], [], [], []
    for pp, g1 in s.groupby("play_pattern"):
        ids.append(pp); labels.append(pp); parents.append(""); values.append(len(g1)); colors.append(NAVY)
        for bp, g2 in g1.groupby("shot_body_part"):
            i2 = f"{pp}/{bp}"
            ids.append(i2); labels.append(bp); parents.append(pp); values.append(len(g2)); colors.append(TEAL)
            for oc, g3 in g2.groupby("outcome"):
                ids.append(f"{i2}/{oc}"); labels.append(oc); parents.append(i2); values.append(len(g3))
                colors.append(SHOT_COLORS.get(oc, SLATE))
    fig = go.Figure(go.Sunburst(ids=ids, labels=labels, parents=parents, values=values, branchvalues="total",
                                marker=dict(colors=colors, line=dict(color="white", width=1)),
                                hovertemplate="<b>%{label}</b><br>%{value} shots (%{percentParent:.0%} of parent)"
                                              "<extra></extra>"))
    fig.update_layout(title=f"{team}: shot anatomy — play pattern → body part → outcome ({len(s)} shots; outer ring "
                            f"amber = goal)", height=560, margin=dict(t=70, l=10, r=10, b=10))
    return fig


# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · gallery — every other chart type
# Gauges, funnel, waterfall, calendar heatmap, dumbbell, area, variability band, polar goal clock, box plots, opponent matrix and bubbles, scatter-plot matrix, marginal scatter, ECDF, split violins, clustered correlation, dendrogram, lollipop, slope chart, radar grid, ternary plot, contribution matrix, density contours, parallel categories and a Gantt chart.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
# ---------------------------------------------------------------------------
# Visual gallery: gauges, funnel, waterfall, calendar, dumbbell, area, bands,
# polar clock, boxes, opponent matrix/bubbles, SPLOM, marginals, ECDF,
# violins, clustered correlation, dendrogram, lollipop, slope, radar grid,
# ternary, contribution matrix, density contours, parallel categories, Gantt
# ---------------------------------------------------------------------------
import plotly.express as px

GALLERY_ACTIONS = {
    "Creating & finishing": ["shots_p90", "key_passes_p90", "dribbles_completed_p90", "box_touches_p90"],
    "Progressing": ["progressive_passes_p90", "progressive_carries_p90", "final_third_passes_p90"],
    "Defending": ["tackles_won_p90", "interceptions_p90", "ball_recoveries_p90", "pressures_p90", "clearances_p90"],
}


def _team(ts: pd.DataFrame, team: str) -> pd.DataFrame | None:
    t = _rows(ts, "team_name", team)
    return t.sort_values("match_date").reset_index(drop=True) if t is not None and len(t) else t


def _mlabel(r) -> str:
    return f"{r['match_date']:%d %b} {'v' if r['home_away'] == 'home' else '@'} {str(r['opponent'])[:14]}"


def kpi_gauges(team_stats: pd.DataFrame, team: str) -> go.Figure:
    """Five gauges: win rate, points per match, xG share, possession proxy, field tilt."""
    t = _team(team_stats, team)
    msg = _missing(t, ["result", "points", "xg", "xg_against", "possession_proxy_pct", "field_tilt_pct"])
    if msg:
        return _empty(msg, "Season gauges")
    vals = [("Win rate", 100 * (t["result"] == "W").mean(), 100, "%"), ("Points / match", t["points"].mean(), 3, ""),
            ("xG share", 100 * t["xg"].sum() / (t["xg"].sum() + t["xg_against"].sum()), 100, "%"),
            ("Possession proxy", t["possession_proxy_pct"].mean(), 100, "%"),
            ("Field tilt", t["field_tilt_pct"].mean(), 100, "%")]
    fig = make_subplots(rows=1, cols=5, specs=[[{"type": "indicator"}] * 5])
    for i, (name, v, mx, suf) in enumerate(vals, 1):
        fig.add_trace(go.Indicator(
            mode="gauge+number", value=float(v), number=dict(suffix=suf, valueformat=".2f" if mx == 3 else ".0f",
                                                              font=dict(size=26, color=NAVY)),
            title=dict(text=name, font=dict(size=13, color=MUTED)),
            gauge=dict(axis=dict(range=[0, mx], tickfont=dict(size=9)), bar=dict(color=TEAL, thickness=0.35),
                       bgcolor="white", borderwidth=0,
                       steps=[dict(range=[0, mx * 0.5], color="#f1efe8"), dict(range=[mx * 0.5, mx * 0.75],
                                                                               color="#e3ece8")],
                       threshold=dict(line=dict(color=AMBER, width=3), value=mx * 0.5))), row=1, col=i)
    fig.update_layout(title=f"{team}: season gauges (amber tick = 50% / 1.5 points)", height=270,
                      margin=dict(t=80, b=10))
    return fig


def attack_funnel(flags: pd.DataFrame, team_id: Any, team: str) -> go.Figure:
    """Approximate attacking funnel from completed passes to goals (stages are overlapping event counts)."""
    f = _rows(flags, "team_id", team_id)
    msg = _missing(f, ["passes_completed", "final_third_passes", "carries_into_final_third", "box_passes",
                       "carries_into_box", "shots", "shots_on_target", "goals"])
    if msg:
        return _empty(msg, "Attacking funnel")
    stages = [("Completed passes", f["passes_completed"].sum()),
              ("Final-third entries", f["final_third_passes"].sum() + f["carries_into_final_third"].sum()),
              ("Box entries", f["box_passes"].sum() + f["carries_into_box"].sum()),
              ("Shots", f["shots"].sum()), ("Shots on target", f["shots_on_target"].sum()), ("Goals", f["goals"].sum())]
    fig = go.Figure(go.Funnel(y=[s[0] for s in stages], x=[int(s[1]) for s in stages], textinfo="value+percent previous",
                              marker=dict(color=[NAVY, "#2b5f86", "#3f8f99", TEAL, AMBER, RED]),
                              connector=dict(line=dict(color=GRID))))
    fig.update_layout(title=f"{team}: attacking funnel — how possession turns into goals (season)", height=420,
                      margin=dict(l=160))
    return fig


def monthly_waterfall(team_stats: pd.DataFrame, team: str) -> go.Figure:
    """Goal difference built up month by month."""
    t = _team(team_stats, team)
    msg = _missing(t, ["match_date", "goal_diff"])
    if msg:
        return _empty(msg, "Goal-difference waterfall")
    m = t.groupby(t["match_date"].dt.to_period("M"))["goal_diff"].sum()
    x = [p.strftime("%b %Y") for p in m.index] + ["Season"]
    fig = go.Figure(go.Waterfall(x=x, y=list(m.values) + [0], measure=["relative"] * len(m) + ["total"],
                                 text=[f"{v:+d}" for v in m.values] + [f"{m.sum():+d}"], textposition="outside",
                                 increasing=dict(marker=dict(color=TEAL)), decreasing=dict(marker=dict(color=RED)),
                                 totals=dict(marker=dict(color=NAVY)), connector=dict(line=dict(color=GRID))))
    fig.update_layout(title=f"{team}: goal difference built month by month", yaxis_title="goal difference",
                      height=420, showlegend=False)
    return fig


def calendar_heatmap(team_stats: pd.DataFrame, team: str) -> go.Figure:
    """Fixture calendar (week × weekday) coloured by xG difference."""
    t = _team(team_stats, team)
    msg = _missing(t, ["match_date", "xg_diff", "result", "opponent"])
    if msg:
        return _empty(msg, "Fixture calendar")
    start = t["match_date"].min() - pd.Timedelta(days=t["match_date"].min().weekday())
    weeks = pd.date_range(start, t["match_date"].max(), freq="7D")
    z = np.full((7, len(weeks)), np.nan)
    txt = np.full((7, len(weeks)), "", dtype=object)
    hov = np.full((7, len(weeks)), "no match", dtype=object)
    for _, r in t.iterrows():
        w = (r["match_date"] - start).days // 7
        d = r["match_date"].weekday()
        z[d, w] = r["xg_diff"]
        txt[d, w] = r["result"]
        hov[d, w] = f"{r['match_date']:%a %d %b %Y} vs {r['opponent']} — {r['goals_for']}-{r['goals_against']}, " \
                    f"xG diff {r['xg_diff']:+.2f}"
    fig = go.Figure(go.Heatmap(z=z, x=[f"{w:%d %b}" for w in weeks], y=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
                               colorscale=DIV_SCALE, zmid=0, text=txt, texttemplate="%{text}", customdata=hov,
                               xgap=3, ygap=3, colorbar=dict(title="xG diff", thickness=12),
                               hovertemplate="%{customdata}<extra></extra>"))
    fig.update_layout(title=f"{team}: fixture calendar — every league match by week and weekday (colour = xG "
                            f"difference, letter = result)", height=330, xaxis=dict(tickangle=-60, tickfont=dict(size=8)),
                      yaxis=dict(autorange="reversed"), plot_bgcolor="white")
    return fig


def xg_dumbbell(team_stats: pd.DataFrame, team: str) -> go.Figure:
    """Dumbbell per match: xG for vs xG against."""
    t = _team(team_stats, team)
    msg = _missing(t, ["xg", "xg_against", "opponent", "home_away"])
    if msg:
        return _empty(msg, "xG dumbbell")
    lab = [_mlabel(r) for _, r in t.iterrows()]
    fig = go.Figure()
    for i, r in t.iterrows():
        fig.add_trace(go.Scatter(x=[r["xg_against"], r["xg"]], y=[lab[i]] * 2, mode="lines",
                                 line=dict(color=RESULT_COLORS[r["result"]], width=3), showlegend=False,
                                 hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=t["xg_against"], y=lab, mode="markers", name="xG against",
                             marker=dict(size=10, color=RED, line=dict(color="white", width=1))))
    fig.add_trace(go.Scatter(x=t["xg"], y=lab, mode="markers", name="xG for",
                             marker=dict(size=10, color=NAVY, line=dict(color="white", width=1)),
                             customdata=np.stack([t["goals_for"], t["goals_against"]], -1),
                             hovertemplate="%{y}: xG %{x:.2f} · score %{customdata[0]}-%{customdata[1]}<extra></extra>"))
    fig.update_layout(title=f"{team}: xG for vs against in every match (bar colour = result)", height=900,
                      yaxis=dict(autorange="reversed", tickfont=dict(size=9)), xaxis_title="xG",
                      legend=dict(orientation="h", y=1.03), margin=dict(l=170))
    return fig


def cumulative_area(team_stats: pd.DataFrame, team: str) -> go.Figure:
    """Cumulative goals and xG for/against as area charts."""
    t = _team(team_stats, team)
    msg = _missing(t, ["xg", "xg_against", "goals_for", "goals_against"])
    if msg:
        return _empty(msg, "Cumulative season")
    n = np.arange(1, len(t) + 1)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=n, y=t["xg"].cumsum(), name="xG for", fill="tozeroy", line=dict(color=NAVY),
                             fillcolor="rgba(29,53,87,0.25)"))
    fig.add_trace(go.Scatter(x=n, y=t["xg_against"].cumsum(), name="xG against", fill="tozeroy",
                             line=dict(color=RED), fillcolor="rgba(200,85,61,0.25)"))
    fig.add_trace(go.Scatter(x=n, y=t["goals_for"].cumsum(), name="Goals for", mode="lines",
                             line=dict(color=NAVY, dash="dot", width=2.5)))
    fig.add_trace(go.Scatter(x=n, y=t["goals_against"].cumsum(), name="Goals against", mode="lines",
                             line=dict(color=RED, dash="dot", width=2.5)))
    fig.update_layout(title=f"{team}: cumulative goals and xG — the gap between areas is the season's dominance",
                      xaxis_title="match number", yaxis_title="cumulative", height=420,
                      legend=dict(orientation="h", y=-0.2))
    return fig


def rolling_band(team_stats: pd.DataFrame, team: str, window: int = 5) -> go.Figure:
    """Rolling xG difference with a ±1 SD band."""
    t = _team(team_stats, team)
    msg = _missing(t, ["xg_diff"])
    if msg:
        return _empty(msg, "Rolling xG difference")
    n = np.arange(1, len(t) + 1)
    mu = t["xg_diff"].rolling(window, min_periods=2).mean()
    sd = t["xg_diff"].rolling(window, min_periods=2).std()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.r_[n, n[::-1]], y=np.r_[mu + sd, (mu - sd)[::-1]], fill="toself",
                             fillcolor="rgba(42,157,143,0.18)", line=dict(width=0), name="±1 SD", hoverinfo="skip"))
    fig.add_trace(go.Bar(x=n, y=t["xg_diff"], name="match xG difference", marker_color=t["result"].map(RESULT_COLORS),
                         opacity=0.45))
    fig.add_trace(go.Scatter(x=n, y=mu, name=f"{window}-match rolling mean", line=dict(color=TEAL, width=3)))
    fig.add_hline(y=0, line=dict(color=INK, width=1))
    fig.update_layout(title=f"{team}: performance trend — rolling xG difference with variability band",
                      xaxis_title="match number", yaxis_title="xG for − against", height=400,
                      legend=dict(orientation="h", y=-0.2))
    return fig


def goal_clock(goals: pd.DataFrame, team_stats: pd.DataFrame, team: str, bin_min: int = 5) -> go.Figure:
    """Polar 'clock' of goals scored and conceded by match minute."""
    t = _team(team_stats, team)
    msg = _missing(t, ["match_id", "team_id"]) or _missing(goals, ["minute", "scoring_team_id", "match_id"])
    if msg:
        return _empty(msg, "Goal clock")
    tid = t["team_id"].iloc[0]
    g = goals[goals["match_id"].isin(t["match_id"])]
    edges = np.arange(0, 100, bin_min)
    fig = go.Figure()
    for name, sub, c in (("Scored", g[g["scoring_team_id"] == tid], NAVY), ("Conceded", g[g["scoring_team_id"] != tid], RED)):
        cnt = pd.cut(sub["minute"].clip(upper=99), np.r_[edges, 100], right=False).value_counts(sort=False)
        fig.add_trace(go.Barpolar(r=cnt.values, theta=(edges + bin_min / 2) * 360 / 100, width=[bin_min * 3.6 * 0.9] * len(edges),
                                  name=f"{name} ({len(sub)})", marker_color=c, opacity=0.8,
                                  customdata=[f"{a}-{a + bin_min - 1}'" for a in edges],
                                  hovertemplate="%{customdata}: %{r} goals<extra>" + name + "</extra>"))
    fig.update_layout(title=f"{team}: goal clock — when goals are scored and conceded ({bin_min}-minute slices)",
                      polar=dict(angularaxis=dict(rotation=90, direction="clockwise", tickvals=[0, 54, 108, 162, 216, 270, 324],
                                                  ticktext=["0'", "15'", "30'", "45'", "60'", "75'", "90'"], gridcolor=GRID),
                                 radialaxis=dict(gridcolor=GRID), bgcolor=PAPER),
                      height=500, legend=dict(orientation="h", y=-0.08))
    return fig


def venue_boxes(team_stats: pd.DataFrame, team: str) -> go.Figure:
    """Box plots of xG for/against and possession by venue."""
    t = _team(team_stats, team)
    msg = _missing(t, ["home_away", "xg", "xg_against", "possession_proxy_pct"])
    if msg:
        return _empty(msg, "Home vs away")
    fig = make_subplots(rows=1, cols=3, subplot_titles=("xG for", "xG against", "Possession proxy %"))
    for i, col in enumerate(["xg", "xg_against", "possession_proxy_pct"], 1):
        for venue, c in (("home", NAVY), ("away", AMBER)):
            d = t[t["home_away"] == venue]
            fig.add_trace(go.Box(y=d[col], name=venue, marker_color=c, boxpoints="all", jitter=0.4, pointpos=0,
                                 showlegend=(i == 1), legendgroup=venue, text=d["opponent"],
                                 hovertemplate="%{text}: %{y:.2f}<extra></extra>"), row=1, col=i)
    fig.update_layout(title=f"{team}: home vs away (each dot is a match)", height=400)
    return fig


def opponent_matrix(team_stats: pd.DataFrame, team: str) -> go.Figure:
    """Opponent × metric matrix (average over meetings; colour standardised per column)."""
    t = _team(team_stats, team)
    msg = _missing(t, ["opponent", "goals_for", "goals_against", "xg", "xg_against"])
    if msg:
        return _empty(msg, "Opponent matrix")
    cols = {"points": "Points", "goals_for": "Goals for", "goals_against": "Goals against", "xg": "xG for",
            "xg_against": "xG against", "possession_proxy_pct": "Possession %", "field_tilt_pct": "Field tilt %",
            "ppda": "PPDA"}
    m = t.groupby("opponent")[list(cols)].mean()
    m = m.loc[m["points"].sort_values().index]
    sign = pd.Series({c: (-1 if c in ("goals_against", "xg_against", "ppda") else 1) for c in cols})
    z = ((m - m.mean()) / m.std(ddof=0).replace(0, 1)) * sign
    fig = go.Figure(go.Heatmap(z=z.values, x=list(cols.values()), y=list(m.index), zmid=0, colorscale=DIV_SCALE,
                               text=m.round(2).values, texttemplate="%{text}", xgap=2, ygap=2, showscale=False,
                               hovertemplate="%{y} · %{x}: %{text}<extra></extra>"))
    fig.update_layout(title=f"{team} vs each opponent (average over meetings; blue = favourable for {team})",
                      height=140 + 28 * len(m), margin=dict(l=190), plot_bgcolor="white", xaxis=dict(side="top"))
    return fig


def opponent_bubbles(team_stats: pd.DataFrame, team: str) -> go.Figure:
    """Bubble chart per opponent: chances created vs conceded, size = goals scored, colour = points."""
    t = _team(team_stats, team)
    msg = _missing(t, ["opponent", "xg", "xg_against", "goals_for", "points"])
    if msg:
        return _empty(msg, "Opponent bubbles")
    m = t.groupby("opponent").agg(xg=("xg", "mean"), xga=("xg_against", "mean"), gf=("goals_for", "sum"),
                                  ppm=("points", "mean"), n=("match_id", "size")).reset_index()
    fig = go.Figure(go.Scatter(x=m["xga"], y=m["xg"], mode="markers+text", text=m["opponent"].str.slice(0, 12),
                               textposition="top center", textfont=dict(size=9),
                               marker=dict(size=12 + 5 * m["gf"], color=m["ppm"], colorscale=[[0, RED], [0.5, AMBER],
                                                                                              [1, TEAL]],
                                           cmin=0, cmax=3, colorbar=dict(title="points/<br>match", thickness=12),
                                           line=dict(color="white", width=1), opacity=0.85),
                               customdata=np.stack([m["gf"], m["n"]], -1),
                               hovertemplate="%{text}<br>xG for %{y:.2f} / against %{x:.2f}<br>%{customdata[0]} goals in "
                                             "%{customdata[1]} meetings<extra></extra>"))
    fig.update_layout(title=f"{team}: opponents by chances created and conceded (bubble = goals scored)",
                      xaxis_title="xG conceded per meeting", yaxis_title="xG created per meeting", height=520)
    return fig


def metric_splom(season_pct: pd.DataFrame, metrics: list[str], min_minutes: float) -> go.Figure:
    """Scatter-plot matrix of several per-90 metrics coloured by role family."""
    d = season_pct[(season_pct["minutes_played"] >= min_minutes) & season_pct["role_family"].notna()]
    metrics = [m for m in metrics if m in d.columns]
    if len(d) < 5 or len(metrics) < 3:
        return _empty("Not enough data.", "Scatter-plot matrix")
    fig = go.Figure(go.Splom(dimensions=[dict(label=label(m), values=d[m]) for m in metrics], text=d["player_name"],
                             marker=dict(size=5, color=d["role_family"].map(FAMILY_COLORS).fillna(SLATE), opacity=0.7,
                                         line=dict(color="white", width=0.4)),
                             diagonal=dict(visible=False), showupperhalf=False,
                             hovertemplate="%{text}<extra></extra>"))
    fig.update_layout(title=f"Scatter-plot matrix of per-90 metrics (n={len(d)}; colour = role family)",
                      height=760, dragmode="select", font=dict(size=10))
    return fig


def marginal_scatter(season_pct: pd.DataFrame, x: str, y: str, min_minutes: float) -> go.Figure:
    """Scatter with marginal histogram and box plot, by role family."""
    d = season_pct[(season_pct["minutes_played"] >= min_minutes) & season_pct["role_family"].notna()]
    if d.empty or x not in d or y not in d:
        return _empty("Not enough data.", "Marginal scatter")
    fig = px.scatter(d, x=x, y=y, color="role_family", color_discrete_map=FAMILY_COLORS, hover_name="player_name",
                     marginal_x="histogram", marginal_y="box", opacity=0.8,
                     labels={x: label(x), y: label(y), "role_family": "role family"})
    fig.update_layout(title=f"{label(x)} vs {label(y)} with marginal distributions (n={len(d)})", height=560)
    return fig


def minutes_ecdf(season: pd.DataFrame, team: str) -> go.Figure:
    """Empirical CDF of minutes played: focus squad vs everyone else."""
    if season is None or "minutes_played" not in season:
        return _empty("No data.", "Minutes ECDF")
    d = season[season["minutes_played"] > 0].assign(group=lambda s: np.where(s["team_name"] == team, team, "Opponents"))
    fig = px.ecdf(d, x="minutes_played", color="group", color_discrete_map={team: TEAL, "Opponents": SLATE},
                  markers=True, labels={"minutes_played": "minutes played"})
    fig.update_layout(title="Distribution of playing time (ECDF): opponents rarely exceed two matches in this sample",
                      yaxis_title="share of players ≤ x minutes", height=400)
    return fig


def split_violin(season_pct: pd.DataFrame, metric: str, team: str, min_minutes: float) -> go.Figure:
    """Violin per role family, split focus squad (left) vs others (right)."""
    d = season_pct[(season_pct["minutes_played"] >= min_minutes) & season_pct["role_family"].notna()]
    if d.empty or metric not in d:
        return _empty("Not enough data.", "Violins")
    fig = go.Figure()
    for grp, side, c in ((team, "negative", TEAL), ("Others", "positive", SLATE)):
        g = d[(d["team_name"] == team) == (grp == team)]
        fig.add_trace(go.Violin(x=g["role_family"], y=g[metric], name=grp, side=side, line_color=c,
                                fillcolor=c, opacity=0.55, points="all", pointpos=-0.4 if side == "negative" else 0.4,
                                jitter=0.2, marker=dict(size=4), text=g["player_name"], meanline_visible=True,
                                hovertemplate="%{text}: %{y:.2f}<extra></extra>"))
    fig.update_layout(violinmode="overlay", title=f"{label(metric)} by role family — {team} (left) vs others (right)",
                      yaxis_title=label(metric), height=440, xaxis=dict(categoryorder="array", categoryarray=[
                          "Defender", "Midfielder", "Forward"]))
    return fig


def clustered_correlation(season_pct: pd.DataFrame, metrics: list[str], min_minutes: float) -> go.Figure:
    """Spearman correlation heatmap reordered by hierarchical clustering of metrics."""
    from scipy.cluster.hierarchy import leaves_list, linkage
    from scipy.spatial.distance import squareform
    d = season_pct[season_pct["minutes_played"] >= min_minutes]
    metrics = [m for m in metrics if m in d]
    if len(d) < 10 or len(metrics) < 3:
        return _empty("Not enough data.", "Clustered correlation")
    c = d[metrics].corr(method="spearman").fillna(0)
    dist = squareform((1 - c.abs()).clip(lower=0).values, checks=False)
    order = [metrics[i] for i in leaves_list(linkage(dist, "average"))]
    c = c.loc[order, order]
    lab = [label(m) for m in order]
    fig = go.Figure(go.Heatmap(z=c.values, x=lab, y=lab, zmin=-1, zmax=1, colorscale=DIV_SCALE, text=c.round(2).values,
                               texttemplate="%{text}", textfont=dict(size=9), colorbar=dict(title="ρ")))
    fig.update_layout(title=f"Metric families — Spearman correlations ordered by hierarchical clustering (n={len(d)})",
                      height=620, xaxis=dict(tickangle=-45), yaxis=dict(autorange="reversed"), margin=dict(l=170, b=150))
    return fig


def player_dendrogram(season_pct: pd.DataFrame, team: str, metrics: list[str], min_minutes: float) -> go.Figure:
    """Hierarchical clustering of the squad's per-90 profiles (who plays like whom)."""
    import plotly.figure_factory as ff
    d = season_pct[(season_pct["team_name"] == team) & (season_pct["minutes_played"] >= min_minutes)
                   & (season_pct["primary_role"] != "GK")].dropna(subset=metrics)
    if len(d) < 4:
        return _empty("Not enough players.", "Squad dendrogram")
    X = ((d[metrics] - d[metrics].mean()) / d[metrics].std(ddof=0).replace(0, 1)).values
    names = [f"{n} ({r})" for n, r in zip(d["player_name"], d["primary_role"])]
    fig = ff.create_dendrogram(X, orientation="left", labels=names, colorscale=[NAVY, TEAL, AMBER, RED, PURPLE, SLATE])
    fig.update_layout(title=f"{team}: squad similarity tree (complete-linkage clustering of standardised per-90 profiles)",
                      height=40 + 26 * len(d), margin=dict(l=260), xaxis_title="distance (lower = more similar)",
                      plot_bgcolor=PAPER, paper_bgcolor=PAPER)
    return fig


def lollipop(season_pct: pd.DataFrame, metric: str, min_minutes: float, n: int = 15) -> go.Figure:
    d = season_pct[season_pct["minutes_played"] >= min_minutes].dropna(subset=[metric])
    if d.empty:
        return _empty("No data.", "Lollipop")
    d = d.nlargest(n, metric).iloc[::-1]
    fig = go.Figure()
    for _, r in d.iterrows():
        fig.add_trace(go.Scatter(x=[0, r[metric]], y=[r["player_name"]] * 2, mode="lines", showlegend=False,
                                 line=dict(color=GRID, width=3), hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=d[metric], y=d["player_name"], mode="markers+text", text=d[metric].round(2),
                             textposition="middle right", showlegend=False,
                             marker=dict(size=14, color=d["role_family"].map(FAMILY_COLORS).fillna(SLATE),
                                         line=dict(color="white", width=1.5)),
                             customdata=d["team_name"], hovertemplate="%{y} (%{customdata}): %{x:.2f}<extra></extra>"))
    fig.update_layout(title=f"Top {len(d)}: {label(metric)} (min {min_minutes:.0f} min)", height=520,
                      xaxis_title=label(metric), margin=dict(l=200))
    return fig


def slope_chart(pms: pd.DataFrame, team: str, base_metric: str, min_half_minutes: float = 400) -> go.Figure:
    """First half vs second half of the season, per 90, for each regular."""
    p = _rows(pms, "team_name", team)
    msg = _missing(p, ["match_date", "minutes_played", base_metric, "player_name"])
    if msg:
        return _empty(msg, "Slope chart")
    cut = p["match_date"].sort_values().unique()
    cut = cut[len(cut) // 2]
    p = p.assign(half=np.where(p["match_date"] < cut, "First half", "Second half"))
    g = p.groupby(["player_name", "half"])[[base_metric, "minutes_played"]].sum().reset_index()
    g["p90"] = g[base_metric] / g["minutes_played"] * 90
    w = g.pivot(index="player_name", columns="half", values="p90")
    mins = g.pivot(index="player_name", columns="half", values="minutes_played")
    w = w[(mins >= min_half_minutes).all(axis=1)].dropna()
    if w.empty:
        return _empty("No player with enough minutes in both halves.", "Slope chart")
    fig = go.Figure()
    for name, r in w.iterrows():
        up = r["Second half"] >= r["First half"]
        fig.add_trace(go.Scatter(x=["First half", "Second half"], y=[r["First half"], r["Second half"]],
                                 mode="lines+markers+text", text=["", name.split()[-1]], textposition="middle right",
                                 line=dict(color=TEAL if up else RED, width=2), marker=dict(size=8), showlegend=False,
                                 hovertemplate=name + ": %{y:.2f}<extra></extra>"))
    fig.update_layout(title=f"{team}: {label(base_metric + '_p90')} — first vs second half of the season "
                            f"(teal = up, red = down; ≥{min_half_minutes:.0f} min in each half)",
                      yaxis_title="per 90", height=520, margin=dict(r=140))
    return fig


def radar_grid(season_pct: pd.DataFrame, player_ids: list, config: Config) -> go.Figure:
    """Small-multiple radars (role-specific metrics) for up to six players."""
    s = season_pct[season_pct["player_id"].isin(player_ids)].drop_duplicates("player_id")
    if s.empty:
        return _empty("No players.", "Radar grid")
    present = set(s["player_id"])
    s = s.set_index("player_id").loc[[p for p in player_ids if p in present]].reset_index().head(6)
    rows = int(np.ceil(len(s) / 3))
    fig = make_subplots(rows=rows, cols=3, specs=[[{"type": "polar"}] * 3] * rows,
                        subplot_titles=[f"{r['player_name']} ({r['primary_role']})" for _, r in s.iterrows()],
                        vertical_spacing=0.12)
    for i, (_, r) in enumerate(s.iterrows()):
        ms = [m for m in radar_metrics(r["primary_role"], config) if f"{m}_pct" in s.columns]
        th = [label(m) for m in ms]
        v = [float(r[f"{m}_pct"]) if pd.notna(r[f"{m}_pct"]) else 0 for m in ms]
        fig.add_trace(go.Scatterpolar(r=v + v[:1], theta=th + th[:1], fill="toself", showlegend=False,
                                      line=dict(color=COLORWAY[i % len(COLORWAY)], width=2),
                                      hovertemplate="%{theta}: %{r:.0f}th pct<extra></extra>"),
                      row=i // 3 + 1, col=i % 3 + 1)
    fig.update_polars(radialaxis=dict(range=[0, 100], showticklabels=False, gridcolor=GRID),
                      angularaxis=dict(tickfont=dict(size=8), gridcolor=GRID), bgcolor=PAPER)
    fig.update_layout(title="Key players at a glance — role-specific percentile radars", height=420 * rows,
                      margin=dict(t=110))
    return fig


def action_ternary(season_pct: pd.DataFrame, min_minutes: float, team: str | None = None) -> go.Figure:
    """Ternary plot of each player's action mix: creating/finishing vs progressing vs defending."""
    d = season_pct[(season_pct["minutes_played"] >= min_minutes) & (season_pct["primary_role"] != "GK")].copy()
    if d.empty:
        return _empty("No data.", "Action mix")
    for k, ms in GALLERY_ACTIONS.items():
        d[k] = d[[m for m in ms if m in d]].fillna(0).sum(axis=1)
    tot = d[list(GALLERY_ACTIONS)].sum(axis=1).replace(0, np.nan)
    a, b, c = (d[k] / tot * 100 for k in GALLERY_ACTIONS)
    hi = (d["team_name"] == team) if team else pd.Series(False, index=d.index)
    fig = go.Figure(go.Scatterternary(a=a, b=b, c=c, mode="markers+text",
                                      text=[n.split()[-1] if h else "" for n, h in zip(d["player_name"], hi)],
                                      textposition="top center", textfont=dict(size=8),
                                      marker=dict(size=np.where(hi, 11, 7), color=d["role_family"].map(FAMILY_COLORS).fillna(SLATE),
                                                  opacity=0.8, line=dict(color="white", width=0.6)),
                                      customdata=np.stack([d["player_name"], d["team_name"]], -1),
                                      hovertemplate="%{customdata[0]} (%{customdata[1]})<br>create %{a:.0f}% · progress "
                                                    "%{b:.0f}% · defend %{c:.0f}%<extra></extra>"))
    fig.update_layout(title=f"Action mix of every outfield player (share of per-90 actions; n={len(d)}; labels = {team})",
                      ternary=dict(sum=100, aaxis=dict(title="Creating & finishing"), baxis=dict(title="Progressing"),
                                   caxis=dict(title="Defending"), bgcolor=PAPER), height=600)
    return fig


def contribution_matrix(pms: pd.DataFrame, matches: pd.DataFrame, team: str) -> go.Figure:
    """Player × match heatmap of xG + xA: where the chances came from, match by match."""
    p = _rows(pms, "team_name", team)
    msg = _missing(p, ["xg", "xa", "player_name", "match_id"])
    if msg:
        return _empty(msg, "Contribution matrix")
    p = p.assign(v=p["xg"] + p["xa"])
    piv = p.pivot_table(index="player_name", columns="match_id", values="v", aggfunc="sum")
    order = matches[matches["match_id"].isin(piv.columns)].sort_values("match_date")
    piv = piv.reindex(columns=order["match_id"])
    piv = piv.loc[piv.sum(axis=1).sort_values(ascending=False).index]
    piv = piv[piv.sum(axis=1) > 0.3]
    lab = [f"{d:%d %b} {(a if h == team else h)[:10]}" for d, h, a in zip(order["match_date"], order["home_team"],
                                                                        order["away_team"])]
    fig = go.Figure(go.Heatmap(z=piv.values, x=lab, y=list(piv.index), colorscale=SEQ_SCALE, xgap=1, ygap=1,
                               colorbar=dict(title="xG+xA", thickness=12),
                               hovertemplate="%{y}<br>%{x}: %{z:.2f} xG+xA<extra></extra>"))
    fig.update_layout(title=f"{team}: chance contribution per player per match (xG + xA; blank = did not play)",
                      height=max(420, 22 * len(piv) + 180), xaxis=dict(tickangle=-60, tickfont=dict(size=8)),
                      yaxis=dict(autorange="reversed"), margin=dict(l=190, b=120), plot_bgcolor="white")
    return fig


def pass_destination_density(flags: pd.DataFrame, player_id: Any, title: str) -> go.Figure:
    """Kernel-style density contour of where a player's completed passes end."""
    p = flags[(flags["player_id"] == player_id) & (flags["passes_completed"] == 1)].dropna(subset=["end_x", "end_y"]) \
        if flags is not None and "player_id" in flags else None
    if p is None or p.empty:
        return _empty("No passes.", title)
    fig = go.Figure(go.Histogram2dContour(x=p["end_x"], y=p["end_y"], colorscale=HEAT_SCALE, ncontours=14,
                                          contours=dict(coloring="fill", showlines=False), showscale=False,
                                          xbins=dict(start=0, end=120, size=6), ybins=dict(start=0, end=80, size=6),
                                          hoverinfo="skip", opacity=0.9))
    draw_pitch(fig, lines_on_top=True)
    attack_arrow(fig)
    fig.update_layout(title=f"{title}<br><sup>{len(p):,} completed passes · density of end locations</sup>", height=520)
    return fig


def shot_density_pair(flags: pd.DataFrame, team_id: Any, match_ids: list, team: str) -> go.Figure:
    """Side-by-side density contours of shots taken and shots conceded (each in its own attacking frame)."""
    s = flags[(flags["type"] == "Shot") & flags["match_id"].isin(match_ids)].dropna(subset=["x", "y"]) \
        if flags is not None and "type" in flags else None
    if s is None or s.empty:
        return _empty("No shots.", "Shot density")
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f"{team} shots", "Shots conceded"), horizontal_spacing=0.04)
    for col, sub in ((1, s[s["team_id"] == team_id]), (2, s[s["team_id"] != team_id])):
        fig.add_trace(go.Histogram2dContour(x=sub["x"], y=sub["y"], colorscale=HEAT_SCALE, ncontours=12,
                                            contours=dict(coloring="fill", showlines=False), showscale=False,
                                            xbins=dict(start=60, end=120, size=4), ybins=dict(start=0, end=80, size=4),
                                            hoverinfo="skip"), row=1, col=col)
        gl = sub[sub["outcome"] == "Goal"]
        fig.add_trace(go.Scatter(x=gl["x"], y=gl["y"], mode="markers", showlegend=False,
                                 marker=dict(symbol="star", size=9, color=AMBER, line=dict(color=INK, width=0.6)),
                                 text=gl["player_name"], hovertemplate="Goal: %{text}<extra></extra>"), row=1, col=col)
        draw_pitch(fig, row=1, col=col, half=True, lines_on_top=True)
    fig.update_layout(title=f"{team}: where shots come from — density of attempts, stars = goals", height=480)
    return fig


def subs_parcats(subs: pd.DataFrame, team: str) -> go.Figure:
    """Parallel categories: game state → timing band → role change of each substitution."""
    s = _rows(subs, "team_name", team)
    msg = _missing(s, ["game_state", "timing_band", "off_role", "on_role"])
    if msg:
        return _empty(msg, "Substitution pathways")
    s = s.dropna(subset=["off_role", "on_role"])
    fam = {r: ROLE_FAMILY.get(r, r) for r in ROLE_FAMILY}
    s = s.assign(off=s["off_role"].map(fam), on=s["on_role"].map(fam))
    code = s["game_state"].map({"leading": 0, "drawing": 1, "trailing": 2})
    fig = go.Figure(go.Parcats(
        dimensions=[dict(label="Game state", values=s["game_state"]), dict(label="Timing band", values=s["timing_band"]),
                    dict(label="Off (role family)", values=s["off"]), dict(label="On (role family)", values=s["on"])],
        line=dict(color=code, colorscale=[[0, TEAL], [0.5, SLATE], [1, RED]], shape="hspline"),
        hoveron="color", arrangement="freeform"))
    fig.update_layout(title=f"{team}: substitution pathways — state → timing → who goes off → who comes on "
                            f"({len(s)} changes; teal = leading, grey = drawing, red = trailing)", height=520,
                      margin=dict(l=80, r=80))
    return fig


def on_pitch_gantt(spells: pd.DataFrame, goals: pd.DataFrame, match_id: Any, team_id: Any, title: str) -> go.Figure:
    """Gantt chart of every player's time on the pitch in one match, with goals marked."""
    s = spells[(spells["match_id"] == match_id) & (spells["team_id"] == team_id)] if spells is not None else None
    msg = _missing(s, ["start", "end", "player_name", "start_reason"])
    if msg:
        return _empty(msg, title)
    s = s.sort_values(["start", "player_name"]).iloc[::-1]
    color = np.where(s["start_reason"] == "starting_xi", NAVY, TEAL)
    fig = go.Figure(go.Bar(y=s["player_name"], x=s["end"] - s["start"], base=s["start"], orientation="h",
                           marker_color=color, customdata=np.stack([s["start"], s["end"], s["end_reason"]], -1),
                           hovertemplate="%{y}: %{customdata[0]:.0f}' → %{customdata[1]:.0f}' (%{customdata[2]})"
                                         "<extra></extra>", showlegend=False))
    for _, g in goals[goals["match_id"] == match_id].iterrows():
        mine = g["scoring_team_id"] == team_id
        fig.add_vline(x=g["elapsed_min"], line=dict(color=TEAL if mine else RED, dash="dot", width=1.5))
        fig.add_annotation(x=g["elapsed_min"], y=1.02, yref="paper", text="⚽" if mine else "✕", showarrow=False,
                           font=dict(color=TEAL if mine else RED))
    fig.update_layout(title=f"{title}<br><sup>navy = starter, teal = substitute · dotted lines = goals for (teal) / "
                            f"against (red)</sup>", xaxis_title="elapsed playing time (min)",
                      height=40 + 26 * len(s) + 120, margin=dict(l=190), bargap=0.25)
    return fig

## 3 · Data collection
**Source:** [StatsBomb Open Data](https://github.com/statsbomb/open-data) (Hudl) — competitions, matches (with
managers, referee, stadium), lineups and event files, fetched as raw JSON over HTTPS. Files are cached unchanged in
`data/raw/`, every read is logged in a manifest (URL, UTC timestamp, bytes, SHA-256, status), and downloads use
timeouts, retries with back-off and graceful failure. Re-runs read from cache, so the notebook also works offline.

The catalogue below is what the source *actually* publishes; the competition/season in Section 2 must be one of
these rows.

In [5]:
from __future__ import annotations

# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · data_sources — source register and attribution
# Source register: what each data provider is used for and under which terms.
#
# The register is reported in the notebook and exported with every run so the
# dataset-collection evidence (rubric: Dataset Collection, 5 marks) is traceable.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
import pandas as pd

SOURCE_REGISTER: list[dict[str, str]] = [
    {
        "source": "StatsBomb Open Data (Hudl)",
        "url": "https://github.com/statsbomb/open-data",
        "role_in_project": "PRIMARY: competitions, matches (incl. managers, referee, stadium), lineups, events",
        "access": "Public GitHub repository, raw JSON over HTTPS",
        "terms": "Free for research/education; attribute StatsBomb as data source and follow the repository "
                 "LICENSE / attribution instructions (use of logo requested for published outputs). "
                 "Verify current terms before redistribution.",
        "status": "used",
    },
    {
        "source": "API-Football v3",
        "url": "https://www.api-football.com/documentation-v3",
        "role_in_project": "OPTIONAL enrichment (coach, injuries, fixtures) — requires the user's own key",
        "access": "REST API; key from Colab Secrets / environment variable API_FOOTBALL_KEY",
        "terms": "Subscription terms of the provider; coverage varies by league",
        "status": "not used by default (no key supplied)",
    },
    {
        "source": "Metrica Sports sample data",
        "url": "https://github.com/metrica-sports/sample-data",
        "role_in_project": "OPTIONAL separate tracking demo only (anonymised; never merged with named players)",
        "access": "Public GitHub repository",
        "terms": "See repository licence",
        "status": "not used",
    },
    {
        "source": "football-data.co.uk",
        "url": "https://www.football-data.co.uk/",
        "role_in_project": "OPTIONAL supplementary team-level results only",
        "access": "CSV downloads",
        "terms": "See website notes",
        "status": "not used",
    },
    {
        "source": "User-supplied files",
        "url": "data/external/",
        "role_in_project": "USER_DATA mode (StatsBomb-format JSON) and optional managers.csv mapping",
        "access": "Local files placed by the user",
        "terms": "User must hold rights to use the files",
        "status": "optional",
    },
]

STATSBOMB_ATTRIBUTION = (
    "Data: StatsBomb Open Data (https://github.com/statsbomb/open-data), provided by StatsBomb / Hudl. "
    "Used for educational, non-commercial research."
)


def source_register() -> pd.DataFrame:
    """Return the source register as a DataFrame."""
    return pd.DataFrame(SOURCE_REGISTER)


def statsbomb_paths(competition_id: int | None = None, season_id: int | None = None,
                    match_id: int | None = None) -> dict[str, str]:
    """Relative paths of StatsBomb open-data resources (same layout locally and remotely)."""
    paths = {"competitions": "competitions.json"}
    if competition_id is not None and season_id is not None:
        paths["matches"] = f"matches/{competition_id}/{season_id}.json"
    if match_id is not None:
        paths["events"] = f"events/{match_id}.json"
        paths["lineups"] = f"lineups/{match_id}.json"
        paths["three_sixty"] = f"three-sixty/{match_id}.json"
    return paths


# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · ingestion — download, cache, provenance manifest
# Data acquisition with caching, retries, timeouts and a provenance manifest.
#
# Raw files are written once to ``data/raw/<mode>/`` in the StatsBomb directory
# layout and never modified. Every read is logged in a manifest with source
# URL, retrieval timestamp, byte size, SHA-256 hash and status.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
import hashlib
import json
import logging
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd


log = logging.getLogger("football_analytics.ingestion")


class DataUnavailableError(RuntimeError):
    """Raised when a resource is neither cached nor retrievable."""


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


@dataclass
class DataStore:
    """Reads StatsBomb-layout JSON from cache, the network, or a local folder.

    * REAL_OPEN_DATA: cache in ``data/raw/statsbomb``; download on cache miss.
    * USER_DATA: read-only from ``config.user_data_dir`` (same layout).
    * DEMO_SYNTHETIC: files generated by :mod:`synthetic` into ``data/raw/synthetic``.
    """

    config: Config
    offline: bool = False
    manifest: list[dict[str, Any]] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.config.ensure_dirs()

    # ------------------------------------------------------------------ paths
    @property
    def root(self) -> Path:
        if self.config.data_mode == USER_DATA:
            p = Path(self.config.user_data_dir)
            return p if p.is_absolute() else self.config.project_root / p
        return self.config.raw_dir

    def _record(self, rel: str, url: str, path: Path | None, status: str, data: bytes | None,
                match_id: int | None, kind: str, from_cache: bool, error: str = "") -> None:
        self.manifest.append({
            "resource": kind,
            "relative_path": rel,
            "source_url": url,
            "local_path": str(path) if path else "",
            "match_id": match_id,
            "retrieved_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            # when the raw file was actually downloaded/written (cache hits keep the original time)
            "file_written_utc": (datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).isoformat(timespec="seconds")
                                 if path is not None and path.exists() else ""),
            "bytes": len(data) if data is not None else 0,
            "sha256": sha256_bytes(data) if data is not None else "",
            "from_cache": from_cache,
            "status": status,
            "error": error,
        })

    # ------------------------------------------------------------------ fetch
    def _download(self, url: str) -> bytes:
        import requests  # imported lazily so offline tests never need it

        last_err: Exception | None = None
        for attempt in range(1, self.config.max_retries + 1):
            try:
                r = requests.get(url, timeout=self.config.request_timeout)
                if r.status_code == 404:
                    raise DataUnavailableError(f"404 Not Found: {url}")
                if r.status_code == 429 or r.status_code >= 500:
                    raise ConnectionError(f"HTTP {r.status_code}")
                r.raise_for_status()
                return r.content
            except DataUnavailableError:
                raise
            except Exception as exc:  # network errors, timeouts, rate limits
                last_err = exc
                wait = min(2 ** attempt, 20)
                log.warning("Attempt %s/%s failed for %s (%s); retrying in %ss",
                            attempt, self.config.max_retries, url, exc, wait)
                time.sleep(wait)
        raise DataUnavailableError(f"Failed after {self.config.max_retries} attempts: {url} ({last_err})")

    def get_json(self, rel: str, kind: str, match_id: int | None = None) -> Any:
        """Return parsed JSON for a relative resource path, caching raw bytes unchanged."""
        path = self.root / rel
        url = (f"{self.config.statsbomb_base_url}/{rel}" if self.config.data_mode == REAL_OPEN_DATA
               else f"file://{path}")
        if path.exists():
            data = path.read_bytes()
            self._record(rel, url, path, "ok", data, match_id, kind, from_cache=True)
        elif self.config.data_mode == REAL_OPEN_DATA and not self.offline:
            try:
                data = self._download(url)
            except DataUnavailableError as exc:
                self._record(rel, url, None, "failed", None, match_id, kind, False, str(exc))
                raise
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(data)
            self._record(rel, url, path, "ok", data, match_id, kind, from_cache=False)
        else:
            msg = f"Not cached and not downloadable in mode {self.config.data_mode} (offline={self.offline})"
            self._record(rel, url, None, "missing", None, match_id, kind, False, msg)
            raise DataUnavailableError(f"{msg}: {path}")
        try:
            return json.loads(data.decode("utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError) as exc:
            self.manifest[-1]["status"] = "malformed"
            self.manifest[-1]["error"] = str(exc)
            raise DataUnavailableError(f"Malformed JSON in {path}: {exc}") from exc

    # --------------------------------------------------------------- catalogue
    def competitions(self) -> pd.DataFrame:
        """The competition/season catalogue actually published by the source."""
        data = self.get_json(statsbomb_paths()["competitions"], "competitions")
        df = pd.DataFrame(data)
        keep = [c for c in ["competition_id", "season_id", "country_name", "competition_name",
                            "competition_gender", "competition_youth", "competition_international",
                            "season_name", "match_available", "match_available_360"] if c in df.columns]
        return df[keep].sort_values(["competition_name", "season_name"]).reset_index(drop=True)

    def matches_raw(self, competition_id: int, season_id: int) -> list[dict]:
        rel = statsbomb_paths(competition_id, season_id)["matches"]
        return self.get_json(rel, "matches")

    def match_files(self, match_ids: list[int], include_360: bool = False,
                    progress: bool = True) -> dict[int, dict[str, Any]]:
        """Fetch events + lineups (+ optional 360) per match. Failures are recorded, not fatal."""
        out: dict[int, dict[str, Any]] = {}
        n = len(match_ids)
        for i, mid in enumerate(match_ids, 1):
            paths = statsbomb_paths(match_id=mid)
            rec: dict[str, Any] = {}
            for kind in ("events", "lineups") + (("three_sixty",) if include_360 else ()):
                try:
                    rec[kind] = self.get_json(paths[kind], kind, match_id=mid)
                except DataUnavailableError as exc:
                    log.warning("match %s %s unavailable: %s", mid, kind, exc)
                    rec[kind] = None
            out[mid] = rec
            if progress and (i % 10 == 0 or i == n):
                print(f"  fetched {i}/{n} matches")
        return out

    def manifest_frame(self) -> pd.DataFrame:
        df = pd.DataFrame(self.manifest)
        if df.empty:
            return df
        # Several reads of the same file in one session -> keep the last record.
        return df.drop_duplicates(subset=["relative_path"], keep="last").reset_index(drop=True)


def select_matches(matches: list[dict], limit: int | None, focus_team: str | None = None) -> list[dict]:
    """Deterministically choose the analysed matches (chronological order).

    The selection is reported in the notebook so truncation is never silent.
    """
    ordered = sorted(matches, key=lambda m: (m.get("match_date", ""), m.get("kick_off") or "", m["match_id"]))
    if focus_team:
        involved = [m for m in ordered if focus_team in (m["home_team"]["home_team_name"],
                                                         m["away_team"]["away_team_name"])]
        if involved:
            ordered = involved
    return ordered if limit is None else ordered[:limit]


def raw_inventory(root: Path) -> pd.DataFrame:
    """Inventory of raw files on disk (count and bytes per resource folder)."""
    rows = []
    if root.exists():
        for p in root.rglob("*.json"):
            rel = p.relative_to(root)
            rows.append({"folder": rel.parts[0] if len(rel.parts) > 1 else "(root)",
                         "file": str(rel), "bytes": p.stat().st_size})
    return pd.DataFrame(rows, columns=["folder", "file", "bytes"])

In [6]:
store = DataStore(cfg, offline=OFFLINE)
catalogue = store.competitions()
print(f"Catalogue: {len(catalogue)} competition-seasons, {catalogue['competition_name'].nunique()} competitions")
sel = catalogue[(catalogue.competition_id == cfg.competition_id) & (catalogue.season_id == cfg.season_id)]
assert len(sel) == 1, "Selected competition/season not in catalogue — pick a row from the table below."
display(sel)
display(catalogue[["competition_id", "season_id", "competition_name", "season_name", "competition_gender",
                   "country_name", "match_available_360"]].head(80))

if WIDGETS_OK and not HEADLESS:
    opts = [(f"{r.competition_name} — {r.season_name}", (r.competition_id, r.season_id)) for r in catalogue.itertuples()]
    pick = W.Dropdown(options=opts, value=(cfg.competition_id, cfg.season_id), description="Competition",
                      layout=W.Layout(width="520px"))
    out = W.Output()
    def _on(change):
        with out:
            out.clear_output()
            c, s = change["new"]
            print(f"Set COMPETITION_ID = {c}, SEASON_ID = {s} in Section 2 and re-run from there.")
    pick.observe(_on, names="value")
    display(pick, out)
else:
    print("Live competition picker available when running interactively (Colab/Jupyter).")

t0 = time.perf_counter()
matches_raw = store.matches_raw(cfg.competition_id, cfg.season_id)
selected = select_matches(matches_raw, cfg.match_limit, cfg.focus_team)
print(f"Matches published for this competition-season: {len(matches_raw)}")
print(f"Matches selected for analysis: {len(selected)}"
      + (f" (focus team filter: {cfg.focus_team})" if cfg.focus_team else "")
      + (f" (limited to first {cfg.match_limit} chronologically)" if cfg.match_limit else " (no truncation)"))
files = store.match_files([m["match_id"] for m in selected], include_360=cfg.include_360)
manifest = store.manifest_frame()
print(f"Acquisition time: {time.perf_counter() - t0:.1f}s")
display(manifest.groupby(["resource", "status", "from_cache"]).agg(files=("relative_path", "size"),
                                                                   MB=("bytes", lambda b: round(b.sum() / 1e6, 2))))
display(manifest[["resource", "relative_path", "bytes", "sha256", "retrieved_utc", "status"]].head(6))

Catalogue: 80 competition-seasons, 24 competitions


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_available,match_available_360
1,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093


,competition_id,season_id,competition_name,season_name,competition_gender,country_name,match_available_360
0,9,27,1. Bundesliga,2015/2016,male,Germany,None
1,9,281,1. Bundesliga,2023/2024,male,Germany,2025-11-15T23:17:41.827093
2,1267,107,African Cup of Nations,2023,male,Africa,2026-05-02T02:07:18.902396
3,16,276,Champions League,1970/1971,male,Europe,None
4,16,71,Champions League,1971/1972,male,Europe,None
...,...,...,...,...,...,...,...
75,35,75,UEFA Europa League,1988/1989,male,Europe,None
76,53,106,UEFA Women's Euro,2022,female,Europe,2026-05-05T03:05:32.480837
77,53,315,UEFA Women's Euro,2025,female,Europe,2026-04-27T22:03:28.087062
78,72,30,Women's World Cup,2019,female,International,None


Live competition picker available when running interactively (Colab/Jupyter).
Matches published for this competition-season: 34
Matches selected for analysis: 34 (focus team filter: Bayer Leverkusen) (no truncation)


  fetched 10/34 matches


  fetched 20/34 matches


  fetched 30/34 matches
  fetched 34/34 matches
Acquisition time: 2.4s


,,,files,MB
resource,status,from_cache,,
competitions,ok,True,1,0.03
events,ok,True,34,119.22
lineups,ok,True,34,0.73
matches,ok,True,1,0.06


,resource,relative_path,bytes,sha256,retrieved_utc,status
0,competitions,competitions.json,34887,e6cd42f5d8956d6aa30fb917ce8d4c3b3df1879a93f02f...,2026-11-01T20:23:53+00:00,ok
1,matches,matches/9/281.json,63079,13dff90f126d9f73da410ae3292b2773744657d3ba968a...,2026-11-01T20:23:53+00:00,ok
2,events,events/3895052.json,3205861,ecc00b4e7ddd0154532329091d5d82fd7de4b9f6aa8da1...,2026-11-01T20:23:53+00:00,ok
3,lineups,lineups/3895052.json,22124,485e5e4f06136578a5dd18648397808110978cadb36a63...,2026-11-01T20:23:53+00:00,ok
4,events,events/3895060.json,3798711,1b2fcf16823fbf04f4ae40492d492304f8d54fdb991ceb...,2026-11-01T20:23:53+00:00,ok
5,lineups,lineups/3895060.json,24228,a94f051cea4356e4a09741cb737d5e6fd80b0b6611bdeb...,2026-11-01T20:23:53+00:00,ok


## 4 · Dataset inventory and collection evidence
Actual dimensions of what was collected and parsed (Dataset-collection rubric). Parsing turns nested JSON into
canonical tables; the figures below are computed, never typed in.

In [7]:
from __future__ import annotations

# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · parsing — JSON → canonical tables
# Schema-aware parsing of StatsBomb-format JSON into canonical analytical tables.
#
# Parser rules were written against observed StatsBomb v4 event / lineup files:
#
# * Event coordinates use a 120 x 80 pitch, each team attacking left -> right.
# * ``minute``/``second`` form a continuous match clock (2nd half restarts at 45:00,
# first-half stoppage time keeps counting: 47:30 in period 1).
# * Lineup-file ``positions`` are NOT reliably chronological (entries are ordered by
# clock string across periods), so on-pitch spells are reconstructed from the
# event stream (Starting XI -> Substitution -> sending-off) instead.
# * Optional keys are frequently absent; every accessor tolerates missing keys.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
from typing import Any, Iterable

import numpy as np
import pandas as pd


SENDING_OFF_CARDS = {"Red Card", "Second Yellow"}
TACKLE_WON_OUTCOMES = {"Won", "Success", "Success In Play", "Success Out"}
SET_PIECE_PASS_TYPES = {"Corner", "Free Kick", "Throw-in", "Goal Kick", "Kick Off"}


def _g(d: Any, *keys: str, default: Any = None) -> Any:
    """Safe nested getter: _g(e, 'pass', 'outcome', 'name')."""
    cur = d
    for k in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k)
        if cur is None:
            return default
    return cur


def _xy(loc: Any, i: int) -> float:
    if isinstance(loc, (list, tuple)) and len(loc) > i and loc[i] is not None:
        try:
            return float(loc[i])
        except (TypeError, ValueError):
            return np.nan
    return np.nan


# ---------------------------------------------------------------------------
# Matches and managers
# ---------------------------------------------------------------------------
def parse_matches(matches_raw: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return (matches, match_managers). Absent fields stay missing, never invented."""
    rows, mgr_rows = [], []
    for m in matches_raw:
        home, away = m.get("home_team", {}), m.get("away_team", {})
        rows.append({
            "match_id": int(m["match_id"]),
            "match_date": m.get("match_date"),
            "kick_off": m.get("kick_off"),
            "competition_id": _g(m, "competition", "competition_id"),
            "competition_name": _g(m, "competition", "competition_name"),
            "country_name": _g(m, "competition", "country_name"),
            "season_id": _g(m, "season", "season_id"),
            "season_name": _g(m, "season", "season_name"),
            "match_week": m.get("match_week"),
            "competition_stage": _g(m, "competition_stage", "name"),
            "home_team_id": home.get("home_team_id"),
            "home_team": home.get("home_team_name"),
            "away_team_id": away.get("away_team_id"),
            "away_team": away.get("away_team_name"),
            "home_score": m.get("home_score"),
            "away_score": m.get("away_score"),
            "stadium": _g(m, "stadium", "name"),
            "referee": _g(m, "referee", "name"),
            "match_status": m.get("match_status"),
            "data_version": _g(m, "metadata", "data_version"),
            "xy_fidelity_version": _g(m, "metadata", "xy_fidelity_version"),
            "shot_fidelity_version": _g(m, "metadata", "shot_fidelity_version"),
        })
        for side, team in (("home", home), ("away", away)):
            for mg in team.get("managers") or []:
                mgr_rows.append({
                    "match_id": int(m["match_id"]),
                    "team_id": team.get(f"{side}_team_id"),
                    "team_name": team.get(f"{side}_team_name"),
                    "manager_id": mg.get("id"),
                    "manager_name": mg.get("nickname") or mg.get("name"),
                    "manager_full_name": mg.get("name"),
                    "manager_country": _g(mg, "country", "name"),
                    "manager_source": "StatsBomb matches.json",
                })
    matches = pd.DataFrame(rows)
    if not matches.empty:
        matches["match_date"] = pd.to_datetime(matches["match_date"], errors="coerce")
        matches = matches.sort_values(["match_date", "match_id"]).reset_index(drop=True)
    managers = pd.DataFrame(mgr_rows, columns=["match_id", "team_id", "team_name", "manager_id", "manager_name",
                                                "manager_full_name", "manager_country", "manager_source"])
    return matches, managers


# ---------------------------------------------------------------------------
# Lineups -> matchday squads
# ---------------------------------------------------------------------------
def parse_lineups(lineups_raw: list[dict] | None, match_id: int) -> pd.DataFrame:
    """Matchday squad rows (starters + bench) from a lineup file."""
    rows = []
    for team in lineups_raw or []:
        for p in team.get("lineup") or []:
            positions = p.get("positions") or []
            first_sub_pos = next((q.get("position") for q in positions
                                  if str(q.get("start_reason", "")).startswith("Substitution - On")), None)
            cards = p.get("cards") or []
            rows.append({
                "match_id": int(match_id),
                "team_id": team.get("team_id"),
                "team_name": team.get("team_name"),
                "player_id": p.get("player_id"),
                "player_name": p.get("player_nickname") or p.get("player_name"),
                "player_full_name": p.get("player_name"),
                "jersey_number": p.get("jersey_number"),
                "nationality": _g(p, "country", "name"),
                "lineup_n_positions": len(positions),
                "lineup_first_position": positions[0].get("position") if positions else None,
                "lineup_sub_on_position": first_sub_pos,
                "lineup_started": any(q.get("start_reason") == "Starting XI" for q in positions),
                "lineup_yellow_cards": sum(c.get("card_type") == "Yellow Card" for c in cards),
                "lineup_sending_offs": sum(c.get("card_type") in SENDING_OFF_CARDS for c in cards),
            })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Events
# ---------------------------------------------------------------------------
def _flatten_event(e: dict, match_id: int) -> dict[str, Any]:
    t = _g(e, "type", "name")
    loc = e.get("location")
    row: dict[str, Any] = {
        "event_id": e.get("id"),
        "match_id": int(match_id),
        "index": e.get("index"),
        "period": e.get("period"),
        "minute": e.get("minute"),
        "second": e.get("second"),
        "timestamp": e.get("timestamp"),
        "type": t,
        "team_id": _g(e, "team", "id"),
        "team_name": _g(e, "team", "name"),
        "player_id": _g(e, "player", "id"),
        "player_name": _g(e, "player", "name"),
        "position_name": _g(e, "position", "name"),
        "possession": e.get("possession"),
        "possession_team_id": _g(e, "possession_team", "id"),
        "play_pattern": _g(e, "play_pattern", "name"),
        "x": _xy(loc, 0), "y": _xy(loc, 1),
        "end_x": np.nan, "end_y": np.nan,
        "duration": e.get("duration"),
        "under_pressure": bool(e.get("under_pressure", False)),
        "counterpress": bool(e.get("counterpress", False)),
        "outcome": None,
        "pass_recipient_id": None, "pass_recipient_name": None, "pass_length": np.nan, "pass_angle": np.nan,
        "pass_height": None, "pass_type": None, "pass_technique": None, "pass_body_part": None,
        "pass_cross": False, "pass_switch": False, "pass_through_ball": False,
        "pass_shot_assist": False, "pass_goal_assist": False, "pass_assisted_shot_id": None,
        "shot_xg": np.nan, "shot_type": None, "shot_body_part": None, "shot_technique": None,
        "shot_end_z": np.nan, "shot_key_pass_id": None,
        "duel_type": None, "aerial_won": False, "card": None,
        "goalkeeper_type": None,
        "sub_replacement_id": None, "sub_replacement_name": None,
        "formation": None,
        "tactics_lineup": None,
    }
    if t == "Pass":
        p = e.get("pass", {})
        end = p.get("end_location")
        row.update({
            "end_x": _xy(end, 0), "end_y": _xy(end, 1),
            "outcome": _g(p, "outcome", "name"),
            "pass_recipient_id": _g(p, "recipient", "id"), "pass_recipient_name": _g(p, "recipient", "name"),
            "pass_length": p.get("length", np.nan), "pass_angle": p.get("angle", np.nan),
            "pass_height": _g(p, "height", "name"), "pass_type": _g(p, "type", "name"),
            "pass_technique": _g(p, "technique", "name"), "pass_body_part": _g(p, "body_part", "name"),
            "pass_cross": bool(p.get("cross", False)), "pass_switch": bool(p.get("switch", False)),
            "pass_through_ball": bool(p.get("through_ball", False)) or _g(p, "technique", "name") == "Through Ball",
            "pass_shot_assist": bool(p.get("shot_assist", False)),
            "pass_goal_assist": bool(p.get("goal_assist", False)),
            "pass_assisted_shot_id": p.get("assisted_shot_id"),
            "aerial_won": bool(p.get("aerial_won", False)),
        })
    elif t == "Shot":
        s = e.get("shot", {})
        end = s.get("end_location")
        row.update({
            "end_x": _xy(end, 0), "end_y": _xy(end, 1), "shot_end_z": _xy(end, 2),
            "outcome": _g(s, "outcome", "name"),
            "shot_xg": s.get("statsbomb_xg", np.nan),
            "shot_type": _g(s, "type", "name"), "shot_body_part": _g(s, "body_part", "name"),
            "shot_technique": _g(s, "technique", "name"), "shot_key_pass_id": s.get("key_pass_id"),
            "aerial_won": bool(s.get("aerial_won", False)),
        })
    elif t == "Carry":
        end = _g(e, "carry", "end_location")
        row.update({"end_x": _xy(end, 0), "end_y": _xy(end, 1)})
    elif t == "Dribble":
        row["outcome"] = _g(e, "dribble", "outcome", "name")
    elif t == "Duel":
        row["duel_type"] = _g(e, "duel", "type", "name")
        row["outcome"] = _g(e, "duel", "outcome", "name")
    elif t == "Interception":
        row["outcome"] = _g(e, "interception", "outcome", "name")
    elif t == "Clearance":
        row["aerial_won"] = bool(_g(e, "clearance", "aerial_won", default=False))
    elif t == "Miscontrol":
        row["aerial_won"] = bool(_g(e, "miscontrol", "aerial_won", default=False))
    elif t == "Ball Recovery":
        row["outcome"] = "Failed" if _g(e, "ball_recovery", "recovery_failure", default=False) else "Success"
    elif t == "Goal Keeper":
        row["goalkeeper_type"] = _g(e, "goalkeeper", "type", "name")
        row["outcome"] = _g(e, "goalkeeper", "outcome", "name")
    elif t == "Foul Committed":
        row["card"] = _g(e, "foul_committed", "card", "name")
    elif t == "Bad Behaviour":
        row["card"] = _g(e, "bad_behaviour", "card", "name")
    elif t == "Substitution":
        row["sub_replacement_id"] = _g(e, "substitution", "replacement", "id")
        row["sub_replacement_name"] = _g(e, "substitution", "replacement", "name")
        row["outcome"] = _g(e, "substitution", "outcome", "name")
    elif t in ("Starting XI", "Tactical Shift"):
        row["formation"] = format_formation(_g(e, "tactics", "formation"))
        row["tactics_lineup"] = [
            {"player_id": _g(q, "player", "id"), "player_name": _g(q, "player", "name"),
             "position_name": _g(q, "position", "name"), "jersey_number": q.get("jersey_number")}
            for q in _g(e, "tactics", "lineup", default=[]) or []
        ]
    return row


def parse_events(events_raw: list[dict] | None, match_id: int) -> tuple[pd.DataFrame, list[dict]]:
    """Flatten one match's events. Malformed records are flagged, not fatal."""
    rows, issues = [], []
    for i, e in enumerate(events_raw or []):
        try:
            if not isinstance(e, dict) or "type" not in e:
                raise ValueError("record is not an event dict with a 'type'")
            rows.append(_flatten_event(e, match_id))
        except Exception as exc:  # keep going; log the record
            issues.append({"match_id": match_id, "record_position": i, "issue": f"malformed event: {exc}",
                           "action": "excluded"})
    df = pd.DataFrame(rows)
    if not df.empty:
        df["coordinate_system"] = COORDINATE_SYSTEM
    return df, issues


def period_bounds(events: pd.DataFrame) -> pd.DataFrame:
    """Start/end clock (seconds) and cumulative offset of each played period per match.

    Elapsed playing time = offset[period] + (clock - start[period]), so stoppage time
    counts and first-half stoppage does not overlap the second half. Period 5
    (penalty shoot-out) is excluded from playing time.
    """
    if events.empty:
        return pd.DataFrame(columns=["match_id", "period", "start_clock", "end_clock", "duration", "offset"])
    ev = events[events["period"].between(1, 4)].copy()
    ev["clock"] = ev["minute"] * 60 + ev["second"]
    out = []
    for mid, g in ev.groupby("match_id"):
        offset = 0.0
        for per, gp in g.groupby("period"):
            hs = gp.loc[gp["type"] == "Half Start", "clock"]
            he = gp.loc[gp["type"] == "Half End", "clock"]
            start = float(hs.min()) if len(hs) else float(gp["clock"].min())
            end = float(he.max()) if len(he) else float(gp["clock"].max())
            dur = max(end - start, 0.0)
            out.append({"match_id": mid, "period": int(per), "start_clock": start, "end_clock": end,
                        "duration": dur, "offset": offset,
                        "bounds_source": "half_start_end_events" if len(hs) and len(he) else "event_clock_range"})
            offset += dur
    return pd.DataFrame(out)


def add_elapsed(events: pd.DataFrame, bounds: pd.DataFrame) -> pd.DataFrame:
    """Add ``elapsed_min``: minutes of playing time since kick-off (period-aware)."""
    if events.empty:
        events["elapsed_min"] = pd.Series(dtype=float)
        return events
    b = bounds[["match_id", "period", "start_clock", "duration", "offset"]].rename(
        columns={"start_clock": "_p_start", "duration": "_p_dur", "offset": "_p_offset"})
    ev = events.merge(b, on=["match_id", "period"], how="left")
    clock = ev["minute"] * 60 + ev["second"]
    rel = (clock - ev["_p_start"]).clip(lower=0)
    rel = np.minimum(rel, ev["_p_dur"])
    ev["elapsed_min"] = (ev["_p_offset"] + rel) / 60.0
    # Shoot-out (period 5) and periods without bounds sit at the end of play.
    totals = bounds.groupby("match_id")["duration"].sum() / 60.0
    shoot = ev["period"] >= 5
    ev.loc[shoot, "elapsed_min"] = ev.loc[shoot, "match_id"].map(totals)
    ev = ev.drop(columns=["_p_start", "_p_dur", "_p_offset"])
    return ev.sort_values(["match_id", "index"]).reset_index(drop=True)


def match_lengths(bounds: pd.DataFrame) -> pd.Series:
    """Total playing time (minutes, periods 1-4) per match."""
    return bounds.groupby("match_id")["duration"].sum() / 60.0


# ---------------------------------------------------------------------------
# Goals, cards, formations
# ---------------------------------------------------------------------------
def goal_timeline(events: pd.DataFrame, matches: pd.DataFrame) -> pd.DataFrame:
    """One row per goal in play (periods 1-4), attributed to the scoring team."""
    cols = ["match_id", "index", "period", "minute", "elapsed_min", "scoring_team_id", "player_name", "goal_type"]
    if events.empty:
        return pd.DataFrame(columns=cols)
    ev = events[events["period"].between(1, 4)]
    shots = ev[(ev["type"] == "Shot") & (ev["outcome"] == "Goal")].copy()
    shots["scoring_team_id"] = shots["team_id"]
    shots["goal_type"] = np.where(shots["shot_type"] == "Penalty", "penalty", "open/set play")
    og_for = ev[ev["type"] == "Own Goal For"].copy()
    og_for["scoring_team_id"] = og_for["team_id"]
    og_for["goal_type"] = "own goal"
    # Fallback: only "Own Goal Against" present -> credit the opponent.
    og_against = ev[ev["type"] == "Own Goal Against"].copy()
    og_against = og_against[~og_against["match_id"].isin(og_for["match_id"])]
    if not og_against.empty:
        opp = matches.set_index("match_id")[["home_team_id", "away_team_id"]]
        og_against["scoring_team_id"] = [
            (opp.loc[m, "away_team_id"] if t == opp.loc[m, "home_team_id"] else opp.loc[m, "home_team_id"])
            for m, t in zip(og_against["match_id"], og_against["team_id"])]
        og_against["goal_type"] = "own goal"
    goals = pd.concat([shots, og_for, og_against], ignore_index=True)
    if goals.empty:
        return pd.DataFrame(columns=cols)
    return goals[cols].sort_values(["match_id", "index"]).reset_index(drop=True)


def sending_offs(events: pd.DataFrame) -> pd.DataFrame:
    cols = ["match_id", "index", "period", "minute", "elapsed_min", "team_id", "player_id", "player_name", "card"]
    if events.empty:
        return pd.DataFrame(columns=cols)
    return events[events["card"].isin(SENDING_OFF_CARDS)][cols].reset_index(drop=True)


def formations(events: pd.DataFrame) -> pd.DataFrame:
    """Starting formation and every in-match tactical shift per team."""
    cols = ["match_id", "team_id", "team_name", "index", "period", "minute", "elapsed_min", "type",
            "formation", "formation_changed"]
    if events.empty:
        return pd.DataFrame(columns=cols)
    f = events[events["type"].isin(["Starting XI", "Tactical Shift"])].copy()
    f = f.sort_values(["match_id", "team_id", "index"])
    f["prev_formation"] = f.groupby(["match_id", "team_id"])["formation"].shift()
    f["formation_changed"] = (f["type"] == "Tactical Shift") & (f["formation"] != f["prev_formation"])
    return f[cols].reset_index(drop=True)


# ---------------------------------------------------------------------------
# Appearances (on-pitch spells) from the event stream
# ---------------------------------------------------------------------------
def build_appearances(events: pd.DataFrame, squads: pd.DataFrame, bounds: pd.DataFrame
                      ) -> tuple[pd.DataFrame, pd.DataFrame, list[dict]]:
    """Reconstruct on-pitch spells and minutes from Starting XI, Substitution and sending-off events.

    Returns (appearances, spells, quality_issues). Minutes are actual playing time
    including stoppage time (periods 1-4). Temporary "Player Off/On" treatment
    breaks are ignored. If a team has no Starting XI event, minutes are left
    unknown (NaN) and flagged rather than assumed.
    """
    issues: list[dict] = []
    spells: list[dict] = []
    lengths = match_lengths(bounds)
    first_pos = (events.dropna(subset=["player_id", "position_name"])
                 .sort_values("index").groupby(["match_id", "player_id"])["position_name"].first())
    mode_pos = (events.dropna(subset=["player_id", "position_name"])
                .groupby(["match_id", "player_id"])["position_name"].agg(lambda s: s.value_counts().index[0]))
    squad_idx = squads.set_index(["match_id", "player_id"]) if not squads.empty else None
    teams_with_xi: set = set()

    for mid, g in events.groupby("match_id"):
        end_min = float(lengths.get(mid, np.nan))
        g = g.sort_values("index")
        for _, xi in g[g["type"] == "Starting XI"].iterrows():
            team = xi["team_id"]
            teams_with_xi.add((mid, team))
            on: dict[Any, dict] = {}
            appeared: set = set()
            for q in xi["tactics_lineup"] or []:
                on[q["player_id"]] = {"start": 0.0, "position": q["position_name"], "start_reason": "starting_xi",
                                      "player_name": q["player_name"], "on_minute": 0, "on_period": 1}
                appeared.add(q["player_id"])
            if len(on) != 11:
                issues.append({"match_id": mid, "team_id": team, "issue": f"Starting XI has {len(on)} players",
                               "action": "flagged"})
            team_ev = g[(g["team_id"] == team) & ((g["type"] == "Substitution") | g["card"].isin(SENDING_OFF_CARDS))]
            for _, e in team_ev.iterrows():
                if e["period"] >= 5:
                    continue
                pid = e["player_id"]
                if e["type"] == "Substitution":
                    if pid in on:
                        s = on.pop(pid)
                        spells.append({**s, "match_id": mid, "team_id": team, "player_id": pid, "end": e["elapsed_min"],
                                       "end_reason": f"substituted ({e['outcome'] or 'unspecified'})",
                                       "off_minute": e["minute"], "off_period": e["period"]})
                    else:
                        issues.append({"match_id": mid, "team_id": team, "player_id": pid,
                                       "issue": "substituted-off player not on pitch in reconstructed timeline",
                                       "action": "flagged; replacement still added"})
                    rep = e["sub_replacement_id"]
                    if rep is None or (isinstance(rep, float) and np.isnan(rep)):
                        issues.append({"match_id": mid, "team_id": team, "issue": "substitution without replacement id",
                                       "action": "flagged"})
                        continue
                    if rep in appeared:
                        issues.append({"match_id": mid, "team_id": team, "player_id": rep,
                                       "issue": "replacement had already appeared", "action": "flagged; ignored"})
                        continue
                    pos = None
                    if squad_idx is not None and (mid, rep) in squad_idx.index:
                        pos = squad_idx.loc[(mid, rep), "lineup_sub_on_position"]
                    pos = pos or first_pos.get((mid, rep)) or e["position_name"]
                    on[rep] = {"start": e["elapsed_min"], "position": pos, "start_reason": "substitute",
                               "player_name": e["sub_replacement_name"], "on_minute": e["minute"],
                               "on_period": e["period"]}
                    appeared.add(rep)
                else:  # sending off
                    if pid in on:
                        s = on.pop(pid)
                        spells.append({**s, "match_id": mid, "team_id": team, "player_id": pid, "end": e["elapsed_min"],
                                       "end_reason": f"sent off ({e['card']})",
                                       "off_minute": e["minute"], "off_period": e["period"]})
            for pid, s in on.items():
                spells.append({**s, "match_id": mid, "team_id": team, "player_id": pid, "end": end_min,
                               "end_reason": "final whistle", "off_minute": None, "off_period": None})

    sp = pd.DataFrame(spells, columns=["match_id", "team_id", "player_id", "player_name", "position", "start", "end",
                                       "start_reason", "end_reason", "on_minute", "on_period", "off_minute",
                                       "off_period"])
    if not sp.empty:
        sp["minutes"] = (sp["end"] - sp["start"]).clip(lower=0)
        if (sp["end"] < sp["start"]).any():
            issues.append({"issue": f"{int((sp['end'] < sp['start']).sum())} spells end before they start",
                           "action": "minutes clipped at 0 and flagged"})

    # Aggregate spells to one appearance row per player-match.
    agg = (sp.groupby(["match_id", "team_id", "player_id"])
           .agg(minutes_played=("minutes", "sum"), on_elapsed=("start", "min"), off_elapsed=("end", "max"),
                start_reason=("start_reason", "first"), entry_position=("position", "first"),
                end_reason=("end_reason", "last"), sub_on_minute=("on_minute", "first"),
                sub_off_minute=("off_minute", "last"), n_spells=("start", "size"))
           .reset_index()) if not sp.empty else pd.DataFrame(
        columns=["match_id", "team_id", "player_id", "minutes_played", "on_elapsed", "off_elapsed", "start_reason",
                 "entry_position", "end_reason", "sub_on_minute", "sub_off_minute", "n_spells"])
    agg["appeared"] = True
    agg["is_starter"] = agg["start_reason"] == "starting_xi"
    agg.loc[agg["is_starter"], "sub_on_minute"] = np.nan
    agg["sent_off"] = agg["end_reason"].astype(str).str.startswith("sent off")
    agg["minutes_flag"] = "derived_from_events"

    # Merge in the full matchday squad (unused substitutes get 0 minutes).
    if not squads.empty:
        base = squads[["match_id", "team_id", "team_name", "player_id", "player_name", "jersey_number",
                       "nationality", "lineup_n_positions", "lineup_started", "lineup_yellow_cards"]]
        app = base.merge(agg, on=["match_id", "team_id", "player_id"], how="outer", indicator=True)
        missing_from_squad = app["_merge"] == "right_only"
        if missing_from_squad.any():
            issues.append({"issue": f"{int(missing_from_squad.sum())} event-timeline players absent from lineup file",
                           "action": "kept; squad fields missing"})
        mismatch = (app["_merge"] == "left_only") & (app["lineup_n_positions"] > 0)
        for _, r in app[mismatch].iterrows():
            issues.append({"match_id": r["match_id"], "team_id": r["team_id"], "player_id": r["player_id"],
                           "issue": "lineup file shows positions but player absent from event timeline",
                           "action": "flagged; minutes unknown"})
        app = app.drop(columns="_merge")
        app["appeared"] = app["appeared"].fillna(False).astype(bool)
        app["is_starter"] = app["is_starter"].fillna(False).astype(bool)
        app["sent_off"] = app["sent_off"].fillna(False).astype(bool)
        unused = ~app["appeared"]
        app.loc[unused, "minutes_played"] = 0.0
        app.loc[unused, "minutes_flag"] = "unused_substitute"
        app.loc[mismatch.values, "minutes_played"] = np.nan
        app.loc[mismatch.values, "minutes_flag"] = "unknown_lineup_event_mismatch"
    else:
        app = agg

    # Teams with no Starting XI event: minutes cannot be derived.
    no_xi = [(m, t) for m, t in app[["match_id", "team_id"]].drop_duplicates().itertuples(index=False)
             if (m, t) not in teams_with_xi]
    for m, t in no_xi:
        sel = (app["match_id"] == m) & (app["team_id"] == t)
        app.loc[sel, "minutes_played"] = np.nan
        app.loc[sel, "minutes_flag"] = "unknown_no_starting_xi_event"
        issues.append({"match_id": m, "team_id": t, "issue": "no Starting XI event", "action": "minutes unknown"})

    app["squad_status"] = np.select([app["is_starter"], app["appeared"]], ["starter", "used_substitute"],
                                    default="unused_substitute")
    app["match_position"] = [mode_pos.get((m, p)) for m, p in zip(app["match_id"], app["player_id"])]
    app["match_position"] = app["match_position"].fillna(app.get("entry_position"))
    app["match_role"] = app["match_position"].map(role_for_position)
    if "player_name" in app.columns and not sp.empty:
        names = sp.drop_duplicates(["match_id", "player_id"]).set_index(["match_id", "player_id"])["player_name"]
        fill = pd.Series([names.get((m, p)) for m, p in zip(app["match_id"], app["player_id"])], index=app.index)
        app["player_name"] = app["player_name"].fillna(fill)
    return app.reset_index(drop=True), sp, issues


def substitutions(events: pd.DataFrame) -> pd.DataFrame:
    """Raw substitution events (direction: ``player`` leaves, ``replacement`` enters)."""
    cols = ["match_id", "event_id", "index", "period", "minute", "second", "elapsed_min", "team_id", "team_name",
            "player_off_id", "player_off_name", "player_on_id", "player_on_name", "off_position", "sub_reason"]
    if events.empty:
        return pd.DataFrame(columns=cols)
    s = events[events["type"] == "Substitution"].rename(columns={
        "player_id": "player_off_id", "player_name": "player_off_name", "sub_replacement_id": "player_on_id",
        "sub_replacement_name": "player_on_name", "position_name": "off_position", "outcome": "sub_reason"})
    return s[[c for c in cols if c in s.columns]].reset_index(drop=True)


def role_family(role: str | None) -> str | None:
    return ROLE_FAMILY.get(role) if isinstance(role, str) else None


def parse_all(matches_raw: list[dict], files: dict[int, dict[str, Any]]) -> dict[str, Any]:
    """Run every parser and return canonical tables plus a parse-issue log."""
    matches, managers = parse_matches(matches_raw)
    ev_frames, sq_frames, issues = [], [], []
    for mid, rec in files.items():
        if rec.get("events") is None:
            issues.append({"match_id": mid, "issue": "events file unavailable", "action": "match excluded"})
            continue
        df, iss = parse_events(rec["events"], mid)
        issues.extend(iss)
        ev_frames.append(df)
        if rec.get("lineups") is None:
            issues.append({"match_id": mid, "issue": "lineups file unavailable", "action": "bench unknown"})
        else:
            sq_frames.append(parse_lineups(rec["lineups"], mid))
    events = pd.concat(ev_frames, ignore_index=True) if ev_frames else pd.DataFrame()
    squads = pd.concat(sq_frames, ignore_index=True) if sq_frames else pd.DataFrame()
    bounds = period_bounds(events)
    events = add_elapsed(events, bounds)
    parsed_ids = set(events["match_id"].unique()) if not events.empty else set()
    matches["parsed"] = matches["match_id"].isin(parsed_ids)
    appearances, spells, app_issues = build_appearances(events, squads, bounds)
    issues.extend(app_issues)
    return {
        "matches": matches, "managers": managers, "events": events, "squads": squads,
        "period_bounds": bounds, "appearances": appearances, "spells": spells,
        "substitutions_raw": substitutions(events), "goals": goal_timeline(events, matches),
        "sending_offs": sending_offs(events), "formations": formations(events),
        "parse_issues": pd.DataFrame(issues),
    }


# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · validation — quality audit
# Data-quality audit and domain-rule validation.
#
# Nothing is dropped silently: every check returns counts, and anything excluded,
# corrected or flagged is written to a quality-action log.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
from typing import Any

import numpy as np
import pandas as pd


def dataset_scale(tables: dict[str, Any], manifest: pd.DataFrame, processing_seconds: float | None = None
                  ) -> pd.DataFrame:
    """Honest data-scale summary (rows, files, bytes) for the coverage cards."""
    ev, app, matches = tables["events"], tables["appearances"], tables["matches"]
    ok = manifest[manifest["status"] == "ok"] if not manifest.empty else manifest
    raw_bytes = float(ok["bytes"].sum()) if not ok.empty else 0.0
    rows = [
        ("Competitions", matches["competition_id"].nunique()),
        ("Seasons", matches["season_id"].nunique()),
        ("Matches listed by source (selected)", len(matches)),
        ("Matches parsed", int(matches["parsed"].sum()) if "parsed" in matches else len(matches)),
        ("Events parsed", len(ev)),
        ("Matchday-squad (lineup) rows", len(app)),
        ("Teams", pd.unique(pd.concat([matches["home_team_id"], matches["away_team_id"]])).size),
        ("Players (unique IDs in squads)", app["player_id"].nunique()),
        ("Substitutions", len(tables["substitutions_raw"])),
        ("Goals (periods 1-4)", len(tables["goals"])),
        ("Raw JSON files", len(ok)),
        ("Raw data size (MB)", round(raw_bytes / 1e6, 2)),
        ("Date range", f"{matches['match_date'].min():%Y-%m-%d} to {matches['match_date'].max():%Y-%m-%d}"
         if len(matches) else "n/a"),
    ]
    if processing_seconds is not None:
        rows.append(("Parse + build time (s)", round(processing_seconds, 1)))
    return pd.DataFrame(rows, columns=["measure", "value"])


def missingness(df: pd.DataFrame, columns: list[str] | None = None) -> pd.DataFrame:
    cols = columns or list(df.columns)
    cols = [c for c in cols if c in df.columns]
    if df.empty:
        return pd.DataFrame(columns=["field", "missing_pct", "non_missing"])
    m = df[cols].isna().mean() * 100
    return (pd.DataFrame({"field": cols, "missing_pct": m.values.round(2),
                          "non_missing": df[cols].notna().sum().values})
            .sort_values("missing_pct", ascending=False).reset_index(drop=True))


def event_type_coverage(events: pd.DataFrame) -> pd.DataFrame:
    """Field completeness per event type (for the coverage heatmap)."""
    fields = ["player_id", "team_id", "x", "y", "end_x", "outcome", "shot_xg", "pass_recipient_id", "position_name"]
    if events.empty:
        return pd.DataFrame()
    top = events["type"].value_counts().head(15).index
    sub = events[events["type"].isin(top)]
    return (sub.groupby("type")[fields].apply(lambda g: g.notna().mean() * 100)).round(1)


def quality_checks(tables: dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run domain-rule checks. Returns (check_results, quality_action_log)."""
    ev, app, matches = tables["events"], tables["appearances"], tables["matches"]
    goals, subs = tables["goals"], tables["substitutions_raw"]
    results: list[dict] = []
    log: list[dict] = []

    def add(check: str, n_bad: int, n_total: int, rule: str, action: str = "flagged") -> None:
        results.append({"check": check, "rule": rule, "violations": int(n_bad), "checked": int(n_total),
                        "status": "PASS" if n_bad == 0 else "FLAG"})
        if n_bad:
            log.append({"check": check, "records": int(n_bad), "action": action})

    # Duplicates
    add("Duplicate event IDs", ev["event_id"].duplicated().sum(), len(ev), "event_id unique")
    add("Duplicate squad rows", app.duplicated(["match_id", "team_id", "player_id"]).sum(), len(app),
        "one row per player per match")
    # Coordinates
    has_xy = ev["x"].notna()
    bad_xy = has_xy & ~(ev["x"].between(0, PITCH_LENGTH) & ev["y"].between(0, PITCH_WIDTH))
    add("Event coordinates outside 120x80", bad_xy.sum(), has_xy.sum(), "0<=x<=120, 0<=y<=80")
    has_end = ev["end_x"].notna()
    bad_end = has_end & ~(ev["end_x"].between(0, PITCH_LENGTH + 2) & ev["end_y"].between(-2, PITCH_WIDTH + 2))
    add("End coordinates outside pitch (+2 tolerance)", bad_end.sum(), has_end.sum(), "end location on/near pitch")
    # xG plausibility
    xg = ev["shot_xg"].dropna()
    add("xG outside [0,1]", (~xg.between(0, 1)).sum(), len(xg), "0<=statsbomb_xg<=1")
    # Negative values
    add("Negative minutes played", (app["minutes_played"] < 0).sum(), app["minutes_played"].notna().sum(),
        "minutes >= 0")
    # Event order
    ordered = ev.groupby("match_id")["index"].apply(lambda s: s.is_monotonic_increasing)
    add("Matches with unordered event index", (~ordered).sum(), len(ordered), "event index increasing")
    clock = ev["minute"] * 60 + ev["second"]
    back = ev.assign(c=clock).groupby(["match_id", "period"])["c"].apply(lambda s: (s.diff() < -5).sum())
    add("Clock steps backwards >5s within period", int(back.sum()), len(ev), "timestamps ordered (flag only)")
    # Starting XI sanity
    starters = app[app["is_starter"]].groupby(["match_id", "team_id"]).size()
    add("Team-matches without exactly 11 starters", (starters != 11).sum(), len(starters), "XI has 11 players")
    gk = (app[app["is_starter"] & (app["match_role"] == "GK")].groupby(["match_id", "team_id"]).size()
          .reindex(starters.index, fill_value=0))
    add("Team-matches without exactly one starting GK", (gk != 1).sum(), len(starters),
        "one goalkeeper slot (event-position based)")
    dup_xi = app[app["is_starter"]].duplicated(["match_id", "team_id", "player_id"]).sum()
    add("Player selected twice in same XI", dup_xi, int(app["is_starter"].sum()), "XI players distinct")
    # Substitutions
    sub_bad = subs["player_off_id"].isna() | subs["player_on_id"].isna() | subs["team_id"].isna() | subs["minute"].isna()
    add("Substitutions missing team/player/time", sub_bad.sum(), len(subs), "complete substitution records")
    # Overlapping spells (same player on pitch twice at once)
    sp = tables["spells"]
    overlap = 0
    if not sp.empty:
        for _, g in sp.groupby(["match_id", "player_id"]):
            if len(g) > 1:
                g = g.sort_values("start")
                overlap += int((g["start"].values[1:] < g["end"].values[:-1]).sum())
    add("Overlapping on-pitch spells", overlap, len(sp), "player cannot be on pitch twice simultaneously")
    # Score reconciliation
    if not goals.empty and not matches.empty:
        gcount = goals.groupby(["match_id", "scoring_team_id"]).size()
        bad = 0
        for _, m in matches[matches["parsed"]].iterrows():
            hg = gcount.get((m["match_id"], m["home_team_id"]), 0)
            ag = gcount.get((m["match_id"], m["away_team_id"]), 0)
            if (hg, ag) != (m["home_score"], m["away_score"]):
                bad += 1
                log.append({"check": "Score reconciliation", "records": 1, "action":
                            f"match {m['match_id']}: events {hg}-{ag} vs metadata {m['home_score']}-{m['away_score']}"})
        results.append({"check": "Event goals != published score", "rule": "goal events reconcile to final score",
                        "violations": bad, "checked": int(matches["parsed"].sum()),
                        "status": "PASS" if bad == 0 else "FLAG"})
    # Unmatched references: event players absent from squads
    squad_ids = set(zip(app["match_id"], app["player_id"]))
    ev_players = ev.dropna(subset=["player_id"])[["match_id", "player_id"]].drop_duplicates()
    unmatched = sum((m, p) not in squad_ids for m, p in ev_players.itertuples(index=False))
    add("Event players missing from matchday squad", unmatched, len(ev_players), "player references resolve")
    # Minutes flags
    unk = app["minutes_flag"].astype(str).str.startswith("unknown").sum()
    add("Appearances with unknown minutes", unk, len(app), "minutes derivable from events")

    pi = tables.get("parse_issues")
    if pi is not None and not pi.empty:
        for _, r in pi.iterrows():
            log.append({"check": "parser", "records": 1, "action": f"{r.get('issue')} -> {r.get('action')}"})
    return pd.DataFrame(results), pd.DataFrame(log, columns=["check", "records", "action"])


def coverage_summary(tables: dict[str, Any]) -> pd.DataFrame:
    ev, app, matches = tables["events"], tables["appearances"], tables["matches"]
    lineup_matches = app["match_id"].nunique() if not app.empty else 0
    on_ball = ev[ev["type"].isin(["Pass", "Carry", "Shot", "Dribble", "Ball Receipt*", "Pressure", "Duel"])]
    rows = [
        ("Matches with lineup data (%)", round(100 * lineup_matches / max(len(matches), 1), 1)),
        ("Squad rows with position (appeared players) (%)",
         round(100 * app.loc[app["appeared"], "match_position"].notna().mean(), 1) if app["appeared"].any() else 0),
        ("On-ball events with coordinates (%)", round(100 * on_ball["x"].notna().mean(), 1) if len(on_ball) else 0),
        ("Shots with xG (%)", round(100 * ev.loc[ev["type"] == "Shot", "shot_xg"].notna().mean(), 1)
         if (ev["type"] == "Shot").any() else 0),
        ("Passes with recipient (%)", round(100 * ev.loc[ev["type"] == "Pass", "pass_recipient_id"].notna().mean(), 1)
         if (ev["type"] == "Pass").any() else 0),
        ("Matches with manager metadata (%)",
         round(100 * tables["managers"]["match_id"].nunique() / max(len(matches), 1), 1)),
        ("Matches with referee (%)", round(100 * matches["referee"].notna().mean(), 1) if len(matches) else 0),
        ("Matches with stadium (%)", round(100 * matches["stadium"].notna().mean(), 1) if len(matches) else 0),
        ("Attendance / weather / injuries", "not provided by source — not analysed"),
    ]
    return pd.DataFrame(rows, columns=["coverage measure", "value"])


def assert_no_nan_ids(df: pd.DataFrame, cols: list[str]) -> dict[str, int]:
    return {c: int(df[c].isna().sum()) for c in cols if c in df.columns}


def numeric_summary(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return pd.DataFrame()
    return df[cols].describe().T.round(3)


def suspicious_values(app: pd.DataFrame, max_minutes: float = 135.0) -> pd.DataFrame:
    """Appearances with implausible minutes (e.g. > 135 incl. extra time)."""
    return app[(app["minutes_played"] > max_minutes) | (app["minutes_played"] < 0)]


# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · run container
# A small container for every table of one run, and the parsing step that fills it.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
import time
from dataclasses import dataclass, field


@dataclass
class Bundle:
    """Every canonical and analytical table of one run."""
    config: Config
    store: DataStore | None = None
    catalogue: pd.DataFrame | None = None
    matches_available: int = 0
    tables: dict = field(default_factory=dict)
    flags: pd.DataFrame | None = None
    pms: pd.DataFrame | None = None
    season: pd.DataFrame | None = None
    season_pct: pd.DataFrame | None = None
    team_stats: pd.DataFrame | None = None
    workload: pd.DataFrame | None = None
    subs: pd.DataFrame | None = None
    quality: pd.DataFrame | None = None
    quality_log: pd.DataFrame | None = None
    timings: dict = field(default_factory=dict)
    manager_source: str = "none"


def build_tables(bundle: Bundle, selected: list, files: dict) -> Bundle:
    """Parse raw JSON into canonical tables (timed)."""
    t0 = time.perf_counter()
    bundle.tables = parse_all(selected, files)
    bundle.timings["parse_s"] = time.perf_counter() - t0
    bundle.manager_source = ("StatsBomb matches.json (managers field)" if not bundle.tables["managers"].empty
                             else "none — manager analysis unavailable")
    return bundle

In [8]:
t0 = time.perf_counter()
bundle = Bundle(config=cfg, store=store, catalogue=catalogue, matches_available=len(matches_raw))
build_tables(bundle, selected, files)
T = bundle.tables
parse_seconds = time.perf_counter() - t0
scale = dataset_scale(T, manifest, parse_seconds)
display(scale)
coverage_cards(scale, banner=BANNER).show()

display(Markdown('**The event stream itself.** What the parsed events are made of, and how many each match produces.'))

event_type_treemap(T["events"]).show()
events_per_match_chart(T["events"], T["matches"]).show()

inv = raw_inventory(cfg.raw_dir)
display(inv.groupby("folder").agg(files=("file", "size"), MB=("bytes", lambda b: round(b.sum() / 1e6, 2))))
m = T["matches"]
print(f"Date coverage: {m.match_date.min():%Y-%m-%d} → {m.match_date.max():%Y-%m-%d} "
      f"({m.match_date.dt.to_period('M').nunique()} calendar months)")
display(Markdown("**Source register and attribution**"))
display(source_register()[["source", "role_in_project", "status"]])
display(Markdown(f"> {STATSBOMB_ATTRIBUTION}" if not cfg.is_synthetic else f"> **{BANNER}**"))

display(Markdown('**Big-data honesty.** The cards report the real scale: tens of MB of JSON and ~10⁵ events for one club-season —\na *scalable football-analytics pipeline / Big Data Visualization case study*, not "massive big data". Pandas is the\nright tool at this volume (parsing + feature building take seconds). The same per-match design maps directly onto a\ndistributed engine: raw files are independent per `match_id`, so parsing is an embarrassingly parallel map\n(e.g. Spark `binaryFiles` → `flatMap`), and every aggregation in Section 7 is a `groupBy` on `(match_id, team_id,\nplayer_id)`. Spark is deliberately *not* added here because it would be decoration at this size — see the\nprocessing times above and the scaling notes in the report.'))

,measure,value
0,Competitions,1
1,Seasons,1
2,Matches listed by source (selected),34
3,Matches parsed,34
4,Events parsed,137765
5,Matchday-squad (lineup) rows,1353
6,Teams,18
7,Players (unique IDs in squads),460
8,Substitutions,301
9,Goals (periods 1-4),113


**The event stream itself.** What the parsed events are made of, and how many each match produces.

,files,MB
folder,,
(root),1,0.03
events,34,119.22
lineups,34,0.73
matches,1,0.06


Date coverage: 2023-08-19 → 2024-05-18 (10 calendar months)


**Source register and attribution**

,source,role_in_project,status
0,StatsBomb Open Data (Hudl),"PRIMARY: competitions, matches (incl. managers...",used
1,API-Football v3,"OPTIONAL enrichment (coach, injuries, fixtures...",not used by default (no key supplied)
2,Metrica Sports sample data,OPTIONAL separate tracking demo only (anonymis...,not used
3,football-data.co.uk,OPTIONAL supplementary team-level results only,not used
4,User-supplied files,USER_DATA mode (StatsBomb-format JSON) and opt...,optional


> Data: StatsBomb Open Data (https://github.com/statsbomb/open-data), provided by StatsBomb / Hudl. Used for educational, non-commercial research.

**Big-data honesty.** The cards report the real scale: tens of MB of JSON and ~10⁵ events for one club-season —
a *scalable football-analytics pipeline / Big Data Visualization case study*, not "massive big data". Pandas is the
right tool at this volume (parsing + feature building take seconds). The same per-match design maps directly onto a
distributed engine: raw files are independent per `match_id`, so parsing is an embarrassingly parallel map
(e.g. Spark `binaryFiles` → `flatMap`), and every aggregation in Section 7 is a `groupBy` on `(match_id, team_id,
player_id)`. Spark is deliberately *not* added here because it would be decoration at this size — see the
processing times above and the scaling notes in the report.

## 5 · Data cleaning and quality audit
Domain rules are tested explicitly; nothing is dropped silently. Key parsing decisions (see the `parsing` pipeline code above): StatsBomb's lineup-file position spells are **not** reliably chronological, so on-pitch time is
rebuilt from the event stream (Starting XI → Substitution → sending-off), period-aware so first-half stoppage time
never overlaps the second half. Temporary "Player Off/On" treatment breaks are ignored.

In [9]:
bundle.quality, bundle.quality_log = quality_checks(T)
display(bundle.quality)
n_flag = int((bundle.quality.status != "PASS").sum())
print(f"{len(bundle.quality) - n_flag}/{len(bundle.quality)} checks pass; {n_flag} flagged.")
display(Markdown("**Quality-action log** (retained / corrected / excluded / flagged records)"))
display(bundle.quality_log if len(bundle.quality_log) else pd.DataFrame({"action": ["no records excluded or corrected"]}))
display(Markdown("**Parser issue log**"))
display(T["parse_issues"] if len(T["parse_issues"]) else pd.DataFrame({"issue": ["none"]}))

display(coverage_summary(T))
missingness_heatmap(event_type_coverage(T["events"]), banner=BANNER).show()

# Minutes reconstruction audit for one match: period bounds and every player's spell.
demo_mid = T["matches"]["match_id"].iloc[0]
display(T["period_bounds"][T["period_bounds"].match_id == demo_mid])
a = T["appearances"][T["appearances"].match_id == demo_mid]
display(a[["team_name", "player_name", "squad_status", "match_position", "minutes_played", "sub_on_minute",
           "sub_off_minute", "end_reason", "minutes_flag"]].sort_values(["team_name", "squad_status"]).round(1))

,check,rule,violations,checked,status
0,Duplicate event IDs,event_id unique,0,137765,PASS
1,Duplicate squad rows,one row per player per match,0,1353,PASS
2,Event coordinates outside 120x80,"0<=x<=120, 0<=y<=80",0,136803,PASS
3,End coordinates outside pitch (+2 tolerance),end location on/near pitch,0,72499,PASS
4,"xG outside [0,1]",0<=statsbomb_xg<=1,0,916,PASS
5,Negative minutes played,minutes >= 0,0,1353,PASS
6,Matches with unordered event index,event index increasing,0,34,PASS
7,Clock steps backwards >5s within period,timestamps ordered (flag only),0,137765,PASS
8,Team-matches without exactly 11 starters,XI has 11 players,0,68,PASS
9,Team-matches without exactly one starting GK,one goalkeeper slot (event-position based),0,68,PASS


16/16 checks pass; 0 flagged.


**Quality-action log** (retained / corrected / excluded / flagged records)

,action
0,no records excluded or corrected


**Parser issue log**

,issue
0,none


,coverage measure,value
0,Matches with lineup data (%),100.0
1,Squad rows with position (appeared players) (%),100.0
2,On-ball events with coordinates (%),100.0
3,Shots with xG (%),100.0
4,Passes with recipient (%),97.5
5,Matches with manager metadata (%),100.0
6,Matches with referee (%),100.0
7,Matches with stadium (%),100.0
8,Attendance / weather / injuries,not provided by source — not analysed


,match_id,period,start_clock,end_clock,duration,offset,bounds_source
0,3895052,1,0.0,2882.0,2882.0,0.0,half_start_end_events
1,3895052,2,2700.0,5749.0,3049.0,2882.0,half_start_end_events


,team_name,player_name,squad_status,match_position,minutes_played,sub_on_minute,sub_off_minute,end_reason,minutes_flag
1,Bayer Leverkusen,Granit Xhaka,starter,Left Defensive Midfield,98.8,NaN,NaN,final whistle,derived_from_events
2,Bayer Leverkusen,Jonathan Tah,starter,Center Back,98.8,NaN,NaN,final whistle,derived_from_events
4,Bayer Leverkusen,Lukáš Hrádecký,starter,Goalkeeper,98.8,NaN,NaN,final whistle,derived_from_events
5,Bayer Leverkusen,Jonas Hofmann,starter,Right Wing,91.3,NaN,88.0,substituted (Tactical),derived_from_events
7,Bayer Leverkusen,Álex Grimaldo,starter,Left Wing Back,98.8,NaN,NaN,final whistle,derived_from_events
8,Bayer Leverkusen,Odilon Kossonou,starter,Right Center Back,98.8,NaN,NaN,final whistle,derived_from_events
10,Bayer Leverkusen,Exequiel Palacios,starter,Right Defensive Midfield,98.8,NaN,NaN,final whistle,derived_from_events
11,Bayer Leverkusen,Edmond Fayçal Tapsoba,starter,Left Center Back,98.8,NaN,NaN,final whistle,derived_from_events
12,Bayer Leverkusen,Victor Boniface,starter,Center Forward,91.6,NaN,88.0,substituted (Tactical),derived_from_events
13,Bayer Leverkusen,Jeremie Frimpong,starter,Right Wing Back,80.4,NaN,77.0,substituted (Injury),derived_from_events


## 6 · Canonical tables
Source-native IDs are kept (never merged on names). Scores and match state are always computed from the team's
perspective; coordinates stay in one declared system (`statsbomb_120x80_attack_left_to_right`).

In [10]:
teams = pd.concat([T["matches"][["home_team_id", "home_team"]].set_axis(["team_id", "team_name"], axis=1),
                   T["matches"][["away_team_id", "away_team"]].set_axis(["team_id", "team_name"], axis=1)]
                  ).drop_duplicates().sort_values("team_name").reset_index(drop=True)
players = (T["appearances"].groupby(["player_id"]).agg(player_name=("player_name", "last"),
           team=("team_name", "last"), squads=("match_id", "nunique")).reset_index())
canon = {"competitions/seasons": catalogue[(catalogue.competition_id == cfg.competition_id)],
         "matches": T["matches"], "teams": teams, "players": players, "appearances (matchday squads)": T["appearances"],
         "events": T["events"], "substitutions": T["substitutions_raw"], "managers": T["managers"]}
display(pd.DataFrame([{"table": k, "rows": len(v), "columns": v.shape[1]} for k, v in canon.items()]))
display(T["matches"][["match_id", "match_date", "home_team", "home_score", "away_score", "away_team", "stadium",
                      "referee"]].head())
display(T["events"][["match_id", "period", "minute", "second", "elapsed_min", "type", "team_name", "player_name",
                     "x", "y", "end_x", "end_y", "outcome", "shot_xg"]].sample(6, random_state=SEED))
display(T["managers"].drop_duplicates(["team_name", "manager_name"]).head(8))

,table,rows,columns
0,competitions/seasons,2,10
1,matches,34,23
2,teams,18,2
3,players,460,4
4,appearances (matchday squads),1353,26
5,events,137765,54
6,substitutions,301,15
7,managers,68,8


,match_id,match_date,home_team,home_score,away_score,away_team,stadium,referee
0,3895052,2023-08-19,Bayer Leverkusen,3,2,RB Leipzig,BayArena,Felix Brych
1,3895060,2023-08-26,Borussia Mönchengladbach,0,3,Bayer Leverkusen,Stadion im BORUSSIA-PARK,Christian Dingert
2,3895067,2023-09-02,Bayer Leverkusen,5,1,Darmstadt 98,BayArena,Sven Jablonski
3,3895074,2023-09-15,Bayern Munich,2,2,Bayer Leverkusen,Allianz Arena,Daniel Schlager
4,3895086,2023-09-24,Bayer Leverkusen,4,1,FC Heidenheim,BayArena,Christian Dingert


,match_id,period,minute,second,elapsed_min,type,team_name,player_name,x,y,end_x,end_y,outcome,shot_xg
135887,3895348,2,45,35,48.583333,Carry,Bayer Leverkusen,Exequiel Alejandro Palacios,61.5,40.0,63.4,35.7,None,NaN
98389,3895258,2,79,34,83.433333,Ball Receipt*,FC Köln,Timo Hübers,44.4,75.8,NaN,NaN,None,NaN
80547,3895220,1,31,28,31.466667,Ball Receipt*,Bayer Leverkusen,Florian Wirtz,104.4,75.1,NaN,NaN,None,NaN
66238,3895194,1,2,3,2.050000,Ball Receipt*,Bayer Leverkusen,Granit Xhaka,42.4,59.8,NaN,NaN,None,NaN
101713,3895266,2,56,5,57.250000,Foul Won,Wolfsburg,Maxence Lacroix,6.3,65.9,NaN,NaN,None,NaN
63799,3895182,1,45,54,45.900000,Duel,Bochum,Erhan Mašović,5.3,37.0,NaN,NaN,None,NaN


,match_id,team_id,team_name,manager_id,manager_name,manager_full_name,manager_country,manager_source
0,3895052,904,Bayer Leverkusen,1000310,Xabi Alonso,Xabier Alonso Olano,Spain,StatsBomb matches.json
1,3895052,182,RB Leipzig,496,Marco Rose,Marco Rose,Germany,StatsBomb matches.json
2,3895060,185,Borussia Mönchengladbach,732,Gerardo Seoane,Gerardo Seoane Castro,Switzerland,StatsBomb matches.json
5,3895067,872,Darmstadt 98,100,Torsten Lieberknecht,Torsten Lieberknecht,Germany,StatsBomb matches.json
6,3895074,169,Bayern Munich,279,Thomas Tuchel,Thomas Tuchel,Germany,StatsBomb matches.json
9,3895086,201,FC Heidenheim,108,Frank Schmidt,Frank Schmidt,Germany,StatsBomb matches.json
10,3895095,177,FSV Mainz 05,4975,Bo Svensson,Bo Svensson,Denmark,StatsBomb matches.json
13,3895107,186,FC Köln,104,Steffen Baumgart,Steffen Baumgart,Germany,StatsBomb matches.json


## 7 · Feature engineering
Every metric has a definition, numerator/denominator, unit and source field (table below). Rules applied:
* **Per-90** = count ÷ minutes × 90, only where minutes > 0 (no division by zero). Minutes are *actual playing time
  including stoppage*, so per-90 values are slightly lower than "nominal 90" conventions — consistently.
* **Role normalisation** — percentiles are computed within the player's role (if ≥ `min_group_size` eligible
  players) or role family otherwise, against the pool of players above `MIN_MINUTES`.
* **Project-defined approximations** (progressive pass/carry, possession proxy, field tilt, PPDA approximation) are
  labelled as such. xT, packing and tracking-based running metrics are **not** computed — the data cannot support them.
* **Workload proxies** (minutes in previous 7/14/28 days, rest days) count only matches in the dataset — they are not
  medical fatigue measures.

In [11]:
from __future__ import annotations

# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · features — metrics, per-90, percentiles, form, workload, team style
# Feature engineering: event flags, player-match / season tables, per-90, percentiles,
# form, workload proxies, team-match style metrics and match state.
#
# Every metric is defined in ``METRIC_DEFINITIONS`` (numerator, denominator, unit,
# source fields) and exported into the data dictionary. Project-defined metrics
# (e.g. progressive pass) are approximations, labelled as such — they are not
# provider-published metrics.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
from typing import Any

import numpy as np
import pandas as pd


GOAL_X, GOAL_Y = 120.0, 40.0
ON_BALL_TYPES = {"Pass", "Ball Receipt*", "Carry", "Shot", "Dribble", "Miscontrol", "Dispossessed", "Clearance",
                 "Interception", "Ball Recovery", "Block", "Goal Keeper", "50/50", "Shield"}
ON_TARGET = {"Goal", "Saved", "Saved To Post", "Saved to Post"}
GK_SAVE_TYPES = {"Shot Saved", "Shot Saved Off T", "Shot Saved To Post", "Penalty Saved", "Penalty Saved To Post",
                 "Save"}
GK_CLAIM_TYPES = {"Collected", "Punch", "Keeper Sweeper", "Smother"}

# name: (definition, numerator, denominator, unit, source fields)
METRIC_DEFINITIONS: dict[str, tuple[str, str, str, str, str]] = {
    "minutes_played": ("Actual playing time on pitch incl. stoppage (periods 1-4) from Starting XI/substitution/"
                       "sending-off events", "sum of spell lengths", "-", "minutes", "events: Starting XI, "
                       "Substitution, Foul Committed/Bad Behaviour cards, Half Start/End"),
    "goals": ("Shots with outcome Goal in periods 1-4 (shoot-out excluded)", "count", "-", "goals", "shot.outcome"),
    "npxg": ("Sum of StatsBomb xG on non-penalty shots", "sum xG", "-", "xG", "shot.statsbomb_xg, shot.type"),
    "xg": ("Sum of StatsBomb xG on all shots (periods 1-4)", "sum xG", "-", "xG", "shot.statsbomb_xg"),
    "xa": ("xG of shots assisted by the player's passes (pass.assisted_shot_id)", "sum xG", "-", "xG",
           "pass.assisted_shot_id, shot.statsbomb_xg"),
    "shots": ("Shot events (periods 1-4)", "count", "-", "shots", "type=Shot"),
    "shots_on_target": ("Shots with outcome Goal / Saved / Saved To Post", "count", "-", "shots", "shot.outcome"),
    "assists": ("Passes flagged goal_assist by StatsBomb", "count", "-", "assists", "pass.goal_assist"),
    "key_passes": ("Passes flagged shot_assist or goal_assist", "count", "-", "passes",
                   "pass.shot_assist, pass.goal_assist"),
    "passes": ("Pass attempts", "count", "-", "passes", "type=Pass"),
    "passes_completed": ("Passes with no outcome (StatsBomb: no outcome = complete)", "count", "-", "passes",
                         "pass.outcome"),
    "pass_completion_pct": ("Completed / attempted passes", "passes_completed", "passes", "%", "pass.outcome"),
    "long_pass_completion_pct": ("Completion of passes >= 30 units (yards)", "completed long", "long passes", "%",
                                 "pass.length, pass.outcome"),
    "progressive_passes": ("PROJECT DEFINITION: completed open-play pass starting outside the defending 40% "
                           "(x >= 48) and ending >=25% closer to the opponent goal centre", "count", "-", "passes",
                           "location, pass.end_location, pass.type, pass.outcome"),
    "final_third_passes": ("Completed passes from x<80 ending x>=80", "count", "-", "passes", "location, end_location"),
    "box_passes": ("Completed passes ending in the penalty area from outside it", "count", "-", "passes",
                   "location, end_location"),
    "crosses_completed": ("Completed passes flagged cross", "count", "-", "passes", "pass.cross, pass.outcome"),
    "through_balls": ("Passes with technique Through Ball", "count", "-", "passes", "pass.technique"),
    "progressive_carries": ("PROJECT DEFINITION: carry ending >=25% closer to goal centre, end x >= 48 and >=5 units "
                            "forward", "count", "-", "carries", "location, carry.end_location"),
    "carries_into_final_third": ("Carries from x<80 ending x>=80", "count", "-", "carries", "carry.end_location"),
    "carries_into_box": ("Carries ending in the penalty area from outside it", "count", "-", "carries",
                         "carry.end_location"),
    "dribbles": ("Dribble (take-on) attempts", "count", "-", "dribbles", "type=Dribble"),
    "dribbles_completed": ("Dribbles with outcome Complete", "count", "-", "dribbles", "dribble.outcome"),
    "touches": ("On-ball actions (pass, receipt, carry, shot, dribble, clearance, interception, recovery, block, ...)",
                "count", "-", "actions", "type, location"),
    "box_touches": ("On-ball actions located in the opponent penalty area", "count", "-", "actions", "location"),
    "final_third_touches": ("On-ball actions with x >= 80", "count", "-", "actions", "location"),
    "tackles": ("Duels of type Tackle", "count", "-", "tackles", "duel.type"),
    "tackles_won": ("Tackles with outcome Won / Success / Success In Play / Success Out", "count", "-", "tackles",
                    "duel.outcome"),
    "interceptions": ("Interception events", "count", "-", "actions", "type=Interception"),
    "blocks": ("Block events", "count", "-", "actions", "type=Block"),
    "clearances": ("Clearance events", "count", "-", "actions", "type=Clearance"),
    "ball_recoveries": ("Ball Recovery events without recovery_failure", "count", "-", "actions", "ball_recovery"),
    "pressures": ("Pressure events", "count", "-", "actions", "type=Pressure"),
    "counterpressures": ("Pressure events flagged counterpress", "count", "-", "actions", "counterpress"),
    "aerials_won": ("aerial_won flag on pass/shot/clearance/miscontrol", "count", "-", "duels", "*.aerial_won"),
    "aerials_lost": ("Duels of type Aerial Lost", "count", "-", "duels", "duel.type"),
    "defensive_actions": ("tackles + interceptions + blocks + clearances + ball recoveries", "sum", "-", "actions",
                          "derived"),
    "fouls_committed": ("Foul Committed events", "count", "-", "fouls", "type"),
    "fouls_won": ("Foul Won events", "count", "-", "fouls", "type"),
    "yellow_cards": ("Yellow cards (foul_committed/bad_behaviour card)", "count", "-", "cards", "card"),
    "red_cards": ("Red or second-yellow cards", "count", "-", "cards", "card"),
    "turnovers": ("Dispossessed + Miscontrol events", "count", "-", "actions", "type"),
    "gk_saves": ("Goal Keeper events whose type is a save", "count", "-", "saves", "goalkeeper.type"),
    "gk_claims": ("Goal Keeper collected / punch / sweeper / smother actions", "count", "-", "actions",
                  "goalkeeper.type"),
    "gk_goals_conceded": ("Goal Keeper events of type Goal Conceded", "count", "-", "goals", "goalkeeper.type"),
}
PER90_BASE = ["goals", "npxg", "xg", "xa", "shots", "shots_on_target", "assists", "key_passes", "passes",
              "progressive_passes", "final_third_passes", "box_passes", "crosses_completed", "through_balls",
              "progressive_carries", "carries_into_final_third", "carries_into_box", "dribbles", "dribbles_completed",
              "touches", "box_touches", "final_third_touches", "tackles", "tackles_won", "interceptions", "blocks",
              "clearances", "ball_recoveries", "pressures", "counterpressures", "aerials_won", "defensive_actions",
              "fouls_committed", "fouls_won", "turnovers", "gk_saves", "gk_claims"]
COUNT_COLS = PER90_BASE + ["passes_completed", "long_passes", "long_passes_completed", "aerials_lost",
                           "yellow_cards", "red_cards", "gk_goals_conceded"]


def _closer(x, y, ex, ey, frac=0.25):
    d0 = np.hypot(GOAL_X - x, GOAL_Y - y)
    d1 = np.hypot(GOAL_X - ex, GOAL_Y - ey)
    return d1 <= (1 - frac) * d0


def _in_box(x, y):
    return (x >= 102) & (y >= 18) & (y <= 62)


def event_flags(events: pd.DataFrame) -> pd.DataFrame:
    """Add one boolean/numeric column per countable action (periods 1-4 only)."""
    ev = events[events["period"].between(1, 4)].copy()
    t = ev["type"]
    is_pass = t == "Pass"
    completed = is_pass & ev["outcome"].isna()
    open_play = ~ev["pass_type"].isin(SET_PIECE_PASS_TYPES)
    ev["passes"] = is_pass
    ev["passes_completed"] = completed
    ev["long_passes"] = is_pass & (ev["pass_length"] >= 30)
    ev["long_passes_completed"] = completed & (ev["pass_length"] >= 30)
    ev["progressive_passes"] = (completed & open_play & (ev["x"] >= 48) & (ev["end_x"] >= 48)
                                & _closer(ev["x"], ev["y"], ev["end_x"], ev["end_y"]))
    ev["final_third_passes"] = completed & (ev["x"] < 80) & (ev["end_x"] >= 80)
    ev["box_passes"] = completed & ~_in_box(ev["x"], ev["y"]) & _in_box(ev["end_x"], ev["end_y"])
    ev["crosses_completed"] = completed & ev["pass_cross"].fillna(False).astype(bool)
    ev["through_balls"] = is_pass & ev["pass_through_ball"].fillna(False).astype(bool)
    ev["key_passes"] = is_pass & (ev["pass_shot_assist"].fillna(False) | ev["pass_goal_assist"].fillna(False)).astype(bool)
    ev["assists"] = is_pass & ev["pass_goal_assist"].fillna(False).astype(bool)
    is_shot = t == "Shot"
    ev["shots"] = is_shot
    ev["shots_on_target"] = is_shot & ev["outcome"].isin(ON_TARGET)
    ev["goals"] = is_shot & (ev["outcome"] == "Goal")
    ev["xg"] = np.where(is_shot, ev["shot_xg"].fillna(0.0), 0.0)
    ev["npxg"] = np.where(is_shot & (ev["shot_type"] != "Penalty"), ev["shot_xg"].fillna(0.0), 0.0)
    shot_xg = ev.loc[is_shot].set_index("event_id")["shot_xg"]
    ev["xa"] = np.where(is_pass, ev["pass_assisted_shot_id"].map(shot_xg).fillna(0.0), 0.0)
    is_carry = t == "Carry"
    ev["progressive_carries"] = (is_carry & _closer(ev["x"], ev["y"], ev["end_x"], ev["end_y"])
                                 & (ev["end_x"] >= 48) & ((ev["end_x"] - ev["x"]) >= 5))
    ev["carries_into_final_third"] = is_carry & (ev["x"] < 80) & (ev["end_x"] >= 80)
    ev["carries_into_box"] = is_carry & ~_in_box(ev["x"], ev["y"]) & _in_box(ev["end_x"], ev["end_y"])
    ev["dribbles"] = t == "Dribble"
    ev["dribbles_completed"] = ev["dribbles"] & (ev["outcome"] == "Complete")
    on_ball = t.isin(ON_BALL_TYPES) & ev["x"].notna()
    ev["touches"] = on_ball
    ev["box_touches"] = on_ball & _in_box(ev["x"], ev["y"])
    ev["final_third_touches"] = on_ball & (ev["x"] >= 80)
    ev["tackles"] = (t == "Duel") & (ev["duel_type"] == "Tackle")
    ev["tackles_won"] = ev["tackles"] & ev["outcome"].isin(TACKLE_WON_OUTCOMES)
    ev["interceptions"] = t == "Interception"
    ev["blocks"] = t == "Block"
    ev["clearances"] = t == "Clearance"
    ev["ball_recoveries"] = (t == "Ball Recovery") & (ev["outcome"] != "Failed")
    ev["pressures"] = t == "Pressure"
    ev["counterpressures"] = ev["pressures"] & ev["counterpress"].fillna(False).astype(bool)
    ev["aerials_won"] = ev["aerial_won"].fillna(False).astype(bool)
    ev["aerials_lost"] = (t == "Duel") & (ev["duel_type"] == "Aerial Lost")
    ev["defensive_actions"] = (ev["tackles"].astype(int) + ev["interceptions"] + ev["blocks"] + ev["clearances"]
                               + ev["ball_recoveries"])
    ev["fouls_committed"] = t == "Foul Committed"
    ev["fouls_won"] = t == "Foul Won"
    ev["yellow_cards"] = ev["card"] == "Yellow Card"
    ev["red_cards"] = ev["card"].isin(SENDING_OFF_CARDS)
    ev["turnovers"] = t.isin(["Dispossessed", "Miscontrol"])
    gk = t == "Goal Keeper"
    ev["gk_saves"] = gk & ev["goalkeeper_type"].isin(GK_SAVE_TYPES)
    ev["gk_claims"] = gk & ev["goalkeeper_type"].isin(GK_CLAIM_TYPES)
    ev["gk_goals_conceded"] = gk & (ev["goalkeeper_type"] == "Goal Conceded")
    for c in COUNT_COLS:
        if ev[c].dtype == bool:
            ev[c] = ev[c].astype(int)
    return ev


# ---------------------------------------------------------------------------
# Player-match and player-season tables
# ---------------------------------------------------------------------------
def player_match_stats(flags: pd.DataFrame, appearances: pd.DataFrame, matches: pd.DataFrame) -> pd.DataFrame:
    """One row per player-match appearance with action counts and minutes."""
    pe = flags.dropna(subset=["player_id"])
    agg = pe.groupby(["match_id", "team_id", "player_id"])[COUNT_COLS].sum().reset_index()
    app = appearances[appearances["appeared"]].copy()
    out = app.merge(agg, on=["match_id", "team_id", "player_id"], how="left")
    out[COUNT_COLS] = out[COUNT_COLS].fillna(0)
    out = out.merge(matches[["match_id", "match_date", "home_team_id", "away_team_id", "home_team", "away_team"]],
                    on="match_id", how="left")
    out["home_away"] = np.where(out["team_id"] == out["home_team_id"], "home", "away")
    out["opponent"] = np.where(out["home_away"] == "home", out["away_team"], out["home_team"])
    return out.sort_values(["match_date", "match_id"]).reset_index(drop=True)


def _safe_rate(num, den, scale=1.0):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(den > 0, num / den * scale, np.nan)
    return r


def per90(count: Any, minutes: Any) -> np.ndarray:
    """count / minutes * 90, NaN when minutes <= 0 (never divides by zero)."""
    return _safe_rate(count, minutes, 90.0)


def player_season_table(pms: pd.DataFrame, appearances: pd.DataFrame, config: Config,
                        asof: pd.Timestamp | None = None, matches: pd.DataFrame | None = None) -> pd.DataFrame:
    """Season-to-date player table. ``asof`` keeps only matches strictly before that date (leakage control)."""
    p = pms.copy()
    a = appearances.copy()
    if asof is not None:
        p = p[p["match_date"] < asof]
        if matches is not None:
            ids = set(matches.loc[matches["match_date"] < asof, "match_id"])
            a = a[a["match_id"].isin(ids)]
    if p.empty:  # e.g. decision before the first analysed match: no history yet
        cols = (["team_id", "player_id", "player_name", "team_name", "minutes_played", "appearances", "starts",
                 "primary_position", "primary_role", "role_family", "last_appearance", "eligible",
                 "sample_reliability", "coverage_label", "pass_completion_pct", "long_pass_completion_pct"]
                + COUNT_COLS + [f"{c}_p90" for c in PER90_BASE])
        return pd.DataFrame(columns=list(dict.fromkeys(cols)))
    g = p.groupby(["team_id", "player_id"])
    season = g[COUNT_COLS + ["minutes_played"]].sum()
    season["appearances"] = g.size()
    season["starts"] = g["is_starter"].sum()
    season["sub_appearances"] = season["appearances"] - season["starts"]
    season["subbed_off"] = g["sub_off_minute"].apply(lambda s: s.notna().sum())
    season["player_name"] = g["player_name"].last()
    season["team_name"] = g["team_name"].last()
    season["last_appearance"] = g["match_date"].max()
    season = season.reset_index()
    unused = (a[a["squad_status"] == "unused_substitute"].groupby(["team_id", "player_id"]).size()
              .rename("unused_sub_count"))
    squad_n = a.groupby(["team_id", "player_id"]).size().rename("matchday_squads")
    season = season.merge(unused, on=["team_id", "player_id"], how="left").merge(
        squad_n, on=["team_id", "player_id"], how="left")
    season["unused_sub_count"] = season["unused_sub_count"].fillna(0).astype(int)
    # Primary position: the match position carrying the most minutes.
    pos = (p.dropna(subset=["match_position"]).groupby(["team_id", "player_id", "match_position"])["minutes_played"]
           .sum().reset_index().sort_values("minutes_played", ascending=False)
           .drop_duplicates(["team_id", "player_id"]))
    season = season.merge(pos[["team_id", "player_id", "match_position"]].rename(
        columns={"match_position": "primary_position"}), on=["team_id", "player_id"], how="left")
    season["primary_role"] = season["primary_position"].map(role_for_position)
    season["role_family"] = season["primary_role"].map(ROLE_FAMILY)
    season["starts_pct"] = _safe_rate(season["starts"], season["appearances"], 100)
    season["avg_minutes"] = _safe_rate(season["minutes_played"], season["appearances"])
    for c in PER90_BASE:
        season[f"{c}_p90"] = per90(season[c], season["minutes_played"])
    season["pass_completion_pct"] = _safe_rate(season["passes_completed"], season["passes"], 100)
    season["long_pass_completion_pct"] = _safe_rate(season["long_passes_completed"], season["long_passes"], 100)
    season["tackle_success_pct"] = _safe_rate(season["tackles_won"], season["tackles"], 100)
    season["dribble_success_pct"] = _safe_rate(season["dribbles_completed"], season["dribbles"], 100)
    season["goals_minus_xg"] = season["goals"] - season["xg"]
    season["eligible"] = season["minutes_played"] >= config.min_minutes
    season["sample_reliability"] = np.clip(season["minutes_played"] / config.reliability_full_minutes, 0, 1) * 100
    season["coverage_label"] = np.select(
        [season["minutes_played"] >= config.reliability_full_minutes, season["minutes_played"] >= config.min_minutes],
        ["high data coverage", "moderate data coverage"], default="low data coverage")
    return season.sort_values("minutes_played", ascending=False).reset_index(drop=True)


def _pct_of(values: np.ndarray, pool: np.ndarray) -> np.ndarray:
    """Percentile (0-100) of each value within a reference pool (mid-rank for ties)."""
    pool = np.sort(pool[~np.isnan(pool)])
    out = np.full(len(values), np.nan)
    if len(pool) == 0:
        return out
    ok = ~np.isnan(values)
    lo = np.searchsorted(pool, values[ok], side="left")
    hi = np.searchsorted(pool, values[ok], side="right")
    out[ok] = (lo + hi) / 2.0 / len(pool) * 100.0
    return out


def add_percentiles(season: pd.DataFrame, metrics: list[str], config: Config) -> pd.DataFrame:
    """Percentile of each metric within the player's benchmark group.

    Benchmark pool = *eligible* players (>= min_minutes). Group = primary role when
    that role has >= ``min_group_size`` eligible players, otherwise role family
    (explicitly labelled). Players below the threshold get a percentile versus the
    pool but carry a low-coverage label.
    """
    s = season.copy()
    metrics = [m for m in metrics if m in s.columns]
    if s.empty:
        for c in ["benchmark_level", "benchmark_group", "benchmark_n", "small_benchmark_warning"] + [
                f"{m}_pct" for m in metrics]:
            s[c] = pd.Series(dtype=float)
        return s
    elig = s[s["eligible"].astype(bool)]
    role_n = elig.groupby("primary_role").size()
    fam_n = elig.groupby("role_family").size()
    s["benchmark_level"] = np.where(s["primary_role"].map(role_n).fillna(0) >= config.min_group_size, "role",
                                    "role family")
    s["benchmark_group"] = np.where(s["benchmark_level"] == "role", s["primary_role"], s["role_family"])
    s["benchmark_n"] = np.where(s["benchmark_level"] == "role", s["primary_role"].map(role_n),
                                s["role_family"].map(fam_n)).astype(float)
    for m in metrics:
        col = np.full(len(s), np.nan)
        for (lvl, grp), idx in s.groupby(["benchmark_level", "benchmark_group"]).groups.items():
            key = "primary_role" if lvl == "role" else "role_family"
            pool = elig.loc[elig[key] == grp, m].to_numpy(dtype=float)
            col[s.index.get_indexer(idx)] = _pct_of(s.loc[idx, m].to_numpy(dtype=float), pool)
        s[f"{m}_pct"] = col
    s["small_benchmark_warning"] = s["benchmark_n"].fillna(0) < config.min_group_size
    return s


def role_fit_scores(season_pct: pd.DataFrame, role: str, config: Config) -> pd.DataFrame:
    """Transparent role-fit = weighted mean of the role's metric percentiles (0-100).

    Missing metrics are dropped and the weights renormalised; the share of weight
    actually available is reported as ``role_fit_completeness``.
    """
    weights = config.role_metric_weights.get(role, {})
    s = season_pct
    num = np.zeros(len(s))
    wsum = np.zeros(len(s))
    for m, w in weights.items():
        col = f"{m}_pct"
        if col not in s.columns:
            continue
        v = s[col].to_numpy(dtype=float)
        ok = ~np.isnan(v)
        num[ok] += w * v[ok]
        wsum[ok] += w
    total_w = sum(weights.values()) or 1.0
    with np.errstate(invalid="ignore", divide="ignore"):
        fit = np.where(wsum > 0, num / np.where(wsum > 0, wsum, 1), np.nan)
    return pd.DataFrame({"player_id": s["player_id"].values, "role": role, "role_fit": fit,
                         "role_fit_completeness": wsum / total_w * 100}, index=s.index)


def all_metric_columns(config: Config) -> list[str]:
    cols = set()
    for w in list(config.role_metric_weights.values()) + list(config.sub_intent_metrics.values()):
        cols.update(w.keys())
    return sorted(cols)


# ---------------------------------------------------------------------------
# Form and workload
# ---------------------------------------------------------------------------
def recent_form(pms: pd.DataFrame, season_pct: pd.DataFrame, config: Config, asof: pd.Timestamp | None = None,
                window: int | None = None) -> pd.DataFrame:
    """Role composite over each player's last ``window`` appearances before ``asof``.

    Per-90 rates over the window are converted to percentiles against the season
    benchmark pool (eligible players), then combined with the player's role weights.
    """
    window = window or config.form_window
    p = pms if asof is None else pms[pms["match_date"] < asof]
    if p.empty:
        return pd.DataFrame(columns=["team_id", "player_id", "form_score", "form_minutes", "form_apps"])
    last = p.sort_values("match_date").groupby(["team_id", "player_id"]).tail(window)
    g = last.groupby(["team_id", "player_id"])
    rec = g[COUNT_COLS + ["minutes_played"]].sum()
    rec["form_apps"] = g.size()
    rec = rec.reset_index()
    for c in PER90_BASE:
        rec[f"{c}_p90"] = per90(rec[c], rec["minutes_played"])
    rec["pass_completion_pct"] = _safe_rate(rec["passes_completed"], rec["passes"], 100)
    rec["long_pass_completion_pct"] = _safe_rate(rec["long_passes_completed"], rec["long_passes"], 100)
    rec = rec.merge(season_pct[["team_id", "player_id", "primary_role", "role_family"]], on=["team_id", "player_id"],
                    how="left")
    elig = season_pct[season_pct["eligible"].astype(bool)]
    scores = np.full(len(rec), np.nan)
    for role, idx in rec.groupby("primary_role").groups.items():
        weights = config.role_metric_weights.get(role, {})
        fam = ROLE_FAMILY.get(role)
        pool_df = elig[elig["role_family"] == fam]
        num = np.zeros(len(idx))
        wsum = np.zeros(len(idx))
        for m, w in weights.items():
            if m not in rec.columns or m not in pool_df.columns:
                continue
            v = _pct_of(rec.loc[idx, m].to_numpy(dtype=float), pool_df[m].to_numpy(dtype=float))
            ok = ~np.isnan(v)
            num[ok] += w * v[ok]
            wsum[ok] += w
        with np.errstate(invalid="ignore", divide="ignore"):
            scores[rec.index.get_indexer(idx)] = np.where(wsum > 0, num / np.where(wsum > 0, wsum, 1), np.nan)
    rec["form_score"] = scores
    rec = rec.rename(columns={"minutes_played": "form_minutes"})
    return rec[["team_id", "player_id", "form_score", "form_minutes", "form_apps"]]


def workload_features(appearances: pd.DataFrame, matches: pd.DataFrame) -> pd.DataFrame:
    """Pre-match workload proxies for every matchday-squad row.

    All values use only matches strictly before the row's match. They describe
    minutes/rest within the analysed dataset only (other competitions are NOT
    included) and are not medical fatigue measures.
    """
    a = appearances.merge(matches[["match_id", "match_date"]], on="match_id", how="left")
    a = a.sort_values(["match_date", "match_id"])
    rows = []
    for team, ta in a.groupby("team_id"):
        team_matches = ta[["match_id", "match_date"]].drop_duplicates().sort_values("match_date").reset_index(drop=True)
        prev_match = dict(zip(team_matches["match_id"], team_matches["match_id"].shift()))
        prev_date = dict(zip(team_matches["match_id"], team_matches["match_date"].shift()))
        team_match_index = {m: i for i, m in enumerate(team_matches["match_id"])}
        hist = ta[ta["appeared"]][["player_id", "match_id", "match_date", "minutes_played", "is_starter"]]
        starters_by_match = ta[ta["is_starter"]].groupby("match_id")["player_id"].apply(set).to_dict()
        for _, r in ta.iterrows():
            d, pid, mid = r["match_date"], r["player_id"], r["match_id"]
            h = hist[(hist["player_id"] == pid) & (hist["match_date"] < d)]
            pm = prev_match.get(mid)
            consec = 0
            i = team_match_index[mid] - 1
            while i >= 0 and pid in starters_by_match.get(team_matches.loc[i, "match_id"], set()):
                consec += 1
                i -= 1
            prior_team_matches = team_match_index[mid]
            rows.append({
                "match_id": mid, "team_id": team, "player_id": pid,
                "minutes_prev_7d": h.loc[h["match_date"] >= d - pd.Timedelta(days=7), "minutes_played"].sum(),
                "minutes_prev_14d": h.loc[h["match_date"] >= d - pd.Timedelta(days=14), "minutes_played"].sum(),
                "minutes_prev_28d": h.loc[h["match_date"] >= d - pd.Timedelta(days=28), "minutes_played"].sum(),
                "rest_days": (d - h["match_date"].max()).days if len(h) else np.nan,
                "team_rest_days": (d - prev_date[mid]).days if pd.notna(prev_date.get(mid)) else np.nan,
                "started_prev_match": (pid in starters_by_match.get(pm, set())) if pm is not None and pd.notna(pm) else np.nan,
                "consecutive_starts_before": consec,
                "starts_before": int(h["is_starter"].sum()),
                "apps_before": len(h),
                "prior_team_matches": prior_team_matches,
                "start_share_before": h["is_starter"].sum() / prior_team_matches if prior_team_matches else np.nan,
                "minutes_share_before": (h["minutes_played"].sum() / (prior_team_matches * 95.0)
                                         if prior_team_matches else np.nan),
            })
    return pd.DataFrame(rows)


def workload_suitability(minutes_prev_7d: Any, rest_days: Any) -> np.ndarray:
    """0-100 proxy: penalise >180 min in the previous 7 days and < 3 rest days.

    A heuristic for rotation discussion only — NOT fatigue or injury risk.
    """
    m7 = np.nan_to_num(np.asarray(minutes_prev_7d, dtype=float), nan=0.0)
    rd = np.asarray(rest_days, dtype=float)
    score = 100 - np.clip((m7 - 180) / 90 * 40, 0, 40)
    score = score - np.where(~np.isnan(rd) & (rd < 3), 25, 0)
    return np.clip(score, 0, 100)


# ---------------------------------------------------------------------------
# Team-match style metrics
# ---------------------------------------------------------------------------
def team_match_stats(flags: pd.DataFrame, matches: pd.DataFrame, goals: pd.DataFrame,
                     formations_df: pd.DataFrame, managers: pd.DataFrame) -> pd.DataFrame:
    """One row per team per match with style indicators (proxies labelled as such)."""
    sums = ["shots", "shots_on_target", "xg", "npxg", "passes", "passes_completed", "progressive_passes",
            "progressive_carries", "final_third_passes", "box_passes", "pressures", "counterpressures",
            "defensive_actions", "final_third_touches", "box_touches", "touches", "crosses_completed",
            "fouls_committed", "yellow_cards", "red_cards"]
    t = flags.groupby(["match_id", "team_id"])[sums].sum().reset_index()
    sp_shot = flags["shots"].astype(bool) & (flags["play_pattern"].isin(["From Corner", "From Free Kick",
                                                                          "From Throw In"])
                                             | flags["shot_type"].isin(["Free Kick", "Penalty", "Corner"]))
    t = t.merge(flags[sp_shot].groupby(["match_id", "team_id"]).size().rename("set_piece_shots").reset_index(),
                on=["match_id", "team_id"], how="left")
    t["set_piece_shots"] = t["set_piece_shots"].fillna(0)
    # PPDA approximation: opponent passes in their own 60% (x <= 72 in their frame) divided by the
    # team's tackles + interceptions + fouls + failed challenges in the same zone (x >= 48 in its frame).
    opp_pass = flags[flags["passes"].astype(bool) & (flags["x"] <= 72)].groupby(["match_id", "team_id"]).size()
    def_act = flags[(flags["tackles"].astype(bool) | flags["interceptions"].astype(bool)
                     | flags["fouls_committed"].astype(bool) | (flags["type"] == "Dribbled Past"))
                    & (flags["x"] >= 48)].groupby(["match_id", "team_id"]).size()
    m = matches.set_index("match_id")
    rows = []
    for _, r in t.iterrows():
        mid, tid = r["match_id"], r["team_id"]
        mm = m.loc[mid]
        home = tid == mm["home_team_id"]
        opp = mm["away_team_id"] if home else mm["home_team_id"]
        rows.append({"match_id": mid, "team_id": tid, "opponent_id": opp,
                     "team_name": mm["home_team"] if home else mm["away_team"],
                     "opponent": mm["away_team"] if home else mm["home_team"],
                     "home_away": "home" if home else "away", "match_date": mm["match_date"],
                     "ppda": (opp_pass.get((mid, opp), 0) / def_act.get((mid, tid), np.nan))
                     if def_act.get((mid, tid), 0) else np.nan})
    t = t.merge(pd.DataFrame(rows), on=["match_id", "team_id"], how="left")
    opp_cols = ["shots", "shots_on_target", "xg", "passes", "final_third_touches"]
    opp = t[["match_id", "team_id"] + opp_cols].rename(columns={"team_id": "opponent_id",
                                                                **{c: f"{c}_against" for c in opp_cols}})
    t = t.merge(opp, on=["match_id", "opponent_id"], how="left")
    gf = goals.groupby(["match_id", "scoring_team_id"]).size()
    t["goals_for"] = [gf.get((mm, tt), 0) for mm, tt in zip(t["match_id"], t["team_id"])]
    t["goals_against"] = [gf.get((mm, oo), 0) for mm, oo in zip(t["match_id"], t["opponent_id"])]
    t["goal_diff"] = t["goals_for"] - t["goals_against"]
    t["result"] = np.select([t["goal_diff"] > 0, t["goal_diff"] == 0], ["W", "D"], default="L")
    t["points"] = t["result"].map({"W": 3, "D": 1, "L": 0})
    t["possession_proxy_pct"] = _safe_rate(t["passes"], t["passes"] + t["passes_against"], 100)
    t["field_tilt_pct"] = _safe_rate(t["final_third_touches"],
                                     t["final_third_touches"] + t["final_third_touches_against"], 100)
    t["pass_completion_pct"] = _safe_rate(t["passes_completed"], t["passes"], 100)
    t["xg_diff"] = t["xg"] - t["xg_against"]
    start_f = formations_df[formations_df["type"] == "Starting XI"][["match_id", "team_id", "formation"]]
    t = t.merge(start_f.rename(columns={"formation": "starting_formation"}), on=["match_id", "team_id"], how="left")
    shifts = formations_df[formations_df["formation_changed"]].groupby(["match_id", "team_id"]).size()
    t["in_match_formation_changes"] = [shifts.get((mm, tt), 0) for mm, tt in zip(t["match_id"], t["team_id"])]
    if not managers.empty:
        mg = managers.drop_duplicates(["match_id", "team_id"])[["match_id", "team_id", "manager_id", "manager_name"]]
        t = t.merge(mg, on=["match_id", "team_id"], how="left")
    else:
        t["manager_id"] = np.nan
        t["manager_name"] = None
    return t.sort_values(["match_date", "match_id", "home_away"], ascending=[True, True, False]).reset_index(drop=True)


# ---------------------------------------------------------------------------
# Match state
# ---------------------------------------------------------------------------
def score_at(goals: pd.DataFrame, match_id: Any, team_id: Any, opponent_id: Any,
             elapsed: float | None = None, before_index: int | None = None) -> tuple[int, int]:
    """(team goals, opponent goals) strictly before an event index or elapsed minute."""
    g = goals[goals["match_id"] == match_id]
    if before_index is not None:
        g = g[g["index"] < before_index]
    elif elapsed is not None:
        g = g[g["elapsed_min"] < elapsed]
    return int((g["scoring_team_id"] == team_id).sum()), int((g["scoring_team_id"] == opponent_id).sum())


def state_label(goal_diff: int) -> str:
    return "leading" if goal_diff > 0 else ("trailing" if goal_diff < 0 else "drawing")


def men_on_pitch(send_offs: pd.DataFrame, match_id: Any, team_id: Any, opponent_id: Any,
                 elapsed: float | None = None, before_index: int | None = None) -> int:
    """Numerical advantage (+) / disadvantage (-) from sending-offs before the moment."""
    s = send_offs[send_offs["match_id"] == match_id]
    if before_index is not None:
        s = s[s["index"] < before_index]
    elif elapsed is not None:
        s = s[s["elapsed_min"] < elapsed]
    return int((s["team_id"] == opponent_id).sum() - (s["team_id"] == team_id).sum())


def assign_band(period: Any, minute: Any, bands: list[dict]) -> str | None:
    """Assign a (period, clock minute) to a configured analytical band."""
    for b in bands:
        if period in b["periods"] and b["min"] <= minute <= b["max"]:
            return b["label"]
    return None


def metric_dictionary() -> pd.DataFrame:
    return pd.DataFrame([{"metric": k, "definition": v[0], "numerator": v[1], "denominator": v[2], "unit": v[3],
                          "source_fields": v[4]} for k, v in METRIC_DEFINITIONS.items()])


# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · tactical_analysis — formations, pass networks, expected points
# Team and tactical intelligence: formation summaries, team style, pass networks.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
from typing import Any

import numpy as np
import pandas as pd


def formation_summary(team_stats: pd.DataFrame, team: str | None = None, min_matches: int = 3) -> pd.DataFrame:
    """Starting-formation usage and observed outcomes (descriptive; small samples flagged)."""
    t = team_stats if not team or team == "All" else team_stats[team_stats["team_name"] == team]
    if t.empty or "starting_formation" not in t:
        return pd.DataFrame()
    g = t.groupby("starting_formation")
    out = pd.DataFrame({
        "matches": g.size(), "wins": g["result"].apply(lambda s: (s == "W").sum()),
        "draws": g["result"].apply(lambda s: (s == "D").sum()), "losses": g["result"].apply(lambda s: (s == "L").sum()),
        "points_per_match": g["points"].mean(), "xg_for": g["xg"].mean(), "xg_against": g["xg_against"].mean(),
        "goals_for": g["goals_for"].mean(), "goals_against": g["goals_against"].mean(),
        "possession_proxy_pct": g["possession_proxy_pct"].mean(), "field_tilt_pct": g["field_tilt_pct"].mean(),
    }).round(2).reset_index().sort_values("matches", ascending=False)
    out["sample_warning"] = np.where(out["matches"] < min_matches, f"n<{min_matches}: descriptive only", "")
    return out.reset_index(drop=True)


def team_style_summary(team_stats: pd.DataFrame) -> pd.DataFrame:
    """Per-team average style indicators across analysed matches."""
    cols = ["goals_for", "goals_against", "xg", "xg_against", "shots", "shots_against", "possession_proxy_pct",
            "field_tilt_pct", "pass_completion_pct", "progressive_passes", "pressures", "ppda", "set_piece_shots",
            "points"]
    g = team_stats.groupby("team_name")
    out = g[cols].mean().round(2)
    out.insert(0, "matches", g.size())
    return out.sort_values("matches", ascending=False).reset_index()


def pass_network(flags: pd.DataFrame, appearances: pd.DataFrame, match_id: Any, team_id: Any,
                 min_passes: int = 3) -> tuple[pd.DataFrame, pd.DataFrame, float | None]:
    """Nodes (average pass origin) and edges (completed passer->recipient counts).

    Restricted to the period before the team's first substitution so all nodes
    represent players on the pitch together. Returns (nodes, edges, cutoff_minute).
    """
    ev = flags[(flags["match_id"] == match_id) & (flags["team_id"] == team_id)]
    if ev.empty:
        return pd.DataFrame(), pd.DataFrame(), None
    subs = ev[ev["type"] == "Substitution"]["elapsed_min"]
    cutoff = float(subs.min()) if len(subs) else float(ev["elapsed_min"].max())
    p = ev[(ev["type"] == "Pass") & (ev["elapsed_min"] < cutoff) & ev["x"].notna()]
    comp = p[p["outcome"].isna() & p["pass_recipient_id"].notna()]
    nodes = (p.groupby(["player_id", "player_name"]).agg(x=("x", "mean"), y=("y", "mean"), passes=("x", "size"))
             .reset_index())
    names = appearances[(appearances["match_id"] == match_id)].set_index("player_id")["player_name"]
    nodes["display_name"] = nodes["player_id"].map(names).fillna(nodes["player_name"])
    edges = (comp.groupby(["player_id", "pass_recipient_id"]).size().rename("count").reset_index())
    edges = edges[edges["count"] >= min_passes]
    pos = nodes.set_index("player_id")[["x", "y"]]
    edges = edges[edges["player_id"].isin(pos.index) & edges["pass_recipient_id"].isin(pos.index)]
    if not edges.empty:
        edges = edges.assign(x0=edges["player_id"].map(pos["x"]), y0=edges["player_id"].map(pos["y"]),
                             x1=edges["pass_recipient_id"].map(pos["x"]), y1=edges["pass_recipient_id"].map(pos["y"]))
    return nodes, edges, cutoff


def match_timeline_events(events: pd.DataFrame, goals: pd.DataFrame, match_id: Any) -> pd.DataFrame:
    """Goals, cards, substitutions and formation changes for one match."""
    ev = events[events["match_id"] == match_id]
    rows = []
    for _, g in goals[goals["match_id"] == match_id].iterrows():
        team = ev.loc[ev["team_id"] == g["scoring_team_id"], "team_name"]
        rows.append({"elapsed_min": g["elapsed_min"], "minute": g["minute"], "period": g["period"],
                     "team_name": team.iloc[0] if len(team) else g["scoring_team_id"], "event": "Goal",
                     "detail": f"{g['player_name']} ({g['goal_type']})"})
    cards = ev[ev["card"].notna()]
    for _, c in cards.iterrows():
        rows.append({"elapsed_min": c["elapsed_min"], "minute": c["minute"], "period": c["period"],
                     "team_name": c["team_name"], "event": c["card"], "detail": c["player_name"]})
    for _, s in ev[ev["type"] == "Substitution"].iterrows():
        rows.append({"elapsed_min": s["elapsed_min"], "minute": s["minute"], "period": s["period"],
                     "team_name": s["team_name"], "event": "Substitution",
                     "detail": f"OFF {s['player_name']} / ON {s['sub_replacement_name']} ({s['outcome']})"})
    for _, f in ev[ev["type"] == "Tactical Shift"].iterrows():
        rows.append({"elapsed_min": f["elapsed_min"], "minute": f["minute"], "period": f["period"],
                     "team_name": f["team_name"], "event": "Tactical shift", "detail": f"formation {f['formation']}"})
    return pd.DataFrame(rows).sort_values("elapsed_min").reset_index(drop=True) if rows else pd.DataFrame(
        columns=["elapsed_min", "minute", "period", "team_name", "event", "detail"])


def cumulative_xg(flags: pd.DataFrame, match_id: Any) -> pd.DataFrame:
    s = flags[(flags["match_id"] == match_id) & (flags["type"] == "Shot")][
        ["team_name", "elapsed_min", "minute", "shot_xg", "outcome", "player_name"]].copy()
    s = s.sort_values("elapsed_min")
    s["cum_xg"] = s.groupby("team_name")["shot_xg"].cumsum()
    return s


def expected_points(flags: pd.DataFrame, team_stats: pd.DataFrame, n_sims: int = 5000, seed: int = 0
                    ) -> pd.DataFrame:
    """Expected points per team-match by simulating every shot as a Bernoulli(xG) trial.

    A shot-based xPts model (periods 1-4; own goals not simulated). It summarises
    chance quality, not what "should" have happened, and assumes shots are independent.
    """
    rng = np.random.default_rng(seed)
    shots = flags[(flags["type"] == "Shot") & flags["period"].between(1, 4)]
    rows = []
    for mid, g in team_stats.groupby("match_id"):
        if len(g) != 2:
            continue
        t0, t1 = g.iloc[0], g.iloc[1]
        sim = {}
        for t in (t0, t1):
            xg = shots.loc[(shots["match_id"] == mid) & (shots["team_id"] == t["team_id"]), "shot_xg"].fillna(0)
            xg = xg.to_numpy(dtype=float)
            sim[t["team_id"]] = (rng.random((n_sims, len(xg))) < xg).sum(axis=1) if len(xg) else np.zeros(n_sims)
        for t, o in ((t0, t1), (t1, t0)):
            a, b = sim[t["team_id"]], sim[o["team_id"]]
            pw, pd_ = float((a > b).mean()), float((a == b).mean())
            rows.append({"match_id": mid, "team_id": t["team_id"], "team_name": t["team_name"],
                         "match_date": t["match_date"], "p_win": pw, "p_draw": pd_, "p_loss": 1 - pw - pd_,
                         "xpts": 3 * pw + pd_, "points": t["points"]})
    return pd.DataFrame(rows).sort_values(["match_date", "match_id"]).reset_index(drop=True)


# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · feature build step
# Runs every feature function above in order and stores the results on the bundle.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
def build_features(bundle: Bundle) -> Bundle:
    cfg = bundle.config
    t = bundle.tables
    t0 = time.perf_counter()
    bundle.flags = event_flags(t["events"])
    bundle.pms = player_match_stats(bundle.flags, t["appearances"], t["matches"])
    bundle.season = player_season_table(bundle.pms, t["appearances"], cfg)
    bundle.season_pct = add_percentiles(bundle.season, all_metric_columns(cfg) + [
        "defensive_actions_p90", "touches_p90", "turnovers_p90"], cfg)
    bundle.team_stats = team_match_stats(bundle.flags, t["matches"], t["goals"], t["formations"], t["managers"])
    bundle.workload = workload_features(t["appearances"], t["matches"])
    bundle.timings["features_s"] = time.perf_counter() - t0
    return bundle

In [12]:
build_features(bundle)
B = bundle
FOCUS = cfg.focus_team or B.team_stats["team_name"].value_counts().index[0]
print(f"Feature tables built in {B.timings['features_s']:.1f}s | player-match rows {len(B.pms)} | "
      f"player-season rows {len(B.season)} | team-match rows {len(B.team_stats)} | focus team: {FOCUS}")
display(metric_dictionary().head(45))

cols = ["player_name", "team_name", "primary_position", "primary_role", "appearances", "starts", "minutes_played",
        "goals", "xg", "xg_p90", "xa_p90", "key_passes_p90", "progressive_passes_p90", "pressures_p90",
        "pass_completion_pct", "benchmark_group", "benchmark_n", "coverage_label"]
display(B.season_pct[B.season_pct.team_name == FOCUS][cols].round(2).head(20))
display(Markdown("**Minimum-minutes sensitivity** — how the xG/90 leaderboard changes with the threshold"))
display(threshold_sensitivity(B.season, "xg_p90"))
threshold_sensitivity_chart(B.season, banner=BANNER).show()

display(Markdown("**Workload proxies (pre-match, dataset matches only)** — focus team, latest match"))
last_mid = B.team_stats[B.team_stats.team_name == FOCUS].sort_values("match_date")["match_id"].iloc[-1]
w = B.workload[B.workload.match_id == last_mid].merge(
    T["appearances"][["match_id", "player_id", "player_name", "squad_status"]], on=["match_id", "player_id"])
display(w[["player_name", "squad_status", "minutes_prev_7d", "minutes_prev_14d", "minutes_prev_28d", "rest_days",
           "started_prev_match", "consecutive_starts_before", "start_share_before"]].round(2)
        .sort_values("start_share_before", ascending=False).head(12))
display(Markdown("**Team style indicators** (averages per analysed match)"))
display(team_style_summary(B.team_stats).head(10))

Feature tables built in 2.4s | player-match rows 1049 | player-season rows 377 | team-match rows 68 | focus team: Bayer Leverkusen


,metric,definition,numerator,denominator,unit,source_fields
0,minutes_played,Actual playing time on pitch incl. stoppage (p...,sum of spell lengths,-,minutes,"events: Starting XI, Substitution, Foul Commit..."
1,goals,Shots with outcome Goal in periods 1-4 (shoot-...,count,-,goals,shot.outcome
2,npxg,Sum of StatsBomb xG on non-penalty shots,sum xG,-,xG,"shot.statsbomb_xg, shot.type"
3,xg,Sum of StatsBomb xG on all shots (periods 1-4),sum xG,-,xG,shot.statsbomb_xg
4,xa,xG of shots assisted by the player's passes (p...,sum xG,-,xG,"pass.assisted_shot_id, shot.statsbomb_xg"
5,shots,Shot events (periods 1-4),count,-,shots,type=Shot
6,shots_on_target,Shots with outcome Goal / Saved / Saved To Post,count,-,shots,shot.outcome
7,assists,Passes flagged goal_assist by StatsBomb,count,-,assists,pass.goal_assist
8,key_passes,Passes flagged shot_assist or goal_assist,count,-,passes,"pass.shot_assist, pass.goal_assist"
9,passes,Pass attempts,count,-,passes,type=Pass


,player_name,team_name,primary_position,primary_role,appearances,starts,minutes_played,goals,xg,xg_p90,xa_p90,key_passes_p90,progressive_passes_p90,pressures_p90,pass_completion_pct,benchmark_group,benchmark_n,coverage_label
0,Lukáš Hrádecký,Bayer Leverkusen,Goalkeeper,GK,33,33,3214.60,0.0,0.00,0.00,0.00,0.00,0.00,0.03,80.16,GK,14.0,high data coverage
1,Granit Xhaka,Bayer Leverkusen,Right Defensive Midfield,DM,33,32,3034.77,3.0,1.57,0.05,0.08,1.22,4.89,9.16,92.30,DM,14.0,high data coverage
2,Álex Grimaldo,Bayer Leverkusen,Left Wing Back,WB,33,31,2994.68,10.0,5.21,0.16,0.27,2.28,4.27,9.65,82.58,Defender,38.0,high data coverage
3,Jonathan Tah,Bayer Leverkusen,Center Back,CB,31,30,2848.12,4.0,1.81,0.06,0.01,0.16,2.02,6.83,94.92,CB,22.0,high data coverage
4,Florian Wirtz,Bayer Leverkusen,Left Attacking Midfield,AM,32,26,2528.87,11.0,8.47,0.30,0.28,2.53,7.15,23.17,83.81,Midfielder,22.0,high data coverage
5,Jeremie Frimpong,Bayer Leverkusen,Right Wing Back,WB,31,27,2378.47,9.0,8.72,0.33,0.17,1.63,2.76,13.13,79.09,Defender,38.0,high data coverage
6,Jonas Hofmann,Bayer Leverkusen,Right Attacking Midfield,AM,32,26,2334.68,4.0,5.71,0.22,0.30,2.93,3.39,20.97,81.09,Midfielder,22.0,high data coverage
7,Edmond Fayçal Tapsoba,Bayer Leverkusen,Left Center Back,CB,28,23,2262.52,0.0,1.01,0.04,0.03,0.28,4.06,8.00,91.62,CB,22.0,high data coverage
8,Odilon Kossonou,Bayer Leverkusen,Right Center Back,CB,22,21,1962.23,1.0,0.39,0.02,0.04,0.50,2.94,8.94,90.58,CB,22.0,high data coverage
9,Exequiel Palacios,Bayer Leverkusen,Left Defensive Midfield,DM,24,21,1949.78,4.0,2.69,0.12,0.15,1.25,4.62,15.79,92.33,DM,14.0,high data coverage


**Minimum-minutes sensitivity** — how the xG/90 leaderboard changes with the threshold

,min_minutes,eligible_players,top_5
0,90,224,"Victor Boniface (0.87), Serhou Guirassy (0.85)..."
1,180,84,"Victor Boniface (0.87), Serhou Guirassy (0.85)..."
2,270,19,"Victor Boniface (0.87), Adam Hložek (0.50), Pa..."
3,450,18,"Victor Boniface (0.87), Adam Hložek (0.50), Pa..."
4,900,17,"Victor Boniface (0.87), Patrik Schick (0.43), ..."


**Workload proxies (pre-match, dataset matches only)** — focus team, latest match

,player_name,squad_status,minutes_prev_7d,minutes_prev_14d,minutes_prev_28d,rest_days,started_prev_match,consecutive_starts_before,start_share_before
2,Felix Uduokhai,starter,0.00,0.00,0.00,126.0,True,1,1.00
3,Jeffrey Gouweleeuw,starter,0.00,0.00,0.00,126.0,True,1,1.00
4,Niklas Dorsch,starter,0.00,0.00,0.00,126.0,True,1,1.00
5,Phillip Tietz,starter,0.00,0.00,0.00,126.0,True,1,1.00
1,Ermedin Demirović,starter,0.00,0.00,0.00,126.0,True,1,1.00
11,Iago,starter,0.00,0.00,0.00,126.0,True,1,1.00
19,Granit Xhaka,used_substitute,80.92,153.30,255.32,6.0,True,2,0.97
22,Lukáš Hrádecký,starter,96.27,192.27,394.05,6.0,True,14,0.97
25,Álex Grimaldo,starter,14.97,14.97,207.17,6.0,False,0,0.91
21,Jonathan Tah,starter,96.27,119.88,268.93,6.0,True,1,0.88


**Team style indicators** (averages per analysed match)

,team_name,matches,goals_for,goals_against,xg,xg_against,shots,shots_against,possession_proxy_pct,field_tilt_pct,pass_completion_pct,progressive_passes,pressures,ppda,set_piece_shots,points
0,Bayer Leverkusen,34,2.62,0.71,2.14,0.82,18.32,8.62,61.70,67.61,87.64,37.85,140.76,16.95,9.59,2.65
1,Augsburg,2,0.50,1.50,0.55,1.87,7.00,22.50,29.92,20.83,78.81,19.00,214.50,20.85,2.50,0.00
2,Werder Bremen,2,0.00,4.00,0.43,2.05,4.50,17.00,42.74,36.26,82.75,18.50,211.00,22.99,2.50,0.00
3,VfB Stuttgart,2,1.50,1.50,2.10,1.59,16.50,16.50,51.31,43.25,84.90,19.00,186.50,12.35,7.50,1.00
4,Union Berlin,2,0.00,2.50,0.29,2.22,5.50,19.00,33.21,29.13,77.20,16.00,200.50,39.03,3.50,0.00
5,RB Leipzig,2,2.00,3.00,1.40,1.70,12.00,14.50,47.88,40.04,82.89,22.00,169.50,13.61,4.50,0.00
6,Hoffenheim,2,1.50,2.50,1.07,2.02,9.00,22.00,39.92,28.93,79.87,19.50,178.00,25.06,3.00,0.00
7,Freiburg,2,1.50,2.50,0.85,2.01,10.50,18.50,31.62,28.55,76.51,16.00,132.00,20.84,8.00,0.00
8,FSV Mainz 05,2,0.50,2.50,0.79,0.95,11.50,11.50,39.42,40.91,77.96,18.00,169.00,16.63,8.00,0.00
9,FC Köln,2,0.00,2.50,0.94,2.12,6.50,20.00,29.36,16.87,77.29,10.50,202.00,36.50,3.00,0.00


## 8 · Exploratory data analysis
Opens with the **season dashboard** (KPI cards, points race against shot-based expected points, match-by-match xG),
then distributions (what is typical, what is extreme), position-group differences (RQ1), relationships (correlation
is association only), team trends, match dominance and a single-match dashboard.

*Expected points (xPts)* simulate each match 5,000 times, treating every shot as a Bernoulli trial with its
StatsBomb xG — a summary of chance quality, not of what "should" have happened (shots are not independent in reality).

In [13]:
from __future__ import annotations

# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · advanced_visuals — heatmaps, matrices, momentum, style radar
# Additional presentation-grade charts: pitch heatmaps, squad matrices, momentum, phase profiles,
# sunburst, pass-direction rose, style radar.
#
# Same contract as :mod:`visualizations`: required columns are checked and an empty-state
# figure is returned instead of an exception; nothing is imputed.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


HEAT_SCALE = [[0.0, "rgba(238,242,236,0)"], [0.15, "#d7e8e1"], [0.4, "#8cc5b8"], [0.65, "#3f8f99"],
              [0.85, "#2b5f86"], [1.0, NAVY]]
ROLE_ORDER = ["GK", "CB", "FB", "WB", "DM", "CM", "AM", "WM", "W", "CF"]


def _match_label(row: pd.Series, team: str) -> str:
    opp = row["away_team"] if row["home_team"] == team else row["home_team"]
    ha = "H" if row["home_team"] == team else "A"
    return f"{row['match_date']:%d %b} {opp[:14]} ({ha})"


# ---------------------------------------------------------------------------
# Pitch heatmap
# ---------------------------------------------------------------------------
def heatmap_layer(ev: pd.DataFrame, bins: tuple[int, int] = (12, 8), row: int | None = None,
                  col: int | None = None, showscale: bool = True) -> go.Heatmap:
    """Zone-share heatmap trace (percent of actions per zone) on the 120 x 80 frame."""
    d = ev.dropna(subset=["x", "y"])
    h, xe, ye = np.histogram2d(d["x"].clip(0, 119.99), d["y"].clip(0, 79.99), bins=bins, range=[[0, 120], [0, 80]])
    share = h / max(h.sum(), 1) * 100
    return go.Heatmap(x=(xe[:-1] + xe[1:]) / 2, y=(ye[:-1] + ye[1:]) / 2, z=share.T, customdata=h.T,
                      colorscale=HEAT_SCALE, zmin=0, zsmooth="best", showscale=showscale,
                      colorbar=dict(title="% of<br>actions", thickness=12, len=0.6),
                      hovertemplate="zone centre x=%{x:.0f}, y=%{y:.0f}<br>%{z:.1f}% of actions "
                                    "(%{customdata:.0f})<extra></extra>")


def action_heatmap(ev: pd.DataFrame, title: str, bins: tuple[int, int] = (12, 8),
                   banner: str | None = None) -> go.Figure:
    """Where a player or team performs the selected actions (share of actions per zone)."""
    msg = _missing(ev, ["x", "y"])
    if msg or ev.dropna(subset=["x", "y"]).empty:
        return _empty(msg or "No located actions for this selection.", title)
    fig = go.Figure(heatmap_layer(ev, bins))
    draw_pitch(fig, lines_on_top=True)
    attack_arrow(fig)
    n = int(ev.dropna(subset=["x", "y"]).shape[0])
    fig.update_layout(title=f"{title}<br><sup>{n:,} located actions · {bins[0]}×{bins[1]} zones · colour = share "
                            f"of actions</sup>", height=520)
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# Squad matrices
# ---------------------------------------------------------------------------
def rotation_matrix(appearances: pd.DataFrame, matches: pd.DataFrame, team: str,
                    banner: str | None = None) -> go.Figure:
    """Player × match minutes matrix: the whole season's selection story in one view."""
    a = _rows(appearances, "team_name", team)
    msg = _missing(a, ["match_id", "player_id", "player_name", "minutes_played", "squad_status"])
    if msg:
        return _empty(msg, f"Squad rotation — {team}")
    a = a.merge(matches[["match_id", "match_date", "home_team", "away_team"]], on="match_id")
    order = a[["match_id", "match_date", "home_team", "away_team"]].drop_duplicates().sort_values("match_date")
    labels = [_match_label(r, team) for _, r in order.iterrows()]
    mins = a.pivot_table(index="player_name", columns="match_id", values="minutes_played", aggfunc="sum")
    stat = a.pivot_table(index="player_name", columns="match_id", values="squad_status", aggfunc="first")
    mins = mins.reindex(columns=order["match_id"])
    stat = stat.reindex(index=mins.index, columns=order["match_id"])
    keep = mins.sum(axis=1).sort_values(ascending=False).index
    mins, stat = mins.loc[keep], stat.loc[keep]
    sym = stat.replace({"starter": "●", "used_substitute": "▲", "unused_substitute": "·"}).fillna("")
    hover = np.where(stat.isna(), "not in matchday squad", stat.fillna("").replace(
        {"starter": "started", "used_substitute": "came on", "unused_substitute": "unused substitute"}))
    fig = go.Figure(go.Heatmap(
        z=mins.values, x=labels, y=list(mins.index), colorscale=[[0, "#f1efe8"], [0.01, "#d7e8e1"], [0.5, "#5aa697"],
                                                                 [1, NAVY]],
        zmin=0, zmax=float(np.nanmax(mins.values)) if np.isfinite(np.nanmax(mins.values)) else 100,
        text=sym.values, texttemplate="%{text}", textfont=dict(size=9, color="white"),
        customdata=hover, xgap=1, ygap=1, colorbar=dict(title="minutes", thickness=12),
        hovertemplate="%{y}<br>%{x}<br>%{customdata} · %{z:.0f} min<extra></extra>"))
    fig.update_layout(title=f"{team}: who played when — minutes per match (● started, ▲ came on, · unused, blank = "
                            f"not in squad)", height=max(420, 20 * len(mins) + 180),
                      xaxis=dict(tickangle=-60, tickfont=dict(size=9), showgrid=False),
                      yaxis=dict(autorange="reversed", tickfont=dict(size=10), showgrid=False),
                      margin=dict(l=180, b=140), plot_bgcolor="white")
    return add_banner(fig, banner)


def lineup_similarity(appearances: pd.DataFrame, matches: pd.DataFrame, team: str,
                      banner: str | None = None) -> go.Figure:
    """Jaccard similarity of starting XIs between every pair of matches (rotation blocks)."""
    a = _rows(appearances, "team_name", team)
    msg = _missing(a, ["match_id", "player_id", "is_starter"])
    if msg:
        return _empty(msg, f"Lineup similarity — {team}")
    st = a[a["is_starter"]].merge(matches[["match_id", "match_date", "home_team", "away_team"]], on="match_id")
    order = st[["match_id", "match_date", "home_team", "away_team"]].drop_duplicates().sort_values("match_date")
    sets = [set(st.loc[st["match_id"] == m, "player_id"]) for m in order["match_id"]]
    n = len(sets)
    if n < 2:
        return _empty("Need at least two matches.", f"Lineup similarity — {team}")
    jac = np.array([[len(sets[i] & sets[j]) / len(sets[i] | sets[j]) for j in range(n)] for i in range(n)])
    shared = np.array([[len(sets[i] & sets[j]) for j in range(n)] for i in range(n)])
    labels = [_match_label(r, team) for _, r in order.iterrows()]
    fig = go.Figure(go.Heatmap(z=jac, x=labels, y=labels, zmin=0, zmax=1, colorscale=SEQ_SCALE, customdata=shared,
                               colorbar=dict(title="Jaccard", thickness=12),
                               hovertemplate="%{y}<br>vs %{x}<br>%{customdata} of 11 starters shared<extra></extra>"))
    fig.update_layout(title=f"{team}: starting-XI similarity between matches (dark blocks = stable XI periods)",
                      height=720, xaxis=dict(tickangle=-60, tickfont=dict(size=8)),
                      yaxis=dict(autorange="reversed", tickfont=dict(size=8)), margin=dict(l=140, b=140))
    return add_banner(fig, banner)


def squad_percentile_heatmap(season_pct: pd.DataFrame, team: str, metrics: list[str], min_minutes: float,
                             banner: str | None = None) -> go.Figure:
    """Squad × metric percentile matrix (benchmark group stated in the hover)."""
    s = _rows(season_pct, "team_name", team)
    cols = [f"{m}_pct" for m in metrics]
    msg = _missing(s, ["player_name", "primary_role", "minutes_played"] + cols)
    if msg:
        return _empty(msg, f"Squad percentile profile — {team}")
    s = s[(s["minutes_played"] >= min_minutes) & (s["primary_role"] != "GK")].copy()
    if s.empty:
        return _empty("No eligible outfield players.", f"Squad percentile profile — {team}")
    s["role_rank"] = s["primary_role"].map({r: i for i, r in enumerate(ROLE_ORDER)})
    s = s.sort_values(["role_rank", "minutes_played"], ascending=[True, False])
    ylab = [f"{n} ({r})" for n, r in zip(s["player_name"], s["primary_role"])]
    raw = s[metrics].to_numpy(dtype=float)
    fig = go.Figure(go.Heatmap(
        z=s[cols].to_numpy(dtype=float), x=[label(m) for m in metrics], y=ylab, zmin=0, zmax=100, colorscale=DIV_SCALE,
        zmid=50, text=np.round(s[cols].to_numpy(dtype=float)), texttemplate="%{text:.0f}", textfont=dict(size=9),
        customdata=np.dstack([raw, np.repeat(s["benchmark_group"].to_numpy()[:, None], len(metrics), axis=1)]),
        xgap=1, ygap=1, colorbar=dict(title="percentile", thickness=12),
        hovertemplate="%{y}<br>%{x}: %{z:.0f}th pct<br>value %{customdata[0]:.2f} · vs %{customdata[1]}"
                      "<extra></extra>"))
    fig.update_layout(title=f"{team}: squad profile — percentile per metric (blue = high, red = low; min "
                            f"{min_minutes:.0f} min)", height=max(420, 24 * len(s) + 200),
                      xaxis=dict(tickangle=-35, side="top"), yaxis=dict(autorange="reversed"),
                      margin=dict(l=230, t=230), title_y=0.99, plot_bgcolor="white")
    return add_banner(fig, banner)


def minutes_sunburst(season: pd.DataFrame, team: str, banner: str | None = None) -> go.Figure:
    """Role family → role → player, sized by minutes: how playing time is distributed."""
    s = _rows(season, "team_name", team)
    msg = _missing(s, ["player_name", "primary_role", "role_family", "minutes_played"])
    if msg:
        return _empty(msg, f"Minutes distribution — {team}")
    s = s[(s["minutes_played"] > 0) & s["primary_role"].notna()]
    ids, labels, parents, values, colors = [], [], [], [], []
    for fam, gf in s.groupby("role_family"):
        ids.append(fam); labels.append(fam); parents.append(""); values.append(gf["minutes_played"].sum())
        colors.append(FAMILY_COLORS.get(fam, SLATE))
        for role, gr in gf.groupby("primary_role"):
            rid = f"{fam}/{role}"
            ids.append(rid); labels.append(ROLE_LABEL.get(role, role)); parents.append(fam)
            values.append(gr["minutes_played"].sum()); colors.append(FAMILY_COLORS.get(fam, SLATE))
            for _, p in gr.iterrows():
                ids.append(f"{rid}/{p['player_id']}"); labels.append(p["player_name"]); parents.append(rid)
                values.append(p["minutes_played"]); colors.append(FAMILY_COLORS.get(fam, SLATE))
    fig = go.Figure(go.Sunburst(ids=ids, labels=labels, parents=parents, values=values, branchvalues="total",
                                marker=dict(colors=colors, line=dict(color="white", width=1)),
                                insidetextorientation="radial",
                                hovertemplate="<b>%{label}</b><br>%{value:,.0f} minutes<br>%{percentRoot:.1%} of "
                                              "team minutes<extra></extra>"))
    fig.update_layout(title=f"{team}: distribution of playing time (role family → role → player; click to zoom)",
                      height=620, margin=dict(t=70, l=10, r=10, b=10))
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# Match / team flow
# ---------------------------------------------------------------------------
def match_momentum(flags: pd.DataFrame, goals: pd.DataFrame, match_id: Any, bucket: float = 5.0,
                   title: str = "Match momentum", banner: str | None = None) -> go.Figure:
    """xG per time bucket, team above / opponent below the axis, with final-third share line."""
    ev = _rows(flags, "match_id", match_id)
    msg = _missing(ev, ["elapsed_min", "team_name", "xg", "final_third_touches"])
    if msg:
        return _empty(msg, title)
    teams = list(dict.fromkeys(ev.dropna(subset=["team_name"])["team_name"]))[:2]
    if len(teams) < 2:
        return _empty("Both teams are needed.", title)
    ev = ev.assign(b=(ev["elapsed_min"] // bucket) * bucket)
    agg = ev.groupby(["b", "team_name"])[["xg", "final_third_touches", "shots"]].sum().unstack(fill_value=0)
    bx = agg.index.to_numpy() + bucket / 2
    a_xg, b_xg = agg["xg"].get(teams[0], 0), agg["xg"].get(teams[1], 0)
    ft = agg["final_third_touches"]
    share = ft.get(teams[0], 0) / (ft.get(teams[0], 0) + ft.get(teams[1], 0)).replace(0, np.nan) * 100
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(x=bx, y=a_xg, name=f"{teams[0]} xG", marker_color=NAVY, width=bucket * 0.85,
                         hovertemplate="%{x:.0f}': %{y:.2f} xG<extra>" + teams[0] + "</extra>"))
    fig.add_trace(go.Bar(x=bx, y=-b_xg, name=f"{teams[1]} xG", marker_color=RED, width=bucket * 0.85,
                         customdata=b_xg, hovertemplate="%{x:.0f}': %{customdata:.2f} xG<extra>" + teams[1]
                                                         + "</extra>"))
    fig.add_trace(go.Scatter(x=bx, y=share, name=f"{teams[0]} share of final-third actions", mode="lines",
                             line=dict(color=AMBER, width=2, shape="spline"), opacity=0.9), secondary_y=True)
    top = max(float(np.max(a_xg)) if np.size(a_xg) else 0, float(np.max(b_xg)) if np.size(b_xg) else 0, 0.3)
    g = goals[goals["match_id"] == match_id] if goals is not None else pd.DataFrame()
    team_ids = ev.dropna(subset=["team_id"]).drop_duplicates("team_name").set_index("team_name")["team_id"]
    for _, r in g.iterrows():
        up = r["scoring_team_id"] == team_ids.get(teams[0])
        fig.add_trace(go.Scatter(x=[r["elapsed_min"]], y=[top * 1.15 if up else -top * 1.15], mode="markers+text",
                                 text=[str(r["player_name"]).split()[-1]], textposition="top center" if up else
                                 "bottom center", marker=dict(symbol="star", size=15, color=AMBER,
                                                              line=dict(color=INK, width=1)),
                                 showlegend=False, hovertemplate=f"Goal: {r['player_name']}<extra></extra>"))
    fig.add_vline(x=float(ev.loc[ev["period"] == 2, "elapsed_min"].min() or 45), line=dict(color=MUTED, dash="dot"))
    fig.update_yaxes(title_text="xG per bucket (opponent below)", range=[-top * 1.4, top * 1.4], secondary_y=False)
    fig.update_yaxes(title_text="final-third share %", range=[0, 100], secondary_y=True, showgrid=False)
    fig.update_layout(title=f"{title}<br><sup>{bucket:.0f}-minute buckets of elapsed playing time · dotted line = "
                            f"second half</sup>", barmode="relative", height=420, legend=dict(orientation="h", y=-0.2),
                      xaxis_title="Elapsed playing time (min)")
    return add_banner(fig, banner)


def phase_profile(flags: pd.DataFrame, goals: pd.DataFrame, team_stats: pd.DataFrame, team: str,
                  phases: list[dict], banner: str | None = None) -> go.Figure:
    """Goals and xG for/against by match phase over the season (diverging heatmap)."""
    ts = _rows(team_stats, "team_name", team)
    msg = _missing(ts, ["match_id", "team_id"]) or _missing(flags, ["period", "minute", "xg", "team_id"])
    if msg:
        return _empty(msg, f"Phase profile — {team}")
    tid = ts["team_id"].iloc[0]
    ev = flags[flags["match_id"].isin(ts["match_id"]) & flags["xg"].gt(0)].copy()
    ev["phase"] = [assign_band(p, m, phases) for p, m in zip(ev["period"], ev["minute"])]
    g = goals[goals["match_id"].isin(ts["match_id"])].copy()
    g["phase"] = [assign_band(p, m, phases) for p, m in zip(g["period"], g["minute"])]
    order = [p["label"] for p in phases]
    rows = {"Goals for": g[g["scoring_team_id"] == tid].groupby("phase").size(),
            "xG for": ev[ev["team_id"] == tid].groupby("phase")["xg"].sum(),
            "xG against": ev[ev["team_id"] != tid].groupby("phase")["xg"].sum(),
            "Goals against": g[g["scoring_team_id"] != tid].groupby("phase").size()}
    vals = pd.DataFrame({k: v.reindex(order, fill_value=0) for k, v in rows.items()}).T
    vals = vals.loc[:, vals.sum(axis=0) > 0]  # e.g. drop extra time when no match went to extra time
    order = list(vals.columns)
    signed = vals.copy()
    signed.loc[["xG against", "Goals against"]] *= -1
    scale = signed.abs().max(axis=1).replace(0, 1)
    fig = go.Figure(go.Heatmap(z=(signed.T / scale).T.values, x=order, y=list(vals.index), zmin=-1, zmax=1,
                               colorscale=DIV_SCALE, text=vals.round(1).values, texttemplate="%{text}",
                               textfont=dict(size=13), showscale=False, xgap=2, ygap=2,
                               hovertemplate="%{y}, %{x}: %{text}<extra></extra>"))
    fig.update_layout(title=f"{team}: when goals and chances happen (season totals by match phase; blue = for, "
                            f"red = against, intensity scaled per row)", height=330, plot_bgcolor="white",
                      margin=dict(l=120),
                      xaxis_title="Match phase (clock minutes, stoppage counted)")
    return add_banner(fig, banner)


def dominance_scatter(team_stats: pd.DataFrame, team: str, banner: str | None = None) -> go.Figure:
    """Each match as xG for vs xG against, coloured by result: how dominant were performances?"""
    t = _rows(team_stats, "team_name", team)
    msg = _missing(t, ["xg", "xg_against", "result", "opponent", "goals_for", "goals_against"])
    if msg:
        return _empty(msg, f"Match dominance — {team}")
    mx = float(max(t["xg"].max(), t["xg_against"].max(), 1)) * 1.1
    fig = go.Figure()
    fig.add_shape(type="path", path=f"M 0 0 L {mx} 0 L {mx} {mx} Z", fillcolor="rgba(42,157,143,0.07)",
                  line=dict(width=0), layer="below")
    fig.add_trace(go.Scatter(x=[0, mx], y=[0, mx], mode="lines", line=dict(color=MUTED, dash="dot"),
                             name="equal xG", hoverinfo="skip"))
    for res, g in t.groupby("result"):
        fig.add_trace(go.Scatter(
            x=g["xg"], y=g["xg_against"], mode="markers", name={"W": "Win", "D": "Draw", "L": "Loss"}[res],
            marker=dict(size=10 + 4 * g["goal_diff"].abs(), color=RESULT_COLORS[res], opacity=0.85,
                        line=dict(color="white", width=1)),
            customdata=np.stack([g["opponent"], g["goals_for"], g["goals_against"], g["match_date"].dt.strftime(
                "%d %b %Y")], axis=-1),
            hovertemplate="%{customdata[3]} vs %{customdata[0]}<br>%{customdata[1]}-%{customdata[2]}<br>"
                          "xG %{x:.2f} – %{y:.2f}<extra></extra>"))
    fig.add_annotation(x=mx * 0.85, y=mx * 0.1, text="out-created opponent", showarrow=False,
                       font=dict(color=TEAL, size=12))
    fig.add_annotation(x=mx * 0.15, y=mx * 0.9, text="out-created by opponent", showarrow=False,
                       font=dict(color=RED, size=12))
    share = (t["xg"] > t["xg_against"]).mean()
    fig.update_layout(title=f"{team}: match dominance — xG created vs conceded ({share:.0%} of matches above the "
                            f"line; size = goal margin)", xaxis=dict(title="xG for", range=[0, mx]),
                      yaxis=dict(title="xG against", range=[0, mx]), height=520)
    return add_banner(fig, banner)


def xg_distribution(flags: pd.DataFrame, team: str, match_ids: list, banner: str | None = None) -> go.Figure:
    """Shot-quality distribution for and against, with conversion annotations."""
    ok = flags is not None and {"type", "match_id"} <= set(flags.columns)
    sh = flags[(flags["type"] == "Shot") & flags["match_id"].isin(match_ids)] if ok else None
    msg = _missing(sh, ["shot_xg", "team_name", "outcome"])
    if msg:
        return _empty(msg, f"Shot quality — {team}")
    fig = go.Figure()
    notes = []
    for name, d, c in ((team, sh[sh["team_name"] == team], NAVY), ("Opponents", sh[sh["team_name"] != team], RED)):
        fig.add_trace(go.Histogram(x=d["shot_xg"], xbins=dict(start=0, end=1, size=0.04), name=f"{name} ({len(d)} "
                                   f"shots)", marker_color=c, opacity=0.65))
        g = int((d["outcome"] == "Goal").sum())
        notes.append(f"{name}: {g} goals from {d['shot_xg'].sum():.1f} xG, mean {d['shot_xg'].mean():.3f} xG/shot")
    fig.update_layout(barmode="overlay", title=f"Shot quality distribution<br><sup>{' · '.join(notes)}</sup>",
                      xaxis_title="StatsBomb xG per shot", yaxis_title="shots", height=400,
                      legend=dict(orientation="h", y=-0.2))
    return add_banner(fig, banner)


def team_style_radar(team_stats: pd.DataFrame, team: str, banner: str | None = None) -> go.Figure:
    """Team-average style indicators as percentiles among all teams in the dataset."""
    msg = _missing(_rows(team_stats, "team_name", team), ["possession_proxy_pct", "field_tilt_pct", "xg"])
    if msg:
        return _empty(msg, f"Team style — {team}")
    metrics = {"possession_proxy_pct": ("Possession proxy", 1), "field_tilt_pct": ("Field tilt", 1),
               "xg": ("xG for", 1), "shots": ("Shots", 1), "pass_completion_pct": ("Pass completion", 1),
               "progressive_passes": ("Progressive passes", 1), "pressures": ("Pressures", 1),
               "ppda": ("Pressing intensity (low PPDA)", -1), "xg_against": ("Low xG against", -1),
               "set_piece_shots": ("Set-piece shots", 1)}
    avg = team_stats.groupby("team_name")[list(metrics)].mean()
    pct = pd.DataFrame({m: (sign * avg[m]).rank(pct=True) * 100 for m, (_, sign) in metrics.items()})
    theta = [v[0] for v in metrics.values()]
    r = pct.loc[team].tolist()
    raw = avg.loc[team].tolist()
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=[50] * (len(r) + 1), theta=theta + theta[:1], name="median team",
                                  line=dict(color=MUTED, dash="dot"), hoverinfo="skip"))
    fig.add_trace(go.Scatterpolar(r=r + r[:1], theta=theta + theta[:1], fill="toself", name=team,
                                  line=dict(color=TEAL, width=2.5), fillcolor="rgba(42,157,143,0.25)",
                                  customdata=raw + raw[:1],
                                  hovertemplate="%{theta}: %{r:.0f}th pct (avg %{customdata:.2f})<extra></extra>"))
    fig.update_layout(title=f"{team}: style fingerprint vs the {len(avg)} teams in this dataset<br><sup>percentile "
                            f"of per-match averages; opponents are measured only in matches against the focus team"
                            f"</sup>", polar=dict(radialaxis=dict(range=[0, 100], gridcolor=GRID),
                                                  angularaxis=dict(gridcolor=GRID), bgcolor=PAPER),
                      height=520, legend=dict(orientation="h", y=-0.1))
    return add_banner(fig, banner)


def pass_direction_rose(passes: pd.DataFrame, title: str, sectors: int = 16,
                        banner: str | None = None) -> go.Figure:
    """Polar histogram of pass directions (forward = up), coloured by completion rate."""
    msg = _missing(passes, ["x", "y", "end_x", "end_y", "outcome"])
    if msg:
        return _empty(msg, title)
    p = passes.dropna(subset=["x", "y", "end_x", "end_y"])
    if p.empty:
        return _empty("No located passes.", title)
    ang = np.degrees(np.arctan2(p["end_y"] - p["y"], p["end_x"] - p["x"])) % 360
    width = 360 / sectors
    sec = ((ang + width / 2) // width % sectors).astype(int)
    df = pd.DataFrame({"sec": sec, "ok": p["outcome"].isna().to_numpy()})
    g = df.groupby("sec").agg(n=("ok", "size"), comp=("ok", "mean")).reindex(range(sectors), fill_value=0)
    fig = go.Figure(go.Barpolar(r=g["n"], theta=g.index * width, width=[width * 0.95] * sectors,
                                marker=dict(color=g["comp"] * 100, colorscale=[[0, RED], [0.5, AMBER], [1, TEAL]],
                                            cmin=50, cmax=100, colorbar=dict(title="completion %", thickness=12),
                                            line=dict(color="white", width=1)),
                                customdata=g["comp"] * 100,
                                hovertemplate="%{theta}°: %{r} passes, %{customdata:.0f}% completed<extra></extra>"))
    fig.update_layout(title=f"{title}<br><sup>{len(p)} passes · forward = up, right flank = right</sup>",
                      polar=dict(bgcolor=PAPER, angularaxis=dict(rotation=90, direction="clockwise",
                                                                  tickvals=[0, 90, 180, 270],
                                                                  ticktext=["Forward", "Right", "Back", "Left"],
                                                                  gridcolor=GRID),
                                 radialaxis=dict(gridcolor=GRID, showticklabels=False)),
                      height=480, showlegend=False)
    return add_banner(fig, banner)


def subs_team_band_heatmap(subs: pd.DataFrame, bands: list[dict], banner: str | None = None) -> go.Figure:
    """Team × timing band share of substitutions (sorted by median substitution minute)."""
    msg = _missing(subs, ["team_name", "timing_band", "minute"])
    if msg:
        return _empty(msg, "Substitution timing by team")
    order = [b["label"] for b in bands]
    ct = pd.crosstab(subs["team_name"], subs["timing_band"]).reindex(columns=order, fill_value=0)
    teams = subs.groupby("team_name")["minute"].median().sort_values().index
    ct = ct.reindex(teams)
    share = ct.div(ct.sum(axis=1), axis=0) * 100
    n = ct.sum(axis=1)
    fig = go.Figure(go.Heatmap(z=share.values, x=order, y=[f"{t} (n={n[t]})" for t in ct.index], colorscale=SEQ_SCALE,
                               zmin=0, text=ct.values, texttemplate="%{text}", xgap=2, ygap=2,
                               colorbar=dict(title="% of team's<br>subs", thickness=12),
                               hovertemplate="%{y}<br>%{x}: %{z:.0f}% (%{text} subs)<extra></extra>"))
    fig.update_layout(title="When each team substitutes (rows sorted earliest → latest median minute; numbers = "
                            "substitutions)", height=max(420, 26 * len(ct) + 160), margin=dict(l=230),
                      yaxis=dict(autorange="reversed"), plot_bgcolor="white")
    source_note(fig, "Opponents appear in 1-2 matches each and always faced the focus team — descriptive only.")
    return add_banner(fig, banner)


# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · dashboards — composite one-screen dashboards
# Composite one-screen dashboards (KPI cards + linked panels) for presentation and the demo.
#
# Each dashboard is a single Plotly figure built from the same tables as the individual
# charts, so every number is traceable. Missing inputs return an empty-state figure.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


KPI_NUM = dict(font=dict(size=30, color=NAVY))
KPI_TITLE = dict(font=dict(size=12, color=MUTED))


def _kpi(fig: go.Figure, value: float, title: str, row: int, col: int, fmt: str = ",.0f", prefix: str = "",
         suffix: str = "") -> None:
    if fmt.startswith("+"):  # the indicator ignores d3's sign flag: add "+" ourselves
        fmt = fmt[1:]
        if pd.notna(value) and value > 0:
            prefix = "+" + prefix
    fig.add_trace(go.Indicator(mode="number", value=float(value) if pd.notna(value) else None,
                               number=dict(valueformat=fmt, prefix=prefix, suffix=suffix, **KPI_NUM),
                               title=dict(text=title, **KPI_TITLE)), row=row, col=col)


# ---------------------------------------------------------------------------
# Season dashboard
# ---------------------------------------------------------------------------
def season_dashboard(team_stats: pd.DataFrame, xpts: pd.DataFrame, team: str, banner: str | None = None
                     ) -> go.Figure:
    """KPIs, points vs expected-points race, and match-by-match xG bars."""
    t = _rows(team_stats, "team_name", team)
    msg = _missing(t, ["match_date", "points", "goals_for", "goals_against", "xg", "xg_against", "result"])
    if msg:
        return _empty(msg, f"Season dashboard — {team}")
    t = t.sort_values("match_date").reset_index(drop=True)
    x = _rows(xpts, "team_name", team)
    x = x.sort_values("match_date") if x is not None and len(x) else None
    rec = t["result"].value_counts().reindex(["W", "D", "L"], fill_value=0)
    fig = make_subplots(rows=3, cols=5, row_heights=[0.16, 0.44, 0.40], vertical_spacing=0.1,
                        specs=[[{"type": "indicator"}] * 5, [{"colspan": 5}, None, None, None, None],
                               [{"colspan": 5, "secondary_y": True}, None, None, None, None]],
                        subplot_titles=["", "", "", "", "", "Points race: actual vs shot-based expected points",
                                        "Chance creation and concession per match"])
    _kpi(fig, t["points"].sum(), f"Points · W{rec['W']} D{rec['D']} L{rec['L']}", 1, 1)
    _kpi(fig, t["goals_for"].sum() - t["goals_against"].sum(),
         f"Goal difference ({t['goals_for'].sum()}-{t['goals_against'].sum()})", 1, 2, "+,.0f")
    _kpi(fig, t["xg"].mean() - t["xg_against"].mean(), "xG difference per match", 1, 3, "+.2f")
    _kpi(fig, t["possession_proxy_pct"].mean(), "Possession proxy", 1, 4, ".1f", suffix="%")
    if x is not None:
        _kpi(fig, t["points"].sum() - x["xpts"].sum(), f"Points vs xPts ({x['xpts'].sum():.1f})", 1, 5, "+.1f")
    else:
        _kpi(fig, t["field_tilt_pct"].mean(), "Field tilt", 1, 5, ".1f", suffix="%")
    md = np.arange(1, len(t) + 1)
    fig.add_trace(go.Scatter(x=md, y=t["points"].cumsum(), name="Actual points", mode="lines+markers",
                             line=dict(color=NAVY, width=3), marker=dict(color=t["result"].map(RESULT_COLORS), size=8,
                                                                         line=dict(color="white", width=1)),
                             customdata=np.stack([t["opponent"], t["goals_for"], t["goals_against"]], axis=-1),
                             hovertemplate="match %{x} vs %{customdata[0]} (%{customdata[1]}-%{customdata[2]})<br>"
                                           "%{y} points<extra></extra>"), row=2, col=1)
    if x is not None:
        fig.add_trace(go.Scatter(x=md, y=x["xpts"].cumsum().values, name="Expected points (xPts)", mode="lines",
                                 line=dict(color=AMBER, width=2.5, dash="dash"),
                                 hovertemplate="match %{x}: %{y:.1f} xPts<extra></extra>"), row=2, col=1)
    fig.add_trace(go.Scatter(x=md, y=md * 2, name="2 points/match pace", mode="lines",
                             line=dict(color=MUTED, width=1, dash="dot"), hoverinfo="skip"), row=2, col=1)
    fig.add_trace(go.Bar(x=md, y=t["xg"], name="xG for", marker_color=NAVY, opacity=0.85,
                         hovertemplate="match %{x}: %{y:.2f} xG for<extra></extra>"), row=3, col=1)
    fig.add_trace(go.Bar(x=md, y=-t["xg_against"], name="xG against", marker_color=RED, opacity=0.8,
                         customdata=t["xg_against"], hovertemplate="match %{x}: %{customdata:.2f} xG against"
                                                                   "<extra></extra>"), row=3, col=1)
    fig.add_trace(go.Scatter(x=md, y=t["goal_diff"], name="Goal difference", mode="markers",
                             marker=dict(symbol="diamond", size=9, color=AMBER, line=dict(color=INK, width=1))),
                  row=3, col=1, secondary_y=True)
    fig.update_xaxes(title_text="Match number (chronological)", row=3, col=1)
    fig.update_yaxes(title_text="cumulative points", row=2, col=1)
    fig.update_yaxes(title_text="xG (against below 0)", row=3, col=1, secondary_y=False)
    fig.update_yaxes(title_text="goal difference", row=3, col=1, secondary_y=True, showgrid=False)
    fig.update_layout(title=f"<b>{team}</b> — season dashboard ({len(t)} matches, "
                            f"{t['match_date'].min():%b %Y} – {t['match_date'].max():%b %Y})",
                      barmode="relative", height=900, legend=dict(orientation="h", y=-0.06),
                      margin=dict(t=110))
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# Player dashboard
# ---------------------------------------------------------------------------
def player_dashboard(season_pct: pd.DataFrame, pms: pd.DataFrame, flags: pd.DataFrame, player_id: Any,
                     config: Config, banner: str | None = None) -> go.Figure:
    """Player card: KPIs, percentile radar, action heatmap and per-match output."""
    s = _rows(season_pct, "player_id", player_id)
    msg = _missing(s, ["player_name", "minutes_played", "primary_role"])
    if msg:
        return _empty(msg, "Player dashboard")
    r = s.sort_values("minutes_played", ascending=False).iloc[0]
    games = pms[pms["player_id"] == player_id].sort_values("match_date")
    fig = make_subplots(rows=3, cols=4, row_heights=[0.14, 0.50, 0.36], vertical_spacing=0.09,
                        horizontal_spacing=0.06,
                        specs=[[{"type": "indicator"}] * 4,
                               [{"type": "polar", "colspan": 2}, None, {"colspan": 2}, None],
                               [{"colspan": 4, "secondary_y": True}, None, None, None]],
                        subplot_titles=["", "", "", "", "Percentile profile vs benchmark", "Where he acts (on-ball "
                                        "actions)", "Match by match: minutes and attacking output"])
    _kpi(fig, r["minutes_played"], f"Minutes · {int(r['starts'])} starts / {int(r['appearances'])} apps", 1, 1)
    _kpi(fig, r["goals"], f"Goals (xG {r['xg']:.1f})", 1, 2)
    _kpi(fig, r["assists"], f"Assists (xA {r['xa']:.1f})", 1, 3)
    _kpi(fig, r.get("key_passes_p90", np.nan), "Key passes per 90", 1, 4, ".2f")
    metrics = [m for m in radar_metrics(r["primary_role"], config) if f"{m}_pct" in s.columns]
    theta = [label(m) for m in metrics]
    vals = [float(r[f"{m}_pct"]) if pd.notna(r[f"{m}_pct"]) else 0.0 for m in metrics]
    fig.add_trace(go.Scatterpolar(r=[50] * (len(vals) + 1), theta=theta + theta[:1], showlegend=False,
                                  line=dict(color=MUTED, dash="dot"), hoverinfo="skip"), row=2, col=1)
    fig.add_trace(go.Scatterpolar(r=vals + vals[:1], theta=theta + theta[:1], fill="toself", showlegend=False,
                                  line=dict(color=NAVY, width=2.5), fillcolor="rgba(29,53,87,0.25)",
                                  hovertemplate="%{theta}: %{r:.0f}th pct<extra></extra>"), row=2, col=1)
    ev = flags[(flags["player_id"] == player_id) & flags["touches"].astype(bool)]
    if len(ev):
        fig.add_trace(heatmap_layer(ev, (12, 8), showscale=False), row=2, col=3)
    draw_pitch(fig, row=2, col=3, lines_on_top=True)
    if len(games):
        lab = [f"{d:%d %b} {o[:10]}" for d, o in zip(games["match_date"], games["opponent"])]
        fig.add_trace(go.Bar(x=lab, y=games["minutes_played"], name="Minutes", marker_color="#d9d4c7",
                             hovertemplate="%{x}: %{y:.0f} min<extra></extra>"), row=3, col=1)
        fig.add_trace(go.Scatter(x=lab, y=games["xg"] + games["xa"], name="xG + xA", mode="lines+markers",
                                 line=dict(color=TEAL, width=2.5),
                                 hovertemplate="%{x}: %{y:.2f} xG+xA<extra></extra>"), row=3, col=1, secondary_y=True)
        g = games[games["goals"] > 0]
        fig.add_trace(go.Scatter(x=[f"{d:%d %b} {o[:10]}" for d, o in zip(g["match_date"], g["opponent"])],
                                 y=g["xg"] + g["xa"], mode="markers", name="Scored", marker=dict(
                                     symbol="star", size=13, color=AMBER, line=dict(color=INK, width=1))),
                      row=3, col=1, secondary_y=True)
    fig.update_xaxes(tickangle=-60, tickfont=dict(size=8), row=3, col=1)
    fig.update_yaxes(title_text="minutes", row=3, col=1, secondary_y=False)
    fig.update_yaxes(title_text="xG + xA", row=3, col=1, secondary_y=True, showgrid=False)
    fig.update_layout(polar=dict(radialaxis=dict(range=[0, 100], tickvals=[25, 50, 75], gridcolor=GRID),
                                 angularaxis=dict(gridcolor=GRID, tickfont=dict(size=10)), bgcolor=PAPER),
                      title=f"<b>{r['player_name']}</b> — {r['team_name']} · {r.get('primary_position')} "
                            f"({ROLE_LABEL.get(r['primary_role'], r['primary_role'])}) · {r['coverage_label']} · "
                            f"benchmark {r.get('benchmark_group')} (n={int(r['benchmark_n'])})",
                      height=980, legend=dict(orientation="h", y=-0.12), margin=dict(t=110))
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# Match dashboard
# ---------------------------------------------------------------------------
def match_dashboard(flags: pd.DataFrame, goals: pd.DataFrame, matches: pd.DataFrame, match_id: Any,
                    bucket: float = 5.0, banner: str | None = None) -> go.Figure:
    """Score header, momentum, cumulative xG and both teams' shot maps for one match."""
    m = _rows(matches, "match_id", match_id)
    ev = _rows(flags, "match_id", match_id)
    msg = _missing(m, ["home_team", "away_team", "home_score", "away_score"]) or _missing(
        ev, ["elapsed_min", "team_name", "xg"])
    if msg:
        return _empty(msg, "Match dashboard")
    m = m.iloc[0]
    home, away = m["home_team"], m["away_team"]
    fig = make_subplots(rows=3, cols=2, row_heights=[0.3, 0.3, 0.4], vertical_spacing=0.1, horizontal_spacing=0.05,
                        specs=[[{"colspan": 2}, None], [{"colspan": 2}, None], [{}, {}]],
                        subplot_titles=[f"Momentum: xG per {bucket:.0f} min ({home} up, {away} down)",
                                        "Cumulative xG", f"{home} shots", f"{away} shots"])
    ev = ev.assign(b=(ev["elapsed_min"] // bucket) * bucket)
    agg = ev.groupby(["b", "team_name"])["xg"].sum().unstack(fill_value=0)
    bx = agg.index.to_numpy() + bucket / 2
    for team, sign, c in ((home, 1, NAVY), (away, -1, RED)):
        y = agg[team] if team in agg else pd.Series(0, index=agg.index)
        fig.add_trace(go.Bar(x=bx, y=sign * y, marker_color=c, name=team, width=bucket * 0.85, customdata=y,
                             hovertemplate="%{x:.0f}': %{customdata:.2f} xG<extra>" + team + "</extra>"), row=1, col=1)
    shots = ev[ev["type"] == "Shot"].sort_values("elapsed_min")
    for team, c in ((home, NAVY), (away, RED)):
        s = shots[shots["team_name"] == team]
        fig.add_trace(go.Scatter(x=np.r_[0, s["elapsed_min"]], y=np.r_[0, s["shot_xg"].fillna(0).cumsum()],
                                 mode="lines", line_shape="hv", line=dict(color=c, width=2.5), showlegend=False,
                                 hovertemplate="%{x:.0f}': %{y:.2f} cumulative xG<extra>" + team + "</extra>"),
                      row=2, col=1)
        gl = s[s["outcome"] == "Goal"]
        fig.add_trace(go.Scatter(x=gl["elapsed_min"], y=s["shot_xg"].fillna(0).cumsum()[gl.index], mode="markers",
                                 marker=dict(symbol="star", size=14, color=AMBER, line=dict(color=INK, width=1)),
                                 text=gl["player_name"], hovertemplate="Goal: %{text}<extra></extra>",
                                 showlegend=False), row=2, col=1)
    for col, team in ((1, home), (2, away)):
        draw_pitch(fig, row=3, col=col, half=True)
        s = shots[shots["team_name"] == team].dropna(subset=["x", "y"])
        fig.add_trace(go.Scatter(x=s["x"], y=s["y"], mode="markers", showlegend=False,
                                 marker=dict(size=6 + 40 * s["shot_xg"].fillna(0.02),
                                             color=[SHOT_COLORS.get(o, SLATE) for o in s["outcome"]],
                                             symbol=["star" if o == "Goal" else "circle" for o in s["outcome"]],
                                             line=dict(color="white", width=1), opacity=0.9),
                                 customdata=np.stack([s["player_name"], s["shot_xg"].round(3), s["outcome"]], axis=-1)
                                 if len(s) else None,
                                 hovertemplate="%{customdata[0]}<br>xG %{customdata[1]} · %{customdata[2]}"
                                               "<extra></extra>"), row=3, col=col)
    xg = ev.groupby("team_name")["xg"].sum()
    fig.update_xaxes(title_text="Elapsed playing time (min)", row=2, col=1)
    fig.update_layout(title=f"<b>{home} {int(m['home_score'])} – {int(m['away_score'])} {away}</b> · "
                            f"{m['match_date']:%d %b %Y} · xG {xg.get(home, 0):.2f} – {xg.get(away, 0):.2f}"
                            + (f" · {m['stadium']}" if pd.notna(m.get("stadium")) else ""),
                      barmode="relative", height=1080, legend=dict(orientation="h", y=1.04, x=1, xanchor="right"),
                      margin=dict(t=120))
    return add_banner(fig, banner)


# ---------------------------------------------------------------------------
# Substitution dashboard
# ---------------------------------------------------------------------------
def substitution_dashboard(subs: pd.DataFrame, team: str, bands: list[dict], banner: str | None = None
                           ) -> go.Figure:
    """KPIs, timing × game state, most-used substitutes, and every change on a minute × score grid."""
    s = _rows(subs, "team_name", team)
    msg = _missing(s, ["minute", "timing_band", "game_state", "player_on_name", "goal_diff", "match_id"])
    if msg:
        return _empty(msg, f"Substitution dashboard — {team}")
    order = [b["label"] for b in bands]
    first = s.sort_values("index").groupby("match_id").first() if "index" in s else s.groupby("match_id").first()
    fig = make_subplots(rows=3, cols=4, row_heights=[0.16, 0.42, 0.42], vertical_spacing=0.11,
                        horizontal_spacing=0.08,
                        specs=[[{"type": "indicator"}] * 4, [{"colspan": 2}, None, {"colspan": 2}, None],
                               [{"colspan": 4}, None, None, None]],
                        subplot_titles=["", "", "", "", "Timing band × game state", "Most-used substitutes",
                                        "Every substitution: minute vs score difference at the change"])
    _kpi(fig, len(s) / s["match_id"].nunique(), "Substitutions per match", 1, 1, ".2f")
    _kpi(fig, s["minute"].median(), "Median substitution minute", 1, 2, ".0f", suffix="'")
    _kpi(fig, first["minute"].median(), "Median first substitution", 1, 3, ".0f", suffix="'")
    _kpi(fig, 100 * s["like_for_like"].mean() if "like_for_like" in s else np.nan, "Like-for-like role changes",
         1, 4, ".0f", suffix="%")
    for st in ["leading", "drawing", "trailing"]:
        c = s[s["game_state"] == st]["timing_band"].value_counts().reindex(order, fill_value=0)
        fig.add_trace(go.Bar(x=order, y=c.values, name=st, marker_color=STATE_COLORS[st],
                             hovertemplate="%{x}: %{y} subs while " + st + "<extra></extra>"), row=2, col=1)
    on_col = "player_on_display" if "player_on_display" in s else "player_on_name"
    off_col = "player_off_display" if "player_off_display" in s else "player_off_name"
    top = s[on_col].value_counts().head(10).iloc[::-1]
    fig.add_trace(go.Bar(y=list(top.index), x=top.values, orientation="h",
                         marker_color=TEAL, showlegend=False, text=top.values, textposition="outside",
                         customdata=top.index, hovertemplate="%{customdata}: %{x} times<extra></extra>"),
                  row=2, col=3)
    rng = np.random.default_rng(0)
    jit = rng.uniform(-0.18, 0.18, len(s))
    fig.add_trace(go.Scatter(x=s["minute"], y=s["goal_diff"] + jit, mode="markers", showlegend=False,
                             marker=dict(size=9, color=s["game_state"].map(STATE_COLORS), opacity=0.75,
                                         symbol=np.where(s.get("sub_reason", "") == "Injury", "x", "circle"),
                                         line=dict(color="white", width=0.5)),
                             customdata=np.stack([s[off_col], s[on_col],
                                                  s.get("sub_reason", pd.Series("", index=s.index))], axis=-1),
                             hovertemplate="%{x}': OFF %{customdata[0]} → ON %{customdata[1]} (%{customdata[2]})"
                                           "<extra></extra>"), row=3, col=1)
    fig.update_yaxes(title_text="goal difference (team view)", dtick=1, row=3, col=1)
    fig.update_xaxes(title_text="clock minute (× = injury change)", row=3, col=1)
    fig.update_layout(title=f"<b>{team}</b> — substitution dashboard ({len(s)} changes in "
                            f"{s['match_id'].nunique()} matches)", barmode="stack", height=950,
                      legend=dict(orientation="v", x=0.01, y=0.72, yanchor="top", bgcolor="rgba(255,255,255,0.8)"),
                      margin=dict(t=110))
    return add_banner(fig, banner)

In [14]:
xpts = expected_points(B.flags, B.team_stats, n_sims=5000, seed=SEED)
season_fig = season_dashboard(B.team_stats, xpts, FOCUS, banner=BANNER)
season_fig.show()
fx = xpts[xpts.team_name == FOCUS]
display(Markdown(f"**Observation (computed):** {FOCUS} collected **{fx.points.sum():.0f} points** against "
                 f"**{fx.xpts.sum():.1f} expected points** from shot quality "
                 f"({fx.points.sum() - fx.xpts.sum():+.1f}); the gap can reflect finishing, goalkeeping, game-state "
                 "management and chance — it is not a measure of luck alone."))

eda_metrics = ["xg_p90", "key_passes_p90", "progressive_passes_p90", "progressive_carries_p90", "pressures_p90",
               "interceptions_p90", "tackles_won_p90", "pass_completion_pct", "box_touches_p90", "aerials_won_p90"]
elig = B.season_pct[B.season_pct.eligible]
feature_distributions(elig, eda_metrics, title=f"Feature distributions (eligible players, n={len(elig)})",
                          banner=BANNER).show()
role_distribution(B.season_pct, eda_metrics, cfg.min_minutes, banner=BANNER).show()

rd = elig.groupby("role_family")[["xg_p90", "progressive_passes_p90", "interceptions_p90", "pass_completion_pct",
                                   "pressures_p90"]].median().round(2)
rd.insert(0, "players", elig.groupby("role_family").size())
display(Markdown("**Observation (computed): median per-90 profile by role family**"))
display(rd)
top_role = rd["xg_p90"].idxmax(); top_int = rd["interceptions_p90"].idxmax()
display(Markdown(f"In this sample, **{top_role}s** have the highest median xG/90 ({rd.loc[top_role, 'xg_p90']}) and "
                 f"**{top_int}s** the highest median interceptions/90 ({rd.loc[top_int, 'interceptions_p90']}) — "
                 "which is why raw totals are never compared across roles."))
correlation_heatmap(elig, eda_metrics, banner=BANNER).show()

display(Markdown("### Distributions and relationships, more ways\nScatter-plot matrix (drag to select across panels), marginal distributions, an ECDF of playing time, split violins and a correlation matrix reordered by hierarchical clustering."))
metric_splom(B.season_pct, ["xg_p90", "key_passes_p90", "progressive_passes_p90", "pressures_p90", "interceptions_p90",
                            "pass_completion_pct"], cfg.min_minutes).show()
marginal_scatter(B.season_pct, "pressures_p90", "ball_recoveries_p90", cfg.min_minutes).show()
minutes_ecdf(B.season, FOCUS).show()
split_violin(B.season_pct, "progressive_passes_p90", FOCUS, cfg.min_minutes).show()
clustered_correlation(B.season_pct, eda_metrics + ["xa_p90", "dribbles_completed_p90", "ball_recoveries_p90",
                                                   "clearances_p90"], cfg.min_minutes).show()

results_overview(B.team_stats, FOCUS, banner=BANNER).show()
team_trend(B.team_stats, FOCUS, banner=BANNER).show()

focus_ids = list(B.team_stats[B.team_stats.team_name == FOCUS].match_id)
dominance_scatter(B.team_stats, FOCUS, banner=BANNER).show()
phase_profile(B.flags, T["goals"], B.team_stats, FOCUS, cfg.match_phases, banner=BANNER).show()
xg_distribution(B.flags, FOCUS, focus_ids, banner=BANNER).show()
team_style_radar(B.team_stats, FOCUS, banner=BANNER).show()

display(Markdown("### The season in shapes\nGauges, a funnel, a waterfall, a calendar, areas, bands, dumbbells, a polar clock, boxes and opponent views — the same season seen through different chart grammars."))
focus_tid = B.team_stats[B.team_stats.team_name == FOCUS].team_id.iloc[0]
kpi_gauges(B.team_stats, FOCUS).show()
attack_funnel(B.flags, focus_tid, FOCUS).show()
monthly_waterfall(B.team_stats, FOCUS).show()
calendar_heatmap(B.team_stats, FOCUS).show()
cumulative_area(B.team_stats, FOCUS).show()
rolling_band(B.team_stats, FOCUS).show()
xg_dumbbell(B.team_stats, FOCUS).show()
goal_clock(T["goals"], B.team_stats, FOCUS).show()
venue_boxes(B.team_stats, FOCUS).show()
opponent_matrix(B.team_stats, FOCUS).show()
opponent_bubbles(B.team_stats, FOCUS).show()

# Match timeline and cumulative xG for the focus team's highest-xG match
fm = B.team_stats[B.team_stats.team_name == FOCUS].sort_values("xg", ascending=False).iloc[0]
mrow = T["matches"].set_index("match_id").loc[fm.match_id]
MATCH_LABEL = f"{mrow.match_date:%d %b %Y}: {mrow.home_team} {mrow.home_score}-{mrow.away_score} {mrow.away_team}"
match_timeline(match_timeline_events(T["events"], T["goals"], fm.match_id), title=f"Timeline — {MATCH_LABEL}",
                   banner=BANNER).show()
cumulative_xg_chart(cumulative_xg(B.flags, fm.match_id), title=f"Cumulative xG — {MATCH_LABEL}",
                        banner=BANNER).show()

match_fig = match_dashboard(B.flags, T["goals"], T["matches"], fm.match_id, banner=BANNER)
match_fig.show()
match_momentum(B.flags, T["goals"], fm.match_id, title=f"Momentum — {MATCH_LABEL}", banner=BANNER).show()

**Observation (computed):** Bayer Leverkusen collected **90 points** against **74.8 expected points** from shot quality (+15.2); the gap can reflect finishing, goalkeeping, game-state management and chance — it is not a measure of luck alone.

**Observation (computed): median per-90 profile by role family**

,players,xg_p90,progressive_passes_p90,interceptions_p90,pass_completion_pct,pressures_p90
role_family,,,,,,
Defender,38,0.02,1.83,1.40,84.44,11.91
Forward,10,0.32,0.60,0.08,74.81,14.45
Goalkeeper,14,0.00,0.00,0.00,74.78,0.00
Midfielder,22,0.03,1.39,0.98,84.83,18.06


In this sample, **Forwards** have the highest median xG/90 (0.32) and **Defenders** the highest median interceptions/90 (1.4) — which is why raw totals are never compared across roles.

### Distributions and relationships, more ways
Scatter-plot matrix (drag to select across panels), marginal distributions, an ECDF of playing time, split violins and a correlation matrix reordered by hierarchical clustering.

### The season in shapes
Gauges, a funnel, a waterfall, a calendar, areas, bands, dumbbells, a polar clock, boxes and opponent views — the same season seen through different chart grammars.

## 9 · Player intelligence centre
Role-aware leaderboards and comparisons (use the dropdowns inside each chart — they work without a running kernel too),
then a profile of one player against his benchmark group. Percentiles are relative to the eligible pool of *this
dataset*; with one club's season, opponents contribute only 1-2 matches each, so benchmarks for the focus squad are
partly internal (the radar title states the benchmark group and its size).

In [15]:
lb_metrics = ["xg_p90", "npxg_p90", "xa_p90", "key_passes_p90", "progressive_passes_p90", "progressive_carries_p90",
              "dribbles_completed_p90", "box_touches_p90", "pressures_p90", "interceptions_p90", "tackles_won_p90",
              "ball_recoveries_p90", "aerials_won_p90"]
leaderboard_chart(B.season_pct, lb_metrics, min_minutes=max(cfg.min_minutes, 450), banner=BANNER).show()
comparison_scatter(B.season_pct, lb_metrics + ["pass_completion_pct"], cfg.min_minutes, x="xg_p90",
                       y="xa_p90", highlight_team=FOCUS, banner=BANNER).show()

display(Markdown('### Finishing, goal involvement and chance creation\nGoals against expected goals, who is involved in goals, who creates the chances, and an animated season race (press ▶).'))

finishing_scatter(B.season_pct, max(cfg.min_minutes, 450), highlight_team=FOCUS).show()
goal_involvement_bars(B.season_pct, FOCUS).show()
chance_treemap(B.season_pct, FOCUS).show()
involvement_race(B.pms, FOCUS).show()

DEFAULT_PLAYER = "Florian Wirtz"
fs = B.season_pct[(B.season_pct.team_name == FOCUS)]
hit = fs[fs.player_name.str.contains(DEFAULT_PLAYER, case=False, na=False)]
P = (hit if len(hit) else fs[fs.primary_role != "GK"]).sort_values("minutes_played", ascending=False).iloc[0]

def player_view(player_id, team_id=None):
    prof = player_profile(B.season_pct, player_id, team_id)
    display(pd.DataFrame([prof]))
    row = B.season_pct[B.season_pct.player_id == player_id].iloc[0]
    player_dashboard(B.season_pct, B.pms, B.flags, player_id, cfg, banner=BANNER).show()
    display(Markdown("**Observed relative strengths / weaknesses** (percentiles of role-fit metrics; not a rating)"))
    display(strengths_weaknesses(B.season_pct, player_id, cfg, team_id))
    metrics = [m for m in radar_metrics(row.primary_role, cfg) if m.endswith("_p90")][:3]
    form_trend(rolling_form_series(B.pms, player_id, metrics, cfg.form_window), metrics, row.player_name,
                   cfg.form_window, banner=BANNER).show()

player_view(P.player_id, P.team_id)

if WIDGETS_OK and not HEADLESS:
    opts = [(f"{r.player_name} ({r.primary_position}, {r.minutes_played:.0f} min)", r.player_id)
            for r in B.season_pct[B.season_pct.team_name == FOCUS].sort_values("minutes_played", ascending=False).itertuples()]
    W.interact(lambda player: player_view(player), player=W.Dropdown(options=opts, value=P.player_id,
                                                                     layout=W.Layout(width="480px")))
else:
    print("Live player selector available when running interactively (Colab/Jupyter).")

# Head-to-head radar with the next most-used team-mate in the same role, plus spatial / directional profile
mates = fs[(fs.primary_role == P.primary_role) & (fs.player_id != P.player_id)].sort_values("minutes_played",
                                                                                          ascending=False)
player_radar(B.season_pct, P.player_id, radar_metrics(P.primary_role, cfg),
                 compare_id=mates.player_id.iloc[0] if len(mates) else None, banner=BANNER).show()
touch = B.flags[(B.flags.player_id == P.player_id) & B.flags.touches.astype(bool)]
action_heatmap(touch, f"{P.player_name}: where he receives and uses the ball (season)", banner=BANNER).show()
passes_p = B.flags[(B.flags.player_id == P.player_id) & (B.flags.type == "Pass")]
pass_direction_rose(passes_p, f"{P.player_name}: pass directions and completion", banner=BANNER).show()

display(Markdown("### Squad-level views\nThe squad matrix shows every eligible outfield player's percentile on each metric (blue = high, red = low) — a\none-glance scouting sheet. The sunburst shows how playing time is spread across role families, roles and players\n(click a segment to zoom)."))

squad_metrics = ["xg_p90", "xa_p90", "key_passes_p90", "progressive_passes_p90", "progressive_carries_p90",
                 "dribbles_completed_p90", "box_touches_p90", "pressures_p90", "interceptions_p90", "tackles_won_p90",
                 "ball_recoveries_p90", "aerials_won_p90", "pass_completion_pct"]
squad_percentile_heatmap(B.season_pct, FOCUS, squad_metrics, max(cfg.min_minutes, 450), banner=BANNER).show()
minutes_sunburst(B.season_pct, FOCUS, banner=BANNER).show()
workload_bars(B.season_pct, FOCUS, banner=BANNER).show()

display(Markdown('### Player archetypes (unsupervised)\nPer-90 profiles of outfield players above the minutes threshold are standardised and clustered with k-means; *k* is chosen by silhouette score and the result is projected onto two principal components (arrows show which metrics pull players in each direction). Archetypes describe playing styles in this sample — they are not quality rankings, and opponents contribute only one or two matches each.'))

arch_metrics = ["xg_p90", "xa_p90", "key_passes_p90", "progressive_passes_p90",
                "progressive_carries_p90", "dribbles_completed_p90", "box_touches_p90", "pressures_p90",
                "interceptions_p90", "tackles_won_p90", "ball_recoveries_p90", "aerials_won_p90",
                "clearances_p90", "pass_completion_pct"]
arch = player_archetypes(B.season_pct, arch_metrics, cfg.min_minutes, seed=SEED)
archetype_map(arch, FOCUS).show()
archetype_profile_heatmap(arch).show()
if arch:
    display(arch['frame'][arch['frame'].team_name == FOCUS][['player_name', 'primary_position', 'archetype']]
            .sort_values('archetype').reset_index(drop=True))
role_parallel(B.season_pct, "Midfielder", squad_metrics, cfg.min_minutes, FOCUS).show()

display(Markdown("### More player views\nLollipop ranking, first-half vs second-half slope chart, a grid of radars, a ternary plot of every player's action mix, a squad similarity tree, a player × match chance matrix and a pass-destination density map."))
lollipop(B.season_pct, "progressive_carries_p90", 450).show()
slope_chart(B.pms, FOCUS, "progressive_passes").show()
radar_grid(B.season_pct, list(fs[fs.primary_role != "GK"].nlargest(6, "minutes_played").player_id), cfg).show()
action_ternary(B.season_pct, cfg.min_minutes, FOCUS).show()
player_dendrogram(B.season_pct, FOCUS, squad_metrics, 450).show()
contribution_matrix(B.pms, T["matches"], FOCUS).show()
pass_destination_density(B.flags, P.player_id, f"{P.player_name}: where his completed passes end").show()

### Finishing, goal involvement and chance creation
Goals against expected goals, who is involved in goals, who creates the chances, and an animated season race (press ▶).

,player,team,primary_position,role,appearances,starts,minutes,goals,xG,assists,xA,coverage,benchmark
0,Florian Wirtz,Bayer Leverkusen,Left Attacking Midfield,Attacking midfielder,32,26,2529.0,11,8.47,10,7.73,high data coverage,"Midfielder (role family, n=22)"


**Observed relative strengths / weaknesses** (percentiles of role-fit metrics; not a rating)

,type,metric,value,percentile
0,relative strength,Prog. carries /90,4.98,98.0
1,relative strength,Completed dribbles /90,2.46,98.0
2,relative strength,Key passes /90,2.53,93.0
3,relative weakness,xA /90,0.28,93.0
4,relative weakness,Box touches /90,15.94,93.0
5,relative weakness,Shots /90,2.53,93.0


Live player selector available when running interactively (Colab/Jupyter).


### Squad-level views
The squad matrix shows every eligible outfield player's percentile on each metric (blue = high, red = low) — a
one-glance scouting sheet. The sunburst shows how playing time is spread across role families, roles and players
(click a segment to zoom).

### Player archetypes (unsupervised)
Per-90 profiles of outfield players above the minutes threshold are standardised and clustered with k-means; *k* is chosen by silhouette score and the result is projected onto two principal components (arrows show which metrics pull players in each direction). Archetypes describe playing styles in this sample — they are not quality rankings, and opponents contribute only one or two matches each.

,player_name,primary_position,archetype
0,Granit Xhaka,Right Defensive Midfield,A1: high Interceptions /90 & Clearances /90
1,Josip Stanišić,Right Center Back,A1: high Interceptions /90 & Clearances /90
2,Piero Hincapié,Left Center Back,A1: high Interceptions /90 & Clearances /90
3,Exequiel Palacios,Left Defensive Midfield,A1: high Interceptions /90 & Clearances /90
4,Odilon Kossonou,Right Center Back,A1: high Interceptions /90 & Clearances /90
5,Edmond Fayçal Tapsoba,Left Center Back,A1: high Interceptions /90 & Clearances /90
6,Robert Andrich,Left Defensive Midfield,A1: high Interceptions /90 & Clearances /90
7,Jonathan Tah,Center Back,A1: high Interceptions /90 & Clearances /90
8,Borja Iglesias,Center Forward,A2: high xG /90 & Aerials won /90
9,Patrik Schick,Center Forward,A2: high xG /90 & Aerials won /90


### More player views
Lollipop ranking, first-half vs second-half slope chart, a grid of radars, a ternary plot of every player's action mix, a squad similarity tree, a player × match chance matrix and a pass-destination density map.

## 10 · Team and tactical intelligence
Formation use and outcomes (descriptive — opponent strength and squad availability confound them), team style,
and spatial views on a 120 × 80 pitch (attacking left → right for the team in question).

In [16]:
fsum = formation_summary(B.team_stats, FOCUS)
display(fsum)
formation_frequency(fsum, FOCUS, banner=BANNER).show()
formation_comparison(fsum, FOCUS, banner=BANNER).show()

shots = B.flags[(B.flags.team_name == FOCUS) & (B.flags.type == "Shot")]
shot_map(shots, f"{FOCUS}: all shots, season", banner=BANNER).show()
opp_shots = B.flags[(B.flags.match_id.isin(B.team_stats[B.team_stats.team_name == FOCUS].match_id))
                    & (B.flags.team_name != FOCUS) & (B.flags.type == "Shot")]
shot_map(opp_shots, f"Opponents' shots against {FOCUS}", banner=BANNER).show()

display(Markdown("### Chance creation and ball circulation\nKey passes (every pass that led to a shot), how completed passes flow between nine pitch zones, and the anatomy of the team's shots."))

team_id_focus = B.team_stats[B.team_stats.team_name == FOCUS].team_id.iloc[0]
key_pass_map(B.flags, team_id_focus, f"{FOCUS}: key passes (season)").show()
pass_flow_matrix(B.flags, team_id_focus, f"{FOCUS}: pass flow between zones").show()
shot_sunburst(B.flags, FOCUS).show()

shot_density_pair(B.flags, team_id_focus, focus_ids, FOCUS).show()

pp = B.flags[(B.flags.match_id == fm.match_id) & (B.flags.player_id == P.player_id) & (B.flags.type == "Pass")]
pass_map(pp, f"{P.player_name}: passes — {MATCH_LABEL}", banner=BANNER).show()
team_id_focus = B.team_stats[B.team_stats.team_name == FOCUS].team_id.iloc[0]
nodes, edges, cut = pass_network(B.flags, T["appearances"], fm.match_id, team_id_focus)
pass_network_chart(nodes, edges, f"{FOCUS} pass network — {MATCH_LABEL}", cut, banner=BANNER).show()
prog = B.flags[(B.flags.player_id == P.player_id)]
progressive_actions_map(prog, f"{P.player_name}: progressive passes and carries (season)", banner=BANNER).show()

display(Markdown("### Where the team presses and defends\nZone heatmaps of the focus team's pressures and ball-winning actions (tackles, interceptions, recoveries, blocks,\nclearances) across the season, in the team's own attacking direction — pressing high shows up as heat in the\nopponent half. The third map shows where opponents managed to have the ball against the focus team."))

team_ev = B.flags[B.flags.team_id == team_id_focus]
action_heatmap(team_ev[team_ev.type == "Pressure"], f"{FOCUS}: pressures (season)", banner=BANNER).show()
action_heatmap(team_ev[team_ev.defensive_actions.astype(bool)], f"{FOCUS}: ball-winning and defensive actions",
                  banner=BANNER).show()
opp_ev = B.flags[B.flags.match_id.isin(focus_ids) & (B.flags.team_id != team_id_focus)]
action_heatmap(opp_ev[opp_ev.touches.astype(bool)], f"Opponents' on-ball actions against {FOCUS}",
                  banner=BANNER).show()
press_high = (team_ev[team_ev.type == "Pressure"].x >= 60).mean()
display(Markdown(f"**Observation (computed):** {press_high:.0%} of {FOCUS}'s pressures happen in the opponent half."))

,starting_formation,matches,wins,draws,losses,points_per_match,xg_for,xg_against,goals_for,goals_against,possession_proxy_pct,field_tilt_pct,sample_warning
0,3-4-2-1,25,23,2,0,2.84,2.05,0.74,2.68,0.64,62.78,68.29,
1,3-4-3,8,4,4,0,2.00,2.44,1.10,2.50,0.88,58.40,66.92,
2,3-4-1-2,1,1,0,0,3.00,1.82,0.46,2.00,1.00,61.07,56.02,n<3: descriptive only


### Chance creation and ball circulation
Key passes (every pass that led to a shot), how completed passes flow between nine pitch zones, and the anatomy of the team's shots.

### Where the team presses and defends
Zone heatmaps of the focus team's pressures and ball-winning actions (tackles, interceptions, recoveries, blocks,
clearances) across the season, in the team's own attacking direction — pressing high shows up as heat in the
opponent half. The third map shows where opponents managed to have the ball against the focus team.

**Observation (computed):** 57% of Bayer Leverkusen's pressures happen in the opponent half.

## 11 · Starting XI decision support
**Decision time:** before kick-off of a selected historical match. **Information used:** only matches *before*
that date. **Candidate pool:** that match's announced matchday squad.

**Score per (player, slot)** = Σ wᵢ·componentᵢ (+2 side-preference tie-breaker) × position compatibility, where the
components (0-100) are: role fit (weighted percentiles of the slot role's metrics), recent form (same composite over
the last 5 appearances; season value if none), sample reliability (minutes/900), workload suitability (penalises
>180 min in the previous 7 days or <3 rest days) and lineup continuity (started previous match).
**Optimiser:** the Hungarian algorithm assigns 11 distinct players to the 11 slots maximising the total — exact,
deterministic, and constrained (one goalkeeper, no duplicates, compatibility > 0, unavailable players excluded,
empty slot + warning when depth is insufficient). Scores are *not* probabilities.

In [17]:
from __future__ import annotations

# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · lineup_optimizer — explainable starting-XI optimiser
# Explainable starting-XI decision support.
#
# Method: every (player, formation slot) pair receives a transparent weighted score
# from season-to-date role fit, recent form, sample reliability, a workload proxy
# and lineup continuity, multiplied by a position-compatibility factor. The XI is
# the assignment of distinct players to the 11 slots that maximises the total
# score (Hungarian algorithm, ``scipy.optimize.linear_sum_assignment``) — exact,
# deterministic for the same inputs, and constrained (one player per slot, no
# duplicates, compatibility > 0, unavailable players excluded).
#
# Only information available *before* the decision date is used. The output is a
# reproducible analytical recommendation, not a claim to out-know a coach.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment


INFEASIBLE = 1e6


@dataclass
class DecisionContext:
    """All pre-decision information for one team at one decision date."""
    team_id: Any
    team_name: str
    asof: pd.Timestamp | None
    candidates: pd.DataFrame            # one row per candidate player with role-independent components
    season_pct: pd.DataFrame            # season-to-date table with percentiles (all players, benchmark pool)
    previous_starters: set
    pool_description: str
    notes: list[str] = field(default_factory=list)


@dataclass
class XIResult:
    formation: str
    lineup: pd.DataFrame
    total_score: float
    warnings: list[str]
    omitted: pd.DataFrame
    depth: pd.DataFrame
    score_matrix: pd.DataFrame
    method: str


def workload_asof(appearances: pd.DataFrame, matches: pd.DataFrame, team_id: Any, asof: pd.Timestamp
                  ) -> pd.DataFrame:
    """Minutes in the previous 7/14/28 days and rest days relative to ``asof`` (dataset matches only)."""
    a = appearances[(appearances["team_id"] == team_id) & appearances["appeared"]].merge(
        matches[["match_id", "match_date"]], on="match_id")
    a = a[a["match_date"] < asof]
    rows = []
    for pid, g in a.groupby("player_id"):
        rows.append({"player_id": pid,
                     "minutes_prev_7d": g.loc[g["match_date"] >= asof - pd.Timedelta(days=7), "minutes_played"].sum(),
                     "minutes_prev_14d": g.loc[g["match_date"] >= asof - pd.Timedelta(days=14), "minutes_played"].sum(),
                     "minutes_prev_28d": g.loc[g["match_date"] >= asof - pd.Timedelta(days=28), "minutes_played"].sum(),
                     "rest_days": (asof - g["match_date"].max()).days})
    return pd.DataFrame(rows, columns=["player_id", "minutes_prev_7d", "minutes_prev_14d", "minutes_prev_28d",
                                       "rest_days"])


def decision_context(bundle: Any, team_name: str, match_id: Any = None, asof: pd.Timestamp | None = None,
                     pool: str = "auto", recent_squads: int = 10) -> DecisionContext:
    """Build the pre-decision candidate pool for a team.

    * ``match_id`` given: decision date = that match's date; pool = its matchday squad
      (the announced available players). Stats use only earlier matches.
    * otherwise: hypothetical next fixture ``asof`` (default: 4 days after the last
      analysed match); pool = players in any of the team's last ``recent_squads`` squads.
    """
    cfg: Config = bundle.config
    matches, app, pms = bundle.tables["matches"], bundle.tables["appearances"], bundle.pms
    team_rows = app[app["team_name"] == team_name]
    if team_rows.empty:
        raise ValueError(f"Team {team_name!r} not found in analysed squads")
    team_id = team_rows["team_id"].iloc[0]
    notes: list[str] = []
    if match_id is not None:
        asof = matches.loc[matches["match_id"] == match_id, "match_date"].iloc[0]
        squad_ids = set(app[(app["match_id"] == match_id) & (app["team_id"] == team_id)]["player_id"])
        pool_desc = f"matchday squad of match {match_id} ({asof:%Y-%m-%d})"
    else:
        if asof is None:
            asof = matches["match_date"].max() + pd.Timedelta(days=4)
            notes.append("Hypothetical fixture dated 4 days after the last analysed match.")
        tm = (team_rows.merge(matches[["match_id", "match_date"]], on="match_id")
              .query("match_date < @asof").sort_values("match_date"))
        last_ids = tm["match_id"].drop_duplicates().tail(recent_squads)
        squad_ids = set(tm[tm["match_id"].isin(last_ids)]["player_id"])
        pool_desc = f"players named in the team's last {len(last_ids)} matchday squads before {asof:%Y-%m-%d}"
    season = player_season_table(pms, app, cfg, asof=asof, matches=matches)
    season_pct = add_percentiles(season, all_metric_columns(cfg), cfg)
    form = recent_form(pms, season_pct, cfg, asof=asof)
    wl = workload_asof(app, matches, team_id, asof)
    prev = (app[(app["team_id"] == team_id)].merge(matches[["match_id", "match_date"]], on="match_id")
            .query("match_date < @asof"))
    prev_starters: set = set()
    if not prev.empty:
        last_mid = prev.sort_values("match_date")["match_id"].iloc[-1]
        prev_starters = set(prev[(prev["match_id"] == last_mid) & prev["is_starter"]]["player_id"])
    names = team_rows.drop_duplicates("player_id", keep="last").set_index("player_id")["player_name"]
    cand = pd.DataFrame({"player_id": sorted(squad_ids)})
    cand["player_name"] = cand["player_id"].map(names)
    st = season_pct[season_pct["team_id"] == team_id]
    cand = cand.merge(st[["player_id", "primary_position", "primary_role", "role_family", "minutes_played",
                          "appearances", "starts", "sample_reliability", "coverage_label"]], on="player_id", how="left")
    cand = cand.merge(form[form["team_id"] == team_id][["player_id", "form_score", "form_minutes", "form_apps"]],
                      on="player_id", how="left")
    cand = cand.merge(wl, on="player_id", how="left")
    cand["minutes_played"] = cand["minutes_played"].fillna(0)
    cand["sample_reliability"] = cand["sample_reliability"].fillna(0)
    cand["coverage_label"] = cand["coverage_label"].fillna("no pre-decision minutes")
    cand["workload_suitability"] = workload_suitability(cand["minutes_prev_7d"], cand["rest_days"])
    cand["lineup_continuity"] = np.where(cand["player_id"].isin(prev_starters), 100.0, 0.0)
    if not prev_starters:
        notes.append("No previous match in dataset: continuity component is 0 for all players.")
    # Players whose primary role is unknown (no minutes) fall back to the role seen in squads, flagged.
    unknown_role = cand["primary_role"].isna()
    if unknown_role.any():
        fallback = (app[app["team_id"] == team_id].dropna(subset=["match_role"])
                    .groupby("player_id")["match_role"].agg(lambda s: s.value_counts().index[0]))
        cand.loc[unknown_role, "primary_role"] = cand.loc[unknown_role, "player_id"].map(fallback)
        cand["role_source"] = np.where(unknown_role, "fallback: squad match role (no pre-decision minutes)",
                                       "season primary position (minutes-weighted)")
    else:
        cand["role_source"] = "season primary position (minutes-weighted)"
    return DecisionContext(team_id=team_id, team_name=team_name, asof=asof, candidates=cand, season_pct=season_pct,
                           previous_starters=prev_starters, pool_description=pool_desc, notes=notes)


def score_candidates(ctx: DecisionContext, formation: str, config: Config,
                     unavailable: list | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Long table of component scores per (player, slot) and the slot templates."""
    slots = pd.DataFrame(config.formation_templates[formation])
    unavailable = set(unavailable or [])
    cand = ctx.candidates[~ctx.candidates["player_id"].isin(unavailable)].copy()
    sp = ctx.season_pct[ctx.season_pct["team_id"] == ctx.team_id].set_index("player_id")
    w = config.xi_score_weights
    rows = []
    fits = {}
    for role in slots["role"].unique():
        rf = role_fit_scores(sp.reset_index(), role, config).set_index("player_id")
        fits[role] = rf
    for _, s in slots.iterrows():
        rf = fits[s["role"]]
        for _, c in cand.iterrows():
            pid = c["player_id"]
            comp = compatibility(c["primary_role"], s["role"], config.role_compatibility)
            role_fit = rf["role_fit"].get(pid, np.nan)
            form = c["form_score"] if pd.notna(c["form_score"]) else role_fit
            parts = {"role_fit": role_fit, "recent_form": form, "sample_reliability": c["sample_reliability"],
                     "workload_suitability": c["workload_suitability"], "lineup_continuity": c["lineup_continuity"]}
            base = sum(w[k] * (0.0 if pd.isna(v) else v) for k, v in parts.items())
            side_ok = position_side(c["primary_position"]) == position_side(s["slot"])
            base += SIDE_MATCH_BONUS if side_ok else 0.0
            feasible = comp > 0 and pd.notna(role_fit)
            rows.append({"slot": s["slot"], "slot_role": s["role"], "player_id": pid, "player_name": c["player_name"],
                         "primary_role": c["primary_role"], "compatibility": comp, **parts,
                         "form_fallback": pd.isna(c["form_score"]), "side_match": side_ok,
                         "base_score": base, "score": base * comp if feasible else np.nan, "feasible": feasible,
                         "role_fit_completeness": rf["role_fit_completeness"].get(pid, 0.0),
                         "coverage_label": c["coverage_label"]})
    return pd.DataFrame(rows), slots


def optimize_xi(ctx: DecisionContext, formation: str, config: Config, unavailable: list | None = None,
                forced: dict | None = None) -> XIResult:
    """Constrained optimal assignment of candidates to the formation's 11 slots.

    ``forced`` maps slot -> player_id for what-if analysis (pinned before optimising).
    """
    if formation not in config.formation_templates:
        raise ValueError(f"Unknown formation {formation!r}; options: {list(config.formation_templates)}")
    long, slots = score_candidates(ctx, formation, config, unavailable)
    warnings: list[str] = []
    forced = forced or {}
    if long.empty:
        return XIResult(formation, pd.DataFrame(), 0.0, ["No eligible candidates."], pd.DataFrame(),
                        pd.DataFrame(), long, "n/a")
    players = sorted(long["player_id"].unique())
    slot_ids = list(slots["slot"])
    mat = long.pivot_table(index="player_id", columns="slot", values="score", aggfunc="first").reindex(
        index=players, columns=slot_ids)
    cost = np.where(mat.isna(), INFEASIBLE, -mat.to_numpy(dtype=float))
    for slot, pid in forced.items():
        j = slot_ids.index(slot)
        if pid not in players:
            warnings.append(f"Forced player {pid} is not an eligible candidate; ignored.")
            continue
        i = players.index(pid)
        cost[i, :] = INFEASIBLE
        cost[:, j] = INFEASIBLE
        cost[i, j] = -(mat.iloc[i, j] if pd.notna(mat.iloc[i, j]) else 0.0)
        if pd.isna(mat.iloc[i, j]):
            warnings.append(f"Forced {slot}: player has zero compatibility/score there (manual override).")
    rows_i, cols_j = linear_sum_assignment(cost)
    chosen = []
    for i, j in zip(rows_i, cols_j):
        if cost[i, j] >= INFEASIBLE:
            continue
        chosen.append((players[i], slot_ids[j]))
    filled = {s for _, s in chosen}
    for s in slot_ids:
        if s not in filled:
            warnings.append(f"Insufficient depth: no compatible candidate with pre-decision data for slot {s} — "
                            f"left empty rather than fabricated.")
    lk = long.set_index(["player_id", "slot"])
    lineup = []
    for pid, s in chosen:
        r = lk.loc[(pid, s)]
        lineup.append({"slot": s, "slot_role": r["slot_role"], "player_id": pid, "player_name": r["player_name"],
                       "primary_role": r["primary_role"], "compatibility": r["compatibility"],
                       "role_fit": r["role_fit"], "recent_form": r["recent_form"],
                       "sample_reliability": r["sample_reliability"],
                       "workload_suitability": r["workload_suitability"],
                       "lineup_continuity": r["lineup_continuity"], "score": r["score"] if pd.notna(r["score"]) else 0.0,
                       "form_fallback": r["form_fallback"], "coverage_label": r["coverage_label"],
                       "forced": forced.get(s) == pid})
    order = {s: k for k, s in enumerate(slot_ids)}
    lineup = pd.DataFrame(lineup)
    if not lineup.empty:
        lineup = lineup.sort_values("slot", key=lambda c: c.map(order)).reset_index(drop=True)
        lineup = lineup.merge(slots[["slot", "x", "y"]], on="slot", how="left")
        if (lineup["slot_role"] == "GK").sum() != 1:
            warnings.append("Lineup does not contain exactly one goalkeeper.")
        low = lineup[lineup["coverage_label"].isin(["low data coverage", "no pre-decision minutes"])]
        for _, r in low.iterrows():
            warnings.append(f"{r['player_name']} ({r['slot']}) selected on low data coverage.")
    depth = (long[long["feasible"] & (long["compatibility"] >= 0.8)].groupby(["slot", "slot_role"])["player_id"]
             .nunique().rename("candidates_compat_ge_0.8").reset_index())
    depth = slots[["slot", "role"]].rename(columns={"role": "slot_role"}).merge(depth, on=["slot", "slot_role"],
                                                                                  how="left").fillna(0)
    for _, d in depth.iterrows():
        if d["candidates_compat_ge_0.8"] < 2:
            warnings.append(f"Thin depth at {d['slot']}: {int(d['candidates_compat_ge_0.8'])} natural candidate(s).")
    omitted = explain_omissions(long, lineup, ctx, unavailable or [])
    total = float(lineup["score"].sum()) if not lineup.empty else 0.0
    method = ("Hungarian assignment (scipy linear_sum_assignment) maximising sum of slot scores; "
              f"weights={config.xi_score_weights}; decision date={ctx.asof:%Y-%m-%d}; pool={ctx.pool_description}")
    return XIResult(formation, lineup, total, warnings, omitted, depth, long, method)


def explain_omissions(long: pd.DataFrame, lineup: pd.DataFrame, ctx: DecisionContext,
                      unavailable: list) -> pd.DataFrame:
    """One readable reason per candidate not selected."""
    selected = set(lineup["player_id"]) if not lineup.empty else set()
    by_slot = lineup.set_index("slot") if not lineup.empty else pd.DataFrame()
    rows = []
    comps = ["role_fit", "recent_form", "sample_reliability", "workload_suitability", "lineup_continuity"]
    for _, c in ctx.candidates.iterrows():
        pid = c["player_id"]
        if pid in selected:
            continue
        if pid in set(unavailable):
            reason = "Marked unavailable by user (manual input)."
        else:
            pl = long[(long["player_id"] == pid)]
            feas = pl[pl["feasible"]]
            if pl.empty or feas.empty:
                if c["minutes_played"] <= 0:
                    reason = "No minutes in analysed matches before decision date (insufficient sample)."
                else:
                    reason = f"No compatible slot in this formation for primary role {c['primary_role']}."
            else:
                best = feas.sort_values("score", ascending=False).iloc[0]
                if best["slot"] in by_slot.index:
                    sel = by_slot.loc[best["slot"]]
                    gap = sel["score"] - best["score"]
                    diffs = {k: float(sel[k]) - float(best[k] if pd.notna(best[k]) else 0) for k in comps}
                    key = max(diffs, key=diffs.get)
                    reason = (f"Best slot {best['slot']} (score {best['score']:.1f}); {sel['player_name']} scored "
                              f"{sel['score']:.1f} (+{gap:.1f}); largest gap in {key.replace('_', ' ')}.")
                else:
                    reason = f"Best slot {best['slot']} (score {best['score']:.1f}) filled by assignment trade-off."
                if c["workload_suitability"] < 80:
                    reason += " Workload proxy elevated."
        rows.append({"player_name": c["player_name"], "primary_role": c["primary_role"],
                     "minutes_before_decision": round(float(c["minutes_played"]), 0), "reason": reason})
    return pd.DataFrame(rows)


def compare_formations(ctx: DecisionContext, config: Config, unavailable: list | None = None) -> pd.DataFrame:
    rows = []
    for f in config.formation_templates:
        r = optimize_xi(ctx, f, config, unavailable)
        n = len(r.lineup)
        rows.append({"formation": f, "slots_filled": n, "total_score": round(r.total_score, 1),
                     "mean_slot_score": round(r.total_score / n, 1) if n else np.nan,
                     "mean_compatibility": round(r.lineup["compatibility"].mean(), 2) if n else np.nan,
                     "low_coverage_picks": int(r.lineup["coverage_label"].isin(
                         ["low data coverage", "no pre-decision minutes"]).sum()) if n else 0,
                     "warnings": len(r.warnings)})
    return pd.DataFrame(rows).sort_values("total_score", ascending=False).reset_index(drop=True)


def actual_xi(appearances: pd.DataFrame, match_id: Any, team_id: Any) -> pd.DataFrame:
    a = appearances[(appearances["match_id"] == match_id) & (appearances["team_id"] == team_id)
                    & appearances["is_starter"]]
    return a[["player_id", "player_name", "match_position", "match_role", "minutes_played"]].reset_index(drop=True)


def compare_with_actual(result: XIResult, appearances: pd.DataFrame, match_id: Any, team_id: Any
                        ) -> tuple[pd.DataFrame, float]:
    """Descriptive agreement between recommended and actual XI (not a quality measure)."""
    act = actual_xi(appearances, match_id, team_id)
    rec_ids = set(result.lineup["player_id"]) if not result.lineup.empty else set()
    act_ids = set(act["player_id"])
    names = dict(zip(act["player_id"], act["player_name"]))
    names.update(dict(zip(result.lineup.get("player_id", []), result.lineup.get("player_name", []))))
    rows = [{"player_name": names.get(p), "in_recommended": p in rec_ids, "in_actual": p in act_ids}
            for p in sorted(rec_ids | act_ids, key=lambda p: str(names.get(p)))]
    agree = len(rec_ids & act_ids) / 11 if act_ids else np.nan
    return pd.DataFrame(rows), agree


def what_if(ctx: DecisionContext, base: XIResult, slot: str, player_id: Any, config: Config,
            unavailable: list | None = None) -> tuple[XIResult, float]:
    """Pin a player to a slot, re-optimise the rest, report the score change."""
    new = optimize_xi(ctx, base.formation, config, unavailable, forced={slot: player_id})
    return new, new.total_score - base.total_score


def xi_historical_agreement(bundle: Any, team_name: str, formation_from_match: bool = True,
                         max_matches: int | None = None) -> pd.DataFrame:
    """Recommendation vs actual XI for each team match (descriptive agreement only)."""
    cfg = bundle.config
    ts = bundle.team_stats[bundle.team_stats["team_name"] == team_name].sort_values("match_date")
    rows = []
    for _, m in ts.iloc[1:].iterrows():  # skip first match: no pre-decision history
        f = m["starting_formation"] if formation_from_match else None
        if f not in cfg.formation_templates:
            f = "3-4-2-1" if "3-4-2-1" in cfg.formation_templates else next(iter(cfg.formation_templates))
            mapped = True
        else:
            mapped = False
        ctx = decision_context(bundle, team_name, match_id=m["match_id"])
        r = optimize_xi(ctx, f, cfg)
        _, agree = compare_with_actual(r, bundle.tables["appearances"], m["match_id"], ctx.team_id)
        rows.append({"match_id": m["match_id"], "match_date": m["match_date"], "opponent": m["opponent"],
                     "actual_formation": m["starting_formation"], "template_used": f,
                     "template_mapped": mapped, "agreement_pct": round(agree * 100, 1)})
        if max_matches and len(rows) >= max_matches:
            break
    return pd.DataFrame(rows)


def role_label(role: str | None) -> str:
    return ROLE_LABEL.get(role, str(role))

In [18]:
focus_matches = B.team_stats[B.team_stats.team_name == FOCUS].sort_values("match_date").reset_index(drop=True)
XI_MATCH_INDEX = min(24, len(focus_matches) - 1)      # editable: which focus-team match to analyse
xm = focus_matches.iloc[XI_MATCH_INDEX]
XI_FORMATION = xm.starting_formation if xm.starting_formation in cfg.formation_templates else "4-3-3"
UNAVAILABLE = []                                      # e.g. [player_id, ...] marked unavailable by the user
ctx = decision_context(B, FOCUS, match_id=xm.match_id)
xi = optimize_xi(ctx, XI_FORMATION, cfg, unavailable=UNAVAILABLE)
print(f"Match: {xm.match_date:%d %b %Y} vs {xm.opponent} ({xm.home_away}) | formation used: {XI_FORMATION} "
      f"(actual {xm.starting_formation}) | candidates: {len(ctx.candidates)} | pool: {ctx.pool_description}")
xi_pitch(xi.lineup, f"Recommended {XI_FORMATION} XI — decision date {ctx.asof:%Y-%m-%d} "
             f"(total score {xi.total_score:.0f})", banner=BANNER).show()
score_decomposition(xi.lineup, cfg.xi_score_weights, "Explainable score decomposition per selected player",
                        banner=BANNER).show()
display(Markdown("**Warnings:** " + ("; ".join(xi.warnings) if xi.warnings else "none")))

cmp, agree = compare_with_actual(xi, T["appearances"], xm.match_id, ctx.team_id)
display(Markdown(f"**Recommended vs actual XI** — descriptive agreement {agree:.0%} "
                 "(agreement is not proof of quality; disagreement is not proof the coach was wrong)."))
display(cmp)
display(Markdown("**Why each squad player was not selected**"))
display(xi.omitted)
display(Markdown("**Squad depth by slot** (candidates with compatibility ≥ 0.8)"))
display(xi.depth)

display(Markdown("**Formation comparison** (same pool and date)"))
display(compare_formations(ctx, cfg, UNAVAILABLE))
# Alternative "rotation" XI: rest the three selected outfield players with the most minutes in the previous 14 days
load = ctx.candidates.set_index("player_id")["minutes_prev_14d"].fillna(0)
rest = list(xi.lineup[xi.lineup.slot_role != "GK"].assign(l=lambda d: d.player_id.map(load))
            .nlargest(3, "l").player_id)
alt = optimize_xi(ctx, XI_FORMATION, cfg, unavailable=UNAVAILABLE + rest)
alternative_xi_comparison(xi.lineup, alt.lineup, "Recommended XI", "Rotation XI (3 highest-load rested)",
                              banner=BANNER).show()
print(f"Total score: recommended {xi.total_score:.1f} vs rotation {alt.total_score:.1f} "
      f"(Δ {alt.total_score - xi.total_score:+.1f})")

# What-if: force the best-scoring unselected player into his best slot and re-optimise
cand_scores = xi.score_matrix[xi.score_matrix.feasible & ~xi.score_matrix.player_id.isin(xi.lineup.player_id)]
if len(cand_scores):
    best = cand_scores.sort_values("score", ascending=False).iloc[0]
    wi, delta = what_if(ctx, xi, best.slot, best.player_id, cfg, UNAVAILABLE)
    display(Markdown(f"**What-if:** pin **{best.player_name}** at **{best.slot}** → total score change "
                     f"**{delta:+.1f}** (the optimiser's choice is optimal under the stated weights, so Δ ≤ 0)."))
    display(wi.lineup[["slot", "player_name", "score", "forced"]].round(1))

if WIDGETS_OK and not HEADLESS:
    names = ctx.candidates.set_index("player_id")["player_name"].to_dict()
    def xi_live(formation, unavailable):
        r = optimize_xi(ctx, formation, cfg, unavailable=list(unavailable))
        xi_pitch(r.lineup, f"{formation} — total {r.total_score:.0f}", banner=BANNER).show()
        print("Warnings:", r.warnings or "none")
    W.interact(xi_live, formation=W.Dropdown(options=list(cfg.formation_templates), value=XI_FORMATION),
               unavailable=W.SelectMultiple(options=[(v, k) for k, v in names.items()], rows=8,
                                            description="Unavailable"))
else:
    print("Live formation / availability controls available when running interactively (Colab/Jupyter).")

t0 = time.perf_counter()
hist_xi = xi_historical_agreement(B, FOCUS)
print(f"Re-ran the optimiser before each of {len(hist_xi)} matches using only prior data "
      f"({time.perf_counter() - t0:.1f}s). Mean descriptive agreement with the actual XI: "
      f"{hist_xi.agreement_pct.mean():.1f}% (median {hist_xi.agreement_pct.median():.0f}%).")
display(hist_xi.tail(8))
# Hypothetical next fixture (4 days after last analysed match)
ctx_next = decision_context(B, FOCUS)
xi_next = optimize_xi(ctx_next, XI_FORMATION, cfg)
xi_pitch(xi_next.lineup, f"Hypothetical next-fixture XI ({XI_FORMATION}), decision date "
             f"{ctx_next.asof:%Y-%m-%d}", banner=BANNER).show()

Match: 10 Mar 2024 vs Wolfsburg (home) | formation used: 3-4-2-1 (actual 3-4-2-1) | candidates: 20 | pool: matchday squad of match 3895266 (2024-03-10)


**Warnings:** Thin depth at GK: 1 natural candidate(s).

**Recommended vs actual XI** — descriptive agreement 91% (agreement is not proof of quality; disagreement is not proof the coach was wrong).

,player_name,in_recommended,in_actual
0,Edmond Fayçal Tapsoba,False,True
1,Exequiel Palacios,True,True
2,Florian Wirtz,True,True
3,Granit Xhaka,True,True
4,Jonas Hofmann,True,False
5,Jonathan Tah,True,True
6,Josip Stanišić,True,True
7,Lukáš Hrádecký,True,True
8,Nathan Tella,True,True
9,Patrik Schick,True,True


**Why each squad player was not selected**

,player_name,primary_role,minutes_before_decision,reason
0,Robert Andrich,DM,1060.0,Best slot LDM (score 69.0); Granit Xhaka score...
1,Borja Iglesias,CF,147.0,Best slot ST (score 60.0); Patrik Schick score...
2,Niklas Lomb,NaN,0.0,No minutes in analysed matches before decision...
3,Odilon Kossonou,CB,1406.0,Best slot RCB (score 67.0); Josip Stanišić sco...
4,Adam Hložek,AM,331.0,Best slot RAM (score 69.7); Jonas Hofmann scor...
5,Edmond Fayçal Tapsoba,CB,1493.0,Best slot LCB (score 72.2); Piero Hincapié sco...
6,Amine Adli,AM,600.0,Best slot LAM (score 74.4); Florian Wirtz scor...
7,Noah Mbamba,DM,23.0,Best slot RDM (score 65.7); Exequiel Palacios ...
8,Gustavo Adolfo Puerta Molano,DM,43.0,Best slot LDM (score 54.5); Granit Xhaka score...


... (6 outputs left out)

,slot,player_name,score,forced
0,GK,Lukáš Hrádecký,69.8,False
1,RCB,Josip Stanišić,72.0,False
2,CB,Jonathan Tah,73.0,False
3,LCB,Piero Hincapié,76.1,False
4,RWB,Nathan Tella,66.4,False
5,RDM,Exequiel Palacios,74.8,False
6,LDM,Granit Xhaka,69.4,False
7,LWB,Álex Grimaldo,85.1,False
8,RAM,Florian Wirtz,93.6,False
9,LAM,Amine Adli,74.4,True


Live formation / availability controls available when running interactively (Colab/Jupyter).


Re-ran the optimiser before each of 33 matches using only prior data (7.1s). Mean descriptive agreement with the actual XI: 77.7% (median 82%).


,match_id,match_date,opponent,actual_formation,template_used,template_mapped,agreement_pct
25,3895286,2024-03-30,Hoffenheim,3-4-2-1,3-4-2-1,False,81.8
26,3895292,2024-04-06,Union Berlin,3-4-2-1,3-4-2-1,False,54.5
27,3895302,2024-04-14,Werder Bremen,3-4-2-1,3-4-2-1,False,72.7
28,3895309,2024-04-21,Borussia Dortmund,3-4-3,3-4-3,False,63.6
29,3895320,2024-04-27,VfB Stuttgart,3-4-2-1,3-4-2-1,False,63.6
30,3895333,2024-05-05,Eintracht Frankfurt,3-4-2-1,3-4-2-1,False,72.7
31,3895340,2024-05-12,Bochum,3-4-3,3-4-3,False,63.6
32,3895348,2024-05-18,Augsburg,3-4-2-1,3-4-2-1,False,72.7


## 12 · Substitution intelligence
Historical substitutions enriched with the **team-perspective score at the moment of the change**, timing band,
roles, window usage (rules user-configured) and manager. Then a live scenario: everything the bench could have known
at the decision minute — and nothing after it.

In [19]:
from __future__ import annotations

# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · substitution_analysis — substitution intelligence
# Substitution decision intelligence.
#
# * Historical substitutions enriched with match state (team perspective), window
# usage under user-configured rules, roles and manager.
# * A live decision context at any elapsed minute built ONLY from events before
# that moment plus pre-match (earlier-match) player history.
# * Transparent ranking of outgoing and incoming candidates with explanations.
# * Descriptive pre/post windows — association, never causation.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
from typing import Any

import numpy as np
import pandas as pd


HUMAN_NOTE = ("Decision support only: the coaching staff makes the final decision, with information "
              "(fitness, instructions, opponent changes, player feedback) that this dataset does not contain.")


# ---------------------------------------------------------------------------
# Historical substitutions
# ---------------------------------------------------------------------------
def _windows(team_subs: pd.DataFrame, rules: dict) -> list[int]:
    """Window number per substitution; interval (period-start) changes optionally free."""
    out, w, last_t = [], 0, None
    for _, s in team_subs.iterrows():
        interval = s["second"] == 0 and s["minute"] in (45, 90, 105) and s["period"] in (2, 3, 4)
        if interval and not rules.get("interval_changes_use_window", False):
            out.append(0)
            continue
        if last_t is None or (s["elapsed_min"] - last_t) * 60 > rules.get("window_grouping_seconds", 90):
            w += 1
        last_t = s["elapsed_min"]
        out.append(w)
    return out


def enrich_substitutions(bundle: Any) -> pd.DataFrame:
    """One row per substitution with game state, timing band, roles, rule usage and manager."""
    cfg: Config = bundle.config
    t = bundle.tables
    subs, matches, goals, so = t["substitutions_raw"].copy(), t["matches"], t["goals"], t["sending_offs"]
    if subs.empty:
        return subs
    app = t["appearances"]
    m = matches.set_index("match_id")
    rows = []
    for _, s in subs.iterrows():
        mm = m.loc[s["match_id"]]
        home = s["team_id"] == mm["home_team_id"]
        opp = mm["away_team_id"] if home else mm["home_team_id"]
        tg, og = score_at(goals, s["match_id"], s["team_id"], opp, before_index=s["index"])
        rows.append({"home_away": "home" if home else "away", "opponent_id": opp,
                     "opponent": mm["away_team"] if home else mm["home_team"], "match_date": mm["match_date"],
                     "team_score": tg, "opponent_score": og, "goal_diff": tg - og, "game_state": state_label(tg - og),
                     "man_advantage": men_on_pitch(so, s["match_id"], s["team_id"], opp, before_index=s["index"])})
    subs = pd.concat([subs.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    subs["timing_band"] = [assign_band(p, mi, cfg.sub_timing_bands) for p, mi in zip(subs["period"], subs["minute"])]
    subs["match_phase"] = [assign_band(p, mi, cfg.match_phases) for p, mi in zip(subs["period"], subs["minute"])]
    on_pos = app.set_index(["match_id", "player_id"])["match_position"]
    subs["on_position"] = [on_pos.get((a, b)) for a, b in zip(subs["match_id"], subs["player_on_id"])]
    subs["off_role"] = subs["off_position"].map(role_for_position)
    subs["on_role"] = subs["on_position"].map(role_for_position)
    subs["like_for_like"] = subs["off_role"] == subs["on_role"]
    # Lineup-file display names (nickname when published) — event names are full legal names.
    disp = app.drop_duplicates(["match_id", "player_id"]).set_index(["match_id", "player_id"])["player_name"]
    subs["player_on_display"] = [disp.get((a, b), n) for a, b, n in zip(subs["match_id"], subs["player_on_id"],
                                                                         subs["player_on_name"])]
    subs["player_off_display"] = [disp.get((a, b), n) for a, b, n in zip(subs["match_id"], subs["player_off_id"],
                                                                          subs["player_off_name"])]
    subs = subs.sort_values(["match_id", "team_id", "index"])
    subs["sub_number"] = subs.groupby(["match_id", "team_id"]).cumcount() + 1
    subs["window_number"] = 0
    for _, idx in subs.groupby(["match_id", "team_id"]).groups.items():
        subs.loc[idx, "window_number"] = _windows(subs.loc[idx], cfg.sub_rules)
    et = subs["period"] >= 3
    subs["max_subs_allowed"] = cfg.sub_rules["max_substitutions"] + np.where(
        et, cfg.sub_rules["extra_time_additional_substitutions"], 0)
    subs["subs_remaining_after"] = subs["max_subs_allowed"] - subs["sub_number"]
    subs["rule_source"] = cfg.sub_rules["source"]
    subs["multi_sub_window"] = subs.duplicated(["match_id", "team_id", "window_number"], keep=False) & (
        subs["window_number"] > 0)
    mg = t["managers"].drop_duplicates(["match_id", "team_id"])
    if not mg.empty:
        subs = subs.merge(mg[["match_id", "team_id", "manager_id", "manager_name"]], on=["match_id", "team_id"],
                          how="left")
    # Identity validation against squads
    squad = set(zip(app["match_id"], app["player_id"]))
    subs["identity_check"] = np.where(
        [(a, b) in squad and (a, c) in squad for a, b, c in zip(subs["match_id"], subs["player_off_id"],
                                                               subs["player_on_id"])], "ok",
        "player not found in matchday squad")
    return subs.sort_values(["match_date", "match_id", "index"]).reset_index(drop=True)


def timing_profile(subs: pd.DataFrame, by: str | None = None, bands: list[dict] | None = None) -> pd.DataFrame:
    """Counts and shares of substitutions per timing band (optionally per group)."""
    if subs.empty:
        return pd.DataFrame()
    order = [b["label"] for b in bands] if bands else sorted(subs["timing_band"].dropna().unique())
    keys = [by, "timing_band"] if by else ["timing_band"]
    c = subs.groupby(keys).size().rename("substitutions").reset_index()
    total = c.groupby(by)["substitutions"].transform("sum") if by else c["substitutions"].sum()
    c["share_pct"] = (c["substitutions"] / total * 100).round(1)
    c["timing_band"] = pd.Categorical(c["timing_band"], categories=order, ordered=True)
    return c.sort_values(keys).reset_index(drop=True)


def state_profile(subs: pd.DataFrame, by: str | None = None) -> pd.DataFrame:
    keys = [by, "game_state"] if by else ["game_state"]
    g = subs.groupby(keys)
    out = pd.DataFrame({"substitutions": g.size(), "median_minute": g["minute"].median(),
                        "mean_minute": g["minute"].mean().round(1),
                        "attacking_change_pct": g.apply(lambda d: (d["on_role"].isin(["CF", "W", "AM"])
                                                                   & ~d["off_role"].isin(["CF", "W", "AM"])).mean()
                                                        * 100).round(1)}).reset_index()
    return out


def role_flow(subs: pd.DataFrame) -> pd.DataFrame:
    """Outgoing role -> incoming role counts (for the flow / Sankey chart)."""
    f = subs.dropna(subset=["off_role", "on_role"]).groupby(["off_role", "on_role"]).size().rename("count")
    return f.reset_index()


# ---------------------------------------------------------------------------
# Live decision context
# ---------------------------------------------------------------------------
def on_pitch_at(spells: pd.DataFrame, match_id: Any, team_id: Any, elapsed: float) -> pd.DataFrame:
    s = spells[(spells["match_id"] == match_id) & (spells["team_id"] == team_id)]
    return s[(s["start"] <= elapsed) & (s["end"] > elapsed)]


def bench_at(appearances: pd.DataFrame, spells: pd.DataFrame, match_id: Any, team_id: Any, elapsed: float
             ) -> pd.DataFrame:
    """Eligible incoming players: matchday-squad members who have not appeared before ``elapsed``.

    A player who came on and was later replaced cannot return; a starter is never on the bench.
    """
    sq = appearances[(appearances["match_id"] == match_id) & (appearances["team_id"] == team_id)]
    s = spells[(spells["match_id"] == match_id) & (spells["team_id"] == team_id)]
    used = set(s.loc[s["start"] <= elapsed, "player_id"])
    return sq[~sq["player_id"].isin(used)]


def scenario_context(bundle: Any, match_id: Any, team_name: str, elapsed: float) -> dict[str, Any]:
    """Everything known at ``elapsed`` minutes of playing time — no later information."""
    cfg: Config = bundle.config
    t = bundle.tables
    matches, flags = t["matches"], bundle.flags
    mm = matches.set_index("match_id").loc[match_id]
    team_id = mm["home_team_id"] if mm["home_team"] == team_name else mm["away_team_id"]
    if mm["home_team"] != team_name and mm["away_team"] != team_name:
        raise ValueError(f"{team_name} did not play match {match_id}")
    opp_id = mm["away_team_id"] if team_id == mm["home_team_id"] else mm["home_team_id"]
    asof = mm["match_date"]
    tg, og = score_at(t["goals"], match_id, team_id, opp_id, elapsed=elapsed)
    men = men_on_pitch(t["sending_offs"], match_id, team_id, opp_id, elapsed=elapsed)
    before = flags[(flags["match_id"] == match_id) & (flags["elapsed_min"] < elapsed)]
    subs_used = before[(before["type"] == "Substitution") & (before["team_id"] == team_id)]
    rules = cfg.sub_rules
    win_used = len(set(w for w in _windows(subs_used, rules) if w > 0)) if len(subs_used) else 0
    period_now = int(before["period"].max()) if len(before) else 1
    max_subs = rules["max_substitutions"] + (rules["extra_time_additional_substitutions"] if period_now >= 3 else 0)
    max_win = rules["max_windows"] + (rules["extra_time_additional_windows"] if period_now >= 3 else 0)

    def team_block(tid):
        b = before[before["team_id"] == tid]
        return {"shots": int(b["shots"].sum()), "xg": round(float(b["xg"].sum()), 2),
                "passes": int(b["passes"].sum()), "pass_completion_pct": round(
                    100 * b["passes_completed"].sum() / max(b["passes"].sum(), 1), 1),
                "final_third_touches": int(b["final_third_touches"].sum()), "pressures": int(b["pressures"].sum()),
                "defensive_actions": int(b["defensive_actions"].sum())}

    pitch = on_pitch_at(t["spells"], match_id, team_id, elapsed)
    bench = bench_at(t["appearances"], t["spells"], match_id, team_id, elapsed)
    # In-match player stats up to the decision moment
    pb = before[before["team_id"] == team_id].dropna(subset=["player_id"])
    inm = pb.groupby("player_id")[COUNT_COLS].sum()
    pitch = pitch.merge(inm, left_on="player_id", right_index=True, how="left").fillna({c: 0 for c in COUNT_COLS})
    pitch["minutes_so_far"] = elapsed - pitch["start"]
    cards = before[(before["team_id"] == team_id) & (before["card"] == "Yellow Card")]["player_id"]
    pitch["booked"] = pitch["player_id"].isin(set(cards))
    pitch["role"] = pitch["position"].map(role_for_position)
    # Pre-match history (earlier matches only)
    season = player_season_table(bundle.pms, t["appearances"], cfg, asof=asof, matches=matches)
    season_pct = add_percentiles(season, all_metric_columns(cfg), cfg)
    form = recent_form(bundle.pms, season_pct, cfg, asof=asof)
    wl = bundle.workload[(bundle.workload["match_id"] == match_id) & (bundle.workload["team_id"] == team_id)]
    return {"match_id": match_id, "team_id": team_id, "team_name": team_name, "opponent_id": opp_id,
            "opponent": mm["away_team"] if team_id == mm["home_team_id"] else mm["home_team"],
            "home_away": "home" if team_id == mm["home_team_id"] else "away", "match_date": asof,
            "elapsed": elapsed, "period": period_now, "team_score": tg, "opponent_score": og,
            "goal_diff": tg - og, "game_state": state_label(tg - og), "man_advantage": men,
            "subs_used": len(subs_used), "subs_remaining": max_subs - len(subs_used),
            "windows_used": win_used, "windows_remaining": max_win - win_used, "rule_source": rules["source"],
            "team_live": team_block(team_id), "opponent_live": team_block(opp_id),
            "on_pitch": pitch, "bench": bench, "season_pct": season_pct, "form": form, "workload": wl}


def infer_intent(goal_diff: int, elapsed: float, man_advantage: int = 0) -> tuple[str, str]:
    """Transparent context prompt (not a law): trailing -> attack, leading -> protect, drawing -> balance."""
    if goal_diff < 0:
        return "attack", "Trailing: consider attacking / progression-focused options."
    if goal_diff > 0:
        if man_advantage < 0:
            return "protect", "Leading with a numerical disadvantage: prioritise defensive stability."
        return "protect", "Leading: consider defensive stability, ball retention and role coverage."
    if elapsed >= 75:
        return "balance", "Drawing late: balance chance creation against defensive risk."
    return "balance", "Drawing: balance control, chance creation and defensive risk."


def _intent_score(season_pct: pd.DataFrame, team_id: Any, player_ids: list, intent: str, config: Config
                  ) -> pd.Series:
    weights = config.sub_intent_metrics[intent]
    sp = season_pct[(season_pct["team_id"] == team_id) & season_pct["player_id"].isin(player_ids)]
    elig = season_pct[season_pct["eligible"].astype(bool)]
    num = np.zeros(len(sp))
    wsum = np.zeros(len(sp))
    for m, w in weights.items():
        if m not in sp.columns:
            continue
        # Intent fit is benchmarked against ALL eligible players (not role groups) because the
        # question is "who adds this quality", e.g. more goal threat when trailing.
        v = _pct_of(sp[m].to_numpy(dtype=float), elig[m].to_numpy(dtype=float))
        ok = ~np.isnan(v)
        num[ok] += w * v[ok]
        wsum[ok] += w
    with np.errstate(invalid="ignore", divide="ignore"):
        vals = np.where(wsum > 0, num / np.where(wsum > 0, wsum, 1), np.nan)
    return pd.Series(vals, index=sp["player_id"].values)


def rank_outgoing(ctx: dict[str, Any], intent: str, config: Config, include_gk: bool = False) -> pd.DataFrame:
    """Rank on-pitch players as substitution-off candidates with reasons.

    Components: minutes already played (proxy for load in this match), yellow-card
    risk, in-match involvement vs the player's OWN pre-match per-90 baseline for
    role-relevant actions (role context, not raw counts), and intent mismatch.
    """
    p = ctx["on_pitch"].copy()
    if p.empty:
        return p
    if not include_gk:
        p = p[p["role"] != "GK"]
    sp = ctx["season_pct"]
    base = sp[sp["team_id"] == ctx["team_id"]].set_index("player_id")
    role_w = config.role_metric_weights
    rows = []
    for _, r in p.iterrows():
        pid, role = r["player_id"], r["role"]
        mins = max(r["minutes_so_far"], 1.0)
        ratios, keys = [], []
        for m in role_w.get(role, {}):
            if not m.endswith("_p90"):
                continue
            b = m[:-4]
            if pid in base.index and pd.notna(base.loc[pid, m]) and base.loc[pid, m] > 0 and b in r:
                live = r[b] / mins * 90
                ratios.append(min(live / base.loc[pid, m], 2.0))
                keys.append(b)
        involvement = float(np.mean(ratios)) if ratios else np.nan
        fatigue_proxy = min(r["minutes_so_far"] / 90, 1.0) * 100
        intent_mismatch = 0.0
        if intent == "attack" and role in ("CB", "DM", "FB"):
            intent_mismatch = 60.0
        elif intent == "protect" and role in ("CF", "W", "AM"):
            intent_mismatch = 60.0
        low_inv = (1 - min(involvement, 1.0)) * 100 if pd.notna(involvement) else 30.0
        card = 100.0 if r["booked"] else 0.0
        score = 0.35 * fatigue_proxy + 0.30 * low_inv + 0.15 * card + 0.20 * intent_mismatch
        reasons = [f"{r['minutes_so_far']:.0f} min played this match"]
        if pd.notna(involvement):
            reasons.append(f"role-action rate at {involvement * 100:.0f}% of own pre-match baseline "
                           f"({len(keys)} role metrics)")
        else:
            reasons.append("no pre-match baseline (insufficient earlier minutes) — involvement not assessed")
        if r["booked"]:
            reasons.append("booked (second-yellow risk)")
        if intent_mismatch:
            reasons.append(f"role {role} less aligned with '{intent}' intent")
        rows.append({"player_id": pid, "player_name": r["player_name"], "position": r["position"], "role": role,
                     "minutes_so_far": round(r["minutes_so_far"], 1), "booked": r["booked"],
                     "involvement_vs_baseline": round(involvement, 2) if pd.notna(involvement) else np.nan,
                     "outgoing_score": round(score, 1), "reasons": "; ".join(reasons),
                     "baseline_coverage": base.loc[pid, "coverage_label"] if pid in base.index else "no history"})
    return pd.DataFrame(rows).sort_values("outgoing_score", ascending=False).reset_index(drop=True)


def rank_incoming(ctx: dict[str, Any], outgoing_role: str | None, intent: str, config: Config) -> pd.DataFrame:
    """Rank eligible bench players for an outgoing player's role and the tactical intent."""
    bench = ctx["bench"]
    if bench.empty:
        return pd.DataFrame()
    sp = ctx["season_pct"]
    team_sp = sp[sp["team_id"] == ctx["team_id"]].set_index("player_id")
    form = ctx["form"]
    form = form[form["team_id"] == ctx["team_id"]].set_index("player_id")
    intent_s = _intent_score(sp, ctx["team_id"], list(bench["player_id"]), intent, config)
    wl = ctx["workload"].set_index("player_id") if not ctx["workload"].empty else pd.DataFrame()
    w = config.sub_candidate_weights
    rows = []
    for _, b in bench.iterrows():
        pid = b["player_id"]
        role = team_sp.loc[pid, "primary_role"] if pid in team_sp.index else b.get("match_role")
        if role == "GK" and outgoing_role != "GK":
            continue
        comp = compatibility(role, outgoing_role, config.role_compatibility) if outgoing_role else 0.5
        # Intent can justify a role change (e.g. forward for a defender when trailing) at reduced weight.
        if comp == 0 and intent == "attack" and role in ("CF", "W", "AM"):
            comp = 0.4
        if comp == 0 and intent == "protect" and role in ("CB", "DM", "FB", "WB"):
            comp = 0.4
        if comp == 0:
            continue
        mins = float(team_sp.loc[pid, "minutes_played"]) if pid in team_sp.index else 0.0
        rel = float(team_sp.loc[pid, "sample_reliability"]) if pid in team_sp.index else 0.0
        intent_fit = float(intent_s.loc[pid]) if pid in intent_s.index else np.nan
        form_v = form.loc[pid, "form_score"] if pid in form.index else np.nan
        parts = {"role_compatibility": comp * 100, "intent_fit": intent_fit, "recent_form": form_v,
                 "sample_reliability": rel}
        score = sum(w[k] * (0.0 if pd.isna(v) else v) for k, v in parts.items())
        m7 = wl.loc[pid, "minutes_prev_7d"] if (not wl.empty and pid in wl.index) else np.nan
        evidence = [f"role {role} vs outgoing {outgoing_role}: compatibility {comp:.2f}"]
        evidence.append(f"'{intent}' metric fit {intent_fit:.0f}th pct" if pd.notna(intent_fit)
                        else "no pre-match per-90 sample for intent metrics")
        evidence.append(f"{mins:.0f} pre-match minutes")
        if pd.notna(form_v):
            evidence.append(f"recent form {form_v:.0f}")
        risks = []
        if rel < 30:
            risks.append("small evidence base")
        if comp < 0.6:
            risks.append("positional change — tactical reshuffle needed")
        if pd.notna(m7) and m7 > 150:
            risks.append(f"{m7:.0f} min in previous 7 days (dataset matches only)")
        rows.append({"player_id": pid, "player_name": b["player_name"], "role": role, **parts,
                     "candidate_score": round(score, 1), "evidence": "; ".join(evidence),
                     "risks": "; ".join(risks) if risks else "none flagged",
                     "injury_status": "unknown (no injury data in source — not assumed fit)"})
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values("candidate_score", ascending=False).reset_index(drop=True)


def recommend_substitution(bundle: Any, match_id: Any, team_name: str, elapsed: float,
                           intent: str | None = None, n_alternatives: int = 3) -> dict[str, Any]:
    """Full explainable substitution recommendation at a decision moment."""
    cfg: Config = bundle.config
    ctx = scenario_context(bundle, match_id, team_name, elapsed)
    auto_intent, why = infer_intent(ctx["goal_diff"], elapsed, ctx["man_advantage"])
    intent = intent or auto_intent
    out = rank_outgoing(ctx, intent, cfg)
    result: dict[str, Any] = {"context": ctx, "intent": intent, "intent_reason": why if intent == auto_intent else
                              "user-selected intent", "outgoing_ranking": out, "human_note": HUMAN_NOTE}
    if ctx["subs_remaining"] <= 0 or ctx["windows_remaining"] <= 0:
        result["status"] = "No substitution allowance remaining under the configured rules."
        return result
    if out.empty or ctx["bench"].empty:
        result["status"] = "No eligible outgoing or incoming players."
        return result
    best = None
    for _, o in out.head(3).iterrows():
        inc = rank_incoming(ctx, o["role"], intent, cfg)
        if inc.empty:
            continue
        pair_score = 0.4 * o["outgoing_score"] + 0.6 * inc.iloc[0]["candidate_score"]
        if best is None or pair_score > best[0]:
            best = (pair_score, o, inc)
    if best is None:
        result["status"] = "No compatible bench candidate for the top outgoing options."
        return result
    _, o, inc = best
    top = inc.iloc[0]
    hist = bundle.subs[(bundle.subs["team_name"] == team_name) & (bundle.subs["game_state"] == ctx["game_state"])]
    if len(hist) >= 5:
        q1, q3 = hist["minute"].quantile([0.25, 0.75])
        window = f"team's historical {ctx['game_state']} substitutions mostly fall between {q1:.0f}' and {q3:.0f}'"
    else:
        window = "insufficient team history for this game state"
    cov = "high" if top["sample_reliability"] >= 70 else ("moderate" if top["sample_reliability"] >= 30 else "low")
    result.update({
        "status": "ok",
        "recommended_off": o.to_dict(), "recommended_on": top.to_dict(),
        "alternatives": inc.iloc[1:1 + n_alternatives][["player_name", "role", "candidate_score", "evidence",
                                                        "risks"]],
        "decision_window": f"now to ~{min(elapsed + 10, 90):.0f}' of playing time; {window}",
        "tactical_objective": {"attack": "increase chance creation and goal threat",
                               "protect": "protect the lead: defensive stability and ball retention",
                               "balance": "maintain control while adding a fresh option"}[intent],
        "coverage": f"{cov} data coverage for the incoming player",
        "rule_status": f"{ctx['subs_remaining']} substitutions and {ctx['windows_remaining']} windows remaining "
                       f"({ctx['rule_source']})",
    })
    return result


def sub_historical_agreement(bundle: Any, team_name: str, top_k: int = 3, max_subs: int | None = None,
                         skip_injury: bool = True) -> pd.DataFrame:
    """Re-run the heuristic one second before each real substitution (pre-decision data only)
    and record whether the actual incoming player was in the top-k. Descriptive agreement only."""
    cfg = bundle.config
    subs = bundle.subs[bundle.subs["team_name"] == team_name]
    if skip_injury:
        subs = subs[subs["sub_reason"] != "Injury"]
    rows = []
    for _, s in subs.iterrows():
        t = s["elapsed_min"] - 1 / 60
        if t <= 0:
            continue
        ctx = scenario_context(bundle, s["match_id"], team_name, t)
        intent, _ = infer_intent(ctx["goal_diff"], t, ctx["man_advantage"])
        out = rank_outgoing(ctx, intent, cfg)
        inc = rank_incoming(ctx, s["off_role"], intent, cfg)
        rank_on = (inc.index[inc["player_id"] == s["player_on_id"]][0] + 1
                   if not inc.empty and (inc["player_id"] == s["player_on_id"]).any() else np.nan)
        rank_off = (out.index[out["player_id"] == s["player_off_id"]][0] + 1
                    if not out.empty and (out["player_id"] == s["player_off_id"]).any() else np.nan)
        rows.append({"match_id": s["match_id"], "minute": s["minute"], "game_state": s["game_state"],
                     "actual_off": s["player_off_name"], "actual_on": s["player_on_name"],
                     "bench_size": len(ctx["bench"]), "rank_actual_on": rank_on,
                     "on_in_top_k": bool(rank_on <= top_k) if pd.notna(rank_on) else False,
                     "rank_actual_off": rank_off,
                     "off_in_top_k": bool(rank_off <= top_k) if pd.notna(rank_off) else False})
        if max_subs and len(rows) >= max_subs:
            break
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Pre/post descriptive windows
# ---------------------------------------------------------------------------
def substitution_decisions(subs: pd.DataFrame, grouping_seconds: float = 90) -> pd.DataFrame:
    """Collapse simultaneous changes (same team, within ``grouping_seconds``) into one decision."""
    if subs.empty:
        return pd.DataFrame()
    rows = []
    for (mid, tid), g in subs.sort_values("elapsed_min").groupby(["match_id", "team_id"]):
        gid, last = 0, None
        for _, s in g.iterrows():
            if last is None or (s["elapsed_min"] - last) * 60 > grouping_seconds:
                gid += 1
            last = s["elapsed_min"]
            rows.append({**s.to_dict(), "decision_id": f"{mid}-{tid}-{gid}"})
    d = pd.DataFrame(rows)
    agg = d.groupby("decision_id").agg(
        match_id=("match_id", "first"), team_id=("team_id", "first"), opponent_id=("opponent_id", "first"),
        team_name=("team_name", "first"), elapsed_min=("elapsed_min", "min"), minute=("minute", "first"),
        game_state=("game_state", "first"), n_changes=("index", "size"),
        players_on=("player_on_name", lambda x: ", ".join(x)), players_off=("player_off_name", lambda x: ", ".join(x)))
    return agg.reset_index()


def impact_windows(bundle: Any, window: float | None = None) -> pd.DataFrame:
    """Team event rates per 15 minutes in equal windows before/after each substitution DECISION.

    The unit is a team's substitution decision (simultaneous changes grouped), so
    changes made together do not "contaminate" each other. Windows are truncated
    at kick-off/full-time and normalised by actual length. ``contaminated`` = a goal,
    sending-off or another decision by the same team inside the window; opponent
    changes are reported separately as a remaining confounder. These are OBSERVED
    ASSOCIATIONS, not causal effects.
    """
    w = window or bundle.config.impact_window_minutes
    flags, subs, goals, so = bundle.flags, bundle.subs, bundle.tables["goals"], bundle.tables["sending_offs"]
    lengths = bundle.tables["period_bounds"].groupby("match_id")["duration"].sum() / 60
    decisions = substitution_decisions(subs, bundle.config.sub_rules.get("window_grouping_seconds", 90))
    metrics = ["shots", "xg", "final_third_touches", "progressive_passes", "pressures"]
    rows = []
    for _, s in decisions.iterrows():
        mid, tid, opp, t = s["match_id"], s["team_id"], s["opponent_id"], s["elapsed_min"]
        end = lengths.get(mid, np.nan)
        pre0, post1 = max(t - w, 0), min(t + w, end)
        if t - pre0 < 3 or post1 - t < 3:
            continue
        ev = flags[flags["match_id"] == mid]
        rec = {"match_id": mid, "team_name": s["team_name"], "minute": s["minute"], "game_state": s["game_state"],
               "n_changes": s["n_changes"], "players_on": s["players_on"], "players_off": s["players_off"],
               "pre_len": t - pre0, "post_len": post1 - t}
        for side, tt in (("team", tid), ("opp", opp)):
            e = ev[ev["team_id"] == tt]
            pre = e[(e["elapsed_min"] >= pre0) & (e["elapsed_min"] < t)]
            post = e[(e["elapsed_min"] >= t) & (e["elapsed_min"] < post1)]
            for m in metrics:
                rec[f"{side}_{m}_pre15"] = pre[m].sum() / (t - pre0) * 15
                rec[f"{side}_{m}_post15"] = post[m].sum() / (post1 - t) * 15
        same = decisions[(decisions["match_id"] == mid) & (decisions["team_id"] == tid)
                         & (decisions["decision_id"] != s["decision_id"]) & decisions["elapsed_min"].between(pre0, post1)]
        opp_d = decisions[(decisions["match_id"] == mid) & (decisions["team_id"] == opp)
                          & decisions["elapsed_min"].between(pre0, post1)]
        g = goals[(goals["match_id"] == mid) & goals["elapsed_min"].between(pre0, post1)]
        r = so[(so["match_id"] == mid) & so["elapsed_min"].between(pre0, post1)]
        rec.update({"goal_in_window": len(g) > 0, "red_card_in_window": len(r) > 0,
                    "other_own_decision_in_window": len(same) > 0, "opponent_changed_in_window": len(opp_d) > 0})
        rec["contaminated"] = bool(len(g) or len(r) or len(same))
        rec["contamination"] = ", ".join(x for x, n in (("own other sub decision", len(same)), ("goal", len(g)),
                                                        ("red card", len(r))) if n) or "none"
        rows.append(rec)
    df = pd.DataFrame(rows)
    for m in metrics:
        if f"team_{m}_pre15" in df:
            df[f"delta_team_{m}"] = df[f"team_{m}_post15"] - df[f"team_{m}_pre15"]
    return df


def impact_summary(impact: pd.DataFrame, clean_only: bool = False) -> pd.DataFrame:
    """Median post-minus-pre change per 15 minutes by game state (association only)."""
    if impact.empty:
        return pd.DataFrame()
    d = impact[~impact["contaminated"]] if clean_only else impact
    if d.empty:
        return pd.DataFrame()
    cols = [c for c in d.columns if c.startswith("delta_team_")]
    g = d.groupby("game_state")
    out = g[cols].median().round(3)
    out.insert(0, "n_decisions", g.size())
    out.insert(1, "opponent_also_changed_pct", (g["opponent_changed_in_window"].mean() * 100).round(0))
    return out.reset_index()


def role_name(role: str | None) -> str:
    return ROLE_LABEL.get(role, str(role))

In [20]:
B.subs = enrich_substitutions(B)
S = B.subs
display(S[["match_date", "team_name", "minute", "player_off_name", "off_role", "player_on_name", "on_role",
           "sub_reason", "game_state", "team_score", "opponent_score", "timing_band", "window_number",
           "manager_name"]].head(10))
print(f"{len(S)} substitutions in {S.match_id.nunique()} matches; identity check: "
      f"{(S.identity_check == 'ok').mean():.0%} resolved against matchday squads; "
      f"injury-coded: {(S.sub_reason == 'Injury').sum()}")
S_grp = S.assign(group=np.where(S.team_name == FOCUS, FOCUS, "Opponents (pooled)"))
sub_timing_histogram(S_grp, cfg.sub_timing_bands, group="group", banner=BANNER).show()
subs_by_state(S, banner=BANNER).show()
display(state_profile(S_grp, by="group"))

subs_fig = substitution_dashboard(S, FOCUS, cfg.sub_timing_bands, banner=BANNER)
subs_fig.show()
subs_team_band_heatmap(S, cfg.sub_timing_bands, banner=BANNER).show()
role_flow_sankey(role_flow(S), banner=BANNER).show()
substitution_timeline(S[S.match_id == fm.match_id], f"Substitutions — {MATCH_LABEL}", banner=BANNER).show()

subs_parcats(S, FOCUS).show()
on_pitch_gantt(T["spells"], T["goals"], fm.match_id, team_id_focus,
               f"{FOCUS}: who was on the pitch — {MATCH_LABEL}").show()

display(Markdown('### Live scenario\nThe scenario is picked automatically as the first focus-team match in which the team was **not winning** at\n~65 minutes of playing time (falls back to any match). Intent is inferred by a transparent rule — trailing → attack,\nleading → protect, drawing → balance — and can be overridden. Outgoing candidates are judged on minutes played *in\nthis match*, booking risk, and in-match role actions relative to **their own pre-match per-90 baseline** (role\ncontext, never a raw low count). Incoming candidates are bench players who have not yet appeared.'))

SCENARIO_MINUTE = 65.0
INTENT_OVERRIDE = None          # "attack" | "protect" | "balance" | None (rule-based)
scen = None
for r in focus_matches.itertuples():
    opp_id = r.opponent_id
    tg, og = score_at(T["goals"], r.match_id, r.team_id, opp_id, elapsed=SCENARIO_MINUTE)
    if tg <= og:
        scen = r; break
scen = scen or focus_matches.iloc[0]
rec = recommend_substitution(B, scen.match_id, FOCUS, SCENARIO_MINUTE, intent=INTENT_OVERRIDE)
c = rec["context"]
display(Markdown(f"**Scenario:** {scen.match_date:%d %b %Y}, {FOCUS} vs {c['opponent']} ({c['home_away']}) at "
                 f"**{SCENARIO_MINUTE:.0f}'** of playing time — score **{c['team_score']}-{c['opponent_score']}** "
                 f"({c['game_state']}), numerical state {c['man_advantage']:+d}, substitutions remaining "
                 f"**{c['subs_remaining']}**, windows remaining **{c['windows_remaining']}** · rules: {c['rule_source']}"))
display(pd.DataFrame([c["team_live"], c["opponent_live"]], index=[FOCUS, c["opponent"]]))
display(Markdown(f"**Intent:** `{rec['intent']}` — {rec['intent_reason']}"))
display(Markdown("**Outgoing-candidate ranking** (data up to the decision minute + pre-match baselines)"))
display(rec["outgoing_ranking"][["player_name", "role", "minutes_so_far", "booked", "involvement_vs_baseline",
                                 "outgoing_score", "reasons"]])

if rec["status"] == "ok":
    off, on = rec["recommended_off"], rec["recommended_on"]
    inc = rank_incoming(c, off["role"], rec["intent"], cfg)
    candidate_comparison(inc, cfg.sub_candidate_weights, f"Incoming candidates for {off['player_name']} "
                             f"({off['role']}) — intent '{rec['intent']}'", banner=BANNER).show()
    card = pd.DataFrame({"field": ["Recommended OFF", "Recommended ON", "Suggested decision window",
                                   "Tactical objective", "Evidence (incoming)", "Evidence (outgoing)",
                                   "Risks / trade-offs", "Rule allowance", "Data coverage", "Injury status",
                                   "Final decision"],
                         "value": [f"{off['player_name']} ({off['role']})", f"{on['player_name']} ({on['role']})",
                                   rec["decision_window"], rec["tactical_objective"], on["evidence"], off["reasons"],
                                   on["risks"], rec["rule_status"], rec["coverage"], on["injury_status"],
                                   rec["human_note"]]})
    display(card.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
    display(Markdown("**Alternatives**"))
    display(rec["alternatives"])
else:
    print("Recommendation status:", rec["status"])

if WIDGETS_OK and not HEADLESS:
    mopts = [(f"{r.match_date:%d %b} vs {r.opponent}", r.match_id) for r in focus_matches.itertuples()]
    def sub_live(match_id, minute, intent):
        r = recommend_substitution(B, match_id, FOCUS, float(minute), intent=None if intent == "auto" else intent)
        cc = r["context"]
        print(f"{cc['team_score']}-{cc['opponent_score']} ({cc['game_state']}) at {minute}' | intent {r['intent']} | "
              f"status {r['status']}")
        if r["status"] == "ok":
            print(f"OFF {r['recommended_off']['player_name']} → ON {r['recommended_on']['player_name']}")
            print("Evidence:", r["recommended_on"]["evidence"]); print("Risks:", r["recommended_on"]["risks"])
    W.interact(sub_live, match_id=W.Dropdown(options=mopts, value=scen.match_id, description="Match"),
               minute=W.IntSlider(value=65, min=46, max=88, step=1), intent=["auto", "attack", "protect", "balance"])
else:
    print("Live substitution scenario controls available when running interactively (Colab/Jupyter).")

t0 = time.perf_counter()
agree_subs = sub_historical_agreement(B, FOCUS, top_k=3)
print(f"Heuristic re-run 1 second before each of {len(agree_subs)} non-injury {FOCUS} substitutions "
      f"({time.perf_counter() - t0:.1f}s), pre-decision data only.")
summary = pd.DataFrame({
    "measure": ["actual incoming player in heuristic top-3", "actual outgoing player in heuristic top-3",
                "median bench size", "random top-3 baseline for incoming (3 / bench size, mean)"],
    "value": [f"{agree_subs.on_in_top_k.mean():.0%}", f"{agree_subs.off_in_top_k.mean():.0%}",
              f"{agree_subs.bench_size.median():.0f}", f"{(3 / agree_subs.bench_size.clip(lower=3)).mean():.0%}"]})
display(summary)
display(Markdown("Agreement describes how often the transparent heuristic points where the coach went; it does not "
                 "measure decision quality (selection bias: we only observe the chosen option's outcome)."))

imp = impact_windows(B)
display(Markdown(f"**Pre/post windows** (±{cfg.impact_window_minutes:.0f} min, rates per 15 min). Unit = a team's "
                 f"substitution *decision* (simultaneous changes grouped): {len(imp)} decisions with ≥3 min on both "
                 f"sides. {imp.contaminated.mean():.0%} contain a goal, red card or another own decision "
                 f"(goal {imp.goal_in_window.mean():.0%}, own other decision {imp.other_own_decision_in_window.mean():.0%}); "
                 f"the opponent also changed in {imp.opponent_changed_in_window.mean():.0%}."))
display(Markdown("All windows:")); display(impact_summary(imp))
display(Markdown("Clean windows only:")); display(impact_summary(imp, clean_only=True))
impact_chart(imp, "xg", banner=BANNER).show()

display(Markdown("> **Causal-inference rule.** These are *observed associations*. Teams that are trailing push forward anyway\n> (score effects), opponents change too, fatigue accumulates, time remaining differs and extreme pre-windows regress\n> to the mean. Nothing here shows that a substitution *caused* a change. A defensible causal design would need, e.g.,\n> matched comparison windows without substitutions in the same game state, which is outside this project's scope."))

,match_date,team_name,minute,player_off_name,off_role,player_on_name,on_role,sub_reason,game_state,team_score,opponent_score,timing_band,window_number,manager_name
0,2023-08-19,RB Leipzig,61,Xavi Simons,WM,Kevin Kampl,DM,Tactical,trailing,1,2,61-70,1,Marco Rose
1,2023-08-19,RB Leipzig,61,Nicolas Seiwald,DM,Junior Castello Lukeba,CB,Tactical,trailing,1,2,61-70,1,Marco Rose
2,2023-08-19,RB Leipzig,61,Timo Werner,WM,Yussuf Yurary Poulsen,CF,Tactical,trailing,1,2,61-70,1,Marco Rose
3,2023-08-19,Bayer Leverkusen,74,Florian Wirtz,W,Robert Andrich,DM,Tactical,leading,3,2,71-80,1,Xabi Alonso
4,2023-08-19,RB Leipzig,74,Ikoma Loïs Openda,CF,Emil Peter Forsberg,AM,Tactical,trailing,2,3,71-80,2,Marco Rose
5,2023-08-19,Bayer Leverkusen,77,Jeremie Frimpong,WB,Arthur Augusto de Matos Soares,WB,Injury,leading,3,2,71-80,2,Xabi Alonso
6,2023-08-19,RB Leipzig,83,Xaver Schlager,DM,Fabio Carvalho,DM,Tactical,trailing,2,3,81-90+,3,Marco Rose
7,2023-08-19,Bayer Leverkusen,88,Jonas Hofmann,CF,Adam Hložek,CF,Tactical,leading,3,2,81-90+,3,Xabi Alonso
8,2023-08-19,Bayer Leverkusen,88,Victor Okoh Boniface,CF,Nadiem Amiri,CF,Tactical,leading,3,2,81-90+,3,Xabi Alonso
9,2023-08-26,Borussia Mönchengladbach,66,Florian Neuhaus,DM,Rocco Reitz,DM,Tactical,trailing,0,3,61-70,1,Gerardo Seoane


301 substitutions in 34 matches; identity check: 100% resolved against matchday squads; injury-coded: 17


,group,game_state,substitutions,median_minute,mean_minute,attacking_change_pct
0,Bayer Leverkusen,drawing,16,67.5,68.2,0.0
1,Bayer Leverkusen,leading,112,79.0,77.0,3.6
2,Bayer Leverkusen,trailing,14,80.5,74.7,28.6
3,Opponents (pooled),drawing,35,74.0,67.8,0.0
4,Opponents (pooled),leading,13,73.0,74.1,15.4
5,Opponents (pooled),trailing,111,71.0,70.3,9.9


### Live scenario
The scenario is picked automatically as the first focus-team match in which the team was **not winning** at
~65 minutes of playing time (falls back to any match). Intent is inferred by a transparent rule — trailing → attack,
leading → protect, drawing → balance — and can be overridden. Outgoing candidates are judged on minutes played *in
this match*, booking risk, and in-match role actions relative to **their own pre-match per-90 baseline** (role
context, never a raw low count). Incoming candidates are bench players who have not yet appeared.

**Scenario:** 15 Sep 2023, Bayer Leverkusen vs Bayern Munich (away) at **65'** of playing time — score **1-1** (drawing), numerical state +0, substitutions remaining **4**, windows remaining **3** · rules: User-configured default (IFAB Law 3 five-substitute option). Verify against the competition's regulations.

,shots,xg,passes,pass_completion_pct,final_third_touches,pressures,defensive_actions
Bayer Leverkusen,7,0.42,393,85.0,240,109,93
Bayern Munich,14,1.88,474,85.9,332,124,92


... (12 outputs left out)

,game_state,n_decisions,opponent_also_changed_pct,delta_team_shots,delta_team_xg,delta_team_final_third_touches,delta_team_progressive_passes,delta_team_pressures
0,drawing,38,61.0,0.0,-0.011,-4.500,-1.5,1.500
1,leading,76,89.0,0.0,-0.007,1.500,0.0,1.952
2,trailing,75,79.0,0.0,0.000,13.315,1.5,-1.500


Clean windows only:

,game_state,n_decisions,opponent_also_changed_pct,delta_team_shots,delta_team_xg,delta_team_final_third_touches,delta_team_progressive_passes,delta_team_pressures
0,drawing,13,69.0,0.000,0.000,6.000,-1.5,0.00
1,leading,16,81.0,0.246,0.041,11.404,1.5,0.75
2,trailing,25,76.0,0.000,0.000,13.315,1.5,3.00


> **Causal-inference rule.** These are *observed associations*. Teams that are trailing push forward anyway
> (score effects), opponents change too, fatigue accumulates, time remaining differs and extreme pre-windows regress
> to the mean. Nothing here shows that a substitution *caused* a change. A defensible causal design would need, e.g.,
> matched comparison windows without substitutions in the same game state, which is outside this project's scope.

## 13 · Managerial decision profile
Measured: **observed tactical flexibility** (formation diversity, in-match shifts), **rotation tendency** (lineup
changes), **substitution timing profile** and **game-state response profile**. Not measured — because no valid
instrument exists in event data — leadership, motivation, man-management or "managerial skill". Manager identity
comes from the source's match metadata; managers with fewer than `min_manager_matches` matches are shown as
descriptive only and are never ranked.

In [21]:
from __future__ import annotations

# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · manager_analysis — observable managerial patterns
# Observable managerial decision patterns (not personality or 'skill' scores).
#
# Indicators are measured per manager-team spell from lineups, formations and
# substitutions: observed tactical flexibility, rotation tendency, substitution
# timing profile and game-state response profile. Every indicator is reported
# with its sample size; comparisons require ``config.min_manager_matches``.
# Manager identity comes from the source's match metadata (or an optional
# user-supplied managers.csv) — never guessed.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def load_manager_mapping(path: Path) -> pd.DataFrame | None:
    """Optional user-supplied mapping: match_id, team_id, manager_name[, manager_id, source]."""
    if not path.exists():
        return None
    df = pd.read_csv(path)
    need = {"match_id", "team_id", "manager_name"}
    if not need.issubset(df.columns):
        raise ValueError(f"managers.csv must contain columns {need}")
    df["manager_source"] = df.get("source", "user-supplied managers.csv")
    return df


def lineup_continuity(appearances: pd.DataFrame, matches: pd.DataFrame) -> pd.DataFrame:
    """Starters retained from the team's previous match IN THE DATASET (gap days reported)."""
    st = appearances[appearances["is_starter"]].merge(matches[["match_id", "match_date"]], on="match_id")
    rows = []
    for team, g in st.groupby("team_id"):
        order = g[["match_id", "match_date"]].drop_duplicates().sort_values("match_date")
        prev_set, prev_date = None, None
        for _, m in order.iterrows():
            cur = set(g.loc[g["match_id"] == m["match_id"], "player_id"])
            if prev_set is not None:
                kept = len(cur & prev_set)
                rows.append({"team_id": team, "match_id": m["match_id"], "match_date": m["match_date"],
                             "starters_retained": kept, "lineup_changes": 11 - kept,
                             "days_since_prev_dataset_match": (m["match_date"] - prev_date).days})
            prev_set, prev_date = cur, m["match_date"]
    return pd.DataFrame(rows)


def _entropy(counts: pd.Series) -> float:
    p = counts / counts.sum()
    if len(p) <= 1:
        return 0.0
    return float(-(p * np.log(p)).sum() / np.log(len(p)))


def manager_profiles(bundle: Any) -> pd.DataFrame:
    """One row per manager-team spell with observable decision indicators and sample sizes."""
    cfg: Config = bundle.config
    ts, subs = bundle.team_stats, bundle.subs
    if ts["manager_name"].isna().all():
        return pd.DataFrame()
    cont = lineup_continuity(bundle.tables["appearances"], bundle.tables["matches"])
    rows = []
    for (mgr, team), g in ts.dropna(subset=["manager_name"]).groupby(["manager_name", "team_name"]):
        ids = set(g["match_id"])
        s = subs[(subs["team_name"] == team) & subs["match_id"].isin(ids)]
        c = cont[cont["match_id"].isin(ids) & (cont["team_id"].isin(g["team_id"]))]
        fcounts = g["starting_formation"].value_counts()
        n = len(g)
        first = s.sort_values("index").groupby("match_id").first()
        rows.append({
            "manager": mgr, "team": team, "matches": n,
            "sample_status": "comparable sample" if n >= cfg.min_manager_matches
            else f"descriptive only (n<{cfg.min_manager_matches})",
            "points_per_match": round(g["points"].mean(), 2), "xg_diff_per_match": round(g["xg_diff"].mean(), 2),
            "formations_used": int(fcounts.size), "most_used_formation": fcounts.index[0] if len(fcounts) else None,
            "most_used_share_pct": round(100 * fcounts.iloc[0] / n, 1) if len(fcounts) else np.nan,
            "formation_diversity_index": round(_entropy(fcounts), 2) if len(fcounts) else np.nan,
            "in_match_formation_changes_per_match": round(g["in_match_formation_changes"].mean(), 2),
            "mean_lineup_changes": round(c["lineup_changes"].mean(), 2) if len(c) else np.nan,
            "continuity_pairs": len(c),
            "subs_per_match": round(len(s) / n, 2),
            "median_sub_minute": float(s["minute"].median()) if len(s) else np.nan,
            "median_first_sub_minute": float(first["minute"].median()) if len(first) else np.nan,
            "early_subs_pct_before_60": round(100 * (s["minute"] < 60).mean(), 1) if len(s) else np.nan,
            "late_subs_pct_75_plus": round(100 * (s["minute"] >= 75).mean(), 1) if len(s) else np.nan,
            "multi_sub_window_pct": round(100 * s["multi_sub_window"].mean(), 1) if len(s) else np.nan,
            "subs_when_trailing_pct": round(100 * (s["game_state"] == "trailing").mean(), 1) if len(s) else np.nan,
            "subs_when_leading_pct": round(100 * (s["game_state"] == "leading").mean(), 1) if len(s) else np.nan,
            "like_for_like_pct": round(100 * s["like_for_like"].mean(), 1) if len(s) else np.nan,
            "injury_subs": int((s["sub_reason"] == "Injury").sum()),
            "substitutions": len(s),
        })
    out = pd.DataFrame(rows).sort_values("matches", ascending=False).reset_index(drop=True)
    return out


def response_after_conceding(bundle: Any) -> pd.DataFrame:
    """Minutes from conceding to the team's next substitution (same match), per manager."""
    goals, subs, ts = bundle.tables["goals"], bundle.subs, bundle.team_stats
    rows = []
    for _, g in goals.iterrows():
        conceding = ts[(ts["match_id"] == g["match_id"]) & (ts["team_id"] != g["scoring_team_id"])]
        if conceding.empty:
            continue
        c = conceding.iloc[0]
        nxt = subs[(subs["match_id"] == g["match_id"]) & (subs["team_id"] == c["team_id"])
                   & (subs["elapsed_min"] > g["elapsed_min"])]
        rows.append({"manager": c.get("manager_name"), "team": c["team_name"], "match_id": g["match_id"],
                     "conceded_minute": g["minute"],
                     "minutes_to_next_sub": (nxt["elapsed_min"].min() - g["elapsed_min"]) if len(nxt) else np.nan})
    return pd.DataFrame(rows)


def pooled_reference(profiles: pd.DataFrame, focus_manager: str, cfg: Config) -> pd.DataFrame:
    """Focus manager vs the pooled small-sample managers (clearly labelled reference, not a ranking)."""
    if profiles.empty or focus_manager not in set(profiles["manager"]):
        return pd.DataFrame()
    f = profiles[profiles["manager"] == focus_manager]
    rest = profiles[profiles["manager"] != focus_manager]
    cols = ["subs_per_match", "median_sub_minute", "median_first_sub_minute", "early_subs_pct_before_60",
            "late_subs_pct_75_plus", "multi_sub_window_pct", "formation_diversity_index",
            "in_match_formation_changes_per_match"]
    weights = rest["matches"]
    pooled = {c: np.average(rest[c].dropna(), weights=weights[rest[c].notna()]) if rest[c].notna().any() else np.nan
              for c in cols}
    out = pd.DataFrame({"indicator": cols, focus_manager: f[cols].iloc[0].values,
                        f"pooled other managers (n={int(weights.sum())} matches, "
                        f"{len(rest)} managers)": [pooled[c] for c in cols]})
    return out.round(2)

In [22]:
display(Markdown(f"**Manager identity source:** {B.manager_source} · coverage "
                 f"{T['managers'].match_id.nunique()}/{len(T['matches'])} matches"))
prof = manager_profiles(B)
display(prof)
FOCUS_MANAGER = B.team_stats[B.team_stats.team_name == FOCUS].manager_name.mode().iloc[0] \
    if B.team_stats.manager_name.notna().any() else None
if FOCUS_MANAGER:
    display(Markdown(f"**{FOCUS_MANAGER} vs pooled small-sample managers** — a reference, not a ranking "
                     "(all pooled managers faced the same opponent, so context differs)."))
    display(pooled_reference(prof, FOCUS_MANAGER, cfg))

if FOCUS_MANAGER:
    manager_formation_distribution(B.team_stats, FOCUS_MANAGER, banner=BANNER).show()
    cont = lineup_continuity(T["appearances"], T["matches"])
    rotation_trend(cont[cont.team_id == team_id_focus], FOCUS, banner=BANNER).show()
    rotation_fig = rotation_matrix(T["appearances"], T["matches"], FOCUS, banner=BANNER)
    rotation_fig.show()
    lineup_similarity(T["appearances"], T["matches"], FOCUS, banner=BANNER).show()
    manager_sub_profile(S, FOCUS_MANAGER, cfg.sub_timing_bands, banner=BANNER).show()
    resp = response_after_conceding(B)
    r = resp[resp.manager == FOCUS_MANAGER]
    display(Markdown(f"**Response after conceding ({FOCUS_MANAGER}):** {len(r)} goals conceded; median "
                     f"{r.minutes_to_next_sub.median():.1f} min to the next substitution where one followed "
                     f"({r.minutes_to_next_sub.notna().sum()} cases) — confounded by when goals happen."))

**Manager identity source:** StatsBomb matches.json (managers field) · coverage 34/34 matches

,manager,team,matches,sample_status,points_per_match,xg_diff_per_match,formations_used,most_used_formation,most_used_share_pct,formation_diversity_index,in_match_formation_changes_per_match,mean_lineup_changes,continuity_pairs,subs_per_match,median_sub_minute,median_first_sub_minute,early_subs_pct_before_60,late_subs_pct_75_plus,multi_sub_window_pct,subs_when_trailing_pct,subs_when_leading_pct,like_for_like_pct,injury_subs,substitutions
0,Xabi Alonso,Bayer Leverkusen,34,comparable sample,2.65,1.32,3,3-4-2-1,73.5,0.61,0.5,2.21,33,4.18,79.0,63.5,7.7,61.3,61.3,9.9,78.9,76.8,5,142
1,Jess Christian Thorup,Augsburg,2,descriptive only (n<10),0.00,-1.32,1,4-1-2-1-2,100.0,0.00,0.5,5.00,1,5.00,79.0,50.5,10.0,60.0,70.0,50.0,0.0,80.0,1,10
2,Torsten Lieberknecht,Darmstadt 98,2,descriptive only (n<10),0.00,-1.95,2,3-5-2,50.0,1.00,1.0,4.00,1,5.00,66.0,44.5,30.0,20.0,90.0,90.0,0.0,20.0,4,10
3,Thomas Tuchel,Bayern Munich,2,descriptive only (n<10),0.50,-0.21,2,4-2-3-1,50.0,1.00,1.5,6.00,1,5.00,69.5,59.5,20.0,40.0,80.0,50.0,0.0,60.0,0,10
4,Sebastian Hoeneß,VfB Stuttgart,2,descriptive only (n<10),1.00,0.51,2,3-4-2-1,50.0,1.00,1.5,3.00,1,5.00,78.0,69.0,0.0,70.0,80.0,0.0,50.0,60.0,1,10
5,Pellegrino Matarazzo,Hoffenheim,2,descriptive only (n<10),0.00,-0.96,1,3-5-2,100.0,0.00,0.0,6.00,1,4.00,77.0,66.5,12.5,62.5,75.0,50.0,50.0,87.5,0,8
6,Ole Werner,Werder Bremen,2,descriptive only (n<10),0.00,-1.61,1,3-5-2,100.0,0.00,0.5,5.00,1,4.00,71.0,61.5,12.5,37.5,75.0,100.0,0.0,87.5,1,8
7,Marco Rose,RB Leipzig,2,descriptive only (n<10),0.00,-0.30,2,4-4-2,50.0,1.00,0.5,3.00,1,5.00,74.0,67.0,0.0,20.0,70.0,50.0,0.0,60.0,0,10
8,Niko Kovač,Wolfsburg,2,descriptive only (n<10),0.00,-1.16,2,3-5-2,50.0,1.00,2.0,4.00,1,4.50,70.0,38.0,22.2,44.4,66.7,77.8,0.0,55.6,0,9
9,Gerardo Seoane,Borussia Mönchengladbach,2,descriptive only (n<10),0.50,-2.15,2,3-4-3,50.0,1.00,0.5,6.00,1,5.00,78.0,63.5,0.0,60.0,70.0,50.0,0.0,70.0,0,10


**Xabi Alonso vs pooled small-sample managers** — a reference, not a ranking (all pooled managers faced the same opponent, so context differs).

,indicator,Xabi Alonso,"pooled other managers (n=34 matches, 21 managers)"
0,subs_per_match,4.18,4.68
1,median_sub_minute,79.00,70.96
2,median_first_sub_minute,63.50,55.79
3,early_subs_pct_before_60,7.70,12.43
4,late_subs_pct_75_plus,61.30,41.94
5,multi_sub_window_pct,61.30,69.81
6,formation_diversity_index,0.61,0.59
7,in_match_formation_changes_per_match,0.50,0.82


**Response after conceding (Xabi Alonso):** 24 goals conceded; median 24.0 min to the next substitution where one followed (24 cases) — confounded by when goals happen.

## 14 · Optional modelling (data-gated): who starts?
| | |
|---|---|
| Prediction unit | player in the focus team's matchday squad, per match |
| Target | named in the Starting XI (1/0) |
| Prediction time | before kick-off |
| Features | only from earlier matches: started previous match, consecutive starts, start share, minutes share, minutes in previous 7/14 days, rest days, goalkeeper flag |
| Split | chronological by match date (first 70 % of matches train, rest test) — never random |
| Baseline | "started the previous match" |
| Gate | ≥ 300 rows and ≥ 15 matches, otherwise no model is fitted |

This models what the coaching staff *did* (observed selection behaviour), not what was optimal.

In [23]:
from __future__ import annotations

# ════════════════════════════════════════════════════════════════════════════════════════════════════
# PIPELINE · modeling — gated, leakage-audited start-selection model
# Optional, data-gated modelling: who starts? (observed selection behaviour).
#
# Prediction unit : player in a team's matchday squad for a match.
# Target          : ``is_starter`` (1 = named in the Starting XI).
# Prediction time : before kick-off.
# Features        : only quantities computed from EARLIER matches (workload proxies,
# start share, continuity, season-to-date minutes share, role).
# Split           : chronological by match date (first 70% of team matches -> train).
# Baseline        : "started the team's previous match".
#
# This models what the coaching staff DID, not what was optimal. A model is fitted
# only if the sample gate passes; otherwise the notebook explains why not.
# ════════════════════════════════════════════════════════════════════════════════════════════════════
from typing import Any

import numpy as np
import pandas as pd

FEATURES = ["started_prev_match", "consecutive_starts_before", "start_share_before", "minutes_share_before",
            "minutes_prev_7d", "minutes_prev_14d", "rest_days_filled", "is_gk"]
LEAKAGE_AUDIT = [
    ("started_prev_match", "team's previous match lineup", "known before kick-off"),
    ("consecutive_starts_before", "previous matches", "known before kick-off"),
    ("start_share_before", "starts / team matches before this match", "known before kick-off"),
    ("minutes_share_before", "minutes / (team matches x 95) before this match", "known before kick-off"),
    ("minutes_prev_7d / 14d", "minutes in earlier matches within 7/14 days", "known before kick-off"),
    ("rest_days_filled", "days since last appearance (60 if none)", "known before kick-off"),
    ("is_gk", "role from squad match position (event-derived)", "position played in this match — mild "
                                                                "contemporaneous information; GK role is stable"),
    ("EXCLUDED: minutes_played, events, goals in the match", "same-match outcomes", "would leak — not used"),
]


def build_start_dataset(bundle: Any, team_name: str) -> pd.DataFrame:
    wl = bundle.workload
    app = bundle.tables["appearances"]
    m = bundle.tables["matches"][["match_id", "match_date"]]
    d = app[app["team_name"] == team_name][["match_id", "team_id", "player_id", "player_name", "is_starter",
                                            "match_role"]].merge(wl, on=["match_id", "team_id", "player_id"])
    d = d.merge(m, on="match_id").sort_values(["match_date", "player_id"])
    d = d[d["prior_team_matches"] >= 1].copy()  # first match has no history
    d["started_prev_match"] = d["started_prev_match"].astype(float).fillna(0.0)
    d["rest_days_filled"] = d["rest_days"].fillna(60).clip(upper=60)
    d["is_gk"] = (d["match_role"] == "GK").astype(float)
    d["y"] = d["is_starter"].astype(int)
    return d.reset_index(drop=True)


def fit_start_model(bundle: Any, team_name: str, min_rows: int = 300, min_matches: int = 15,
                    train_frac: float = 0.7) -> dict[str, Any]:
    """Fit logistic regression with a chronological split if the sample gate passes."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (brier_score_loss, confusion_matrix, f1_score, precision_score, recall_score,
                                 roc_auc_score, roc_curve)
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    d = build_start_dataset(bundle, team_name)
    n_matches = d["match_id"].nunique()
    gate = {"rows": len(d), "matches": n_matches, "min_rows": min_rows, "min_matches": min_matches,
            "class_balance_starter_pct": round(100 * d["y"].mean(), 1) if len(d) else np.nan}
    if len(d) < min_rows or n_matches < min_matches:
        return {"fitted": False, "gate": gate, "reason": "Sample gate not met: descriptive analysis only."}
    order = d[["match_id", "match_date"]].drop_duplicates().sort_values("match_date")
    cut = int(len(order) * train_frac)
    train_ids, test_ids = set(order["match_id"].iloc[:cut]), set(order["match_id"].iloc[cut:])
    tr, te = d[d["match_id"].isin(train_ids)], d[d["match_id"].isin(test_ids)]
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=0))
    model.fit(tr[FEATURES], tr["y"])
    prob = model.predict_proba(te[FEATURES])[:, 1]
    # Decision rule respecting the domain constraint: exactly 11 starters per match.
    te = te.assign(prob=prob)
    te["pred_top11"] = te.groupby("match_id")["prob"].rank(ascending=False, method="first") <= 11
    base = te["started_prev_match"].astype(int)

    def metrics(pred, name, p=None):
        out = {"model": name, "precision": precision_score(te["y"], pred), "recall": recall_score(te["y"], pred),
               "f1": f1_score(te["y"], pred)}
        if p is not None:
            out["roc_auc"] = roc_auc_score(te["y"], p)
            out["brier"] = brier_score_loss(te["y"], p)
        return out

    res = pd.DataFrame([metrics(base, "Baseline: started previous match"),
                        metrics(te["pred_top11"].astype(int), "Logistic regression (top-11 per match)", prob)]).round(3)
    coef = pd.DataFrame({"feature": FEATURES, "coefficient_std": model[-1].coef_[0]}).sort_values(
        "coefficient_std", key=np.abs, ascending=False).round(3)
    fpr, tpr, _ = roc_curve(te["y"], prob)
    bins = pd.cut(te["prob"], [0, .2, .4, .6, .8, 1.0], include_lowest=True)
    calib = te.groupby(bins, observed=True).agg(mean_pred=("prob", "mean"), observed_rate=("y", "mean"),
                                                 n=("y", "size")).reset_index()
    return {"fitted": True, "gate": gate, "train_matches": len(train_ids), "test_matches": len(test_ids),
            "train_rows": len(tr), "test_rows": len(te),
            "split": f"chronological: train up to {order['match_date'].iloc[cut - 1]:%Y-%m-%d}, "
                     f"test from {order['match_date'].iloc[cut]:%Y-%m-%d}",
            "metrics": res, "coefficients": coef,
            "confusion_matrix": confusion_matrix(te["y"], te["pred_top11"].astype(int)),
            "roc": pd.DataFrame({"fpr": fpr, "tpr": tpr}), "calibration": calib, "test_predictions": te,
            "leakage_audit": pd.DataFrame(LEAKAGE_AUDIT, columns=["feature", "computed from", "availability"])}

In [24]:
model = fit_start_model(B, FOCUS)
display(pd.DataFrame([model["gate"]]))
if model["fitted"]:
    print(model["split"], f"| train rows {model['train_rows']} | test rows {model['test_rows']}")
    display(model["metrics"])
    display(model["coefficients"])
    display(Markdown("**Leakage audit**")); display(model["leakage_audit"])
    confusion_matrix_chart(model["confusion_matrix"], banner=BANNER).show()
    roc_chart(model["roc"], float(model["metrics"].roc_auc.dropna().iloc[0]), banner=BANNER).show()
    calibration_chart(model["calibration"], banner=BANNER).show()
else:
    display(Markdown(f"**No model fitted:** {model['reason']} More matches for one team (e.g. a full season with "
                     "cup fixtures) would be required."))

,rows,matches,min_rows,min_matches,class_balance_starter_pct
0,660,33,300,15,55.0


chronological: train up to 2024-03-03, test from 2024-03-10 | train rows 460 | test rows 200


,model,precision,recall,f1,roc_auc,brier
0,Baseline: started previous match,0.743,0.682,0.711,NaN,NaN
1,Logistic regression (top-11 per match),0.745,0.745,0.745,0.775,0.197


,feature,coefficient_std
3,minutes_share_before,1.357
7,is_gk,0.924
0,started_prev_match,0.525
1,consecutive_starts_before,-0.323
2,start_share_before,0.321
5,minutes_prev_14d,0.259
4,minutes_prev_7d,0.143
6,rest_days_filled,-0.123


**Leakage audit**

,feature,computed from,availability
0,started_prev_match,team's previous match lineup,known before kick-off
1,consecutive_starts_before,previous matches,known before kick-off
2,start_share_before,starts / team matches before this match,known before kick-off
3,minutes_share_before,minutes / (team matches x 95) before this match,known before kick-off
4,minutes_prev_7d / 14d,minutes in earlier matches within 7/14 days,known before kick-off
5,rest_days_filled,days since last appearance (60 if none),known before kick-off
6,is_gk,role from squad match position (event-derived),position played in this match — mild contempor...
7,"EXCLUDED: minutes_played, events, goals in the...",same-match outcomes,would leak — not used


## 15 · Findings and research interpretation
Answers are generated from the executed data (re-running with another competition updates them). They are
descriptive statements about this sample, with the main alternative explanation alongside.

In [25]:
fam = rd
fs_ = formation_summary(B.team_stats, FOCUS)
fsub = S[S.team_name == FOCUS]
osub = S[S.team_name != FOCUS]
cont_f = lineup_continuity(T["appearances"], T["matches"]).query("team_id == @team_id_focus")
rows = [
 ("RQ1 Role-distinguishing attributes",
  f"Median xG/90 peaks for {fam['xg_p90'].idxmax()}s ({fam['xg_p90'].max()}), interceptions/90 for "
  f"{fam['interceptions_p90'].idxmax()}s, pass completion for {fam['pass_completion_pct'].idxmax()}s "
  f"({fam['pass_completion_pct'].max()}%).", "Role labels are event-position based; tactical roles vary within labels."),
 ("RQ2 Per-90 and role normalisation",
  f"{int(B.season.eligible.sum())} of {len(B.season)} players pass {cfg.min_minutes:.0f} min; raising the threshold "
  f"to 900 leaves {(B.season.minutes_played >= 900).sum()} — leaderboards change (sensitivity table, §7).",
  "Small minutes inflate per-90 extremes."),
 ("RQ3 Formations and outcomes",
  f"{FOCUS} used {len(fs_)} starting formations; most used {fs_.iloc[0]['starting_formation']} "
  f"({fs_.iloc[0]['matches']} matches, {fs_.iloc[0]['points_per_match']} ppm, xG diff "
  f"{fs_.iloc[0]['xg_for'] - fs_.iloc[0]['xg_against']:+.2f}/match).",
  "Formation choice depends on opponent and availability — not a causal comparison."),
 ("RQ4 Substitution timing by state",
  f"{FOCUS}: median sub minute {fsub.minute.median():.0f}', {(fsub.game_state == 'leading').mean():.0%} made while "
  f"leading. Opponents: median {osub.minute.median():.0f}', {(osub.game_state == 'trailing').mean():.0%} while trailing.",
  "Score state is itself driven by team strength (the focus team led most matches)."),
 ("RQ5 Substitute matching",
  f"Heuristic top-3 contained the actual incoming player in {agree_subs.on_in_top_k.mean():.0%} of non-injury subs "
  f"(random ≈ {(3 / agree_subs.bench_size.clip(lower=3)).mean():.0%}).", "Agreement ≠ quality; bench composition varies."),
 ("RQ6 Rotation, minutes and rest",
  f"Mean {cont_f.lineup_changes.mean():.1f} starting-XI changes between consecutive dataset matches; "
  f"pre-match start model F1 {model['metrics'].f1.iloc[1]:.2f} vs 'started previous match' baseline "
  f"{model['metrics'].f1.iloc[0]:.2f} (ROC-AUC {model['metrics'].roc_auc.dropna().iloc[0]:.2f})." if model.get("fitted") else
  f"Mean {cont_f.lineup_changes.mean():.1f} lineup changes; model gate not met.",
  "Cup/European fixtures are missing, so rest days are over-estimated."),
 ("RQ7 Observable manager patterns",
  f"{FOCUS_MANAGER}: {prof.iloc[0]['subs_per_match']} subs/match, first sub median "
  f"{prof.iloc[0]['median_first_sub_minute']:.0f}', formation diversity {prof.iloc[0]['formation_diversity_index']}."
  if FOCUS_MANAGER else "Manager identity unavailable.", "Other managers have ≤2 matches — descriptive only."),
 ("RQ8 Limits to causal claims",
  f"{imp.contaminated.mean():.0%} of pre/post substitution-decision windows contain a goal, red card or another "
  f"own decision, and the opponent also changed in {imp.opponent_changed_in_window.mean():.0%}; deltas are "
  "associations only.", "Score effects, opponent changes, fatigue, regression to the mean."),
]
findings = pd.DataFrame(rows, columns=["research question", "observed answer (this dataset)", "main caveat"])
display(findings.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

display(Markdown("**Recommendations for further data collection.** (1) All competitions a team plays (cups, Europe) for valid\nworkload/rotation proxies; (2) injury/availability feeds from a licensed provider; (3) several full league seasons\n(e.g. StatsBomb's complete 2015/16 leagues) for role benchmarks independent of one club; (4) tracking or 360 data\nfor off-ball and pressing metrics; (5) a manager-tenure table to separate team and manager effects."))

research question,observed answer (this dataset),main caveat
RQ1 Role-distinguishing attributes,"Median xG/90 peaks for Forwards (0.32), interceptions/90 for Defenders, pass completion for Midfielders (84.83%).",Role labels are event-position based; tactical roles vary within labels.
RQ2 Per-90 and role normalisation,"84 of 377 players pass 180 min; raising the threshold to 900 leaves 17 — leaderboards change (sensitivity table, §7).",Small minutes inflate per-90 extremes.
RQ3 Formations and outcomes,"Bayer Leverkusen used 3 starting formations; most used 3-4-2-1 (25 matches, 2.84 ppm, xG diff +1.31/match).",Formation choice depends on opponent and availability — not a causal comparison.
RQ4 Substitution timing by state,"Bayer Leverkusen: median sub minute 79', 79% made while leading. Opponents: median 72', 70% while trailing.",Score state is itself driven by team strength (the focus team led most matches).
RQ5 Substitute matching,Heuristic top-3 contained the actual incoming player in 82% of non-injury subs (random ≈ 43%).,Agreement ≠ quality; bench composition varies.
"RQ6 Rotation, minutes and rest",Mean 2.2 starting-XI changes between consecutive dataset matches; pre-match start model F1 0.74 vs 'started previous match' baseline 0.71 (ROC-AUC 0.78).,"Cup/European fixtures are missing, so rest days are over-estimated."
RQ7 Observable manager patterns,"Xabi Alonso: 4.18 subs/match, first sub median 64', formation diversity 0.61.",Other managers have ≤2 matches — descriptive only.
RQ8 Limits to causal claims,"71% of pre/post substitution-decision windows contain a goal, red card or another own decision, and the opponent also changed in 79%; deltas are associations only.","Score effects, opponent changes, fatigue, regression to the mean."


**Recommendations for further data collection.** (1) All competitions a team plays (cups, Europe) for valid
workload/rotation proxies; (2) injury/availability feeds from a licensed provider; (3) several full league seasons
(e.g. StatsBomb's complete 2015/16 leagues) for role benchmarks independent of one club; (4) tracking or 360 data
for off-ball and pressing metrics; (5) a manager-tenure table to separate team and manager effects.

## 17 · Conclusion and viva preparation
**Plain-language summary.** The notebook turns StatsBomb event and lineup files into audited tables, role-aware
player metrics and interactive views, and uses them for two decisions coaches actually make — who starts and who comes
on — with every number traceable to a definition and every recommendation limited to information available at the
time.

| The system **can** | The system **cannot** |
|---|---|
| Rank squad players for formation slots with transparent, editable weights | Know fitness, injuries, training, tactics instructions or dressing-room context |
| Enforce lineup constraints and explain omissions | Prove a lineup or substitution is optimal or caused a result |
| Reconstruct who was on the pitch and the score at any minute | Measure leadership, motivation or "managerial skill" |
| Describe managers' observable timing / rotation / formation patterns | Rank managers from 1-2 matches |
| Fit a leakage-free, gated model of observed selection behaviour | Forecast results or player value |

**Key terms.** *Event data* (time-stamped on-ball actions) vs *aggregated statistics*; *per-90*; *percentile
within role*; *xG* (StatsBomb model probability that a shot is scored — provider output); *game state*; *data leakage*;
*chronological split*; *association vs causation*; *Hungarian assignment*.

### Viva question bank (answer prompts use this run's numbers)
1. **What is the problem statement?** Event and lineup data are rich but hard to use consistently for XI and
   substitution decisions; the project makes them auditable, role-aware, time-aware and interactive, with limits shown.
2. **Why is football analytics suitable for Big Data Visualization?** High-volume, high-dimensional, spatio-temporal
   event streams (≈4,000 events per match, x/y coordinates, nested qualifiers) that only become interpretable through
   aggregation and interactive visuals. Honest scale: 137,765 events, 120 MB raw — a scalable pipeline case study,
   not "massive" data.
3. **Raw event data vs aggregated match statistics?** Events are time-stamped on-ball actions with location, player
   and qualifiers; aggregates (shots, possession) are sums over events. Events allow per-90, spatial, match-state and
   before/after analysis that aggregates cannot.
4. **Which dataset and why?** StatsBomb Open Data (public, documented, lineups with bench, managers, xG). Bundesliga
   2023/24: 34 matches with manager metadata covering one manager's whole season.
5. **How many competitions, matches, events, teams, players?** 1 competition-season, 34 matches, 137,765 events,
   18 teams, 460 players in matchday squads (377 appeared), 301 substitutions, 113 goals.
6. **How did you collect the data?** HTTPS download of raw JSON from the repository via `DataStore`: timeout,
   retries with back-off, cache, SHA-256 manifest with UTC timestamps; raw files never modified.
7. **What does a lineup record contain?** Per team: player id/name, jersey, country, cards and position spells with
   start/end reasons; unused substitutes have no positions — that is how the bench is identified.
8. **How did you calculate minutes?** From the event stream: Starting XI → substitutions → red/second-yellow cards,
   converted to elapsed playing time per period (Half Start/End), so stoppage time counts and first-half stoppage
   never overlaps the second half. Treatment breaks (Player Off/On) ignored. Lineup spells were rejected because they
   are not chronological (example: a second yellow at 47:30 of the first half listed after a 45:00 second-half entry).
9. **Per-90 and why?** count ÷ minutes × 90 — compares players with different playing time; NaN for zero minutes;
   minimum-minutes threshold with a sensitivity table (84 eligible at 180 min, 17 at 900).
10. **Why within position groups?** A centre-back's interceptions and a forward's xG are not comparable; percentiles
    within role (≥8 players) or role family, labelled with the group size.
11. **What is xG and where did it come from?** Probability a shot is scored, from StatsBomb's model
    (`shot.statsbomb_xg`); we never compute or alter it.
12. **How does the XI recommendation work?** Per player-slot score = Σ weights × (role fit, recent form, sample
    reliability, workload suitability, continuity) × position compatibility; Hungarian assignment maximises the total.
13. **What constraints?** 11 distinct players, one goalkeeper, each slot filled once, compatibility > 0, unavailable
    excluded, players without pre-decision data excluded, empty slot + warning if depth is insufficient — all unit-tested.
14. **How does the substitution recommendation use score state?** State from the team's perspective just before the
    decision sets the intent (trailing → attack metrics, leading → protect, drawing → balance); user can override.
15. **How do you know who was on the pitch at a minute?** Spells from the event timeline; `on_pitch_at` / `bench_at`;
    a replaced player cannot return (tested).
16. **Substitution timing profile?** Distribution of substitution minutes across configurable bands and by score
    state. Alonso: 4.18 subs/match, median minute 79, first-sub median 63.5'.
17. **Can the system determine leadership skills?** No — event data contain no valid measurement instrument; we report
    observable decision patterns only.
18. **Correlation vs causation in substitution impact?** Pre/post windows are associations: 71 % of decision windows
    contain a goal, red card or another own decision, and opponents also changed in most; score effects and regression
    to the mean remain.
19. **How did you avoid leakage?** Every decision uses only earlier matches (`asof` filters) or events before the
    decision minute; model features audited; chronological split.
20. **How did you evaluate?** XI: descriptive agreement with actual XIs (77.7 % mean). Substitutions: actual incoming in
    top-3 82 % vs 43 % random. Start model: F1 0.745 vs baseline 0.711, ROC-AUC 0.775, Brier 0.197.
21. **Major data limitations?** One club's season (opponents 1-2 matches), no cup fixtures, no injuries/fitness/
    tracking, user-configured substitution rules, xG is a model.
22. **How would you scale?** Raw files are independent per match → parallel parsing (Spark/Dask `flatMap`), Parquet
    partitioned by competition/season/match, `groupBy` aggregations; switch `MATCH_LIMIT=None` across many seasons.
23. **Why Plotly and interactive filtering?** Hover provenance (minutes, percentile, coverage), dropdowns that survive
    HTML export, consistent theme, accessible zoom/pan; widgets for live what-if analysis.
24. **What would tracking data add?** Off-ball runs, defensive shape, pressing intensity, physical load — better
    workload proxies and role descriptors.
25. **What should a coach do with a recommendation?** Treat it as a structured second opinion: check the evidence and
    coverage, add information we lack (fitness, opponent plan), then decide.

26. **What are expected points (xPts)?** Each match is simulated 5,000 times, treating every shot as a Bernoulli
    trial with its xG; xPts = 3·P(win) + P(draw). Leverkusen: 90 points vs 74.8 xPts (+15.2). It summarises
    chance quality; it assumes independent shots and ignores game state, so it is not a 'luck' measure.
27. **What do the pitch heatmaps show?** Share of a player's or team's located actions per 10×10-unit zone
    (StatsBomb 120×80 frame, attacking left → right). Leverkusen applied 57 % of pressures in the opponent half.

### 5–7 minute presentation outline
| Time | Content | Notebook |
|---|---|---|
| 0:00-0:30 | Problem and objective; disclaimer | §0 |
| 0:30-1:15 | Dataset and collection: catalogue, manifest, 34 matches / 137,765 events / 120 MB, attribution | §3-4 |
| 1:15-2:00 | Architecture and preprocessing: minutes reconstruction, 16/16 quality checks | §5-6 |
| 2:00-3:00 | Player intelligence: leaderboard dropdown, radar, scatter; tactical: shot map, pass network | §9-10 |
| 3:00-4:00 | Starting XI: pitch, score decomposition, omissions, vs actual XI, rotation what-if | §11 |
| 4:00-5:00 | Substitution scenario: live context, intent, ranked candidates, recommendation card | §12 |
| 5:00-5:30 | Manager decision patterns: timing heatmap, rotation | §13 |
| 5:30-6:00 | Findings, limitations, causality note, conclusion | §15, §17 |

In [26]:
facts = {"Competition": f"{T['matches'].competition_name.iloc[0]} {T['matches'].season_name.iloc[0]}",
         "Matches analysed": len(T["matches"]), "Events": len(T["events"]), "Teams": len(teams),
         "Players (squads)": T["appearances"].player_id.nunique(), "Substitutions": len(S),
         "Quality checks passed": f"{int((B.quality.status == 'PASS').sum())}/{len(B.quality)}",
         "Total pipeline time (s)": round(parse_seconds + B.timings['features_s'], 1)}
display(Markdown("**Numbers to quote in the viva (from this run)**"))
display(pd.DataFrame(list(facts.items()), columns=["fact", "value"]))
status_badge()

**Numbers to quote in the viva (from this run)**

,fact,value
0,Competition,1. Bundesliga 2023/2024
1,Matches analysed,34
2,Events,137765
3,Teams,18
4,Players (squads),460
5,Substitutions,301
6,Quality checks passed,16/16
7,Total pipeline time (s),6.0
